# Reproducing TimesNet (ICLR 2023) - Table 14: short-term forecasting (M4)


## Overview of Changes

All modifications are applied in memory before `run.py` starts so TSLib's source files remain unchanged. Results for each configuration (`NO_LEAK`, `PREPROC_FIX`, `ROBUST_NORM`) are stored in separate folders on Google Drive.

Sections 1-7 outputs: re-run on an L4 using the saved Table 14 runs. Section 8 outputs: earlier run on an RTX PRO 6000 (Blackwell).

## 1. Wrap-up helper functions

The helper functions:

1) Set up the Colab environment and avoid collisions (e.g. library versions, GPUs, folders).

2) Wrap the original repository code, reproduce it, and present the results for evaluation (e.g. seeds, metrics, errors found, differences from the paper's results).

In [1]:
%%writefile repro_utils.py
import glob
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time


# Settings

SEEDS = [2021, 2022, 2023]   # 2021 = TSLib's own seed; the paper averages 3 runs
SMOKE = False                # True = 1 training epoch per run, to test the pipeline
USE_DRIVE = True             # keeps the results on Google Drive if the Colab runtime disconnects
HF_REVISION = None           # None = latest on first run, then pinned automatically
TSLIB_URL = "https://github.com/thuml/Time-Series-Library.git"      # TSLib repository
TSLIB_COMMIT = "4e938a1767106324dd753b2a44832bf870a0252e"           # pinned TSLib commit (2026-04-18)
HF_REPO = "thuml/Time-Series-Library"                               # Hugging Face dataset repository

# Versions from TSLib's README / requirements.txt.
PINNED_PACKAGES = ["numpy==2.1.2",
                   "pandas==2.3.3",
                   "scipy==1.16.3",
                   "scikit-learn==1.7.2",
                   "matplotlib==3.10.8",
                   "sktime==0.40.1",
                   "patool==4.0.3",
                   "tqdm==4.66.5",
                   "datasets==4.5.0",
                   "huggingface_hub==0.36.0"]

TORCH_PROFILES = {"tslib": ("torch==2.5.1", "https://download.pytorch.org/whl/cu121", ["sympy==1.13.1"]),
                  "blackwell": ("torch==2.7.1", "https://download.pytorch.org/whl/cu128", [])}

_RUN_PY_SEED = "fix_seed = 2021"
_RUN_PY_SEED_NEW = "fix_seed = int(os.environ.get('TSLIB_SEED', '2021'))"
_RUN_PY_NP_SEED = "np.random.seed(fix_seed)"
_DETERMINISM_BLOCK = """np.random.seed(fix_seed)
    # [repro patch] deterministic kernels and no TF32 (TITAN RTX used in the paper has no TF32)
    if os.environ.get('TSLIB_DETERMINISTIC', '1') == '1':
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.use_deterministic_algorithms(True, warn_only=True)"""


# Helper functions

def sh(cmd,
       cwd=None,
       env=None,
       check=True,
       quiet=False):
    """Run a shell command and return everything it prints."""

    if env is None:
        env = dict(os.environ, MPLBACKEND="Agg")
    if not quiet:
        print("$", cmd)
    res = subprocess.run(cmd,
                         shell=True,
                         cwd=cwd,
                         env=env,
                         text=True,
                         stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT)
    if res.returncode != 0 and check:
        print(res.stdout[-4000:])
        raise RuntimeError(f"command failed ({res.returncode}): {cmd}")

    return res.stdout


def in_colab():
    """Return True when running in Google Colab."""
    try:
        import google.colab
        return True
    except ImportError:
        return False


def results_root(name,
                 use_drive=True):
    """Return the folder where the reproduced table results are stored."""

    if in_colab() and use_drive:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        root = f"/content/drive/MyDrive/TimesNet_reproduction/{name}"
    else:
        base = "/content" if os.path.exists("/content") else os.getcwd()
        root = os.path.join(base, "TimesNet_reproduction", name)
    os.makedirs(root, exist_ok=True)

    return root


def load_json(path,
              default=None):
    """Return the content of a JSON file, or default if it does not exist."""

    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default


def save_json(path, obj):
    """Save an object to a JSON file."""

    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


# Environment functions

def gpu_compute_capability():
    """Return the compute capability of the GPU."""

    res = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
                         shell=True, text=True, capture_output=True)
    try:
        return float(res.stdout.strip().splitlines()[0]) if res.returncode == 0 else None
    except (ValueError, IndexError):
        return None


def setup_environment(env_dir="/content/tslib_env",
                      python_version="3.11",
                      profile=None):
    """Create the pinned Python environment and return its path and a description."""

    cap = gpu_compute_capability()
    if profile is None:
        profile = "blackwell" if cap is not None and cap >= 10 else "tslib"
    if profile != "tslib":
        env_dir = f"{env_dir}_{profile}"
        print(f"NOTE: GPU compute capability {cap} requires torch 2.7.1 (CUDA 12.8) instead of TSLib's default 2.5.1.")

    torch_pkg, index, extra = TORCH_PROFILES[profile]
    py = os.path.join(env_dir, "bin", "python")
    marker = os.path.join(env_dir, ".repro_ready")

    if not os.path.exists(marker):
        sh(f"{sys.executable} -m pip install -q uv")
        sh(f"uv venv --python {python_version} {env_dir}")
        sh(f"uv pip install --python {py} {torch_pkg} --index-url {index}")
        sh(f"uv pip install --python {py} " + " ".join(PINNED_PACKAGES + extra))
        open(marker, "w").close()

    info = sh(f"{py} -c \"import torch, numpy, pandas, sys; "
              f"print(sys.version.split()[0], torch.__version__, torch.version.cuda, "
              f"torch.cuda.is_available(), numpy.__version__, pandas.__version__)\"",
              quiet=True).split()

    env = {"python": info[0], "torch": info[1], "cuda": info[2],
           "cuda_available": info[3] == "True", "numpy": info[4], "pandas": info[5],
           "torch_profile": profile, "compute_capability": cap}
    print("Environment:", env)

    if cap is None or not env["cuda_available"]:
        print("WARNING: no GPU visible.")
    else:
        test = subprocess.run(
            [py, "-c", "import torch; x = torch.randn(64, 64, device='cuda'); "
                       "print(float((x @ x).sum()), torch.cuda.get_device_name(0))"],
            text=True, capture_output=True)
        if test.returncode != 0:
            raise RuntimeError(
                "The GPU test failed, so training would fail too:\n"
                + test.stdout[-1500:] + test.stderr[-1500:]
                + "\nFix: Runtime > Change runtime type > T4, L4, A100 or H100, then run all cells.")
        print("GPU test passed:", test.stdout.strip().split(" ", 1)[-1])

    return py, env


def gpu_name():
    """Return the name of the GPU."""
    res = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                         text=True, capture_output=True)
    lines = res.stdout.strip().splitlines()
    return lines[0] if res.returncode == 0 and lines else "no GPU"


# Repository functions

def prepare_repo(repo_dir="/content/Time-Series-Library",
                 commit=TSLIB_COMMIT):
    """Clone TSLib at a fixed commit and apply the reproducibility patches."""

    if not os.path.exists(os.path.join(repo_dir, ".git")):
        sh(f"git clone -q {TSLIB_URL} {repo_dir}")
    sh(f"git -C {repo_dir} fetch -q origin {commit}", check=False, quiet=True)
    sh(f"git -C {repo_dir} checkout -q -f {commit}")
    head = sh(f"git -C {repo_dir} rev-parse HEAD", quiet=True).strip()
    assert head == commit, f"expected {commit}, got {head}"

    run_py = os.path.join(repo_dir, "run.py")
    with open(run_py) as f:
        src = f.read()
    if "[repro patch]" not in src:
        assert src.count(_RUN_PY_SEED) == 1 and src.count(_RUN_PY_NP_SEED) == 1, \
            "run.py layout changed; patches cannot be applied safely"
        src = src.replace(_RUN_PY_SEED, _RUN_PY_SEED_NEW)
        src = src.replace(_RUN_PY_NP_SEED, _DETERMINISM_BLOCK)
        with open(run_py, "w") as f:
            f.write(src)
    _patch_m4_numpy(repo_dir)
    print(f"TSLib ready at {repo_dir} (commit {commit[:10]}), run.py patched "
          f"(seed from TSLIB_SEED, deterministic kernels, TF32 off), M4 NumPy fix applied.")
    return repo_dir


_STACK_HELPER = '''
def _repro_stack(rows):
    """[repro patch] NumPy<1.24 behaviour of np.array(list_of_arrays): a 2-D array when all
    rows have the same length, otherwise a 1-D object array (NumPy>=1.24 raises instead)."""
    rows = list(rows)
    if len({len(r) for r in rows}) <= 1:
        return np.array(rows)
    out = np.empty(len(rows), dtype=object)
    out[:] = rows
    return out
'''


def _patch_m4_numpy(repo_dir):
    """TSLib's M4 code builds arrays from series of different lengths. That raises an
    error on NumPy>=1.24 (TSLib pins 2.1.2), so M4 would crash on the real data.
    This restores the old NumPy behaviour exactly; nothing else changes."""
    targets = {
        "utils/m4_summary.py": [
            "return np.array([v[~np.isnan(v)] for v in values[groups == group_name]])",
            "naive2_forecasts = np.array([v[~np.isnan(v)] for v in naive2_forecasts])"],
        "data_provider/data_loader.py": [
            "training_values = np.array(\n            [v[~np.isnan(v)] for v in"],
    }
    for rel, olds in targets.items():
        path = os.path.join(repo_dir, rel)
        with open(path) as f:
            src = f.read()
        if "[repro patch] NumPy<1.24" in src:
            continue
        for old in olds:
            assert src.count(old) == 1, f"{rel}: unexpected layout, cannot patch"
            src = src.replace(old, old.replace("np.array(", "_repro_stack(", 1))
        anchor = "import numpy as np\n"
        assert anchor in src, f"{rel}: numpy import not found"
        src = src.replace(anchor, anchor + _STACK_HELPER, 1)
        with open(path, "w") as f:
            f.write(src)


def download_data(repo_dir,
                  python,
                  folders,
                  revision=None):

    """Download dataset folders from the TSLib Hugging Face repo into ./dataset.
    Returns the exact dataset revision (commit sha) used."""
    patterns = json.dumps([f"{d}/*" for d in folders])
    code = (
        "from huggingface_hub import snapshot_download, HfApi\n"
        f"rev = {revision!r} or HfApi().dataset_info({HF_REPO!r}).sha\n"
        f"snapshot_download(repo_id={HF_REPO!r}, repo_type='dataset', revision=rev,"
        f" allow_patterns={patterns}, local_dir='dataset')\n"
        "print('REVISION', rev)\n"
    )
    out = sh(f"{python} -c {shlex.quote(code)}", cwd=repo_dir, quiet=True)
    rev = re.search(r"REVISION (\S+)", out).group(1)
    missing = [d for d in folders if not os.listdir(os.path.join(repo_dir, "dataset", d))]
    if missing:
        raise RuntimeError(f"dataset folders empty after download: {missing}")
    print(f"Datasets {folders} ready (Hugging Face revision {rev[:10]}).")

    return rev


def record_environment(root,
                       env,
                       commit,
                       data_rev):

    """Store the environment; warn if this session's GPU differs from earlier runs."""
    path = os.path.join(root, "environment.json")
    info = load_json(path, {})
    current = {"gpu": gpu_name(), "tslib_commit": commit, "hf_dataset_revision": data_rev,
               **{f"env_{k}": v for k, v in env.items()}}
    runs_dir = os.path.join(root, "runs")
    has_runs = os.path.isdir(runs_dir) and any(f.endswith(".json") for f in os.listdir(runs_dir))
    if info and not has_runs:
        info = {}   # nothing finished yet: the new session simply becomes the reference
    if info:
        diffs = {k: (info.get(k), v) for k, v in current.items() if info.get(k) != v}
        if diffs:
            print("WARNING - this session differs from the one that produced earlier "
                  "results in this folder:")
            for k, (old, new) in diffs.items():
                print(f"   {k}: {old} -> {new}")
            print("Runs are only bit-for-bit repeatable on the same GPU model and "
                  "software. Use a new results folder or the same GPU type.")
    else:
        save_json(path, current)
    return current


def expand_script(repo_dir,
                  script):
    """Return one argument list per training command in a TSLib script."""

    shim = ("python() { for a in \"$@\"; do printf '%s\\037' \"$a\"; done; printf '\\036'; }; "
            "python3() { python \"$@\"; }; "
            f"source {shlex.quote(script)}")
    out = subprocess.run(["bash", "-c", shim], cwd=repo_dir, text=True,
                         capture_output=True, check=True).stdout
    commands = []
    for rec in out.split("\x1e"):
        parts = [p for p in rec.split("\x1f") if p != ""]
        if "run.py" in parts:
            commands.append(parts[parts.index("run.py") + 1:])

    return commands


def arg(args,
        key,
        default=None):
    """Return the value that follows --<key> in an argument list, or default if absent."""

    flag = "--" + key
    return args[args.index(flag) + 1] if flag in args else default


def set_arg(args,
            key,
            value):
    """Return a copy of the argument list with --<key> set to value."""

    args = list(args)
    flag = "--" + key
    if flag in args:
        args[args.index(flag) + 1] = str(value)
    else:
        args += [flag, str(value)]

    return args


def run_tslib(python,
              repo_dir,
              args,
              seed,
              log_path,
              smoke=False,
              entry=("run.py",)):
    """Run one TSLib command with a fixed seed and return (success, output)."""

    args = list(args)
    if smoke:
        args = set_arg(args, "train_epochs", 1)
    if os.environ.get("REPRO_FORCE_CPU") == "1":   # only for testing without a GPU
        args = args + ["--no_use_gpu", "--num_workers", "0"]
    env = dict(os.environ, TSLIB_SEED=str(seed), TSLIB_DETERMINISTIC="1",
               MPLBACKEND="Agg",   # Colab sets an inline backend the isolated env lacks
               CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONHASHSEED=str(seed),
               PYTHONUNBUFFERED="1")
    env.pop("CUDA_VISIBLE_DEVICES", None)
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    lines = []
    with open(log_path, "w") as log:
        shown = "run.py" if entry[0] != "-c" else "-c <code from the notebook>"
        log.write(f"python -u {shown} " + " ".join(shlex.quote(a) for a in args) + "\n")
        log.write(f"# TSLIB_SEED={seed}\n")
        proc = subprocess.Popen([python, "-u", *entry, *args], cwd=repo_dir, env=env,
                                text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            lines.append(line)
        proc.wait()

    return proc.returncode == 0, "".join(lines)


def clean_outputs(repo_dir,
                  keep_checkpoint_to=None):
    """Delete TSLib's output folders, optionally copying the latest checkpoint first."""

    if keep_checkpoint_to:
        ckpts = sorted(glob.glob(os.path.join(repo_dir, "checkpoints", "*", "checkpoint.pth")),
                       key=os.path.getmtime)
        if ckpts:
            os.makedirs(os.path.dirname(keep_checkpoint_to), exist_ok=True)
            shutil.copy2(ckpts[-1], keep_checkpoint_to)
    for d in ["checkpoints", "results", "test_results"]:
        shutil.rmtree(os.path.join(repo_dir, d), ignore_errors=True)


def run_all(python,
            repo_dir,
            root,
            commands,
            seeds,
            parse,
            meta,
            smoke=False,
            key_fn=lambda a: arg(a, "model_id"),
            keep_checkpoints=False,
            after_run=None,
            max_commands=None,
            entry=("run.py",)):

    """Run every (command, seed) pair not already stored in `root/runs`.

    parse(output) -> dict of metrics or None; after_run(args, seed) is an optional hook.
    """
    runs_dir = os.path.join(root, "smoke_runs" if smoke else "runs")
    todo = commands[:max_commands] if max_commands else commands
    total, n = len(todo) * len(seeds), 0
    for seed in seeds:
        for args in todo:
            n += 1
            key = key_fn(args)
            out_json = os.path.join(runs_dir, f"{key}__seed{seed}.json")
            if os.path.exists(out_json):
                print(f"[{n}/{total}] {key} seed {seed}: already done, skipping")
                continue
            print(f"[{n}/{total}] {key} seed {seed}: running ...", end="", flush=True)
            t0 = time.time()
            ok, output = run_tslib(python, repo_dir, args, seed,
                                   os.path.join(root, "logs", f"{key}__seed{seed}.log"), smoke, entry)
            metrics = parse(output) if ok else None
            if after_run:
                after_run(args, seed)
            ckpt = (os.path.join(root, "checkpoints", f"{key}__seed{seed}.pth")
                    if keep_checkpoints and ok and not smoke else None)
            clean_outputs(repo_dir, ckpt)
            minutes = (time.time() - t0) / 60
            if not metrics:
                print(f" FAILED after {minutes:.1f} min - see logs/{key}__seed{seed}.log")
                print(output[-2500:])
                continue
            save_json(out_json, {"key": key, "seed": seed, "metrics": metrics,
                                 "minutes": round(minutes, 2), "args": args,
                                 "smoke": smoke, **meta})
            shown = ", ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}"
                              for k, v in metrics.items())
            print(f" done in {minutes:.1f} min -> {shown}")


def load_runs(root,
              smoke=False):
    """Return the stored results of every run as a table."""

    import pandas as pd
    rows = []
    for p in glob.glob(os.path.join(root, "smoke_runs" if smoke else "runs", "*.json")):
        r = load_json(p)
        rows.append({"key": r["key"], "seed": r["seed"], **r["metrics"]})
    return pd.DataFrame(rows)


def verdict(delta,
            rel,
            kind):
    """Return a label (matches, close, differs) for a difference from the paper."""

    if kind == "rel":
        a = abs(rel)
        return "matches" if a <= 2 else ("close" if a <= 5 else "differs")
    a = abs(delta)
    return "matches" if a <= 1 else ("close" if a <= 3 else "differs")


def determinism_check(python,
                      repo_dir,
                      root,
                      args,
                      seed,
                      parse,
                      smoke=False):

    """Re-run one stored configuration and check the metric is identical."""
    key = arg(args, "model_id")
    stored = load_json(os.path.join(root, "smoke_runs" if smoke else "runs",
                                    f"{key}__seed{seed}.json"))
    if not stored:
        print("No stored run found for this configuration."); return None
    ok, output = run_tslib(python, repo_dir, args, seed,
                           os.path.join(root, "logs", f"{key}__seed{seed}__recheck.log"), smoke)
    clean_outputs(repo_dir)
    again = parse(output) if ok else None
    same = again == stored["metrics"]
    print(f"stored : {stored['metrics']}\nre-run : {again}\n"
          + ("IDENTICAL - the run is bit-for-bit repeatable on this setup." if same else
             "DIFFERENT - check GPU model / environment against environment.json."))

    return same

Writing repro_utils.py


## 2. Environment, original code, data and optimizer

The cells in this section:

1) Create an isolated Python 3.11 environment (torch 2.5.1 with CUDA 12.1 and TSLib's pinned versions; torch 2.7.1 with CUDA 12.8 on newer GPUs).

2) Clone TSLib at a fixed commit.

3) Download the original datasets from Hugging Face.


In [2]:
import importlib, os, sys
sys.path.insert(0, os.getcwd())
import repro_utils as R
importlib.reload(R)
SEEDS, SMOKE, USE_DRIVE, HF_REVISION = R.SEEDS, R.SMOKE, R.USE_DRIVE, R.HF_REVISION

# Settings (switches for the fixes)
NO_LEAK = True        # True = validate on the end of the training series (2b), False = TSLib default
PREPROC_FIX = True    # True = aligned training windows + first-value padding (2c)
ROBUST_NORM = False   # True = use median / MAD instead of mean / std inside TimesNet (2d)
ROBUST_SUBSETS = ["Yearly", "Quarterly", "Monthly", "Weekly", "Daily", "Hourly"]
AUTO_DISCONNECT = True    # release the GPU at the end so Colab compute units aren't wasted on long runs

ALL_SUBSETS = ["Yearly", "Quarterly", "Monthly", "Weekly", "Daily", "Hourly"]
ROBUST_TAG = ('_robust' + ('' if sorted(ROBUST_SUBSETS) == sorted(ALL_SUBSETS)
                           else '_' + '-'.join(s.lower() for s in ROBUST_SUBSETS))) if ROBUST_NORM else ''
ROOT = R.results_root(name='table14' + ('_noleak' if NO_LEAK else '') + ('_prepfix' if PREPROC_FIX else '') + ROBUST_TAG,
                      use_drive=USE_DRIVE)
print("Results folder:", ROOT)

def _release_gpu_on_error(result):
    if result.error_in_exec is not None:
        print(f"A cell failed ({result.error_in_exec!r}) - releasing the GPU. Results so far are in", ROOT)
        from google.colab import runtime
        runtime.unassign()

# Remove the old copy first, so running this cell twice does not add it twice
_ip = get_ipython()
for _f in [f for f in _ip.events.callbacks["post_run_cell"]
           if getattr(f, "__name__", "") == "_release_gpu_on_error"]:
    _ip.events.unregister("post_run_cell", _f)
if AUTO_DISCONNECT and USE_DRIVE and R.in_colab():
    _ip.events.register("post_run_cell", _release_gpu_on_error)

# Create the pinned Python environment
PY, ENV = R.setup_environment()

# Clone TSLib at the fixed commit
REPO = R.prepare_repo()
os.chdir(REPO)

previous = R.load_json(os.path.join(ROOT, "environment.json"), {})

# Download the M4 data
DATA_REV = R.download_data(REPO, PY, ['m4'],
                           revision=HF_REVISION or previous.get("hf_dataset_revision"))

# Record the environment of this session
META = R.record_environment(root=ROOT,
                            env=ENV,
                            commit=R.TSLIB_COMMIT,
                            data_rev=DATA_REV)
print(META)

Mounted at /content/drive
Results folder: /content/drive/MyDrive/TimesNet_reproduction/table14_noleak_prepfix
$ /usr/bin/python3 -m pip install -q uv
$ uv venv --python 3.11 /content/tslib_env
$ uv pip install --python /content/tslib_env/bin/python torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
$ uv pip install --python /content/tslib_env/bin/python numpy==2.1.2 pandas==2.3.3 scipy==1.16.3 scikit-learn==1.7.2 matplotlib==3.10.8 sktime==0.40.1 patool==4.0.3 tqdm==4.66.5 datasets==4.5.0 huggingface_hub==0.36.0 sympy==1.13.1
Environment: {'python': '3.11.18', 'torch': '2.5.1+cu121', 'cuda': '12.1', 'cuda_available': True, 'numpy': '2.1.2', 'pandas': '2.3.3', 'torch_profile': 'tslib', 'compute_capability': 8.9}
GPU test passed: NVIDIA L4
$ git clone -q https://github.com/thuml/Time-Series-Library.git /content/Time-Series-Library
$ git -C /content/Time-Series-Library checkout -q -f 4e938a1767106324dd753b2a44832bf870a0252e
TSLib ready at /content/Time-Series-Library (commit 

In [3]:
p = os.path.join(REPO, "exp/exp_short_term_forecasting.py")
src = open(p).read()
assert "optim.Adam(self.model.parameters(), lr=self.args.learning_rate)" in src, (
    "Expected Adam optimizer in exp_short_term_forecasting.py"
)
print("Optimizer: Adam (paper)")

Optimizer: Adam (paper)


## 2b. Data-Leakage Fix (Validation Split)

Hold out the last `pred_len` steps of each training series for validation instead of validating on the M4 test file.

In [4]:
# In-memory patch for Dataset_M4 and Exp_Short_Term_Forecast.test
NOLEAK_DEFS = r"""
import runpy, sys
import data_provider.data_factory as data_factory
from data_provider.data_loader import Dataset_M4
from exp.exp_short_term_forecasting import Exp_Short_Term_Forecast


# 'train': training series without their last pred_len points
# 'val': the last pred_len points of every training series (validation answers)
# 'train_full': full training series (input of the final test forecast)
# 'test': the M4 test file (used once, at the end)
class Dataset_M4_NoLeak(Dataset_M4):
    def __read_data__(self):
        flag = self.flag
        self.flag = 'test' if flag == 'test' else 'train'
        super().__read_data__()
        self.flag = flag
        if flag == 'train':
            self.timeseries = [ts[:-self.pred_len] for ts in self.timeseries]
        elif flag == 'val':
            self.timeseries = [ts[-self.pred_len:] for ts in self.timeseries]

data_factory.data_dict['m4'] = Dataset_M4_NoLeak


# test() takes its input windows from the 'train' data; give it the full history instead
_test = Exp_Short_Term_Forecast.test
def _test_from_full_history(self, setting, test=0):
    get_data = self._get_data
    self._get_data = lambda flag: get_data('train_full' if flag == 'train' else flag)
    try:
        return _test(self, setting, test)
    finally:
        del self._get_data

Exp_Short_Term_Forecast.test = _test_from_full_history
"""

# Entry snippet to run TSLib's run.py after applying patches
LAUNCH = r"""
import runpy, sys
sys.argv = ['run.py'] + sys.argv[1:]
runpy.run_path('run.py', run_name='__main__')
"""

print("Validation:", "held-out end of the training series (no leak)" if NO_LEAK
      else "M4 test set (TSLib original, leaks)")


Validation: held-out end of the training series (no leak)


In [5]:
# Verify the train / val splits on the Yearly subset
import json
CHECK = NOLEAK_DEFS + r"""
import json
import numpy as np
from data_provider.m4 import M4Meta

sp = 'Yearly'
h = M4Meta.horizons_map[sp]
def load(cls, flag):
    return cls(args=None, root_path='./dataset/m4', flag=flag, size=[2 * h, h, h],
               seasonal_patterns=sp).timeseries

train, val, full, test = (load(Dataset_M4_NoLeak, f) for f in ['train', 'val', 'train_full', 'test'])
tslib_val = load(Dataset_M4, 'val')

same = lambda a, b: all(np.array_equal(x, y) for x, y in zip(a, b))
assert len(train) == len(val) == len(full) == len(test), 'different number of series'
assert same(train, [f[:-h] for f in full]), 'train is not the history without the last horizon'
assert same(val, [f[-h:] for f in full]), 'val is not the last horizon of the training series'
assert same(tslib_val, test), "TSLib's own 'val' should be the test file"
overlap = float(np.mean([np.array_equal(v, t) for v, t in zip(val, test)]))
assert overlap < 0.01, 'val answers equal the test answers'
print('CHECKJSON' + json.dumps({'series': len(full), 'horizon': h, 'val_equal_test': overlap}))
"""
out = R.sh(f"{PY} -c {R.shlex.quote(CHECK)}", cwd=REPO, quiet=True)
info = json.loads(out.split("CHECKJSON")[-1])
print(f"OK - {info['series']} Yearly series: train = history without the last {info['horizon']} points, "
      f"val = those {info['horizon']} points ({100 * info['val_equal_test']:.2f}% equal to test answers). "
      f"TSLib's original 'val' is the test file.")


OK - 23000 Yearly series: train = history without the last 6 points, val = those 6 points (0.00% equal to test answers). TSLib's original 'val' is the test file.


## 2c. Preprocessing Check and Fix

Align training target windows for short series in `Dataset_M4.__getitem__` and pad missing history with the first observed value instead of 0.

In [6]:
# Check percentage of affected training windows per M4 subset
import json, pandas as pd
DIAG = r"""
import json
import numpy as np
from data_provider.m4 import M4Dataset, M4Meta

# Share of each series' cut points below the given limit, averaged over all series
def share(lengths, limit, below):
    lo = np.maximum(1, lengths - limit)
    return float(np.mean(np.clip(np.minimum(below, lengths) - lo, 0, None) / (lengths - lo)))

train = M4Dataset.load(training=True, dataset_file='./dataset/m4')
rows = {}
for sp in M4Meta.seasonal_patterns:
    h = M4Meta.horizons_map[sp]
    limit = int(M4Meta.history_size[sp] * h)
    series = [v[~np.isnan(v)] for v in train.values[train.groups == sp]]
    L = np.array([len(s) for s in series])
    values = np.concatenate(series)
    row = {'series': len(L), 'min length': int(L.min()), 'median length': int(np.median(L))}
    for name, lengths in [('TSLib', L), ('no-leak', L - h)]:
        row[f'shifted targets % ({name})'] = 100 * share(lengths, limit, h)
        row[f'zero-padded inputs % ({name})'] = 100 * share(lengths, limit, 2 * h)
    row['padded val inputs % (no-leak)'] = 100 * float(np.mean(L - h < 2 * h))
    row['padded test inputs %'] = 100 * float(np.mean(L < 2 * h))
    row['zeros in data'] = int((values == 0).sum())
    row['negatives in data'] = int((values < 0).sum())
    rows[sp] = row
print('DIAGJSON' + json.dumps(rows))
"""
out = R.sh(f"{PY} -c {R.shlex.quote(DIAG)}", cwd=REPO, quiet=True)
DIAGNOSTICS = pd.DataFrame(json.loads(out.split("DIAGJSON")[-1])).T
DIAGNOSTICS.round(2)


,series,min length,median length,shifted targets % (TSLib),zero-padded inputs % (TSLib),shifted targets % (no-leak),zero-padded inputs % (no-leak),padded val inputs % (no-leak),padded test inputs %,zeros in data,negatives in data
Yearly,23000.0,13.0,29.0,0.24,13.38,12.83,34.23,17.13,0.0,0.0,0.0
Quarterly,24000.0,16.0,88.0,0.00,0.62,0.62,2.81,1.00,0.0,0.0,0.0
Monthly,48000.0,42.0,202.0,0.00,0.06,0.05,10.21,0.03,0.0,0.0,0.0
Weekly,359.0,80.0,934.0,2.75,5.73,3.29,6.86,0.00,0.0,0.0,0.0
Daily,4227.0,93.0,2940.0,0.31,0.70,0.41,0.98,0.00,0.0,0.0,0.0
Hourly,414.0,700.0,960.0,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0


In [7]:
# In-memory patch for target window alignment and first-value padding in Dataset_M4
PREPROC_DEFS = r"""
import numpy as np
import data_provider.data_factory as data_factory


# Builds on the class already registered for 'm4' (TSLib's, or the no-leak one from section 2b)
class Dataset_M4_Prep(data_factory.data_dict['m4']):

    # One training example for a cut point: input = values before it, target = values after it
    def _window(self, ts, cut):
        insample = np.zeros((self.seq_len, 1))
        insample_mask = np.zeros((self.seq_len, 1))
        outsample = np.zeros((self.pred_len + self.label_len, 1))
        outsample_mask = np.zeros((self.pred_len + self.label_len, 1))

        # Input: missing history is filled with the first observed value, not 0 (mask stays 0 there)
        insample_window = ts[max(0, cut - self.seq_len):cut]
        insample[:, 0] = insample_window[0]
        insample[-len(insample_window):, 0] = insample_window
        insample_mask[-len(insample_window):, 0] = 1.0

        # Target: placed so that position label_len is always the cut point
        start = max(0, cut - self.label_len)
        outsample_window = ts[start:min(len(ts), cut + self.pred_len)]
        offset = self.label_len - (cut - start)
        outsample[offset:offset + len(outsample_window), 0] = outsample_window
        outsample_mask[offset:offset + len(outsample_window), 0] = 1.0
        return insample, outsample, insample_mask, outsample_mask

    # Same random cut point as TSLib (same random numbers), then the corrected example
    def __getitem__(self, index):
        ts = self.timeseries[index]
        cut = np.random.randint(low=max(1, len(ts) - self.window_sampling_limit),
                                high=len(ts),
                                size=1)[0]
        return self._window(ts, cut)

    # Inputs of validation / test forecasts: same first-value padding for short series
    def last_insample_window(self):
        insample, insample_mask = super().last_insample_window()
        for i, ts in enumerate(self.timeseries):
            if len(ts) < self.seq_len:
                insample[i, :-len(ts)] = ts[0]
        return insample, insample_mask

data_factory.data_dict['m4'] = Dataset_M4_Prep
"""

# Combine active patches
CODE = (NOLEAK_DEFS if NO_LEAK else "") + (PREPROC_DEFS if PREPROC_FIX else "")

# Subprocess entry point
ENTRY = ("-c", CODE + LAUNCH) if CODE else ("run.py",)
print("Validation:", "held-out end of the training series (no leak)" if NO_LEAK
      else "M4 test set (TSLib original, leaks)")
print("Preprocessing:", "aligned targets + first-value padding" if PREPROC_FIX
      else "TSLib original (shifted targets for short series, zero padding)")


Validation: held-out end of the training series (no leak)
Preprocessing: aligned targets + first-value padding


In [8]:
# Test window alignment and padding across synthetic series lengths
UNIT = PREPROC_DEFS + r"""
import json, types
from data_provider.data_loader import Dataset_M4

h = 6
ds = Dataset_M4_Prep.__new__(Dataset_M4_Prep)
ds.seq_len, ds.label_len, ds.pred_len = 2 * h, h, h
cuts = old_shifted = old_zero_padded = 0
for L in range(2, 40):
    ts = np.arange(1, L + 1, dtype=float)
    for cut in range(1, L):
        cuts += 1
        x, y, xm, ym = ds._window(ts, cut)
        for j in range(h):
            assert ym[h + j, 0] == (cut + j < L), 'target mask is wrong'
            assert ym[h + j, 0] == 0 or y[h + j, 0] == ts[cut + j], 'target is shifted'
        assert x[-1, 0] == ts[cut - 1], 'input does not end right before the cut'
        assert (x[:, 0] >= 1).all(), 'zero padding left'
        assert xm.sum() == min(cut, 2 * h), 'input mask is wrong'

        # TSLib's original __getitem__ with the same cut point
        orig = types.SimpleNamespace(seq_len=2 * h, label_len=h, pred_len=h,
                                     window_sampling_limit=10 ** 9, timeseries=[ts])
        randint, np.random.randint = np.random.randint, lambda **kw: np.array([cut])
        try:
            ox, oy, oxm, oym = Dataset_M4.__getitem__(orig, 0)
        finally:
            np.random.randint = randint
        old_shifted += any(oym[h + j, 0] == 1 and oy[h + j, 0] != ts[cut + j] for j in range(h))
        old_zero_padded += bool((oxm[:, 0] == 0).any())

ds.timeseries = [np.arange(1, L + 1, dtype=float) for L in range(1, 20)]
w, m = ds.last_insample_window()
assert (w > 0).all(), 'zero padding left in last_insample_window'
assert all(m[i].sum() == min(i + 1, 2 * h) for i in range(len(w))), 'mask changed'
print('UNITJSON' + json.dumps({'cuts': cuts, 'old_shifted': int(old_shifted),
                               'old_zero_padded': int(old_zero_padded)}))
"""
out = R.sh(f"{PY} -c {R.shlex.quote(UNIT)}", cwd=REPO, quiet=True)
info = json.loads(out.split("UNITJSON")[-1])
print(f"OK - {info['cuts']} cut points checked: targets aligned, no zero padding, masks unchanged.\n"
      f"TSLib original on the same cut points: {info['old_shifted']} with shifted targets, "
      f"{info['old_zero_padded']} with zero padding.")


OK - 741 cut points checked: targets aligned, no zero padding, masks unchanged.
TSLib original on the same cut points: 165 with shifted targets, 363 with zero padding.


## 2d. Outlier Diagnostics and Robust Standardization

Check input windows for spikes and std inflation against Gaussian noise, with optional median/MAD instance normalization inside TimesNet.

In [9]:
# Check outlier statistics on M4 input windows per subset
import numpy as np
OUTLIER = r"""
import json
import numpy as np
from data_provider.m4 import M4Dataset, M4Meta

NO_LEAK, PREPROC_FIX = __FLAGS__

# Robust z-scores: distance from the median in units of 1.4826 * MAD (= std for normal data)
def robust_z(x):
    dev = np.abs(x - np.median(x))
    mad = 1.4826 * np.median(dev)
    if mad == 0:                        # more than half the values are equal: use the mean deviation instead
        mad = 1.2533 * dev.mean()
    return dev / mad if mad > 0 else np.zeros_like(x)

# observed = real values of the window, window = what TimesNet gets (padded to seq_len)
def window_stats(observed, window):
    std = window.std()
    mad = 1.4826 * np.median(np.abs(window - np.median(window)))
    return (len(observed) >= 4 and bool((robust_z(np.diff(observed)) > 3.5).any()),   # spike
            bool((robust_z(observed) > 3.5).any()),                                    # level outlier
            std / mad if mad > 0 else np.nan,                                          # std inflation
            bool(mad < 0.1 * std))                                                     # floor used

def pad(observed, n):
    window = np.full(n, observed[0] if PREPROC_FIX else 0.0)
    window[-len(observed):] = observed
    return window

def summary(stats):
    spike, level, inflation, floor = (np.array(s, dtype=float) for s in zip(*stats))
    return {'spike windows %': 100 * spike.mean(), 'level-outlier windows %': 100 * level.mean(),
            'std/MAD median': float(np.nanmedian(inflation)),
            'std/MAD p90': float(np.nanpercentile(inflation, 90)), 'floor used %': 100 * floor.mean()}

rng = np.random.default_rng(0)
train = M4Dataset.load(training=True, dataset_file='./dataset/m4')
rows = {}
for sp in M4Meta.seasonal_patterns:
    h = M4Meta.horizons_map[sp]
    n = 2 * h                                                   # seq_len of TimesNet for M4
    limit = int(M4Meta.history_size[sp] * h)
    series = [v[~np.isnan(v)] for v in train.values[train.groups == sp]]

    # Test inputs: the last n points of every full training series
    test = [window_stats(s[-n:], pad(s[-n:], n)) for s in series]

    # Training inputs: one random cut point (like Dataset_M4) for up to 20000 series
    samples = []
    for k in rng.choice(len(series), min(20000, len(series)), replace=False):
        s = series[k][:-h] if NO_LEAK else series[k]
        cut = rng.integers(max(1, len(s) - limit), len(s))
        observed = s[max(0, cut - n):cut]
        samples.append(window_stats(observed, pad(observed, n)))

    # Reference: pure Gaussian noise windows of the same length (no outliers at all)
    noise = [window_stats(w, w) for w in rng.normal(size=(20000, n))]

    row = {'series': len(series), 'window length': n}
    for name, stats in [('test', test), ('train', samples), ('noise', noise)]:
        row.update({f'{k} ({name})': v for k, v in summary(stats).items()})
    rows[sp] = row
print('OUTLIERJSON' + json.dumps(rows))
""".replace("__FLAGS__", repr((NO_LEAK, PREPROC_FIX)))
out = R.sh(f"{PY} -c {R.shlex.quote(OUTLIER)}", cwd=REPO, quiet=True)
OUTLIERS = pd.DataFrame(json.loads(out.split("OUTLIERJSON")[-1])).T

# Flag subsets where test windows exceed 1.5x noise spikes and 1.2x noise std/MAD p90
worth = ((OUTLIERS["spike windows % (test)"] > 1.5 * OUTLIERS["spike windows % (noise)"])
         & (OUTLIERS["std/MAD p90 (test)"] > 1.2 * OUTLIERS["std/MAD p90 (noise)"]))
OUTLIERS["verdict"] = np.where(worth, "robust worth testing", "mean/std is fine")
OUTLIERS.round(2).to_csv(os.path.join(ROOT, "table14_outliers.csv"))
print("Subsets where robust normalization is worth testing:", list(OUTLIERS.index[worth]) or "none")
print("To test them: ROBUST_NORM = True and ROBUST_SUBSETS =", list(OUTLIERS.index[worth]), "in section 2.")
display(OUTLIERS.T.round(2))


Subsets where robust normalization is worth testing: ['Monthly']
To test them: ROBUST_NORM = True and ROBUST_SUBSETS = ['Monthly'] in section 2.


,Yearly,Quarterly,Monthly,Weekly,Daily,Hourly
series,23000.0,24000.0,48000.0,359.0,4227.0,414.0
window length,12.0,16.0,36.0,26.0,28.0,96.0
spike windows % (test),27.965217,25.995833,37.458333,34.818942,24.982257,38.164251
level-outlier windows % (test),12.673913,15.4,21.927083,19.498607,12.230897,10.628019
std/MAD median (test),0.910379,0.970102,1.00559,0.993424,0.944923,0.751997
std/MAD p90 (test),1.562079,1.590978,1.633476,1.489805,1.408866,1.110742
floor used % (test),0.213043,0.133333,0.735417,0.557103,0.283889,0.241546
spike windows % (train),23.98,28.12,40.435,40.111421,27.63189,40.57971
level-outlier windows % (train),11.79,15.82,23.14,24.512535,12.964277,11.352657
std/MAD median (train),0.917899,0.972832,1.017872,0.990721,0.964999,0.760685


In [10]:
# Robust median / MAD instance normalization patch for TimesNet
ROBUST_DEFS = r"""
import sys
import torch
import models.TimesNet as timesnet

ROBUST_SUBSETS = __ROBUST_SUBSETS__
_argv = sys.argv[1:]
_subset = _argv[_argv.index('--seasonal_patterns') + 1] if '--seasonal_patterns' in _argv else None


# Center and scale of every input window (over time): median and 1.4826 * MAD;
# the scale is kept >= 0.1 * std so mostly flat (e.g. padded) windows are not divided by ~0
def robust_stats(x_enc):
    center = torch.quantile(x_enc, 0.5, dim=1, keepdim=True)
    mad = torch.quantile((x_enc - center).abs(), 0.5, dim=1, keepdim=True)
    std = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
    return center.detach(), torch.maximum(1.4826 * mad, 0.1 * std).detach()


# TimesNet.forecast with median / MAD instead of mean / std; everything else is TSLib's code unchanged
def _robust_forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
    center, scale = robust_stats(x_enc)
    x_enc = (x_enc - center) / scale

    enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]
    enc_out = self.predict_linear(enc_out.permute(0, 2, 1)).permute(0, 2, 1)  # align temporal dimension
    for i in range(self.layer):
        enc_out = self.layer_norm(self.model[i](enc_out))
    dec_out = self.projection(enc_out)

    # Back to the original scale (same center / scale for every time step)
    return dec_out * scale + center

_forecast = timesnet.Model.forecast
if _subset in ROBUST_SUBSETS:
    timesnet.Model.forecast = _robust_forecast
    print('Normalization: median / MAD (robust) for', _subset)
"""

# Add it to the code that runs before run.py (nothing changes when ROBUST_NORM = False)
if ROBUST_NORM:
    CODE += ROBUST_DEFS.replace("__ROBUST_SUBSETS__", repr(ROBUST_SUBSETS))
    ENTRY = ("-c", CODE + LAUNCH)
print("Normalization:", f"median / MAD (robust) for {ROBUST_SUBSETS}" if ROBUST_NORM
      else "mean / std (TSLib original)")


Normalization: mean / std (TSLib original)


In [11]:
# Test robust normalization on a small TimesNet model
RCHECK = "import sys\nsys.argv = ['run.py', '--seasonal_patterns', 'Yearly']\n" + \
         ROBUST_DEFS.replace("__ROBUST_SUBSETS__", "['Yearly']") + r"""
import json, types
assert timesnet.Model.forecast is _robust_forecast, 'patch not applied for a chosen subset'
timesnet.Model.forecast = _forecast

torch.manual_seed(0)
cfg = types.SimpleNamespace(task_name='short_term_forecast', seq_len=12, label_len=6, pred_len=6,
                            top_k=5, d_model=16, d_ff=16, num_kernels=6, e_layers=2, enc_in=1, c_out=1,
                            embed='timeF', freq='h', dropout=0.0)
model = timesnet.Model(cfg).eval()
robust = lambda x: _robust_forecast(model, x, None, None, None)
x = 100 + torch.randn(64, 12, 1).cumsum(1)                  # clean random-walk windows

with torch.no_grad():
    # Same shape as TSLib's forecast, and the same answer after shifting / scaling the input
    assert robust(x).shape == _forecast(model, x, None, None, None).shape
    assert torch.allclose(robust(3 * x + 1000), 3 * robust(x) + 1000, rtol=1e-4, atol=1e-2), 'de-normalization wrong'

    # One spike: robust center / scale stay close to the clean ones, mean / std move a lot
    spiked = x.clone()
    spiked[:, 5] += 50
    c0, s0 = robust_stats(x)
    c1, s1 = robust_stats(spiked)
    robust_move = float(((c1 - c0).abs() / s0).median())
    mean_move = float(((spiked.mean(1, keepdim=True) - x.mean(1, keepdim=True)).abs() / x.std(1, keepdim=True)).median())
    assert robust_move < mean_move, 'robust center moved more than the mean'

    # Test flat window and a 4-point toy example
    assert torch.isfinite(robust(torch.full((2, 12, 1), 7.0))).all(), 'flat window gives NaN / inf'
    c, s = robust_stats(torch.tensor([10., 11., 12., 50.]).view(1, 4, 1))
print('RCHECKJSON' + json.dumps({'robust_move': robust_move, 'mean_move': mean_move,
                                 'example': [round(float(c), 2), round(float(s), 2)]}))
"""
out = R.sh(f"{PY} -c {R.shlex.quote(RCHECK)}", cwd=REPO, quiet=True)
info = json.loads(out.split("RCHECKJSON")[-1])
print(f"OK - robust normalization: same output shape, correct back-scaling, finite on flat windows, "
      f"only patched for the chosen subsets.\nOne spike of +50 moves the robust center by "
      f"{info['robust_move']:.2f} scales and the mean by {info['mean_move']:.2f} std. "
      f"Example [10, 11, 12, 50]: median {info['example'][0]}, scale {info['example'][1]}.")


OK - robust normalization: same output shape, correct back-scaling, finite on flat windows, only patched for the chosen subsets.
One spike of +50 moves the robust center by 0.18 scales and the mean by 3.07 std. Example [10, 11, 12, 50]: median 11.5, scale 1.69.


## 3. Paper values (TimesNet column)

In [12]:
# Paper values, TimesNet column of Table 14 (SMAPE, MASE, OWA)
PAPER_T14 = {"Yearly": (13.387, 2.996, 0.786),
             "Quarterly": (10.100, 1.182, 0.890),
             "Monthly": (12.670, 0.933, 0.878),
             "Others": (4.891, 3.302, 1.035),
             "Average": (11.829, 1.585, 0.851)}

# Earlier reproduction with TSLib's original validation on the test set (mean of 3 seeds)
LEAKY_T14 = {"Yearly": (13.510, 3.044, 0.797),
             "Quarterly": (10.139, 1.192, 0.895),
             "Monthly": (12.763, 0.940, 0.884),
             "Others": (5.014, 3.343, 1.055),
             "Average": (11.918, 1.604, 0.859)}

## 4. The original commands

The cells in this section:

1) Use the commands from the original repository.

2) Use the original per-dataset settings for the model.


In [13]:
import re, json, shutil, pandas as pd

# Read the original commands from the TSLib script
COMMANDS = R.expand_script(REPO, "scripts/short_term_forecast/TimesNet_M4.sh")
assert len(COMMANDS) == 6, len(COMMANDS)

# Check the M4 Naive2 file (needed for OWA) was downloaded
assert os.path.exists("dataset/m4/submission-Naive2.csv"), "M4 Naive2 file missing"

# Show the official settings per subset
pd.DataFrame([{k: R.arg(c, k) for k in ["model_id",
                                        "seasonal_patterns",
                                        "e_layers",
                                        "d_model",
                                        "d_ff",
                                        "top_k",
                                        "batch_size",
                                        "learning_rate",
                                        "train_epochs"]} for c in COMMANDS])

,model_id,seasonal_patterns,e_layers,d_model,d_ff,top_k,batch_size,learning_rate,train_epochs
0,m4_Monthly,Monthly,2,32,32,5,16,0.001,None
1,m4_Yearly,Yearly,2,16,32,5,16,0.001,None
2,m4_Quarterly,Quarterly,2,64,64,5,16,0.001,None
3,m4_Daily,Daily,2,16,16,5,16,0.001,None
4,m4_Weekly,Weekly,2,32,32,5,16,0.001,None
5,m4_Hourly,Hourly,2,32,32,5,16,0.001,None


In [14]:
# Paper settings (d_ff is not given in the paper, so TSLib's value is kept)
PAPER_COMMANDS = []
for c in COMMANDS:
    for k, v in {"e_layers": 2,
                 "top_k": 5,
                 "d_model": 16,
                 "batch_size": 16,
                 "learning_rate": 0.001,
                 "train_epochs": 10}.items():
        c = R.set_arg(c, k, v)
    PAPER_COMMANDS.append(c)

# Show the paper settings per subset
pd.DataFrame([{k: R.arg(c, k) for k in ["model_id",
                                        "seasonal_patterns",
                                        "e_layers",
                                        "d_model",
                                        "d_ff",
                                        "top_k",
                                        "batch_size",
                                        "learning_rate",
                                        "train_epochs"]} for c in PAPER_COMMANDS])

,model_id,seasonal_patterns,e_layers,d_model,d_ff,top_k,batch_size,learning_rate,train_epochs
0,m4_Monthly,Monthly,2,16,32,5,16,0.001,10
1,m4_Yearly,Yearly,2,16,32,5,16,0.001,10
2,m4_Quarterly,Quarterly,2,16,64,5,16,0.001,10
3,m4_Daily,Daily,2,16,16,5,16,0.001,10
4,m4_Weekly,Weekly,2,16,32,5,16,0.001,10
5,m4_Hourly,Hourly,2,16,32,5,16,0.001,10


## 5. Run

In [15]:
# Folder for the forecasts of each seed
FORECASTS = os.path.join(ROOT, "smoke_forecasts" if SMOKE else "forecasts")

# A run counts as successful if TSLib printed the test shape
def parse_m4(output):
    return {"status": "ok"} if "test shape" in output else None

# TSLib overwrites m4_results/<model>/ on every run, so move each seed's forecasts
def keep_forecast(args, seed):
    sp = R.arg(args, "seasonal_patterns").strip("'")
    src = os.path.join(REPO, "m4_results", "TimesNet", f"{sp}_forecast.csv")
    if os.path.exists(src):
        dst = os.path.join(FORECASTS, f"seed{seed}")
        os.makedirs(dst, exist_ok=True)
        shutil.move(src, os.path.join(dst, f"{sp}_forecast.csv"))

# Run the experiments
R.run_all(python=PY,
          repo_dir=REPO,
          root=ROOT,
          commands=PAPER_COMMANDS,
          seeds=SEEDS,
          parse=parse_m4,
          meta=META,
          smoke=SMOKE,
          after_run=keep_forecast,
          entry=ENTRY)

# Compute SMAPE, MASE, OWA per seed
EVAL = '''
import json, sys
from utils.m4_summary import M4Summary
smape, owa, mape, mase = M4Summary(sys.argv[1], "./dataset/m4").evaluate()
print("M4JSON" + json.dumps({"smape": smape, "mase": mase, "owa": owa}, default=float))
'''
scores = {}
for seed in SEEDS:
    folder = os.path.join(FORECASTS, f"seed{seed}") + "/"
    if len(os.listdir(folder) if os.path.exists(folder) else []) < 6:
        print(f"seed {seed}: not all 6 subsets finished yet")
        continue
    out = R.sh(f"{PY} -c {R.shlex.quote(EVAL)} {folder}", cwd=REPO, quiet=True)
    scores[seed] = json.loads(out.split("M4JSON")[-1])
    R.save_json(os.path.join(ROOT, "smoke_scores" if SMOKE else "scores", f"seed{seed}.json"),
                scores[seed])
    print(f"seed {seed}: weighted average SMAPE {scores[seed]['smape']['Average']:.3f}, "
          f"MASE {scores[seed]['mase']['Average']:.3f}, OWA {scores[seed]['owa']['Average']:.3f}")

[1/18] m4_Monthly seed 2021: already done, skipping
[2/18] m4_Yearly seed 2021: already done, skipping
[3/18] m4_Quarterly seed 2021: already done, skipping
[4/18] m4_Daily seed 2021: already done, skipping
[5/18] m4_Weekly seed 2021: already done, skipping
[6/18] m4_Hourly seed 2021: already done, skipping
[7/18] m4_Monthly seed 2022: already done, skipping
[8/18] m4_Yearly seed 2022: already done, skipping
[9/18] m4_Quarterly seed 2022: already done, skipping
[10/18] m4_Daily seed 2022: already done, skipping
[11/18] m4_Weekly seed 2022: already done, skipping
[12/18] m4_Hourly seed 2022: already done, skipping
[13/18] m4_Monthly seed 2023: already done, skipping
[14/18] m4_Yearly seed 2023: already done, skipping
[15/18] m4_Quarterly seed 2023: already done, skipping
[16/18] m4_Daily seed 2023: already done, skipping
[17/18] m4_Weekly seed 2023: already done, skipping
[18/18] m4_Hourly seed 2023: already done, skipping
seed 2021: weighted average SMAPE 12.103, MASE 1.659, OWA 0.880


## 6. Comparison with the paper

In [16]:
assert scores, "No seed has all 6 subsets finished yet - run section 5 first."
rows = []
for group, (p_smape, p_mase, p_owa) in PAPER_T14.items():
    l_smape, l_mase, l_owa = LEAKY_T14[group]
    for metric, paper, leaky in [("SMAPE", p_smape, l_smape), ("MASE", p_mase, l_mase),
                                 ("OWA", p_owa, l_owa)]:
        vals = {s: scores[s][metric.lower()][group] for s in scores}
        row = {"group": group, "metric": metric, "paper": paper, "leaky": leaky,
               **{f"seed_{s}": v for s, v in vals.items()}}
        if vals:
            mean = sum(vals.values()) / len(vals)
            std = (sum((v - mean) ** 2 for v in vals.values()) / len(vals)) ** 0.5
            rel = 100 * (mean - paper) / paper
            row.update({"mean": mean, "std": std, "delta": mean - paper, "delta %": rel,
                        "delta vs leaky": mean - leaky,
                        "verdict": R.verdict(None, rel, "rel")})
        rows.append(row)
table = pd.DataFrame(rows).set_index(["group", "metric"])
table.round(3).to_csv(os.path.join(ROOT, "table14_reproduction.csv"))
open(os.path.join(ROOT, "table14_reproduction.md"), "w").write(table.round(3).to_markdown())
table.round(3)

paper   leaky  seed_2021  seed_2022  seed_2023    mean  \
group     metric                                                            
Yearly    SMAPE   13.387  13.510     13.901     13.818     13.671  13.797   
          MASE     2.996   3.044      3.216      3.209      3.150   3.192   
          OWA      0.786   0.797      0.830      0.826      0.815   0.824   
Quarterly SMAPE   10.100  10.139     10.277     10.329     10.279  10.295   
          MASE     1.182   1.192      1.221      1.224      1.218   1.221   
          OWA      0.890   0.895      0.912      0.915      0.911   0.913   
Monthly   SMAPE   12.670  12.763     12.887     12.954     12.975  12.939   
          MASE     0.933   0.940      0.950      0.958      0.958   0.955   
          OWA      0.878   0.884      0.893      0.899      0.900   0.897   
Others    SMAPE    4.891   5.014      5.079      5.024      5.116   5.073   
          MASE     3.302   3.343      3.406      3.367      3.519   3.431   
          OWA      1.035   1.055      1.072      1.059      1.093   1.075   
Average   SMAPE   11.829  11.918     12.103     12.126     12.095  12.108   
          MASE     1.585   1.604      1.659      1.660      1.653   1.657   
          OWA      0.851   0.859      0.880      0.881      0.878   0.880   

                    std  delta  delta %  delta vs leaky  verdict  
group     metric                                                  
Yearly    SMAPE   0.095  0.410    3.060           0.287    close  
          MASE    0.030  0.196    6.531           0.148  differs  
          OWA     0.006  0.038    4.792           0.027    close  
Quarterly SMAPE   0.024  0.195    1.931           0.156  matches  
          MASE    0.002  0.039    3.299           0.029    close  
          OWA     0.002  0.023    2.547           0.018    close  
Monthly   SMAPE   0.038  0.269    2.120           0.176    close  
          MASE    0.004  0.022    2.394           0.015    close  
          OWA     0.003  0.019    2.202           0.013    close  
Others    SMAPE   0.038  0.182    3.721           0.059    close  
          MASE    0.064  0.129    3.897           0.088    close  
          OWA     0.014  0.040    3.833           0.020    close  
Average   SMAPE   0.013  0.279    2.359           0.190    close  
          MASE    0.003  0.072    4.564           0.053    close  
          OWA     0.001  0.029    3.369           0.021    close

## 6b. Ablation: Effect of Each Fix


In [17]:
import glob

# Map result folders to table labels
VARIANTS = {"table14": "TSLib original (leaky)",
            "table14_prepfix": "preprocessing fix only",
            "table14_noleak": "no leak",
            "table14_noleak_prepfix": "no leak + preprocessing fix"}

# Include any robust normalization runs
for path in sorted(glob.glob(os.path.join(os.path.dirname(ROOT), "table14*_robust*"))):
    folder = os.path.basename(path)
    VARIANTS[folder] = folder.replace("table14_", "") + " (robust normalization)"

# Load saved seed scores per variant
found = {}
for folder, name in VARIANTS.items():
    files = glob.glob(os.path.join(os.path.dirname(ROOT), folder,
                                   "smoke_scores" if SMOKE else "scores", "seed*.json"))
    if files:
        found[f"{name} ({len(files)} seeds)"] = [R.load_json(f) for f in files]

# Build comparison table across variants
rows = []
for group, (p_smape, p_mase, p_owa) in PAPER_T14.items():
    rows.append({"group": group, "variant": "paper", "SMAPE": p_smape, "MASE": p_mase, "OWA": p_owa})
    for name, runs in found.items():
        rows.append({"group": group, "variant": name,
                     **{k.upper(): sum(r[k][group] for r in runs) / len(runs)
                        for k in ["smape", "mase", "owa"]}})
ablation = pd.DataFrame(rows).set_index(["group", "variant"])
ablation.round(3)


SMAPE   MASE    OWA
group     variant                                                    
Yearly    paper                                  13.387  2.996  0.786
          no leak + preprocessing fix (3 seeds)  13.797  3.192  0.824
Quarterly paper                                  10.100  1.182  0.890
          no leak + preprocessing fix (3 seeds)  10.295  1.221  0.913
Monthly   paper                                  12.670  0.933  0.878
          no leak + preprocessing fix (3 seeds)  12.939  0.955  0.897
Others    paper                                   4.891  3.302  1.035
          no leak + preprocessing fix (3 seeds)   5.073  3.431  1.075
Average   paper                                  11.829  1.585  0.851
          no leak + preprocessing fix (3 seeds)  12.108  1.657  0.880

## 7. Determinism check
Re-runs one configuration and confirms the result is identical to the stored one.

In [18]:
# Re-run Yearly with seed 2021 and check that the forecasts are identical
import filecmp
yearly = [c for c in COMMANDS if "Yearly" in R.arg(c, "seasonal_patterns")][0]
ok, _ = R.run_tslib(PY, REPO, yearly, SEEDS[0],
                    os.path.join(ROOT, "logs", "m4_Yearly__recheck.log"), SMOKE, ENTRY)
R.clean_outputs(REPO)
new = os.path.join(REPO, "m4_results", "TimesNet", "Yearly_forecast.csv")
old = os.path.join(FORECASTS, f"seed{SEEDS[0]}", "Yearly_forecast.csv")
same = ok and filecmp.cmp(new, old, shallow=False)
os.remove(new) if os.path.exists(new) else None
print("IDENTICAL forecasts - bit-for-bit repeatable." if same else
      "DIFFERENT - check GPU model / environment against environment.json.")

IDENTICAL forecasts - bit-for-bit repeatable.


## 8. TimesNet (GELU) vs TimesNet-KANB on M4: explainability, drift and novelty (any group)

This section compares the original TimesNet (GELU activation) with TimesNet-KANB, a variant that uses the trainable KANB activation, on the M4 data. It looks at which periods the models use, what drives their forecasts, how much the data drifts, and where the models fail.

**Questions**
1. Which cycles (periods) does TimesNet use, and do they match each group's seasonality?
2. Which part of the past drives the forecasts (global and per series)? Are the explanations **trustworthy**
   (faithfulness) and **stable** (over seeds, folds, calendar time, between the two models)?
3. Which part of the past does each model really **need** (segment ablation)?
4. How much **drift / novelty** is there (per series and over the whole population; covariate and concept drift),
   and **which model copes better with small vs large drift**?
5. Where do the models fail (worst series)?

**Experimental Protocol**

| Part | Choice | Rationale |
|---|---|---|
| Models | TimesNet (GELU) and TimesNet-KANB, **identical except for the activation** (KANB2 inside the TimesBlocks; KANB1 is defined but inactive, the forecasting head has no activation) | only the activation differs |
| CV | **calendar-aware sliding CV with a purge gap** (`S9_CV = "calendar"`): one **calendar cut-off** per fold, the same for all series (M4-info dates). Each fold trains on the values before the cut-off − gap of **every** series and validates on the first horizon after the cut-off; **gap = 2H = seq_len**, so the validation input never overlaps a training target. `S9_CV = "series"` = the earlier per-series CV (last K horizons of every series; does not remove the leak) | removes M4's **cross-series leak**: no model ever sees data later than a series' forecast start, from any series |
| Folds | **5** for Monthly, Weekly, Daily, Hourly; **3** for Yearly and Quarterly. Cut-offs = quantiles (10% … 50%) of the series' last possible validation start, so the early folds validate most series and later folds have more recent training data (if many series share dates, distinct dates are chosen automatically) | |
| Test | the **M4 test set, untouched**; every fold model forecasts every test series from its full history, **one series at a time** (as TSLib). **A test series counts only for fold models whose cut-off is not later than its forecast start** (leak-free); its score = mean over those fold models and seeds. Series that no fold may score, or with unreadable dates, are counted and left out | leak-free test, not directly comparable with the paper (less recent data) |
| Preprocessing | the preprocessing fix from Section 2c (aligned targets, first-value padding); validation = the fold's own blocks (no leak) | |
| Seeds | 2021, 2022, 2023 | |
| Grad-CAM | Grad-CAM of the mean forecast on TimesNet's **2D period images** (output of each TimesBlock's Conv2d stack; rows = cycles, columns = position inside one period), folded back to time and weighted by the period weights; future part (created by the linear layer before the convolutions) shaded | looks inside the part that makes TimesNet TimesNet |
| SHAP | permutation SHAP on **time segments** of the input (exactly additive) + GradientSHAP per lag, globally and **one signed bar per input step** for single series (red raised, blue lowered); replacements = **real training windows of other series rescaled to the series' level and scale**; results in each series' normalised units (TimesNet's own units) | univariate, very different scales |
| Drift | **per series**: forecast period vs the series' own input (energy distance), conformal p-value against the series' own history, Benjamini-Hochberg; **population**: MMD + energy distance (permutation tests) and a domain classifier with SHAP, inputs only (covariate) vs inputs + future (concept); **residual-based** concept drift; **novelty** (kNN) | non-parametric throughout |
| Drift bins | terciles of the per-series drift score | |
| KANB vs GELU | **paired Wilcoxon signed-rank** on per-series sMAPE / MASE differences, **bootstrap 95% CI**, **Holm** correction; plus folds won | series are independent, so Diebold-Mariano (time order) does not apply |
| Leak | **removed** by the calendar CV, and **measured** by the leak experiment (8.6b): in each fold a random half of the series keeps its values after the cut-off (the leak), the other half is scored, then swapped; same series and same amount of data, so leaky − clean error = the leak's effect. **Placebo** (optional) = the same with shuffled calendar positions. Wilcoxon + bootstrap + Holm; and whether **KANB profits more from the leak than GELU** | turns the leak into a measured quantity |

**How to use:** 8.1 settings → 8.2–8.4 write the scripts → 8.5 helpers → 8.6 train (one cell per group, resumable)
→ 8.6b leak experiment (optional, per group) → 8.7 analyse (one cell per group) → 8.8 report. Results go to
`table14_s9/<cv>/` (on Drive with `USE_DRIVE = True`).

**Limits:** the calendar CV is only as good as M4's dates (2-digit years, ~100 unreadable dates in Yearly/Monthly,
some artificial dates, e.g. Hourly's start dates cluster on a few days). Models see less recent data than in the paper's
protocol, so leak-free scores are expected to be somewhat worse; the leak experiment measures how much of that gap the
leak explains.
`S9_ACTIVE = False` skips the whole section (e.g. for "Run all").

**Cost (L4):** 2 models × folds × 3 seeds per group. Rough guide per training: Hourly/Weekly < 1 min, Daily ~3 min,
Yearly ~5–10 min, Quarterly ~15 min, Monthly ~30 min (fewer series in the early folds).

In [ ]:
# ------------------------------ 8.1 Settings ------------------------------

import subprocess, json, time, glob
import numpy as np
import pandas as pd

S9_ACTIVE = True                                # False = skip Section 8 (e.g. when using "Run all")
S9_MODELS = ["TimesNet", "TimesNet_KANB"]       # first = reference (GELU), second = ours
S9_SEEDS = SEEDS                                # [2021, 2022, 2023]
S9_CV = "calendar"                              # "calendar" = leak-free calendar CV; "series" = per-series CV
S9_GAP_H = 2.0                                  # purge gap in horizons (2 = seq_len)
S9_PLACEBO = False                              # leak experiment: also run the placebo (shuffled calendar)
S9_MIN_TRAIN_H = 1.0                            # a series joins a fold only if >= this x H training values remain
S9_FOLDS = {"Yearly": 3, "Quarterly": 3, "Monthly": 5, "Weekly": 5, "Daily": 5, "Hourly": 5}

# Analysis settings (passed to xai_m4.py); larger = more precise, slower
S9_XAI = {"n_segments": 8,        # input segments (SHAP features)
          "shap_series": 300,     # test series explained with permutation SHAP (same series for every model)
          "shap_samples": 16,     # random orders per series (antithetic pairs)
          "background": 64,       # real training windows used as replacements
          "grad_series": 100,     # series for GradientSHAP
          "grad_samples": 32,
          "grad_seeds": "2021",
          "faith_series": 100,    # series for the faithfulness curves
          "cam_series": 100,      # series for the global Grad-CAM profile
          "period_series": 2000,  # series for the period analysis
          "ablation_series": 500, # series for the segment ablation
          "ablation_repeats": 2,
          "pop_series": 3000,     # series per side for the population drift tests
          "perms": 199,           # permutations for the drift tests
          "clf_perms": 29,        # permutations for the domain classifier's AUC
          "gallery": 6}

# Model settings for Section 8 only (empty = the paper settings of Section 4; later entries override earlier ones).
# Each combination of settings gets its own results folder.
S9_HP = {"all": {},                      # both models, every group, e.g. {"train_epochs": 20}
         "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {}, "Hourly": {}}
S9_HP_MODEL = {"TimesNet":      {"all": {}, "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {},
                                 "Hourly": {}},
               "TimesNet_KANB": {"all": {}, "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {},
                                 "Hourly": {}}}

S9_ROOT = R.results_root(name = "table14_s9", use_drive = USE_DRIVE)
print("Results folder:", S9_ROOT)
if S9_ACTIVE:
    R.sh(f"uv pip install --python {PY} -q plotly==5.24.1 kaleido==0.2.1 shap==0.48.0 statsmodels")

Results folder: /content/TimesNet_reproduction/table14_s9
$ uv pip install --python /content/tslib_env_blackwell/bin/python -q plotly==5.24.1 kaleido==0.2.1 shap==0.48.0 statsmodels


In [ ]:
%%writefile models/TimesNet_KANB.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from layers.Embed import DataEmbedding
from layers.Conv_Blocks import Inception_Block_V1

# ---------------------------------------------------------------------------
# TimesNet with KANB activations ("Ours").
# Identical to TSLib's models/TimesNet.py (commit 4e938a1) except for the two
# activation functions, which are the model's only non-linear activations:
#   * KANB2 replaces nn.GELU() inside every TimesBlock (between the two Inception blocks)
#   * KANB1 replaces F.gelu in the classification head (before the final Linear layer)
#
# KANB: Kanakoudis, Mandalis & Pelekis, "KANB: A Family of Parametric Activation
# Functions With Trainable Parameters", IEEE Access 13 (2025), 146081-146087.
#   s = sigmoid(y)
#   KANB1(y) = 3(1-s)^2 s P1 + 3(1-s) s^2 P2 + s^3        (bounded in [0, 1])
#   KANB2(y) = 3(1-s)^2 s P1 + 3(1-s) s^2 P2 + y s^3      (-> 0 for y -> -inf, ~y for y -> +inf)
#   P1, P2: trainable, kept in [0, 1] (clip after every optimiser step, as in the paper),
#           initialised at KANB1: P1 = 0.2, P2 = 0.8 and KANB2: P1 = 0.8, P2 = 0.2 (every layer).
# ---------------------------------------------------------------------------


class KANB(nn.Module):
    """KANB activation with two trainable Bezier ordinates (one pair per layer)."""

    # initial (P1, P2) per variant; both parameters stay trainable
    INIT = {1: (0.2, 0.8),          # KANB1 (classification head)
            2: (0.8, 0.2)}          # KANB2 (inside every TimesBlock)

    def __init__(self, variant):
        super(KANB, self).__init__()
        assert variant in (1, 2), "variant must be 1 (KANB1) or 2 (KANB2)"
        self.variant = variant
        # Fixed starting values, no random draw: every other weight is initialised exactly as
        # in the original (GELU) TimesNet with the same seed -> a fair comparison.
        p1, p2 = KANB.INIT[variant]
        self.p1 = nn.Parameter(torch.tensor([p1]))
        self.p2 = nn.Parameter(torch.tensor([p2]))

    def forward(self, y):
        s = torch.sigmoid(y)
        t = 1.0 - s
        last = s ** 3 if self.variant == 1 else y * s ** 3
        return 3.0 * t * t * s * self.p1 + 3.0 * t * s * s * self.p2 + last

    @torch.no_grad()
    def clip_(self):
        """Clip01 constraint of the paper: keep P1, P2 in [0, 1]."""
        self.p1.clamp_(0.0, 1.0)
        self.p2.clamp_(0.0, 1.0)

    def extra_repr(self):
        return f"variant=KANB{self.variant}"


def clip_kanb_(model):
    """Call after every optimiser step."""
    for m in model.modules():
        if isinstance(m, KANB):
            m.clip_()


def kanb_parameters(model):
    """{module name: (variant, P1, P2)} for every KANB in the model."""
    return {name: (m.variant, float(m.p1), float(m.p2))
            for name, m in model.named_modules() if isinstance(m, KANB)}


def FFT_for_Period(x, k=2):
    # [B, T, C]
    xf = torch.fft.rfft(x, dim=1)
    # find period by amplitudes
    frequency_list = abs(xf).mean(0).mean(-1)
    frequency_list[0] = 0
    _, top_list = torch.topk(frequency_list, k)
    top_list = top_list.detach().cpu().numpy()
    period = x.shape[1] // top_list
    return period, abs(xf).mean(-1)[:, top_list]


class TimesBlock(nn.Module):
    def __init__(self, configs):
        super(TimesBlock, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.k = configs.top_k
        # parameter-efficient design
        self.conv = nn.Sequential(
            Inception_Block_V1(configs.d_model, configs.d_ff,
                               num_kernels=configs.num_kernels),
            KANB(2),  # [Ours] was nn.GELU()
            Inception_Block_V1(configs.d_ff, configs.d_model,
                               num_kernels=configs.num_kernels)
        )

    def forward(self, x):
        B, T, N = x.size()
        period_list, period_weight = FFT_for_Period(x, self.k)

        res = []
        for i in range(self.k):
            period = period_list[i]
            # padding
            if (self.seq_len + self.pred_len) % period != 0:
                length = (
                                 ((self.seq_len + self.pred_len) // period) + 1) * period
                padding = torch.zeros([x.shape[0], (length - (self.seq_len + self.pred_len)), x.shape[2]]).to(x.device)
                out = torch.cat([x, padding], dim=1)
            else:
                length = (self.seq_len + self.pred_len)
                out = x
            # reshape
            out = out.reshape(B, length // period, period,
                              N).permute(0, 3, 1, 2).contiguous()
            # 2D conv: from 1d Variation to 2d Variation
            out = self.conv(out)
            # reshape back
            out = out.permute(0, 2, 3, 1).reshape(B, -1, N)
            res.append(out[:, :(self.seq_len + self.pred_len), :])
        res = torch.stack(res, dim=-1)
        # adaptive aggregation
        period_weight = F.softmax(period_weight, dim=1)
        period_weight = period_weight.unsqueeze(
            1).unsqueeze(1).repeat(1, T, N, 1)
        res = torch.sum(res * period_weight, -1)
        # residual connection
        res = res + x
        return res


class Model(nn.Module):
    """
    TimesNet (paper link: https://openreview.net/pdf?id=ju_Uqw384Oq) with KANB2 in the
    TimesBlocks and KANB1 in the classification head.
    """

    def __init__(self, configs):
        super(Model, self).__init__()
        self.configs = configs
        self.task_name = configs.task_name
        self.seq_len = configs.seq_len
        self.label_len = configs.label_len
        self.pred_len = configs.pred_len
        self.model = nn.ModuleList([TimesBlock(configs)
                                    for _ in range(configs.e_layers)])
        self.enc_embedding = DataEmbedding(configs.enc_in, configs.d_model, configs.embed, configs.freq,
                                           configs.dropout)
        self.layer = configs.e_layers
        self.layer_norm = nn.LayerNorm(configs.d_model)
        if self.task_name == 'long_term_forecast' or self.task_name == 'short_term_forecast':
            self.predict_linear = nn.Linear(
                self.seq_len, self.pred_len + self.seq_len)
            self.projection = nn.Linear(
                configs.d_model, configs.c_out, bias=True)
        if self.task_name == 'imputation' or self.task_name == 'anomaly_detection':
            self.projection = nn.Linear(
                configs.d_model, configs.c_out, bias=True)
        if self.task_name == 'classification':
            self.act = KANB(1)  # [Ours] was F.gelu
            self.dropout = nn.Dropout(configs.dropout)
            self.projection = nn.Linear(
                configs.d_model * configs.seq_len, configs.num_class)

    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
        # Normalization from Non-stationary Transformer
        means = x_enc.mean(1, keepdim=True).detach()
        x_enc = x_enc.sub(means)
        stdev = torch.sqrt(
            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
        x_enc = x_enc.div(stdev)

        # embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]
        enc_out = self.predict_linear(enc_out.permute(0, 2, 1)).permute(
            0, 2, 1)  # align temporal dimension
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # project back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out.mul(
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        dec_out = dec_out.add(
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        return dec_out

    def imputation(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask):
        # Normalization from Non-stationary Transformer
        means = torch.sum(x_enc, dim=1) / torch.sum(mask == 1, dim=1)
        means = means.unsqueeze(1).detach()
        x_enc = x_enc.sub(means)
        x_enc = x_enc.masked_fill(mask == 0, 0)
        stdev = torch.sqrt(torch.sum(x_enc * x_enc, dim=1) /
                           torch.sum(mask == 1, dim=1) + 1e-5)
        stdev = stdev.unsqueeze(1).detach()
        x_enc = x_enc.div(stdev)

        # embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # project back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out.mul(
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        dec_out = dec_out.add(
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        return dec_out

    def anomaly_detection(self, x_enc):
        # Normalization from Non-stationary Transformer
        means = x_enc.mean(1, keepdim=True).detach()
        x_enc = x_enc.sub(means)
        stdev = torch.sqrt(
            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
        x_enc = x_enc.div(stdev)

        # embedding
        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # project back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out.mul(
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        dec_out = dec_out.add(
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        return dec_out

    def classification(self, x_enc, x_mark_enc):
        # embedding
        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))

        # Output
        # the output transformer encoder/decoder embeddings don't include non-linearity
        output = self.act(enc_out)
        output = self.dropout(output)
        # zero-out padding embeddings
        output = output * x_mark_enc.unsqueeze(-1)
        # (batch_size, seq_length * d_model)
        output = output.reshape(output.shape[0], -1)
        output = self.projection(output)  # (batch_size, num_classes)
        return output

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
        if self.task_name == 'long_term_forecast' or self.task_name == 'short_term_forecast':
            dec_out = self.forecast(x_enc, x_mark_enc, x_dec, x_mark_dec)
            return dec_out[:, -self.pred_len:, :]  # [B, L, D]
        if self.task_name == 'imputation':
            dec_out = self.imputation(
                x_enc, x_mark_enc, x_dec, x_mark_dec, mask)
            return dec_out  # [B, L, D]
        if self.task_name == 'anomaly_detection':
            dec_out = self.anomaly_detection(x_enc)
            return dec_out  # [B, L, D]
        if self.task_name == 'classification':
            dec_out = self.classification(x_enc, x_mark_enc)
            return dec_out  # [B, N]
        return None

Writing models/TimesNet_KANB.py


In [ ]:
%%writefile cv_m4.py
"""
Sliding-window cross-validation with a purge gap for TSLib short-term forecasting on M4
(TimesNet and TimesNet_KANB), evaluated on the untouched M4 test set.

Protocol (Section 8 of this notebook: M4 cross-validation):
* --cv calendar (default; removes M4's cross-series leak): one CALENDAR cut-off C_k per fold, the same for all
  series (quantiles of the series' last possible validation start, i.e. the date of value L - H). Fold k trains on the values before C_k - gap of every series and
  validates on the first H values from C_k. A test series is scored only by fold models with C_k <= its forecast
  start, so no model has seen anything later than a series' forecast start, from any series.
  The leak experiment (--variant leakA/leakB, placebo*) retrains a fold with one random half of the series
  keeping its values after C_k, and scores the other half's validation block (see fold_sets).
* --cv series: inside every series, K validation blocks of one horizon (H) each, consecutive and non-overlapping,
  ending where the series' training data ends (does not remove the cross-series leak).
* Purge gap (default 2H = seq_len) between a fold's training data and its validation block: the fold trains
  only on the series cut at  val_start - gap. The validation INPUT (the seq_len values before the block) lies
  inside the gap, so no training target ever overlaps a validation input or target.
* TSLib cuts M4 training examples only from the last history_size x H values of each (cut) series, so every
  fold has the same amount of training data per series (a sliding window).
* K = 5 for Monthly, Weekly, Daily, Hourly; K = 3 for Yearly and Quarterly (short series). A series joins a
  fold only if at least min_train_h x H values remain for training; counts are reported per fold.
* The M4 test set is never used for training, validation or early stopping. Every fold model forecasts every
  test series from its full training history (as TSLib), ONE SERIES AT A TIME (TimesNet picks its FFT periods
  from the whole batch, so batch 1 makes each forecast depend only on its own input; TSLib's test also uses 1).
* Preprocessing = the notebook's fix of Section 2c: aligned targets and first-value padding.
* Training = TSLib's own Exp_Short_Term_Forecast.train (Adam, SMAPE loss, lradj, EarlyStopping). Its vali()
  is given the fold's own validation inputs (the values just before each validation block).
* KANB: P1, P2 are clipped to [0, 1] after every optimizer step.
* Metrics per series: sMAPE, MASE (M4 definitions); per group: sMAPE, MASE, OWA (vs Naive2).

Usage (inside the TSLib repository):
  python cv_m4.py --make_folds --folds_file folds_m4/Yearly.json -- <TSLib args>
  python cv_m4.py --folds_file folds_m4/Yearly.json --fold 0 --seed 2021 --out_dir runs/Yearly/TimesNet -- <args>
"""
import argparse
import json
import os
import random
import shutil
import sys
import tempfile
import textwrap
import time

import numpy as np
import pandas as pd
import torch

GROUPS = ["Yearly", "Quarterly", "Monthly", "Weekly", "Daily", "Hourly"]
HORIZON = {"Yearly": 6, "Quarterly": 8, "Monthly": 18, "Weekly": 13, "Daily": 14, "Hourly": 48}
FREQ = {"Yearly": 1, "Quarterly": 4, "Monthly": 12, "Weekly": 1, "Daily": 1, "Hourly": 24}
HISTORY = {"Yearly": 1.5, "Quarterly": 1.5, "Monthly": 1.5, "Weekly": 10, "Daily": 10, "Hourly": 10}
DEFAULT_FOLDS = {"Yearly": 3, "Quarterly": 3, "Monthly": 5, "Weekly": 5, "Daily": 5, "Hourly": 5}
STEP = {"Yearly": pd.DateOffset(years=1), "Quarterly": pd.DateOffset(months=3), "Monthly": pd.DateOffset(months=1),
        "Weekly": pd.Timedelta(days=7), "Daily": pd.Timedelta(days=1), "Hourly": pd.Timedelta(hours=1)}


# ---------------------------------------------------------------------------------------------------------
# Arguments
# ---------------------------------------------------------------------------------------------------------
def get_tslib_parser():
    """TSLib's own argparse parser, built from run.py, so every flag and default is identical."""
    with open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "run.py")) as f:
        src = f.read()
    # Re-use run.py's own argument block (from 'parser = ...' to 'parse_args()'), so every flag and default
    # matches TSLib exactly and the saved tslib_args stay comparable
    start = src.rfind("\n", 0, src.index("parser = argparse.ArgumentParser")) + 1
    end = src.rfind("\n", 0, src.index("args = parser.parse_args()")) + 1
    namespace = {"argparse": argparse}
    exec(textwrap.dedent(src[start:end]), namespace)
    return namespace["parser"]


def parse_all_args(argv=None):
    argv = sys.argv[1:] if argv is None else argv
    if "--" not in argv:
        raise SystemExit("Put TSLib arguments after a `--` separator.")
    k = argv.index("--")
    p = argparse.ArgumentParser(description="Sliding-window CV with purge gap on M4")
    p.add_argument("--make_folds", action="store_true")
    p.add_argument("--folds_file", required=True)
    p.add_argument("--n_folds", type=int, default=0, help="0 = 3 for Yearly/Quarterly, 5 otherwise")
    p.add_argument("--gap_h", type=float, default=2.0, help="purge gap in horizons (2 = seq_len)")
    p.add_argument("--min_train_h", type=float, default=1.0, help="min training values per series, in horizons")
    p.add_argument("--cv", default="calendar", choices=["calendar", "series"])
    p.add_argument("--quantiles", default="", help="calendar CV: quantiles of the last validation starts (cut-offs)")
    p.add_argument("--ab_seed", type=int, default=0, help="calendar CV: seed of the A/B halves and the placebo")
    p.add_argument("--fold", type=int, default=0)
    p.add_argument("--seed", type=int, default=2021)
    p.add_argument("--variant", default="clean", choices=VARIANTS)
    p.add_argument("--out_dir", default="./runs_m4")
    p.add_argument("--save_checkpoint", type=int, default=1)
    own = p.parse_args(argv[:k])
    tslib = get_tslib_parser().parse_args(argv[k + 1:])
    tslib.seasonal_patterns = tslib.seasonal_patterns.strip("'\"")
    H = HORIZON[tslib.seasonal_patterns]                     # as TSLib's Exp_Short_Term_Forecast._build_model
    tslib.pred_len, tslib.seq_len, tslib.label_len = H, 2 * H, H
    tslib.frequency_map = FREQ[tslib.seasonal_patterns]
    return own, tslib


# ---------------------------------------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------------------------------------
def load_group(root, group):
    """Training series, test series (lists of arrays), ids and starting dates of one M4 group."""
    from data_provider.m4 import M4Dataset
    tr = M4Dataset.load(training=True, dataset_file=root)
    te = M4Dataset.load(training=False, dataset_file=root)
    sel = tr.groups == group
    clean = lambda vals: [np.asarray(v, dtype=np.float64)[~np.isnan(np.asarray(v, dtype=np.float64))] for v in vals]
    info = pd.read_csv(os.path.join(root, "M4-info.csv"))
    info = info.set_index("M4id").loc[tr.ids[sel]]
    return clean(tr.values[sel]), clean(te.values[sel]), np.asarray(tr.ids[sel]), info


VARIANTS = ["clean", "leakA", "leakB", "placebo", "placebo_leakA", "placebo_leakB"]


def fold_sets(lengths, spec, k, variant="clean"):
    """Who trains (and on how many values) and who is validated in fold k, for a variant of the fold.

    series CV  : the last K horizons of every series (only 'clean').
    calendar CV: one calendar cut-off C_k for all series; training = values before C_k - gap of every series,
                 validation = the first H values from C_k of every series that has them. A test series may be
                 scored by this fold model only if its forecast starts at or after C_k (no leak).
    leakA/leakB: the leak experiment. Half A (B) is validated and cut as in 'clean'; the other half keeps its
                 values AFTER the cut-off (up to its end) - the M4 cross-series leak, on purpose. Same series
                 and, because TSLib only samples the last history_size x H values, the same amount of data.
    placebo*   : the same with every series' calendar position shuffled (start dates permuted), so the 'leak'
                 carries no real calendar information.
    Returns train_idx, t1 (training length per training series), val_idx, v0 (validation start), test_eligible."""
    H, gap, min_train = spec["H"], spec["gap"], spec["min_train"]
    L = np.asarray(lengths)
    N = len(L)
    if spec.get("cv", "series") == "series":
        if variant != "clean":
            raise SystemExit("the leak experiment needs the calendar CV (--cv calendar)")
        K = spec["n_folds"]
        v0 = L - (K - k) * H
        t1 = v0 - gap
        idx = np.where(t1 >= min_train)[0]
        return idx, t1[idx], idx, v0[idx], np.ones(N, bool)
    cal = spec["cal"]
    key = "j_placebo" if variant.startswith("placebo") else "j"
    j = np.asarray(cal[key][k])
    dated = np.asarray(cal["dated"], bool)
    t1_raw = j - gap
    t1_all = np.minimum(L, t1_raw)
    train_mask = dated & (t1_all >= min_train)
    val_mask = dated & (t1_raw >= min_train) & (j + H <= L)
    eligible = dated & (L >= j)
    t1 = t1_all.copy()
    if variant.endswith("leakA") or variant.endswith("leakB"):
        ab = np.asarray(cal["ab"])
        ev = 0 if variant.endswith("leakA") else 1
        val_mask = val_mask & (ab == ev)
        leaky = train_mask & (ab != ev)
        t1[leaky] = L[leaky]                        # the other half keeps its values after the cut-off
    tr = np.where(train_mask)[0]
    va = np.where(val_mask)[0]
    return tr, t1[tr], va, j[va], eligible


def parse_dates(info, group):
    d = pd.to_datetime(info["StartingDate"], format="%d-%m-%y %H:%M", errors="coerce")
    return d


def date_at(start, step, n):
    try:
        return start + step * int(n)
    except Exception:  # out of bounds
        return pd.NaT


MONTHS = {"Yearly": 12, "Quarterly": 3, "Monthly": 1}


def steps_until(start, cutoff, group):
    """Index of the first value of a series (starting at `start`) whose date is >= cutoff."""
    step = STEP[group]
    try:
        if group in MONTHS:
            md = (cutoff.year - start.year) * 12 + (cutoff.month - start.month)
            j = int(np.floor(md / MONTHS[group])) - 1
            while start + step * j < cutoff:
                j += 1
            while start + step * (j - 1) >= cutoff:
                j -= 1
            return j
        return int(np.ceil((cutoff - start) / step))
    except Exception:
        return None


def calendar_report(train, info, spec, group, rng):
    """How much training data lies, in calendar time, AFTER the first predicted value of OTHER series (the
    cross-series leak of M4): share of (last training value, other series' first predicted value) pairs where
    the training value is later. Per fold (should be ~0 for the calendar CV) and for the paper's test protocol
    (models trained on the full histories)."""
    starts = parse_dates(info, group)
    step = STEP[group]
    L = np.array([len(s) for s in train])
    out = {"unparsable_dates": int(starts.isna().sum()), "series": len(L)}

    def ns(x):
        return np.array([v.value if not pd.isna(v) else np.nan for v in x], dtype=float)

    def share(train_dates, target_dates):
        a, b = train_dates[np.isfinite(train_dates)], np.sort(target_dates[np.isfinite(target_dates)])
        if len(a) == 0 or len(b) == 0:
            return None
        return float(np.mean(np.searchsorted(b, a, side="right") / len(b)))

    sub = np.sort(rng.choice(len(L), min(len(L), 3000), replace=False))
    rows = []
    for k in list(range(spec["n_folds"])) + ["test"]:
        if k == "test":
            tr_idx, t1, va_idx, v0 = np.arange(len(L)), L, np.arange(len(L)), L
        else:
            tr_idx, t1, va_idx, v0, _ = fold_sets(L, spec, k)
        kt, kv = np.isin(tr_idx, sub), np.isin(va_idx, sub)
        tr_dates = ns([date_at(starts.iloc[i], step, t - 1) for i, t in zip(tr_idx[kt], t1[kt])])
        target = ns([date_at(starts.iloc[i], step, v) for i, v in zip(va_idx[kv], v0[kv])])
        rows.append({"fold": "M4 test (paper protocol)" if k == "test" else k,
                     "share of training values after other series' target start": share(tr_dates, target),
                     "median target start": str(pd.to_datetime(np.nanmedian(target))) if np.isfinite(target).any()
                     else None})
    out["folds"] = rows
    return out


def make_folds(args, own):
    group = args.seasonal_patterns
    train, test, ids, info = load_group(args.root_path, group)
    H = HORIZON[group]
    K = own.n_folds or DEFAULT_FOLDS[group]
    spec = {"group": group, "cv": own.cv, "H": H, "seq_len": 2 * H, "freq": FREQ[group], "n_folds": K,
            "gap": int(round(own.gap_h * H)), "gap_h": own.gap_h, "min_train": int(round(own.min_train_h * H)),
            "series": len(train), "history_size": HISTORY[group]}
    L = np.array([len(s) for s in train])
    if own.cv == "calendar":
        starts = parse_dates(info, group)
        step = STEP[group]
        dated = starts.notna().values
        # last possible validation start of each series = the date of its value L - H. Cut-off C_k = the q_k
        # quantile of these dates: a share 1 - q_k of the series has a full validation block after C_k, and every
        # series whose forecast starts at or after C_k (at least that share) is scored leak-free on the M4 test.
        last_val = pd.Series([date_at(s, step, n - H) if d else pd.NaT for s, n, d in zip(starts, L, dated)])
        dated &= last_val.notna().values
        qs = [float(q) for q in own.quantiles.split(",")] if own.quantiles else \
            ([0.10, 0.30, 0.50] if K == 3 else list(np.round(np.linspace(0.10, 0.50, K), 3)))
        assert len(qs) == K, "one quantile per fold"
        ordinal = np.array([t.value for t in last_val[dated]], dtype=np.int64)
        cutoffs = [pd.Timestamp(int(np.quantile(ordinal, q, method="lower"))) for q in qs]
        if len(set(cutoffs)) < len(cutoffs) and not own.quantiles:
            # many M4 series share the same dates: take K distinct dates from a finer grid of quantiles instead
            grid = np.round(np.linspace(0.05, 0.70, 131), 4)
            cand = {}
            for q in grid:
                cand.setdefault(int(np.quantile(ordinal, q, method="lower")), q)
            if len(cand) >= K:
                keys = sorted(cand)
                pick = [keys[int(round(i))] for i in np.linspace(0, len(keys) - 1, K)]
                cutoffs, qs = [pd.Timestamp(v) for v in pick], [float(cand[v]) for v in pick]
                print(f"note: cut-offs coincided (shared M4 dates); using quantiles {qs} instead")
            else:
                print("note: some calendar cut-offs coincide (many series share the same dates) - those folds are "
                      "identical apart from the seed")
        rng = np.random.default_rng(own.ab_seed)
        perm = np.arange(len(L))
        di = np.where(dated)[0]
        perm[di] = rng.permutation(di)                      # placebo: start dates shuffled among dated series
        J, JP = [], []
        for C in cutoffs:
            jj = [steps_until(starts.iloc[i], C, group) if dated[i] else None for i in range(len(L))]
            jp = [steps_until(starts.iloc[perm[i]], C, group) if dated[i] else None for i in range(len(L))]
            J.append([int(v) if v is not None else -10 ** 9 for v in jj])
            JP.append([int(v) if v is not None else -10 ** 9 for v in jp])
        bad = np.array([v == -10 ** 9 for v in J[0]])
        dated &= ~bad
        ab = rng.integers(0, 2, len(L))
        spec["cal"] = {"quantiles": qs, "cutoffs": [str(c) for c in cutoffs], "dated": dated.astype(int).tolist(),
                       "j": J, "j_placebo": JP, "ab": ab.astype(int).tolist(), "ab_seed": own.ab_seed}
        spec["protocol"] = ("calendar-aware sliding CV: one calendar cut-off per fold for all series (quantiles of the "
                            "series' last possible validation start); training = values before cut-off - gap; validation = the first H values from the cut-off; a test series is "
                            "scored only by fold models whose cut-off is not later than its forecast start (no "
                            "cross-series leak)")
    else:
        spec["protocol"] = ("per-series sliding-window CV: K consecutive validation blocks of H values ending at the end "
                            "of each training series; purge gap before every block; test = M4 test set, untouched")
    folds = []
    for k in range(K):
        tr, t1, va, v0, el = fold_sets(L, spec, k)
        row = {"fold": k, "series": int(len(va)), "training_series": int(len(tr)),
               "share_of_group": float(len(va) / len(L)), "median_train_length": int(np.median(t1)) if len(t1) else 0,
               "test_series_scored": int(el.sum())}
        if own.cv == "calendar":
            row["cutoff"] = spec["cal"]["cutoffs"][k]
            row["validation_block"] = f"first {H} values from {spec['cal']['cutoffs'][k][:10]}"
            assert np.all(v0 + H <= L[va]) and np.all(v0 - spec["gap"] >= spec["min_train"])
        else:
            row["validation_block"] = (f"values L-{(K - k) * H} .. L-{(K - k - 1) * H - 1}" if k < K - 1
                                       else f"values L-{H} .. L-1")
            assert np.all(t1 + spec["gap"] == v0) and np.all(v0 + H <= L[va])
        folds.append(row)
    spec["folds"] = folds
    spec["calendar"] = calendar_report(train, info, spec, group, np.random.default_rng(0))
    spec["length_quantiles"] = {str(q): int(np.quantile(L, q)) for q in (0.0, 0.25, 0.5, 0.75, 1.0)}
    return spec


def window(ts, cut, seq_len, label_len, pred_len):
    """One training example with the notebook's preprocessing fix (Section 2c): input = the seq_len values
    before `cut`, padded with the first observed value; target placed so position label_len is the cut."""
    insample = np.zeros((seq_len, 1))
    insample_mask = np.zeros((seq_len, 1))
    outsample = np.zeros((pred_len + label_len, 1))
    outsample_mask = np.zeros((pred_len + label_len, 1))
    w = ts[max(0, cut - seq_len):cut]
    insample[:, 0] = w[0]
    insample[-len(w):, 0] = w
    insample_mask[-len(w):, 0] = 1.0
    start = max(0, cut - label_len)
    o = ts[start:min(len(ts), cut + pred_len)]
    off = label_len - (cut - start)
    outsample[off:off + len(o), 0] = o
    outsample_mask[off:off + len(o), 0] = 1.0
    return insample, outsample, insample_mask, outsample_mask


def last_window(series, seq_len):
    """Inputs [n, seq_len] = the last seq_len values of each series (first-value padding), and masks."""
    x = np.zeros((len(series), seq_len))
    m = np.zeros((len(series), seq_len))
    for i, ts in enumerate(series):
        w = ts[-seq_len:]
        x[i, :] = w[0]
        x[i, -len(w):] = w
        m[i, -len(w):] = 1.0
    return x, m


class FoldM4:
    """Datasets of one fold (and variant), in TSLib's interface (Dataset_M4)."""

    def __init__(self, train, spec, k, variant="clean"):
        tr, t1, va, v0, el = fold_sets([len(s) for s in train], spec, k, variant)
        self.train_idx, self.t1, self.idx, self.v0, self.test_eligible = tr, t1, va, v0, el
        self.train = [train[i][:t] for i, t in zip(tr, t1)]
        self.val_in = [train[i][:v] for i, v in zip(va, v0)]
        self.val_out = [train[i][v:v + spec["H"]] for i, v in zip(va, v0)]


def make_dataset_class(fold, args):
    from torch.utils.data import Dataset
    limit = int(HISTORY[args.seasonal_patterns] * args.pred_len)

    class Dataset_M4_CV(Dataset):
        def __init__(self, args, root_path, flag="train", size=None, **kw):
            self.seq_len, self.label_len, self.pred_len = size
            self.flag = flag
            self.window_sampling_limit = limit
            if flag == "train":
                self.timeseries = fold.train
            else:                                   # 'val' (and never the M4 test set)
                self.timeseries = fold.val_out
                self.inputs = fold.val_in
            self.ids = fold.idx if flag != "train" else fold.train_idx

        def __len__(self):
            return len(self.timeseries)

        def __getitem__(self, index):                # same random cut point as TSLib
            ts = self.timeseries[index]
            cut = np.random.randint(low=max(1, len(ts) - self.window_sampling_limit), high=len(ts), size=1)[0]
            return window(ts, cut, self.seq_len, self.label_len, self.pred_len)

        def last_insample_window(self):
            return last_window(self.inputs if self.flag != "train" else self.timeseries, self.seq_len)

    return Dataset_M4_CV


# ---------------------------------------------------------------------------------------------------------
# Metrics (M4 definitions) and Naive2
# ---------------------------------------------------------------------------------------------------------
def smape_series(y, f):
    d = np.abs(y) + np.abs(f)
    return 200 * np.mean(np.where(d > 0, np.abs(y - f) / np.where(d > 0, d, 1), 0.0), axis=-1)


def mase_scale(x, m):
    if len(x) <= m:
        return np.nan
    return float(np.mean(np.abs(x[m:] - x[:-m])))


def _acf(x, k):
    m = x.mean()
    return np.sum((x[:-k] - m) * (x[k:] - m)) / np.sum((x - m) ** 2)


def naive2(x, h, m):
    """M4's Naive2 benchmark (seasonal naive on the classical multiplicative decomposition, 90% test)."""
    x = np.asarray(x, float)
    si = np.full(m, 100.0)
    if m > 1 and len(x) > m:
        s = _acf(x, 1)
        for i in range(2, m):
            s += _acf(x, i) ** 2
        limit = 1.645 * np.sqrt((1 + 2 * s) / len(x))
        if abs(_acf(x, m)) > limit:
            ts = pd.Series(x)
            if len(x) % 2 == 0:
                ma = ts.rolling(m, center=True).mean().rolling(2, center=True).mean()
                ma = np.roll(ma.values, -1)
            else:
                ma = ts.rolling(m, center=True).mean().values
            le = x * 100 / ma
            le = np.hstack((le, np.full((m - (len(le) % m)) % m, np.nan)))
            si = np.nanmean(le.reshape(-1, m), 0)
            si = si / (np.sum(si) / (m * 100))
    full = np.tile(si, (len(x) + h) // m + 1)[:len(x) + h]
    des = x * 100 / full[:len(x)]
    return des[-1] * full[len(x):] / 100


def naive2_forecasts(root, group, train, ids, h, m):
    path = os.path.join(root, "submission-Naive2.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df.set_index(df.columns[0])
        if np.isin(ids, df.index).all():
            return df.loc[ids].values[:, :h].astype(float), "submission-Naive2.csv"
        print("note: submission-Naive2.csv ids do not match - Naive2 computed with the M4 benchmark code")
    return np.stack([naive2(x, h, m) for x in train]), "computed (M4 benchmark code)"


# ---------------------------------------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------------------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def model_module(model):
    return sys.modules[type(model).__module__]


def kanb_parameters(model):
    f = getattr(model_module(model), "kanb_parameters", None)
    return f(model) if f else None


@torch.no_grad()
def forecast(model, args, x, device, batch_size=1):
    """Forecasts [n, H] from inputs x [n, seq_len] (TSLib's decoder input), batch 1 by default."""
    model.eval()
    out = np.zeros((len(x), args.pred_len))
    for i in range(0, len(x), batch_size):
        xb = torch.tensor(x[i:i + batch_size], dtype=torch.float32, device=device).unsqueeze(-1)
        dec = torch.zeros((len(xb), args.pred_len, 1), device=device)
        dec = torch.cat([xb[:, -args.label_len:, :], dec], dim=1)
        out[i:i + batch_size] = model(xb, None, dec, None)[:, -args.pred_len:, 0].cpu().numpy()
    return out


def main():
    own, args = parse_all_args()
    if own.make_folds:
        spec = make_folds(args, own)
        os.makedirs(os.path.dirname(os.path.abspath(own.folds_file)), exist_ok=True)
        with open(own.folds_file, "w") as fh:
            json.dump(spec, fh, indent=1)
        print("FOLDS", json.dumps({k: spec[k] for k in ("group", "H", "n_folds", "gap", "series")}),
              "| series per fold:", [f["series"] for f in spec["folds"]])
        return

    spec = json.load(open(own.folds_file))
    group = args.seasonal_patterns
    assert spec["group"] == group, "folds file of another group"
    args.use_gpu = bool(torch.cuda.is_available() and args.use_gpu)
    args.device = torch.device("cuda:0" if args.use_gpu else "cpu")
    set_seed(own.seed)
    train, test, ids, info = load_group(args.root_path, group)
    fold = FoldM4(train, spec, own.fold, own.variant)
    tmp = tempfile.mkdtemp(prefix="cvm4_")
    args.checkpoints = tmp

    import data_provider.data_factory as DF
    import exp.exp_short_term_forecasting as EST
    import utils.tools as T
    DF.data_dict["m4"] = make_dataset_class(fold, args)

    def vali(self, train_loader, vali_loader, criterion):
        """TSLib's vali(), with the fold's own validation inputs (the values just before each block)."""
        x, _ = vali_loader.dataset.last_insample_window()
        y = vali_loader.dataset.timeseries
        x = torch.tensor(x, dtype=torch.float32).to(self.device).unsqueeze(-1)
        self.model.eval()
        with torch.no_grad():
            B, _, C = x.shape
            dec = torch.zeros((B, self.args.pred_len, C)).float().to(self.device)
            dec = torch.cat([x[:, -self.args.label_len:, :], dec], dim=1).float()
            out = torch.zeros((B, self.args.pred_len, C)).float()
            ids_ = np.append(np.arange(0, B, 500), B)
            for i in range(len(ids_) - 1):
                out[ids_[i]:ids_[i + 1]] = self.model(x[ids_[i]:ids_[i + 1]], None, dec[ids_[i]:ids_[i + 1]],
                                                      None).detach().cpu()[:, -self.args.pred_len:, :]
            true = torch.from_numpy(np.array(y))
            loss = criterion(x.detach().cpu()[:, :, 0], self.args.frequency_map, out[:, :, 0], true,
                             torch.ones(true.shape))
        self.model.train()
        return loss

    EST.Exp_Short_Term_Forecast.vali = vali
    history = {"vali": []}
    original_call = T.EarlyStopping.__call__

    def recording_call(self, val_loss, model, path):
        history["vali"].append(float(val_loss))
        return original_call(self, val_loss, model, path)

    EST.EarlyStopping.__call__ = recording_call
    original_opt = EST.Exp_Short_Term_Forecast._select_optimizer

    def select_optimizer(self):
        opt = original_opt(self)
        clip = getattr(model_module(self.model), "clip_kanb_", None)
        if clip is not None:
            opt.register_step_post_hook(lambda *_: clip(self.model))
        return opt

    EST.Exp_Short_Term_Forecast._select_optimizer = select_optimizer

    t0 = time.time()
    exp = EST.Exp_Short_Term_Forecast(args)
    n_params = sum(p.numel() for p in exp.model.parameters())
    kanb_start = kanb_parameters(exp.model)
    setting = f"m4_{group}_{args.model}_fold{own.fold}_seed{own.seed}_{own.variant}"
    exp.train(setting)
    train_min = (time.time() - t0) / 60
    model = exp.model

    H, m = args.pred_len, FREQ[group]
    # ---- validation blocks of this fold (early-stopping data; also the leak experiment's evaluation)
    x_val, _ = last_window(fold.val_in, args.seq_len)
    vpred = forecast(model, args, x_val, exp.device)
    vy = np.stack(fold.val_out)
    vscale = np.array([mase_scale(s, m) for s in fold.val_in])
    v_sm = smape_series(vy, vpred)
    v_ms = np.mean(np.abs(vy - vpred), axis=1) / vscale
    common = {"group": group, "model": args.model, "fold": own.fold, "seed": own.seed, "variant": own.variant, "H": H,
              "cv": spec.get("cv", "series"), "val_smape": float(np.mean(v_sm)), "val_mase": float(np.nanmean(v_ms)),
              "n_val_series": int(len(fold.idx)), "n_train_series": int(len(fold.train_idx)),
              "epochs": len(history["vali"]), "vali_losses": history["vali"],
              "best_vali": float(min(history["vali"])) if history["vali"] else None, "train_minutes": train_min,
              "n_params": int(n_params), "kanb_start": kanb_start, "kanb_end": kanb_parameters(model),
              "gap": spec["gap"], "tslib_args": {k: v for k, v in vars(args).items()
                                                 if isinstance(v, (int, float, str, bool, type(None)))}}
    if own.variant != "clean":
        # leak experiment / placebo: only the validation half is scored; nothing else is needed
        out_dir = os.path.join(own.out_dir, "leak")
        os.makedirs(out_dir, exist_ok=True)
        stem = os.path.join(out_dir, f"fold{own.fold}__seed{own.seed}__{own.variant}")
        np.savez_compressed(stem + ".npz", val_idx=fold.idx, val_smape=v_sm.astype(np.float32),
                            val_mase=v_ms.astype(np.float32), val_pred=vpred.astype(np.float32))
        with open(stem + ".json.part", "w") as fh:
            json.dump(common, fh, indent=1)
        os.replace(stem + ".json.part", stem + ".json")
        shutil.rmtree(tmp, ignore_errors=True)
        print("FOLD_RESULT", json.dumps({k: common[k] for k in ("model", "group", "fold", "seed", "variant", "val_smape",
                                                                 "epochs", "train_minutes")}))
        return
    # ---- test: every series of the group, from its full training history, one series at a time; a series counts
    #      only if this fold model is leak-free for it (calendar CV: its forecast starts at or after the cut-off)
    t1 = time.time()
    x_test, _ = last_window(train, args.seq_len)
    pred = forecast(model, args, x_test, exp.device)
    test_min = (time.time() - t1) / 60
    y = np.stack(test)
    el = fold.test_eligible
    scale = np.array([mase_scale(s, m) for s in train])
    sm = smape_series(y, pred)
    ms = np.mean(np.abs(y - pred), axis=1) / scale
    n2, n2_src = naive2_forecasts(args.root_path, group, train, ids, H, m)
    sm_n2, ms_n2 = smape_series(y, n2), np.mean(np.abs(y - n2), axis=1) / scale
    owa = 0.5 * (np.nanmean(sm[el]) / np.nanmean(sm_n2[el]) + np.nanmean(ms[el]) / np.nanmean(ms_n2[el]))
    res = dict(common, smape=float(np.mean(sm[el])), mase=float(np.nanmean(ms[el])), owa=float(owa),
               naive2_smape=float(np.mean(sm_n2[el])), naive2_mase=float(np.nanmean(ms_n2[el])),
               naive2_source=n2_src, n_test_series=int(len(train)), n_test_scored=int(el.sum()),
               test_minutes=test_min, test_batch_size=1)
    os.makedirs(own.out_dir, exist_ok=True)
    stem = os.path.join(own.out_dir, f"fold{own.fold}__seed{own.seed}")
    np.savez_compressed(stem + "_test.npz", pred=pred.astype(np.float32), smape=sm.astype(np.float32),
                        mase=ms.astype(np.float32), scale=scale.astype(np.float32), test_eligible=el,
                        err_scaled=((pred - y) / scale[:, None]).mean(1).astype(np.float32),
                        val_idx=fold.idx, val_pred=vpred.astype(np.float32), val_smape=v_sm.astype(np.float32),
                        val_mase=v_ms.astype(np.float32),
                        val_err_scaled=((vpred - vy) / vscale[:, None]).mean(1).astype(np.float32),
                        naive2_smape=sm_n2.astype(np.float32), naive2_mase=ms_n2.astype(np.float32))
    if own.save_checkpoint:
        shutil.copy2(os.path.join(tmp, setting, "checkpoint.pth"), stem + ".pth")
    with open(stem + ".json.part", "w") as fh:
        json.dump(res, fh, indent=1)
    os.replace(stem + ".json.part", stem + ".json")
    shutil.rmtree(tmp, ignore_errors=True)
    print("FOLD_RESULT", json.dumps({k: res[k] for k in ("model", "group", "fold", "seed", "smape", "mase", "owa",
                                                          "n_test_scored", "epochs", "train_minutes")}))


if __name__ == "__main__":
    main()

Writing cv_m4.py


In [ ]:
%%writefile xai_m4.py
"""
Section 8 of this notebook (M4): explainability, drift and TimesNet (GELU) vs TimesNet-KANB, for the fold
models of cv_m4.py. No training happens here.

M4 has three properties that shape every analysis:
* one variable per series -> the features are TIME SEGMENTS of the input (lags), not variables;
* ~100,000 short, separate series of very different scales -> everything is expressed in each series'
  own normalized units (value - input mean) / input std, exactly the units TimesNet works in internally,
  and replacements for SHAP are real training windows of OTHER series, rescaled to the series' level/scale;
* the unit of comparison is the SERIES (independent) -> paired Wilcoxon signed-rank tests with bootstrap
  confidence intervals and Holm's correction (Diebold-Mariano needs a time order, which series do not have).

Analyses (--analyses, comma separated or "all"):
  periods   which periods TimesNet's FFT picks per series vs the group's seasonality, and over calendar time
  shap      permutation SHAP on time segments of the input (exact additivity), global, over calendar time,
            beeswarm, local waterfall + decision plots
  gradshap  GradientSHAP per lag (fine grained), seasonal lags marked
  gradshap_local  GradientSHAP of single series: one signed bar per input step (red raised, blue lowered)
  gradcam   Grad-CAM on TimesNet's 2D period images (rows = cycles, columns = position in the period),
            folded back to time, future part shaded; global profile and single series
  faith     faithfulness (deletion with real replacements) + stability across seeds, folds and models
  ablation  segment ablation: one input segment replaced by the same segment of another real series
  drift     per-series drift (conformal p-values against the series' own history, Benjamini-Hochberg),
            population covariate and concept drift (MMD, energy distance, domain classifier with SHAP),
            novelty, residual-based concept drift, drift terciles with Wilcoxon + bootstrap + Holm, calendar leak
  gallery   worst series: input, truth and both models' forecasts
"""
import argparse
import glob
import importlib
import json
import os
import time
from argparse import Namespace

import numpy as np
import pandas as pd
import torch

import cv_m4 as CVM

COLORS = {"TimesNet": "#2B6CB0", "TimesNet_KANB": "#C05621"}
LIGHT = {"TimesNet": "rgba(43,108,176,0.18)", "TimesNet_KANB": "rgba(192,86,33,0.18)"}
LABEL = {"TimesNet": "TimesNet (GELU)", "TimesNet_KANB": "TimesNet-KANB (ours)"}
PAL = ["#2B6CB0", "#DD6B20", "#2F855A", "#6B46C1", "#B7791F", "#C53030", "#319795", "#D53F8C", "#4A5568",
       "#975A16", "#2C5282", "#9C4221"]
label = lambda m: LABEL.get(m, m)
color = lambda m: COLORS.get(m, "#4A5568")
FIGURES = []


def save(fig, out_dir, name, width=1200, height=650):
    os.makedirs(out_dir, exist_ok=True)
    fig.update_layout(template="plotly_white", font=dict(family="Arial", size=12), title_font=dict(size=16),
                      width=width, height=height)
    fig.write_html(os.path.join(out_dir, name + ".html"), include_plotlyjs="directory")
    fig.write_json(os.path.join(out_dir, name + ".json"))
    try:
        fig.write_image(os.path.join(out_dir, name + ".png"), scale=2)
    except Exception as e:
        print(f"PNG export failed for {name}: {e}")
    FIGURES.append(name)


def title(main, sub):
    return f"<b>{main}</b><br><sup>{sub}</sup>"


# =========================================================================================================
# Statistics
# =========================================================================================================
def bh(p):
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(pv)
    out[order] = np.minimum(ranked, 1)
    q[ok] = out
    return q


def holm(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = np.where(np.isfinite(p))[0]
    order = ok[np.argsort(p[ok])]
    run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (len(order) - r) * p[i]))
        out[i] = run
    return out


def paired_test(d, rng, n_boot=2000):
    """Paired comparison on per-series differences d (KANB - GELU): Wilcoxon signed-rank p, mean and median
    difference with bootstrap 95% CIs (over series), share of series where KANB is better."""
    from scipy.stats import wilcoxon
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    n = len(d)
    if n < 6:
        return {"n": n, "mean": float(np.mean(d)) if n else np.nan, "p": np.nan}
    p = wilcoxon(d, zero_method="zsplit").pvalue if np.any(d != 0) else 1.0
    idx = rng.integers(0, n, size=(n_boot, n))
    bm = d[idx].mean(1)
    bmed = np.median(d[idx], axis=1)
    return {"n": n, "mean": float(d.mean()), "mean_lo": float(np.quantile(bm, 0.025)),
            "mean_hi": float(np.quantile(bm, 0.975)), "median": float(np.median(d)),
            "median_lo": float(np.quantile(bmed, 0.025)), "median_hi": float(np.quantile(bmed, 0.975)),
            "kanb_better_share": float((d < 0).mean()), "p": float(p)}


def median_gamma(ref, rng, max_n=600):
    from scipy.spatial.distance import pdist
    r = ref[rng.choice(len(ref), min(len(ref), max_n), replace=False)]
    d = pdist(r, "sqeuclidean")
    med = np.median(d[d > 0]) if (d > 0).any() else 1.0
    return 1.0 / med


def mmd2(A, B, gamma):
    from scipy.spatial.distance import cdist
    Kaa = np.exp(-gamma * cdist(A, A, "sqeuclidean"))
    Kbb = np.exp(-gamma * cdist(B, B, "sqeuclidean"))
    Kab = np.exp(-gamma * cdist(A, B, "sqeuclidean"))
    m, n = len(A), len(B)
    return float((Kaa.sum() - np.trace(Kaa)) / (m * (m - 1)) + (Kbb.sum() - np.trace(Kbb)) / (n * (n - 1))
                 - 2 * Kab.mean())


def energy(A, B):
    from scipy.spatial.distance import cdist
    A = A.reshape(len(A), -1)
    B = B.reshape(len(B), -1)
    return float(2 * cdist(A, B).mean() - cdist(A, A).mean() - cdist(B, B).mean())


def energy_1d(a, b):
    """Energy distance between two samples of numbers: 2E|A-B| - E|A-A'| - E|B-B'|."""
    a, b = np.asarray(a, float).ravel(), np.asarray(b, float).ravel()
    return float(2 * np.abs(a[:, None] - b[None, :]).mean() - np.abs(a[:, None] - a[None, :]).mean()
                 - np.abs(b[:, None] - b[None, :]).mean())


def lowess_line(x, y, frac=0.5):
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        r = lowess(y, x, frac=frac)
        return r[:, 0], r[:, 1]
    except Exception:
        o = np.argsort(x)
        k = max(3, len(x) // 5)
        return np.asarray(x)[o], pd.Series(np.asarray(y)[o]).rolling(k, center=True, min_periods=1).mean().values


# =========================================================================================================
# Runs, models and TimesNet's period selection
# =========================================================================================================
def find_runs(run_root, model, folds=None, seeds=None):
    out = []
    for js in sorted(glob.glob(os.path.join(run_root, model, "fold*__seed*.json"))):
        stem = js[:-5]
        res = json.load(open(js))
        if folds is not None and res["fold"] not in folds:
            continue
        if seeds is not None and res["seed"] not in seeds:
            continue
        if os.path.exists(stem + ".pth") and os.path.exists(stem + "_test.npz"):
            out.append({"model": model, "fold": res["fold"], "seed": res["seed"], "res": res,
                        "pth": stem + ".pth", "npz": stem + "_test.npz"})
    return out


PER = {"mode": "normal", "rec": [], "i": 0, "log": None}


def fft_for_period(x, k=2):
    xf = torch.fft.rfft(x, dim=1)
    amp = abs(xf).mean(-1)
    if PER["mode"] == "force":
        top = PER["rec"][PER["i"]]
        PER["i"] += 1
    else:                                                       # TSLib's own rule, unchanged
        fl = abs(xf).mean(0).mean(-1)
        fl[0] = 0
        _, top = torch.topk(fl, k)
        top = top.detach().cpu().numpy()
        if PER["mode"] == "record":
            PER["rec"].append(top)
    period = x.shape[1] // top
    if PER["log"] is not None:
        w = torch.softmax(amp[:, top], dim=1)[0].detach().cpu().numpy()
        PER["log"].append((period.tolist(), w.tolist()))
    return period, amp[:, top]


def forecast_full_gradient(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
    """TSLib's forecast() without .detach() on the window mean (same values; full gradient for SHAP)."""
    means = x_enc.mean(1, keepdim=True)
    x_enc = x_enc.sub(means)
    stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
    x_enc = x_enc.div(stdev)
    enc_out = self.enc_embedding(x_enc, x_mark_enc)
    enc_out = self.predict_linear(enc_out.permute(0, 2, 1)).permute(0, 2, 1)
    for i in range(self.layer):
        enc_out = self.layer_norm(self.model[i](enc_out))
    dec_out = self.projection(enc_out)
    dec_out = dec_out.mul(stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len + self.seq_len, 1))
    dec_out = dec_out.add(means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len + self.seq_len, 1))
    return dec_out


def patch_module(mod):
    if getattr(mod, "_xaim4", False):
        return
    mod.FFT_for_Period = fft_for_period
    mod._original_forecast = mod.Model.forecast
    mod._xaim4 = True


class Runner:
    """One fold model and forward passes in each series' own normalized units."""

    def __init__(self, run, D, device):
        self.run, self.D, self.device = run, D, device
        args = Namespace(**run["res"]["tslib_args"])
        args.use_gpu = torch.cuda.is_available()
        self.args = args
        self.mod = importlib.import_module(f"models.{args.model}")
        patch_module(self.mod)
        self.model = self.mod.Model(args).float().to(device)
        self.model.load_state_dict(torch.load(run["pth"], map_location=device, weights_only=True))
        self.model.eval()
        self.L, self.H = args.seq_len, args.pred_len
        self.fold = CVM.FoldM4(D["train"], D["spec"], run["fold"])

    def forward(self, xs, mode="normal", rec=None, grad=False, chunk=512):
        """Forecasts [B, H] (original units) for inputs xs [B, L] (numpy or tensor)."""
        self.mod.Model.forecast = forecast_full_gradient if grad else self.mod._original_forecast
        outs = []
        for i in range(0, len(xs), chunk):
            xb = xs[i:i + chunk]
            xt = xb if torch.is_tensor(xb) else torch.tensor(np.asarray(xb), dtype=torch.float32, device=self.device)
            xt = xt.unsqueeze(-1)
            dec = torch.zeros((len(xt), self.L // 2 + self.H, 1), device=self.device)
            PER.update(mode=mode, rec=rec if rec is not None else [], i=0)
            with torch.set_grad_enabled(grad):
                o = self.model(xt, None, dec, None)[:, -self.H:, 0]
            if mode == "record":
                self.last_rec = list(PER["rec"])
            PER.update(mode="normal", rec=[], i=0)
            outs.append(o)
        self.mod.Model.forecast = self.mod._original_forecast
        return torch.cat(outs)

    def own_periods(self, x):
        with torch.no_grad():
            self.forward(x[None], mode="record")
        return self.last_rec

    def train_windows(self, rng, n):
        """Real training windows (input L) of this fold's training data, z-normalized: [n, L]."""
        out = []
        series = self.fold.train
        lim = int(CVM.HISTORY[self.D["group"]] * self.H)
        while len(out) < n:
            ts = series[rng.integers(len(series))]
            if len(ts) < self.L + 1:
                continue
            c = rng.integers(max(self.L, len(ts) - lim), len(ts) + 1)
            w = ts[c - self.L:c]
            sd = w.std()
            if sd <= 1e-8:
                continue
            out.append((w - w.mean()) / sd)
        return np.stack(out)


def norm_stats(x):
    mu = x.mean(-1, keepdims=True)
    sd = np.sqrt(((x - mu) ** 2).mean(-1, keepdims=True) + 1e-5)
    return mu, sd


def outputs_of(pred_n, H):
    """Explained outputs from normalized forecasts [B, H]: mean over the horizon, and steps 1, H/2, H."""
    steps = sorted({0, H // 2 - 1, H - 1})
    return torch.cat([pred_n.mean(1, keepdim=True), pred_n[:, steps]], dim=1)


def output_names(H):
    return ["mean forecast"] + [f"t+{h}" for h in sorted({1, H // 2, H})]


def segments(L, n_seg):
    n_seg = min(n_seg, L)
    edges = np.linspace(0, L, n_seg + 1).round().astype(int)
    names = [f"lag {L - edges[j + 1] + 1}-{L - edges[j]}" for j in range(n_seg)]
    masks = []
    for j in range(n_seg):
        m = np.zeros(L, bool)
        m[edges[j]:edges[j + 1]] = True
        masks.append(m)
    return names, masks, edges


# =========================================================================================================
# 0. Periods
# =========================================================================================================
def analysis_periods(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    g, L, H, m_ = D["group"], D["L"], D["H"], D["freq"]
    sample = ctx["period_sample"]
    rows = []
    for m, runners in R_by_model.items():
        for R in runners:
            for i in sample:
                PER["log"] = []
                with torch.no_grad():
                    R.forward(D["x_test"][i][None])
                for layer, (periods, weights) in enumerate(PER["log"]):
                    for rank, (p, w) in enumerate(zip(periods, weights)):
                        rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "series": int(i),
                                     "layer": layer + 1, "rank": rank + 1, "period": int(p), "weight": float(w)})
                PER["log"] = None
    P = pd.DataFrame(rows)
    P.to_csv(os.path.join(out_dir, "periods.csv"), index=False)
    models = list(R_by_model)
    # the data's own cycles: mean autocorrelation of the (normalized) last 4H values of the series
    xs = [s[-min(len(s), 4 * L):] for s in D["train"]]
    max_lag = min(2 * L, min(len(x) for x in xs) - 2)
    xs = [(x - x.mean()) / (x.std() + 1e-9) for x in xs[:3000]]
    acf = [1.0]
    for k in range(1, max(2, max_lag) + 1):
        acf.append(float(np.mean([np.corrcoef(x[:-k], x[k:])[0, 1] for x in xs if len(x) > k + 2 and x[k:].std() > 0
                                  and x[:-k].std() > 0])))
    acf = np.array(acf)
    peaks = [k for k in range(2, len(acf) - 1) if acf[k] > acf[k - 1] and acf[k] >= acf[k + 1] and acf[k] > 0.1]
    top = P.groupby("period").size().sort_values(ascending=False).index[:12]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.1,
                        subplot_titles=["share of series choosing each period (any rank, any layer)",
                                        "data's own cycles: mean autocorrelation of the series"])
    for m in models:
        d = P[P.model == m].drop_duplicates(["fold", "seed", "series", "layer", "period"])   # 3H//k can repeat
        n_w = d.groupby(["fold", "seed", "series", "layer"]).ngroups
        share = d.groupby("period").size().reindex(top).fillna(0) / max(n_w, 1)
        fig.add_trace(go.Bar(x=[str(p) for p in top], y=share.values, name=label(m), marker_color=color(m)), 1, 1)
    fig.add_trace(go.Scatter(x=list(range(len(acf))), y=acf, mode="lines", name="autocorrelation",
                             line=dict(color="#4A5568")), 1, 2)
    if m_ > 1:
        for k in range(m_, len(acf), m_):
            fig.add_vline(x=k, line=dict(color="#C05621", dash="dot"), row=1, col=2)
    for k in peaks[:6]:
        fig.add_annotation(x=k, y=acf[k], text=str(k), showarrow=False, yshift=10, row=1, col=2)
    fig.update_xaxes(title_text="period (time steps)", type="category", row=1, col=1)
    fig.update_xaxes(title_text="lag (time steps)", row=1, col=2)
    fig.update_yaxes(title_text="share of series", row=1, col=1)
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.2), margin=dict(t=120), title=title(
        f"M4 {g} · Which cycles does TimesNet use?",
        f"left: the periods TimesNet's FFT picks for {len(sample)} test series (top-k, both TimesBlocks; fold models and "
        f"seeds pooled). Right: the series' own autocorrelation" + (f"; dotted = multiples of the seasonal period {m_}"
                                                                    if m_ > 1 else "") +
        "<br>Note: TimesNet's FFT runs on 2H + H = 3H steps, so it can only pick periods 3H/k"))
    save(fig, out_dir, "01_periods_chosen", 1400, 560)
    # (b) over calendar time
    yr = ctx["test_year_bin"]
    fig = make_subplots(rows=1, cols=len(models), shared_yaxes=True, subplot_titles=[label(m) for m in models])
    for j, m in enumerate(models, 1):
        d = P[(P.model == m) & (P.layer == 1)].copy()
        d["when"] = yr[d.series.values]
        piv = d.groupby(["when", "period"]).size().unstack(fill_value=0)
        piv = piv.div(piv.sum(1), axis=0).reindex(columns=list(top), fill_value=0.0)
        fig.add_trace(go.Heatmap(z=piv.values.T, x=[str(w) for w in piv.index], y=[str(p) for p in piv.columns],
                                 colorscale="Blues", zmin=0, zmax=1, showscale=(j == 1), colorbar=dict(title="share")),
                      1, j)
    fig.update_yaxes(type="category")
    fig.update_yaxes(title_text="period", row=1, col=1)
    fig.update_xaxes(title_text="calendar period of the forecast (M4-info dates)")
    fig.update_layout(margin=dict(t=110), title=title(
        f"M4 {g} · Period choice over calendar time (first TimesBlock)",
        "share of series (fold models and seeds pooled) whose forecast starts in each calendar period that pick each "
        "period. M4 series are not aligned in time; their starting dates place them on one calendar"))
    save(fig, out_dir, "02_periods_over_time", 1400, 520)
    out = {}
    for m in models:
        d = P[(P.model == m) & (P["rank"] == 1)]
        if len(d):
            t = d.period.value_counts(normalize=True)
            out[m] = {"most_chosen_period": int(t.index[0]), "share": float(t.iloc[0]),
                      "seasonal_period_chosen_share": float((P[(P.model == m)].groupby(["fold", "seed", "series"])
                                                             .period.apply(lambda s: (s % m_ == 0).any() if m_ > 1
                                                                           else False)).mean())}
    out["acf_peaks"] = peaks[:6]
    return out


# =========================================================================================================
# 1. SHAP
# =========================================================================================================
def perm_shap_series(R, x, masks, bank, samples, rng, H):
    """Permutation SHAP (antithetic orders) of one series; replacements = real training windows of other
    series rescaled to this series' input mean / std. Outputs in this series' normalized units."""
    mu, sd = norm_stats(x)
    rec = R.own_periods(x)
    F = len(masks)
    inputs, orders = [], []
    for _ in range(max(1, samples // 2)):
        b = bank[rng.integers(len(bank))] * sd + mu
        o = rng.permutation(F)
        for order in (o, o[::-1]):
            cur = b.copy()
            seq = [cur.copy()]
            for j in order:
                cur = np.where(masks[j], x, cur)
                seq.append(cur.copy())
            inputs.extend(seq)
            orders.append(order)
    with torch.no_grad():
        pred = R.forward(np.stack(inputs), mode="force", rec=rec).cpu().numpy()
    out = outputs_of(torch.tensor((pred - mu) / sd), H).numpy().reshape(len(orders), F + 1, -1)
    phi = np.zeros((F, out.shape[-1]))
    for k, order in enumerate(orders):
        phi[order] += out[k, 1:] - out[k, :-1]
    phi /= len(orders)
    return phi, out[0, -1], out[:, 0].mean(0)


def analysis_shap(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from scipy.stats import spearmanr
    g, L, H = D["group"], D["L"], D["H"]
    names, masks, edges = segments(L, opt.n_segments)
    F = len(names)
    onames = output_names(H)
    sample = ctx["shap_sample"]
    rng0 = np.random.default_rng(opt.seed)
    store, checks = {}, []
    for m, runners in R_by_model.items():
        for R in runners:
            rng = np.random.default_rng(rng0.integers(1 << 31))
            bank = R.train_windows(rng, opt.background)
            phis, fulls, bases = [], [], []
            for i in sample:
                phi, full, base = perm_shap_series(R, D["x_test"][i], masks, bank, opt.shap_samples, rng, H)
                phis.append(phi); fulls.append(full); bases.append(base)
                checks.append(float(np.abs(phi.sum(0) - (full - base)).max()))
            store[(m, R.run["fold"], R.run["seed"])] = {"phi": np.stack(phis), "full": np.stack(fulls),
                                                        "base": np.stack(bases)}
            print(f"  SHAP {label(m)} fold {R.run['fold']} seed {R.run['seed']}: {len(phis)} series", flush=True)
    np.savez_compressed(os.path.join(out_dir, "shap_values.npz"), series=np.array(sample),
                        **{f"{m}__fold{f}__seed{s}__{k}": v[k] for (m, f, s), v in store.items() for k in v},
                        feature_names=np.array(names), output_names=np.array(onames))
    models = list(R_by_model)
    rows = []
    for (m, f, s), v in store.items():
        imp = np.abs(v["phi"]).mean(axis=(0, 2))
        share = imp / imp.sum()
        for j in range(F):
            rows.append({"model": m, "fold": f, "seed": s, "segment": j, "lags": names[j], "importance": imp[j],
                         "share": share[j]})
    I = pd.DataFrame(rows)
    I.to_csv(os.path.join(out_dir, "shap_importance.csv"), index=False)
    # (a) global importance per segment, and per explained output
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                        subplot_titles=["share of importance per input segment", "importance per explained output"])
    for m in models:
        ps = I[I.model == m].pivot_table(index=["fold", "seed"], columns="segment", values="share")
        fig.add_trace(go.Bar(x=names, y=ps.mean().values, error_y=dict(type="data", array=ps.std().fillna(0).values),
                             name=label(m), marker_color=color(m), legendgroup=m), 1, 1)
        po = np.mean([np.abs(v["phi"]).sum(1).mean(0) for (mm, f, s), v in store.items() if mm == m], axis=0)
        fig.add_trace(go.Bar(x=onames, y=po, name=label(m), marker_color=color(m), legendgroup=m, showlegend=False), 1, 2)
    fig.update_yaxes(tickformat=".0%", row=1, col=1)
    fig.update_yaxes(title_text="total |SHAP| (normalised units)", row=1, col=2)
    fig.update_xaxes(title_text="input segment (lag 1 = most recent value)", row=1, col=1)
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.22), margin=dict(t=120), title=title(
        f"M4 {g} · Global importance of the past (permutation SHAP)",
        f"{len(sample)} test series; mean ± sd over the {I.groupby(['fold', 'seed']).ngroups} fold models and seeds of "
        f"each model. Features = {F} segments of the {L}-step input; replacements = real training windows of other "
        "series rescaled to the series' level and scale; units = the series' own normalised units"))
    save(fig, out_dir, "03_shap_global_importance", 1400, 560)
    # (b) segment x output heatmaps + difference
    nm = len(models)
    fig = make_subplots(rows=1, cols=nm + (1 if nm == 2 else 0), horizontal_spacing=0.07,
                        subplot_titles=[label(m) for m in models] + (["KANB − GELU"] if nm == 2 else []))
    mats = {}
    for j, m in enumerate(models, 1):
        mat = np.mean([np.abs(v["phi"]).mean(0) for (mm, f, s), v in store.items() if mm == m], axis=0)  # [F, out]
        mat = mat / mat.sum(0, keepdims=True)
        mats[m] = mat
        fig.add_trace(go.Heatmap(z=100 * mat, x=onames, y=names, colorscale="Viridis", showscale=(j == 1),
                                 colorbar=dict(title="% of<br>output", x=-0.1),
                                 hovertemplate="%{y} → %{x}<br>%{z:.1f}%<extra></extra>"), 1, j)
    if nm == 2:
        dm = 100 * (mats[models[1]] - mats[models[0]])
        zmax = max(1e-9, float(np.abs(dm).max()))
        fig.add_trace(go.Heatmap(z=dm, x=onames, y=names, colorscale="RdBu_r", zmin=-zmax, zmax=zmax,
                                 colorbar=dict(title="points"), hovertemplate="%{y} → %{x}<br>%{z:+.1f}<extra></extra>"),
                      1, 3)
    fig.update_layout(margin=dict(t=120), title=title(
        f"M4 {g} · Which part of the past drives which part of the forecast?",
        "share of each explained output's importance per input segment. Right: KANB minus GELU (red = KANB relies more "
        "on that segment)"))
    save(fig, out_dir, "04_shap_segment_output", 1500, 520)
    # (c) over calendar time + stability
    when = ctx["test_year_bin"][sample]
    bins = sorted(set(when))
    fig = make_subplots(rows=nm, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=[label(m) for m in models])
    stab = []
    for r, m in enumerate(models, 1):
        per = np.mean([np.abs(v["phi"]).mean(2) for (mm, f, s), v in store.items() if mm == m], axis=0)   # [n, F]
        share = per / per.sum(1, keepdims=True)
        for j in range(F):
            y = [share[when == b, j].mean() for b in bins]
            fig.add_trace(go.Scatter(x=[str(b) for b in bins], y=y, mode="lines", stackgroup=f"g{r}", name=names[j],
                                     legendgroup=names[j], showlegend=(r == 1),
                                     line=dict(width=0.5, color=PAL[j % len(PAL)])), r, 1)
        sub = per[rng0.choice(len(per), min(len(per), 200), replace=False)]
        rho = [spearmanr(sub[a], sub[b]).statistic for a in range(len(sub)) for b in range(a + 1, len(sub))]
        cv = share.std(0) / np.maximum(share.mean(0), 1e-12)
        stab.append({"model": m, "median Spearman between series": float(np.nanmedian(rho)),
                     **{f"CV share {n}": float(c) for n, c in zip(names, cv)}})
    fig.update_yaxes(tickformat=".0%", title_text="share of importance")
    fig.update_xaxes(title_text="calendar period of the forecast")
    fig.update_layout(margin=dict(t=120), legend=dict(orientation="h", y=-0.12), title=title(
        f"M4 {g} · Is the same part of the past important over calendar time?",
        "share of importance of each input segment for series whose forecast starts in each calendar period (mean over "
        "fold models and seeds). Flat bands = stable explanations"))
    save(fig, out_dir, "05_shap_importance_over_time", 1400, 300 + 280 * nm)
    S = pd.DataFrame(stab).set_index("model")
    S.round(4).to_csv(os.path.join(out_dir, "shap_stability.csv"))
    # (d) beeswarm for the mean forecast
    fig = make_subplots(rows=1, cols=nm, shared_yaxes=True, horizontal_spacing=0.06, subplot_titles=[label(m) for m in models])
    xn = (D["x_test"][sample] - norm_stats(D["x_test"][sample])[0]) / norm_stats(D["x_test"][sample])[1]
    for j, m in enumerate(models, 1):
        keys = [k for k in store if k[0] == m]
        for f_ in range(F):
            xs = np.concatenate([store[k]["phi"][:, f_, 0] for k in keys])
            vals = np.tile(xn[:, masks[f_]].mean(1), len(keys))
            z = np.clip(vals, -2, 2)
            jit = np.random.default_rng(f_).uniform(-0.3, 0.3, len(xs))
            fig.add_trace(go.Scatter(x=xs, y=f_ + jit, mode="markers", showlegend=False,
                                     marker=dict(size=4, color=z, colorscale="RdBu_r", cmin=-2, cmax=2, opacity=0.6,
                                                 showscale=(j == 1 and f_ == 0),
                                                 colorbar=dict(title="segment<br>level (z)"))), 1, j)
        fig.add_vline(x=0, line=dict(color="#A0AEC0"), row=1, col=j)
        fig.update_xaxes(title_text="SHAP on the mean forecast (normalised units)", row=1, col=j)
    fig.update_yaxes(tickvals=list(range(F)), ticktext=names, col=1)
    fig.update_layout(margin=dict(t=120, l=120), title=title(
        f"M4 {g} · Beeswarm: how each part of the past moves the forecast",
        "one dot = one series of one fold model / seed; colour = the segment's level relative to the input window "
        "(red above the window mean). Red on the right = a high level there raises the forecast"))
    save(fig, out_dir, "06_shap_beeswarm", 1500, 600)
    # (e) local
    ctx["shap_store"], ctx["seg_names"], ctx["onames"] = store, names, onames
    for m in models:
        e = ctx["series_smape"][m][sample]
        picks = [("worst series", int(np.argmax(e))), ("best series", int(np.argmin(e)))]
        if "drift_series" in ctx:
            picks.append(("largest drift", int(np.nanargmax(ctx["drift_series"][sample]))))
        for why, w in picks:
            local_figure(m, w, why, store, names, onames, D, ctx, out_dir)
    msg = {}
    for m in models:
        d = I[I.model == m].groupby("lags").share.mean().sort_values(ascending=False)
        msg[m] = f"most important segment {d.index[0]} ({100 * d.iloc[0]:.0f}% of importance)"
    return {"messages": msg, "efficiency_max_error": float(np.max(checks)) if checks else None,
            "stability": S.round(4).to_dict(orient="index")}


def local_figure(m, w, why, store, names, onames, D, ctx, out_dir):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    keys = [k for k in store if k[0] == m]
    phi = np.mean([store[k]["phi"][w] for k in keys], axis=0)
    full = np.mean([store[k]["full"][w] for k in keys], axis=0)
    base = np.mean([store[k]["base"][w] for k in keys], axis=0)
    i = ctx["shap_sample"][w]
    t = 0
    order = np.argsort(-np.abs(phi[:, t]))
    xs = ["average forecast<br>(training windows)"] + [names[j] for j in order] + ["forecast"]
    ys = [base[t]] + [phi[j, t] for j in order] + [0.0]
    meas = ["absolute"] + ["relative"] * len(order) + ["total"]
    texts = [f"{v:+.2f}" if mm == "relative" else f"{v:.2f}" for v, mm in zip(ys, meas)]
    texts[-1] = f"{full[t]:.2f}"
    run_ = np.cumsum([base[t]] + [phi[j, t] for j in order])
    lo, hi = min(0.0, run_.min()), max(0.0, run_.max())
    pad = 0.2 * (hi - lo + 1e-9)
    fig = make_subplots(rows=1, cols=3, column_widths=[0.34, 0.33, 0.33], horizontal_spacing=0.09,
                        subplot_titles=["waterfall: mean forecast (normalised units)",
                                        "decision plot: every explained output", "the series"])
    fig.add_trace(go.Waterfall(orientation="h", y=xs, x=ys, measure=meas, text=texts, textposition="outside",
                               cliponaxis=False, increasing=dict(marker=dict(color="#C53030")),
                               decreasing=dict(marker=dict(color="#2B6CB0")), totals=dict(marker=dict(color="#2D3748")),
                               showlegend=False), 1, 1)
    fig.update_xaxes(range=[lo - pad, hi + pad], row=1, col=1)
    fig.update_yaxes(autorange="reversed", row=1, col=1)
    imp = np.argsort(-np.abs(phi).sum(1))[::-1]
    labs = ["average"] + [names[j] for j in imp]
    for k, on in enumerate(onames):
        steps = [base[k]] + list(base[k] + np.cumsum(phi[imp, k]))
        fig.add_trace(go.Scatter(x=steps, y=labs, mode="lines+markers", name=on,
                                 line=dict(color=PAL[k % len(PAL)], width=3 if k == 0 else 1.3)), 1, 2)
    x = D["x_test"][i]
    y = D["y_test"][i]
    L, H = len(x), len(y)
    fig.add_trace(go.Scatter(x=list(range(-L + 1, 1)), y=x, mode="lines", line=dict(color="#718096"), name="input"), 1, 3)
    fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=y, mode="lines", line=dict(color="black", width=2), name="truth"),
                  1, 3)
    fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=ctx["series_pred"][m][i], mode="lines",
                             line=dict(color=color(m), width=2), name=label(m)), 1, 3)
    fig.update_layout(margin=dict(t=130, l=150), legend=dict(orientation="h", y=-0.15), title=title(
        f"M4 {D['group']} · {label(m)} · series {D['ids'][i]}: {why}",
        f"sMAPE {ctx['series_smape'][m][i]:.1f} (mean over fold models and seeds). From the average forecast on real "
        "training windows, each input segment adds (red) or removes (blue); the bars add up exactly to the forecast"))
    save(fig, out_dir, f"07_shap_local_{m}_{D['ids'][i]}", 1700, 620)


def analysis_gradshap(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    g, L, H, mf = D["group"], D["L"], D["H"], D["freq"]
    rng = np.random.default_rng(opt.seed + 1)
    sample = ctx["shap_sample"][:opt.grad_series]
    maps = {}
    for m, runners in R_by_model.items():
        acc = []
        for R in runners:
            if R.run["seed"] not in opt.grad_seeds:
                continue
            bank = R.train_windows(rng, opt.background)
            for i in sample:
                x = D["x_test"][i]
                mu, sd = norm_stats(x)
                rec = R.own_periods(x)
                pick = bank[rng.integers(0, len(bank), opt.grad_samples)] * sd + mu
                alpha = rng.uniform(size=(opt.grad_samples, 1))
                diff = x[None] - pick
                xb = torch.tensor(pick + alpha * diff, dtype=torch.float32, device=R.device, requires_grad=True)
                pred = R.forward(xb, mode="force", rec=rec, grad=True, chunk=10 ** 6)
                outs = outputs_of((pred - mu.item()) / sd.item(), H)
                ph = []
                for t in range(outs.shape[1]):
                    gr, = torch.autograd.grad(outs[:, t].sum(), xb, retain_graph=t < outs.shape[1] - 1)
                    ph.append((torch.tensor(diff, dtype=torch.float32, device=R.device) * gr).mean(0).detach().cpu().numpy())
                acc.append(np.abs(np.stack(ph, -1)))                       # [L, out]
        if acc:
            maps[m] = np.mean(acc, axis=0)
    if not maps:
        return {}
    np.savez_compressed(os.path.join(out_dir, "gradshap_maps.npz"), **maps)
    models = list(maps)
    onames = output_names(H)
    fig = make_subplots(rows=2, cols=len(models), row_heights=[0.55, 0.45], vertical_spacing=0.14,
                        horizontal_spacing=0.08, subplot_titles=[label(m) for m in models] + [""] * len(models))
    for j, m in enumerate(models, 1):
        mp = maps[m][::-1]                                                # row 0 = lag 1
        share = mp / mp.sum(0, keepdims=True)
        fig.add_trace(go.Heatmap(z=100 * share.T, x=list(range(1, L + 1)), y=onames, colorscale="Viridis",
                                 showscale=(j == 1), colorbar=dict(title="%", len=0.5, y=0.78),
                                 hovertemplate="lag %{x} → %{y}<br>%{z:.2f}%<extra></extra>"), 1, j)
        fig.add_trace(go.Bar(x=list(range(1, L + 1)), y=100 * share[:, 0], marker_color=color(m), showlegend=False), 2, j)
        if mf > 1:
            for k in range(mf, L + 1, mf):
                fig.add_vline(x=k, line=dict(color="#A0AEC0", dash="dot"), row=2, col=j)
        fig.update_xaxes(title_text="lag (1 = most recent value)", row=2, col=j)
    fig.update_yaxes(title_text="% of importance (mean forecast)", row=2, col=1)
    fig.update_layout(margin=dict(t=120), title=title(
        f"M4 {g} · Importance of every past value (GradientSHAP)",
        f"expected gradients with rescaled real training windows as references ({len(sample)} series; seed(s) "
        + ", ".join(map(str, opt.grad_seeds)) + ")." + (f" Dotted = multiples of the seasonal period {mf}" if mf > 1 else "")))
    save(fig, out_dir, "08_gradshap_lags", 1500, 720)
    out = {}
    for m in models:
        prof = maps[m][::-1][:, 0]
        k = mf if mf > 1 else max(1, H // 2)
        out[m] = {f"share of importance in the last {k} values": float(prof[:k].sum() / prof.sum())}
    return out


# =========================================================================================================
# 1b. Local GradientSHAP (one signed bar per input step) and Grad-CAM on TimesNet's 2D period images
# =========================================================================================================
def local_picks(D, ctx, models, k_extra=True):
    """Series explained locally: worst, best and largest-drift series of the SHAP sample (mean of both models)."""
    pool = np.asarray(ctx["shap_sample"])
    e = np.nanmean([ctx["series_smape"][m][pool] for m in models], axis=0)
    picks = [("worst series", int(pool[np.nanargmax(e)])), ("best series", int(pool[np.nanargmin(e)]))]
    if k_extra and "drift_series" in ctx:
        picks.append(("largest drift", int(pool[np.nanargmax(ctx["drift_series"][pool])])))
    out, seen = [], set()
    for why, i in picks:
        if i not in seen:
            out.append((why, i))
            seen.add(i)
    return out


def gradshap_signed(R, x, bank, n, rng):
    """Expected gradients of the mean forecast (series' normalized units) w.r.t. every input step, signed.
    Returns phi [L], f(x), mean f(baselines) (all normalized); phi sums approximately to f(x) - E f(baseline)."""
    mu, sd = norm_stats(x)
    rec = R.own_periods(x)
    pick = bank[rng.integers(0, len(bank), n)] * sd + mu
    alpha = rng.uniform(size=(n, 1))
    diff = x[None] - pick
    xb = torch.tensor(pick + alpha * diff, dtype=torch.float32, device=R.device, requires_grad=True)
    pred = R.forward(xb, mode="force", rec=rec, grad=True, chunk=10 ** 6)
    out = ((pred - mu.item()) / sd.item()).mean(1)
    gr, = torch.autograd.grad(out.sum(), xb)
    phi = (torch.tensor(diff, dtype=torch.float32, device=R.device) * gr).mean(0).detach().cpu().numpy()
    with torch.no_grad():
        fx = ((R.forward(x[None], mode="force", rec=rec).cpu().numpy() - mu) / sd).mean()
        fb = ((R.forward(pick, mode="force", rec=rec).cpu().numpy() - mu) / sd).mean(1).mean()
    return phi, float(fx), float(fb)


def analysis_gradshap_local(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    g, L, H, mf = D["group"], D["L"], D["H"], D["freq"]
    models = list(R_by_model)
    rng = np.random.default_rng(opt.seed + 7)
    picks = local_picks(D, ctx, models)
    out = {}
    for why, i in picks:
        x, y = D["x_test"][i], D["y_test"][i]
        res = {}
        for m in models:
            phis, fxs, fbs = [], [], []
            for R in R_by_model[m]:
                if R.run["seed"] not in opt.grad_seeds:
                    continue
                bank = R.train_windows(rng, opt.background)
                phi, fx, fb = gradshap_signed(R, x, bank, opt.grad_samples * 4, rng)
                phis.append(phi); fxs.append(fx); fbs.append(fb)
            if phis:
                res[m] = (np.mean(phis, 0), float(np.mean(fxs)), float(np.mean(fbs)))
        if not res:
            continue
        cols = list(res)
        fig = make_subplots(rows=2, cols=len(cols), row_heights=[0.45, 0.55], shared_xaxes=True, vertical_spacing=0.1,
                            horizontal_spacing=0.07, subplot_titles=[label(m) for m in cols] + [""] * len(cols))
        mu, sd = norm_stats(x)
        steps_in = list(range(-L + 1, 1))
        for j, m in enumerate(cols, 1):
            phi, fx, fb = res[m]
            fig.add_trace(go.Scatter(x=steps_in, y=(x - mu) / sd, mode="lines", line=dict(color="#718096"), name="input",
                                     legendgroup="in", showlegend=(j == 1)), 1, j)
            fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=(y - mu) / sd, mode="lines", line=dict(color="black"),
                                     name="truth", legendgroup="tr", showlegend=(j == 1)), 1, j)
            fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=(ctx["series_pred"][m][i] - mu) / sd, mode="lines",
                                     line=dict(color=color(m), width=2), name=label(m)), 1, j)
            fig.add_trace(go.Bar(x=steps_in, y=phi, marker_color=np.where(phi >= 0, "#C53030", "#2B6CB0"),
                                 showlegend=False, hovertemplate="step %{x}<br>SHAP %{y:+.3f}<extra></extra>"), 2, j)
            if mf > 1:
                for k in range(0, -L, -mf):
                    fig.add_vline(x=k, line=dict(color="#CBD5E0", dash="dot"), row=2, col=j)
            fig.layout.annotations[j - 1].text = (f"{label(m)}<br><sup>sum of bars {phi.sum():+.3f} vs forecast − "
                                                  f"average forecast {fx - fb:+.3f}</sup>")
        fig.update_yaxes(title_text="normalised units", row=1, col=1)
        fig.update_yaxes(title_text="SHAP on the mean forecast", row=2, col=1)
        fig.update_xaxes(title_text="input step (0 = last observed value)", row=2)
        fig.update_layout(legend=dict(orientation="h", y=-0.12), margin=dict(t=140), title=title(
            f"M4 {g} · series {D['ids'][i]} ({why}): which past steps pushed the forecast up or down?",
            "GradientSHAP (expected gradients), one bar per input step: red = raised the mean forecast, blue = lowered "
            "it.<br>Baselines = other series' training windows at this series' level and scale; the bars add up "
            "(approximately) to forecast − average forecast of the baselines. Mean over the fold models of seed(s) "
            + ", ".join(map(str, opt.grad_seeds)) + (f"; dotted = every {mf} steps (seasonal period)" if mf > 1 else "")))
        save(fig, out_dir, f"20_gradshap_steps_{D['ids'][i]}", 1600, 700)
        out[str(D["ids"][i])] = {why: {m: {"sum": float(res[m][0].sum()), "forecast_minus_baseline":
                                             res[m][1] - res[m][2]} for m in res}}
    return out


def gradcam_series(R, x):
    """Grad-CAM of the mean forecast on every 2D period image of every TimesBlock (one series, its own periods).
    Returns a list per TimesBlock of (period, weight, cam [rows, period]) and the time profile per block [L+H]."""
    acts = []
    hooks = []
    for blk in R.model.model:
        def hook(module, inp, out):
            out.retain_grad()
            acts.append(out)
        hooks.append(blk.conv.register_forward_hook(hook))
    PER["log"] = []
    try:
        xt = torch.tensor(x[None], dtype=torch.float32, device=R.device)
        R.mod.Model.forecast = R.mod._original_forecast
        with torch.set_grad_enabled(True):
            o = R.model(xt.unsqueeze(-1), None, torch.zeros((1, R.L // 2 + R.H, 1), device=R.device), None)
            o[:, -R.H:, 0].mean().backward()
    finally:
        for h in hooks:
            h.remove()
        log, PER["log"] = PER["log"], None
    T = R.L + R.H
    blocks, profiles, a = [], [], 0
    for periods, weights in log:
        items, prof = [], np.zeros(T)
        for p, w in zip(periods, weights):
            A, G = acts[a], acts[a].grad
            a += 1
            alpha = G.mean(dim=(2, 3), keepdim=True)
            cam = torch.relu((alpha * A).sum(1))[0].detach().cpu().numpy()          # [rows, period]
            items.append((int(p), float(w), cam))
            prof += w * cam.reshape(-1)[:T]
        blocks.append(items)
        profiles.append(prof)
    return blocks, profiles


def analysis_gradcam(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    g, L, H = D["group"], D["L"], D["H"]
    T = L + H
    models = list(R_by_model)
    # ---- global: mean time profile over the SHAP sample, per model and TimesBlock
    sample = np.asarray(ctx["shap_sample"])[:opt.cam_series]
    glob = {}
    for m in models:
        profs = []
        for R in R_by_model[m]:
            for i in sample:
                _, pr = gradcam_series(R, D["x_test"][i])
                profs.append([p / (p.sum() + 1e-12) for p in pr])
        glob[m] = np.mean(profs, axis=0)                                         # [blocks, T]
    nb = glob[models[0]].shape[0]
    fig = make_subplots(rows=1, cols=nb, shared_yaxes=True, horizontal_spacing=0.06,
                        subplot_titles=[f"TimesBlock {b + 1}" for b in range(nb)])
    out = {}
    for m in models:
        for b in range(nb):
            fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=100 * glob[m][b], mode="lines",
                                     line=dict(color=color(m), width=2.5), name=label(m), legendgroup=m,
                                     showlegend=(b == 0)), 1, b + 1)
        out[m] = {f"block {b + 1}: share of Grad-CAM on the future part": float(glob[m][b][L:].sum()) for b in range(nb)}
    for b in range(nb):
        fig.add_vrect(x0=0.5, x1=H + 0.5, fillcolor="#EDF2F7", opacity=0.7, line_width=0, layer="below", row=1, col=b + 1)
        fig.update_xaxes(title_text="time step (≤ 0 input, > 0 future part)", row=1, col=b + 1)
    fig.update_yaxes(title_text="% of Grad-CAM", row=1, col=1)
    fig.update_layout(legend=dict(orientation="h", y=-0.2), margin=dict(t=120), title=title(
        f"M4 {g} · Where do TimesNet's convolutions look? (Grad-CAM, folded back to time)",
        f"{len(sample)} test series, all fold models and seeds; each period image's Grad-CAM is folded back to time and "
        "weighted by TimesNet's period weight. Shaded = the future part, which TimesNet's linear layer creates before "
        "the convolutions"))
    save(fig, out_dir, "21_gradcam_global", 1400, 520)
    # ---- local: the 2D images of chosen series
    rng = np.random.default_rng(opt.seed + 8)
    for why, i in local_picks(D, ctx, models):
        x, y = D["x_test"][i], D["y_test"][i]
        mu, sd = norm_stats(x)
        ncols = 3
        comp, titles = {}, []
        for m in models:
            Rs = [R for R in R_by_model[m] if R.run["seed"] in opt.grad_seeds] or R_by_model[m]
            blocks, profs = gradcam_series(Rs[0], x)
            items = sorted(blocks[0], key=lambda t: -t[1])[:ncols]
            comp[m] = (items, profs)
            titles += [f"{label(m)} · TimesBlock 1 · period {p} (weight {w:.2f})" for p, w, _ in items]
            titles += [""] * (ncols - len(items))
            titles.append(f"{label(m)} · folded back to time (shaded = future part)")
        fig = make_subplots(rows=2 * len(models), cols=ncols, vertical_spacing=0.07, horizontal_spacing=0.06,
                            specs=[[{}] * ncols, [{"colspan": ncols}, None, None]] * len(models),
                            row_heights=[0.6, 0.4] * len(models), subplot_titles=titles)
        for r_, m in enumerate(models):
            items, profs = comp[m]
            row = 2 * r_ + 1
            for c, (p, w, cam) in enumerate(items, 1):
                z = cam / (cam.max() + 1e-12)
                fig.add_trace(go.Heatmap(z=z, colorscale="Inferno", zmin=0, zmax=1, showscale=(r_ == 0 and c == 1),
                                         colorbar=dict(title="Grad-CAM", len=0.3, y=0.85),
                                         hovertemplate="cycle %{y} · position %{x}<br>%{z:.2f}<extra></extra>"), row, c)
                fig.update_xaxes(title_text="position inside one period", row=row, col=c)
                fig.update_yaxes(title_text="cycle (row)", autorange="reversed", row=row, col=c)
            prof = profs[0] / (profs[0].max() + 1e-12)
            fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=prof, mode="lines", fill="tozeroy",
                                     line=dict(color=color(m)), name=f"{label(m)} Grad-CAM (block 1)"), row + 1, 1)
            if len(profs) > 1:
                p2 = profs[1] / (profs[1].max() + 1e-12)
                fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=p2, mode="lines",
                                         line=dict(color=color(m), dash="dot"), name=f"{label(m)} Grad-CAM (block 2)"),
                              row + 1, 1)
            series = np.r_[(x - mu) / sd, (y - mu) / sd]
            sn = (series - series.min()) / (series.max() - series.min() + 1e-12)
            fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=sn, mode="lines", line=dict(color="#A0AEC0"),
                                     name="series (input + truth, rescaled)", showlegend=(r_ == 0)), row + 1, 1)
            fig.add_vrect(x0=0.5, x1=H + 0.5, fillcolor="#EDF2F7", opacity=0.6, line_width=0, layer="below", row=row + 1, col=1)
        fig.update_layout(legend=dict(orientation="h", y=-0.06), margin=dict(t=130), title=title(
            f"M4 {g} · series {D['ids'][i]} ({why}): which part of the 2D period images did the convolutions use?",
            "TimesNet folds the (embedded) series into 2D images, rows = cycles, columns = position inside one period, "
            "before its Conv2d blocks.<br>Bright = important for the mean forecast (Grad-CAM on the convolution output). "
            "The three periods with the largest weights in TimesBlock 1; one fold model (first seed)"))
        save(fig, out_dir, f"22_gradcam_{D['ids'][i]}", 1500, 520 * len(models) + 120)
    _ = rng
    return out


# =========================================================================================================
# 2. Faithfulness and stability
# =========================================================================================================
def analysis_faith(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from scipy.stats import spearmanr
    from itertools import combinations
    store = ctx["shap_store"]
    g, L, H = D["group"], D["L"], D["H"]
    names, masks, _ = segments(L, opt.n_segments)
    F = len(masks)
    sample = ctx["shap_sample"][:opt.faith_series]
    rng = np.random.default_rng(opt.seed + 2)
    rows = []
    for m, runners in R_by_model.items():
        for R in runners:
            v = store[(m, R.run["fold"], R.run["seed"])]
            bank = R.train_windows(rng, 3)
            for w, i in enumerate(sample):
                x = D["x_test"][i]
                mu, sd = norm_stats(x)
                rec = R.own_periods(x)
                imp = np.abs(v["phi"][w]).sum(1)
                orders = {"SHAP order": list(np.argsort(-imp)), "random order": list(rng.permutation(F))}
                inputs, tags = [], []
                for kind, order in orders.items():
                    for b in bank:
                        b = b * sd + mu
                        for n in range(F + 1):
                            cur = x.copy()
                            for j in order[:n]:
                                cur = np.where(masks[j], b, cur)
                            inputs.append(cur)
                            tags.append((kind, n))
                with torch.no_grad():
                    pred = R.forward(np.stack(inputs), mode="force", rec=rec).cpu().numpy()
                out = outputs_of(torch.tensor((pred - mu) / sd), H).numpy()
                fx = out[0]
                ref = np.mean([out[k] for k, (kind, n) in enumerate(tags) if n == F and kind == "SHAP order"], axis=0)
                scale = np.linalg.norm(ref - fx) + 1e-9
                for k, (kind, n) in enumerate(tags):
                    rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "series": w, "order": kind,
                                 "fraction": n / F, "change": float(np.linalg.norm(out[k] - fx) / scale)})
    Df = pd.DataFrame(rows).groupby(["model", "fold", "seed", "series", "order", "fraction"]).change.mean().reset_index()
    Df.to_csv(os.path.join(out_dir, "faithfulness.csv"), index=False)
    models = list(R_by_model)
    res = {}
    fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.12,
                        subplot_titles=["forecast change while segments are replaced", "area under the curve"])
    for m in models:
        d = Df[Df.model == m]
        for kind, dash in (("SHAP order", "solid"), ("random order", "dot")):
            c = d[d.order == kind].groupby("fraction").change.mean()
            fig.add_trace(go.Scatter(x=100 * c.index, y=c.values, mode="lines+markers", name=f"{label(m)} · {kind}",
                                     line=dict(color=color(m), dash=dash, width=3 if dash == "solid" else 2)), 1, 1)
        auc = d.groupby(["order", "fold", "seed", "series"]).change.mean().unstack("order")
        res[m] = {"auc_shap": float(auc["SHAP order"].mean()), "auc_random": float(auc["random order"].mean()),
                  "shap_better_share": float((auc["SHAP order"] > auc["random order"]).mean())}
    fig.add_trace(go.Bar(x=[label(m) for m in res], y=[res[m]["auc_shap"] for m in res], name="SHAP order",
                         marker_color="#2D3748"), 1, 2)
    fig.add_trace(go.Bar(x=[label(m) for m in res], y=[res[m]["auc_random"] for m in res], name="random order",
                         marker_color="#CBD5E0"), 1, 2)
    fig.update_xaxes(title_text="% of input segments replaced by real (rescaled) training values", row=1, col=1)
    fig.update_yaxes(title_text="forecast change (1 = all replaced)", row=1, col=1)
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.22), margin=dict(t=120), title=title(
        f"M4 {g} · Faithfulness: does SHAP find what the model really uses?",
        "segments replaced most-important-first vs at random; a faster rise for the SHAP order = trustworthy. Share of "
        "series where SHAP beats random: " + ", ".join(f"{label(m)} {100 * r['shap_better_share']:.0f}%" for m, r in res.items())))
    save(fig, out_dir, "09_faithfulness", 1400, 560)
    imp = {k: np.abs(v["phi"]).mean(axis=2) for k, v in store.items()}
    rows = []
    for m in models:
        keys = [k for k in imp if k[0] == m]
        for a, b in combinations(keys, 2):
            kind = "across seeds (same fold)" if a[1] == b[1] else ("across folds (same seed)" if a[2] == b[2] else None)
            if kind:
                for w in range(imp[a].shape[0]):
                    rows.append({"pair": f"{label(m)}: {kind}", "rho": spearmanr(imp[a][w], imp[b][w]).statistic})
    if len(models) == 2:
        for k in [k for k in imp if k[0] == models[0]]:
            k2 = (models[1], k[1], k[2])
            if k2 in imp:
                for w in range(imp[k].shape[0]):
                    rows.append({"pair": "GELU vs KANB (same fold, seed)", "rho": spearmanr(imp[k][w], imp[k2][w]).statistic})
    S = pd.DataFrame(rows).dropna()
    fig = go.Figure()
    for j, (pair, gg) in enumerate(S.groupby("pair", sort=False)):
        fig.add_trace(go.Box(y=gg.rho, name=pair, boxmean=True, marker_color=PAL[j % len(PAL)]))
    fig.add_hline(y=0, line=dict(color="#A0AEC0", dash="dash"))
    fig.update_yaxes(range=[-1, 1], title_text="Spearman correlation of the segment rankings (per series)")
    fig.update_layout(showlegend=False, margin=dict(t=120), title=title(
        f"M4 {g} · Stability of the explanations",
        "1 = the two explanations rank the input segments identically for the same series"))
    save(fig, out_dir, "10_explanation_stability", 1300, 560)
    res["stability"] = S.groupby("pair").rho.median().round(3).to_dict() if len(S) else {}
    return res


# =========================================================================================================
# 3. Segment ablation
# =========================================================================================================
def analysis_ablation(R_by_model, D, out_dir, ctx, opt):
    import plotly.graph_objects as go
    g, L, H = D["group"], D["L"], D["H"]
    names, masks, _ = segments(L, opt.n_segments)
    sample = np.array(ctx["ablation_sample"])
    rng = np.random.default_rng(opt.seed + 3)
    X, Y = D["x_test"][sample], D["y_test"][sample]
    mu, sd = norm_stats(X)
    rows = []
    for m, runners in R_by_model.items():
        for R in runners:
            with torch.no_grad():
                base = R.forward(X).cpu().numpy()
            base_s = CVM.smape_series(Y, base).mean()
            for j, nm in enumerate(names):
                for rep in range(opt.ablation_repeats):
                    donor = rng.permutation(len(X))
                    Xd = X.copy()
                    zd = (X[donor] - mu[donor]) / sd[donor]
                    Xd[:, masks[j]] = (zd * sd + mu)[:, masks[j]]
                    with torch.no_grad():
                        p = np.concatenate([R.forward(Xd[k:k + 1]).cpu().numpy() for k in range(len(Xd))])
                    s = CVM.smape_series(Y, p).mean()
                    rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "segment": nm, "repeat": rep,
                                 "base_smape": base_s, "smape": s, "increase_pct": 100 * (s - base_s) / base_s})
    A = pd.DataFrame(rows)
    A.to_csv(os.path.join(out_dir, "segment_ablation.csv"), index=False)
    gA = A.groupby(["model", "fold", "seed", "segment"]).increase_pct.mean().reset_index()
    fig = go.Figure()
    for m in R_by_model:
        d = gA[gA.model == m].groupby("segment").increase_pct
        mean, lo, hi = d.mean().reindex(names), d.quantile(0.025).reindex(names), d.quantile(0.975).reindex(names)
        fig.add_trace(go.Bar(x=names, y=mean.values, name=label(m), marker_color=color(m),
                             error_y=dict(type="data", symmetric=False, array=(hi - mean).values, arrayminus=(mean - lo).values)))
    fig.add_hline(y=0, line=dict(color="#4A5568"))
    fig.update_yaxes(title_text="test sMAPE increase when the segment is replaced (%)")
    fig.update_xaxes(title_text="input segment (lag 1 = most recent value)")
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.18), margin=dict(t=120), title=title(
        f"M4 {g} · Which part of the past does each model need? (segment ablation)",
        f"{len(sample)} test series; each segment is replaced by the same segment of another real series, rescaled to "
        f"the series' level and scale ({opt.ablation_repeats} repeats). Bars = mean over fold models and seeds, "
        "whiskers = 2.5–97.5% range"))
    save(fig, out_dir, "11_segment_ablation", 1300, 560)
    out = {}
    for m in R_by_model:
        d = gA[gA.model == m].groupby("segment").increase_pct.mean().sort_values(ascending=False)
        out[m] = {"most needed": d.index[0], "increase_pct": float(d.iloc[0])}
    return out


# =========================================================================================================
# 4. Drift, novelty, concept drift, KANB vs GELU per drift tercile
# =========================================================================================================
def window_features(xin, fut=None, m=1):
    """Scale-free features of input windows [n, L] (and their futures [n, H]) in input-normalized units."""
    mu, sd = norm_stats(xin)
    z = (xin - mu) / sd
    L = z.shape[1]
    t = np.arange(L) - (L - 1) / 2
    slope = (z * t).sum(1) / (t ** 2).sum()
    d = np.diff(z, axis=1)
    half = L // 2
    lvl = z[:, half:].mean(1) - z[:, :half].mean(1)
    vol = np.log((z[:, half:].std(1) + 1e-3) / (z[:, :half].std(1) + 1e-3))

    def ac(k):
        if k >= L - 2:
            return np.zeros(len(z))
        a, b = z[:, :-k], z[:, k:]
        a = a - a.mean(1, keepdims=True)
        b = b - b.mean(1, keepdims=True)
        return (a * b).sum(1) / (np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1)) + 1e-9)

    cols = {"slope": slope, "noise (std of changes)": d.std(1), "level change (2nd vs 1st half)": lvl,
            "volatility change": vol, "autocorrelation lag 1": ac(1), "last value": z[:, -1],
            "max |value|": np.abs(z).max(1)}
    if m > 1:
        cols[f"autocorrelation lag {m}"] = ac(m)
    marg = pd.DataFrame(cols)
    if fut is None:
        return marg, None
    zf = (fut - mu) / sd
    tf = np.arange(zf.shape[1]) - (zf.shape[1] - 1) / 2
    joint = marg.copy()
    joint["future level"] = zf.mean(1)
    joint["future slope"] = (zf * tf).sum(1) / (tf ** 2).sum()
    joint["future spread"] = zf.std(1)
    return marg, joint


def series_drift(D, opt):
    """Per series: drift of the forecast period vs the input (energy distance of the values, in input-normalized
    units) and a conformal p-value against the same score for earlier blocks of the series' own history."""
    L, H = D["L"], D["H"]
    scores, pvals, nnull = [], [], []
    for x, y in zip(D["train"], D["y_test"]):
        w = x[-L:]
        mu, sd = norm_stats(w)
        s = energy_1d((w - mu) / sd, (y - mu) / sd)
        null = []
        for c in range(len(x) - H, L - 1, -H):
            if len(null) >= opt.max_null_blocks:
                break
            wi = x[c - L:c]
            m_, s_ = norm_stats(wi)
            null.append(energy_1d((wi - m_) / s_, (x[c:c + H] - m_) / s_))
        null = np.array(null)
        scores.append(s)
        pvals.append((1 + (null >= s).sum()) / (1 + len(null)) if len(null) else np.nan)
        nnull.append(len(null))
    return np.array(scores), np.array(pvals), np.array(nnull)


def analysis_drift(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from scipy.stats import gaussian_kde
    from scipy.spatial.distance import cdist
    g, L, H, mf = D["group"], D["L"], D["H"], D["freq"]
    rng = np.random.default_rng(opt.seed + 4)
    models = list(R_by_model)
    # ---- per-series drift
    t0 = time.time()
    score, pval, nnull = series_drift(D, opt)
    qval = bh(pval)
    ctx["drift_series"] = score
    q1, q2 = np.nanquantile(score, [1 / 3, 2 / 3])
    bins = np.where(score <= q1, "small drift", np.where(score <= q2, "medium drift", "large drift"))
    print(f"  drift: per-series scores ({(time.time() - t0) / 60:.1f} min)", flush=True)
    # ---- population drift: training windows (per fold) vs test windows
    folds = sorted({R.run["fold"] for rs in R_by_model.values() for R in rs})
    test_idx = rng.choice(len(D["train"]), min(opt.pop_series, len(D["train"])), replace=False)
    te_m, te_j = window_features(D["x_test"][test_idx], D["y_test"][test_idx], mf)
    glob_rows, clf_rows, shap_rows, feat_store = [], [], [], {}
    for f in folds:
        fo = CVM.FoldM4(D["train"], D["spec"], f)
        ok = [k for k, ts in enumerate(fo.train) if len(ts) >= L + H]
        ok = rng.choice(ok, min(opt.pop_series, len(ok)), replace=False)
        xin = np.stack([fo.train[k][-H - L:-H] for k in ok])          # training input with its (training) future
        fut = np.stack([fo.train[k][-H:] for k in ok])
        tr_m, tr_j = window_features(xin, fut, mf)
        feat_store[f] = (tr_m, tr_j)
        for kind, A, B in (("covariate", tr_m, te_m), ("joint", tr_j, te_j)):
            mu, sd = A.values.mean(0), A.values.std(0) + 1e-9
            Az, Bz = (A.values - mu) / sd, (B.values - mu) / sd
            n = min(len(Az), len(Bz), opt.mmd_rows)
            Az, Bz = Az[rng.choice(len(Az), n, replace=False)], Bz[rng.choice(len(Bz), n, replace=False)]
            gamma = median_gamma(np.vstack([Az, Bz]), rng)
            obs = (mmd2(Az, Bz, gamma), energy(Az, Bz))
            pool = np.vstack([Az, Bz])
            perm = []
            for _ in range(opt.perms):
                o = rng.permutation(len(pool))
                perm.append((mmd2(pool[o[:n]], pool[o[n:]], gamma), energy(pool[o[:n]], pool[o[n:]])))
            perm = np.array(perm)
            glob_rows.append({"fold": f, "kind": kind, "mmd2": obs[0], "energy": obs[1],
                              "p_mmd": (1 + (perm[:, 0] >= obs[0]).sum()) / (1 + opt.perms),
                              "p_energy": (1 + (perm[:, 1] >= obs[1]).sum()) / (1 + opt.perms)})
            auc, p_auc, sh = domain_classifier(A.values, B.values, opt, rng)
            clf_rows.append({"fold": f, "kind": kind, "auc": auc, "p": p_auc})
            if sh is not None:
                for nm, v in zip(A.columns, sh):
                    shap_rows.append({"fold": f, "kind": kind, "feature": nm, "mean_abs_shap": v})
        print(f"  drift: fold {f} done", flush=True)
    G, CL, SH = pd.DataFrame(glob_rows), pd.DataFrame(clf_rows), pd.DataFrame(shap_rows)
    G.to_csv(os.path.join(out_dir, "drift_global_tests.csv"), index=False)
    CL.to_csv(os.path.join(out_dir, "domain_classifier.csv"), index=False)
    # ---- novelty of each test input (kNN in feature space vs the training windows, leave-series-out null)
    tr_m = pd.concat([feat_store[f][0] for f in folds])
    ref = tr_m.values[rng.choice(len(tr_m), min(len(tr_m), 4000), replace=False)]
    mu, sd = ref.mean(0), ref.std(0) + 1e-9
    refz = (ref - mu) / sd
    allm, _ = window_features(D["x_test"], None, mf)
    tz = (allm.values - mu) / sd
    k = opt.knn
    Dr = cdist(refz, refz)
    np.fill_diagonal(Dr, np.inf)
    null = np.sort(Dr, axis=1)[:, :k].mean(1)
    nov = np.empty(len(tz))
    for s0 in range(0, len(tz), 2000):
        dt = np.sort(cdist(tz[s0:s0 + 2000], refz), axis=1)[:, :k].mean(1)
        nov[s0:s0 + 2000] = np.searchsorted(np.sort(null), dt) / len(null)
    # ---- per-series table, KANB vs GELU per drift tercile
    W = pd.DataFrame({"id": D["ids"], "length": [len(s) for s in D["train"]], "drift_score": score,
                      "drift_p": pval, "drift_q": qval, "null_blocks": nnull, "drift_bin": bins, "novelty": nov,
                      "forecast_period": ctx["test_year_bin"]})
    for m in models:
        W[f"smape_{m}"] = ctx["series_smape"][m]
        W[f"mase_{m}"] = ctx["series_mase"][m]
    rows = []
    if len(models) == 2:
        for metric in ("smape", "mase"):
            d_all = ctx[f"series_diff_{metric}"]
            for bname in ("small drift", "medium drift", "large drift", "all series"):
                sel = np.ones(len(d_all), bool) if bname == "all series" else (bins == bname)
                r = paired_test(d_all[sel], rng)
                base = np.nanmean(ctx[f"series_{metric}"][models[0]][sel])
                rows.append({"metric": metric, "drift": bname, "series": r["n"], "GELU": base,
                             "KANB": np.nanmean(ctx[f"series_{metric}"][models[1]][sel]),
                             "mean diff (KANB-GELU)": r["mean"], "95% CI low": r.get("mean_lo"),
                             "95% CI high": r.get("mean_hi"), "change %": 100 * r["mean"] / base if base else np.nan,
                             "KANB better (series)": r.get("kanb_better_share"), "Wilcoxon p": r["p"],
                             "KANB better in folds": ctx["fold_bin_wins"](sel, metric)})
        T = pd.DataFrame(rows)
        T["Holm p"] = np.nan
        for metric in ("smape", "mase"):
            s = T.metric == metric
            T.loc[s, "Holm p"] = holm(T.loc[s, "Wilcoxon p"].values)
        T["verdict"] = np.where((T["Holm p"] < 0.05) & (T["mean diff (KANB-GELU)"] < 0), "KANB better",
                                np.where((T["Holm p"] < 0.05) & (T["mean diff (KANB-GELU)"] > 0), "GELU better",
                                         "no significant difference"))
    else:
        T = pd.DataFrame()
    T.to_csv(os.path.join(out_dir, "kanb_vs_gelu_by_drift.csv"), index=False)
    W.to_csv(os.path.join(out_dir, "series_drift_novelty.csv"), index=False)
    # ---- residual-based concept drift: each model's validation errors vs its test errors
    res_rows = []
    for m, runners in R_by_model.items():
        for R in runners:
            z = np.load(R.run["npz"])
            a = z["val_err_scaled"]
            b = z["err_scaled"][z["val_idx"]]
            ok = np.isfinite(a) & np.isfinite(b)
            a, b = a[ok], b[ok]
            lo, hi = np.quantile(np.r_[a, b], [0.01, 0.99])
            a, b = np.clip(a, lo, hi), np.clip(b, lo, hi)
            n = min(len(a), opt.mmd_rows)
            sel = rng.choice(len(a), n, replace=False)
            obs = energy_1d(a[sel], b[sel])
            pool = np.r_[a[sel], b[sel]]
            perm = []
            for _ in range(opt.perms):
                o = rng.permutation(len(pool))
                perm.append(energy_1d(pool[o[:n]], pool[o[n:]]))
            res_rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "energy": obs,
                             "p": (1 + (np.array(perm) >= obs).sum()) / (1 + opt.perms),
                             "validation sMAPE": float(np.mean(z["val_smape"])),
                             "test sMAPE (same series)": float(np.mean(z["smape"][z["val_idx"]]))})
    RS = pd.DataFrame(res_rows)
    if len(RS):
        RS["q"] = bh(RS.p)
    RS.to_csv(os.path.join(out_dir, "residual_concept_drift.csv"), index=False)
    drift_figures(W, G, CL, SH, T, RS, feat_store, te_m, D, ctx, out_dir, models, folds)
    p_min = np.nanmedian(1 / (1 + nnull[nnull > 0])) if (nnull > 0).any() else np.nan
    msgs = {"series": f"{100 * np.nanmean(pval <= 0.05):.1f}% of series have conformal p ≤ 0.05 (5% expected without "
                      f"drift); {100 * np.nanmean(qval < 0.05):.1f}% after Benjamini-Hochberg (the smallest possible p "
                      f"is 1/(1 + history blocks), median {p_min:.3f})"}
    if len(T):
        for r in T[T.metric == "smape"].itertuples():
            msgs[f"sMAPE {r.drift}"] = (f"KANB {r._9:+.2f}% sMAPE vs GELU (Wilcoxon-Holm p = {r._13:.3g}, "
                                        f"{r.series} series) → {r.verdict}")
    msgs["population"] = "; ".join(f"fold {r.fold} {r.kind}: MMD p = {r.p_mmd:.3f}" for r in G.itertuples())
    msgs["classifier"] = "; ".join(f"fold {r.fold} {r.kind}: AUC {r.auc:.2f} (p = {r.p:.3f})" for r in CL.itertuples())
    return {"messages": msgs, "comparison": T.round(4).to_dict(orient="records") if len(T) else []}


def domain_classifier(A, B, opt, rng):
    """Training windows (0) vs test windows (1), one row per series: 5-fold CV AUC, permutation null, SHAP."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold
    n = min(len(A), len(B), opt.clf_rows)
    X = np.vstack([A[rng.choice(len(A), n, replace=False)], B[rng.choice(len(B), n, replace=False)]])
    y = np.r_[np.zeros(n), np.ones(n)]

    def oof(lab):
        p = np.zeros(len(lab))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, lab):
            clf = HistGradientBoostingClassifier(max_iter=opt.clf_iter, max_depth=3, learning_rate=0.1, random_state=0)
            clf.fit(X[tr], lab[tr])
            p[te] = clf.predict_proba(X[te])[:, 1]
        return p

    auc = roc_auc_score(y, oof(y))
    null = [roc_auc_score(yp, oof(yp)) for yp in (rng.permutation(y) for _ in range(opt.clf_perms))]
    p_auc = (1 + (np.array(null) >= auc).sum()) / (1 + opt.clf_perms)
    sh = None
    try:
        import shap
        clf = HistGradientBoostingClassifier(max_iter=opt.clf_iter, max_depth=3, learning_rate=0.1, random_state=0)
        clf.fit(X, y)
        sv = shap.TreeExplainer(clf).shap_values(X[rng.choice(len(X), min(len(X), 800), replace=False)])
        sv = sv[1] if isinstance(sv, list) else sv
        sh = np.abs(sv).mean(0)
    except Exception as e:
        print("domain-classifier SHAP unavailable:", e)
    return float(auc), float(p_auc), sh


def drift_figures(W, G, CL, SH, T, RS, feat_store, te_m, D, ctx, out_dir, models, folds):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from scipy.stats import gaussian_kde
    g = D["group"]
    # (a) per-series drift: distribution, significance, over calendar time
    per = W.groupby("forecast_period").agg(sig=("drift_p", lambda q: float(np.nanmean(q <= 0.05))),
                                           sig_bh=("drift_q", lambda q: float(np.nanmean(q < 0.05))),
                                           score=("drift_score", "median"), n=("drift_score", "size")).reset_index()
    fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], horizontal_spacing=0.1,
                        subplot_titles=["drift score of every series (forecast period vs its input)",
                                        "over calendar time: share of series drifting significantly (dashed = 5% expected without drift)"])
    fig.add_trace(go.Histogram(x=W.drift_score, nbinsx=60, marker_color="#4A5568", name="all series"), 1, 1)
    for q, nm in zip(np.nanquantile(W.drift_score, [1 / 3, 2 / 3]), ["tercile 1/3", "tercile 2/3"]):
        fig.add_vline(x=q, line=dict(color="#C05621", dash="dash"), row=1, col=1)
    fig.add_trace(go.Bar(x=per.forecast_period.astype(str), y=per.sig, marker_color="#C05621", name="p ≤ 0.05",
                         text=[f"n={n}" for n in per.n], textposition="outside"), 1, 2)
    fig.add_trace(go.Bar(x=per.forecast_period.astype(str), y=per.sig_bh, marker_color="#2D3748",
                         name="BH q < 0.05"), 1, 2)
    fig.add_hline(y=0.05, line=dict(color="#A0AEC0", dash="dash"), row=1, col=2)
    fig.update_xaxes(title_text="drift score (energy distance, normalised units)", row=1, col=1)
    fig.update_yaxes(title_text="series", row=1, col=1)
    fig.update_yaxes(title_text="share of series (BH q < 0.05)", tickformat=".0%", row=1, col=2)
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.18), margin=dict(t=120), title=title(
        f"M4 {g} · How much does each series drift in its forecast period?",
        "score = energy distance between the forecast period's values and the input window (both in the input's "
        "normalised units); p-value = rank of the score among the same score for earlier blocks of the SAME series "
        "(conformal), Benjamini-Hochberg over series. Dashed = tercile borders (small / medium / large drift)"))
    save(fig, out_dir, "12_drift_series", 1400, 560)
    # (b) train/test shift of the window features (population)
    tr = feat_store[folds[-1]][0]
    cols = list(tr.columns)
    ncol = 4
    nrow = -(-len(cols) // ncol)
    fig = make_subplots(rows=nrow, cols=ncol, subplot_titles=cols, horizontal_spacing=0.06, vertical_spacing=0.16)
    shift = []
    for i, c in enumerate(cols):
        a, b = tr[c].values, te_m[c].values
        lo, hi = np.percentile(np.r_[a, b], [1, 99])
        if hi <= lo:
            continue
        grid = np.linspace(lo, hi, 200)
        try:
            fa, fb = gaussian_kde(a)(grid), gaussian_kde(b)(grid)
        except Exception:
            continue
        ovl = float(np.trapezoid(np.minimum(fa, fb), grid))
        shift.append({"feature": c, "overlap (OVL)": ovl, "train median": float(np.median(a)), "test median": float(np.median(b))})
        r, cc = i // ncol + 1, i % ncol + 1
        fig.add_trace(go.Scatter(x=grid, y=fa, fill="tozeroy", name="training windows (last fold)", legendgroup="a",
                                 showlegend=(i == 0), line=dict(color="#2B6CB0")), r, cc)
        fig.add_trace(go.Scatter(x=grid, y=fb, fill="tozeroy", name="test inputs", legendgroup="b", showlegend=(i == 0),
                                 line=dict(color="#C53030")), r, cc)
        fig.layout.annotations[i].text = f"{c} · OVL {ovl:.2f}"
    pd.DataFrame(shift).round(4).to_csv(os.path.join(out_dir, "train_test_shift.csv"), index=False)
    fig.update_layout(legend=dict(orientation="h", y=-0.08), margin=dict(t=120), title=title(
        f"M4 {g} · Train/test shift of the input windows (no model)",
        "scale-free features of the training windows of the last fold vs the test inputs. OVL = overlap of the two "
        "densities (1 = identical)"))
    save(fig, out_dir, "13_train_test_shift", 1400, 260 * nrow + 160)
    # (c) population tests + domain classifier
    fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6], horizontal_spacing=0.16,
                        subplot_titles=["domain classifier AUC per fold (0.5 = no drift)",
                                        "what gives the test windows away (SHAP of the classifier)"])
    for kind, col_ in (("covariate", "#2B6CB0"), ("joint", "#C05621")):
        d = CL[CL.kind == kind]
        gg = G[G.kind == kind].set_index("fold")
        fig.add_trace(go.Bar(x=[f"fold {f}" for f in d.fold], y=d.auc, marker_color=col_,
                             name="inputs only" if kind == "covariate" else "inputs + future",
                             text=[f"p={p:.2f}<br>MMD p={gg.loc[f, 'p_mmd']:.2f}" for f, p in zip(d.fold, d.p)],
                             textposition="outside"), 1, 1)
    fig.add_hline(y=0.5, line=dict(color="#A0AEC0", dash="dash"), row=1, col=1)
    if len(SH):
        s = SH.groupby(["kind", "feature"]).mean_abs_shap.mean().reset_index()
        for kind, col_ in (("covariate", "#2B6CB0"), ("joint", "#C05621")):
            d = s[s.kind == kind].sort_values("mean_abs_shap")
            fig.add_trace(go.Bar(y=d.feature, x=d.mean_abs_shap, orientation="h", marker_color=col_, showlegend=False), 1, 2)
    fig.update_yaxes(range=[0, 1.15], row=1, col=1)
    fig.update_xaxes(title_text="mean |SHAP| (log-odds of 'test')", row=1, col=2)
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.18), margin=dict(t=120, l=60), title=title(
        f"M4 {g} · Population drift: training windows vs test windows",
        "gradient-boosted trees on scale-free window features, 5-fold CV, p from label permutations (series are "
        "independent); MMD p from permutations. 'Inputs + future' clearly above 'inputs only' = the relation "
        "past → future changed (concept drift)"))
    save(fig, out_dir, "14_population_drift", 1400, 600)
    # (d) error vs drift / novelty, terciles
    fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.36, 0.28], horizontal_spacing=0.08,
                        subplot_titles=["sMAPE vs drift score (binned)", "sMAPE vs novelty of the input (binned)",
                                        "sMAPE per drift tercile"])
    for mdl in models:
        y = W[f"smape_{mdl}"].values
        for j, xcol in ((1, "drift_score"), (2, "novelty")):
            x = W[xcol].values
            if xcol == "drift_score":
                x = np.maximum(x, 1e-4)
            ok = np.isfinite(x) & np.isfinite(y)
            qs = np.unique(np.nanquantile(x[ok], np.linspace(0, 1, 21)))
            idx = np.clip(np.searchsorted(qs, x[ok], side="right") - 1, 0, len(qs) - 2)
            bx = [x[ok][idx == b].mean() for b in range(len(qs) - 1) if (idx == b).any()]
            by = [y[ok][idx == b].mean() for b in range(len(qs) - 1) if (idx == b).any()]
            be = [1.96 * y[ok][idx == b].std() / np.sqrt((idx == b).sum()) for b in range(len(qs) - 1) if (idx == b).any()]
            fig.add_trace(go.Scatter(x=bx, y=by, mode="lines+markers", error_y=dict(type="data", array=be),
                                     line=dict(color=color(mdl), width=2.5), name=label(mdl), legendgroup=mdl,
                                     showlegend=(j == 1)), 1, j)
        order = ["small drift", "medium drift", "large drift"]
        gm = W.groupby("drift_bin")[f"smape_{mdl}"].mean().reindex(order)
        fig.add_trace(go.Bar(x=order, y=gm.values, marker_color=color(mdl), showlegend=False, legendgroup=mdl), 1, 3)
    if len(T):
        for r in T[(T.metric == "smape") & (T.drift != "all series")].itertuples():
            fig.add_annotation(x=r.drift, y=max(r.GELU, r.KANB), text=f"{r._9:+.1f}%<br>p={r._13:.2g}",
                               showarrow=False, yshift=22, row=1, col=3, font=dict(size=10))
    fig.update_xaxes(title_text="drift score (energy distance, log scale)", type="log", row=1, col=1)
    fig.update_xaxes(title_text="novelty (share of training windows less unusual)", row=1, col=2)
    fig.update_yaxes(title_text="sMAPE (mean over fold models and seeds)", row=1, col=1)
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.2), margin=dict(t=130), title=title(
        f"M4 {g} · Which model copes better with drift and novelty?",
        "every test series, sMAPE averaged over the fold models and seeds; 20 equal-count bins with 95% CIs. Right: "
        "sMAPE per drift tercile, KANB's change vs GELU and the Wilcoxon signed-rank p (Holm-corrected)"))
    save(fig, out_dir, "15_error_vs_drift", 1600, 600)
    # (e) residual-based concept drift
    if len(RS):
        fig = go.Figure()
        for mdl in models:
            d = RS[RS.model == mdl]
            fig.add_trace(go.Box(y=d.energy, name=label(mdl), marker_color=color(mdl), boxpoints="all", jitter=0.4,
                                 pointpos=0, hovertext=[f"fold {f} seed {s}: p={p:.3f}" for f, s, p in zip(d.fold, d.seed, d.p)]))
        fig.update_yaxes(title_text="energy distance: test vs validation scaled errors")
        fig.update_layout(showlegend=False, margin=dict(t=120), title=title(
            f"M4 {g} · Did the relation each model learned break on the test period?",
            "per fold model and seed, on the same series: mean scaled error (error / MASE scale) of the validation "
            "block vs of the test period; permutation p-values in the hover. Lower = errors behave as on validation"))
        save(fig, out_dir, "16_residual_concept_drift", 1100, 540)
    # (f) calendar leak
    cal = D["spec"].get("calendar", {})
    if cal.get("folds"):
        c = pd.DataFrame(cal["folds"])
        fig = go.Figure(go.Bar(x=c.fold.astype(str), y=c.iloc[:, 1], marker_color="#6B46C1",
                               text=[f"{v:.0%}" if v is not None else "" for v in c.iloc[:, 1]], textposition="outside"))
        fig.update_yaxes(tickformat=".0%", range=[0, 1], title_text="share of (training value, other series) pairs")
        fig.update_xaxes(title_text="fold (and the M4 test)")
        fig.update_layout(margin=dict(t=120), title=title(
            f"M4 {g} · Calendar overlap: the cross-series leak of M4",
            "share of training values that lie, in calendar time, after the first predicted value of ANOTHER series.<br>"
            "Calendar-CV folds should be 0 (no leak); the paper protocol (full histories, M4 test) shows the leak. "
            f"{cal.get('unparsable_dates', 0)} series with unreadable dates left out"))
        save(fig, out_dir, "18_calendar_overlap", 1000, 520)


# =========================================================================================================
# 5. Worst series
# =========================================================================================================
def analysis_gallery(R_by_model, D, out_dir, ctx, opt):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    models = list(R_by_model)
    ens = np.nanmean([ctx["series_smape"][m] for m in models], axis=0)
    worst = list(np.argsort(-np.nan_to_num(ens, nan=-np.inf))[:opt.gallery])
    nrow = -(-len(worst) // 2)
    titles = []
    for i in worst:
        extra = f" · drift {ctx['drift_series'][i]:.2f}" if "drift_series" in ctx else ""
        titles.append(f"{D['ids'][i]} (length {len(D['train'][i])})<br><sup>" +
                      " · ".join(f"{label(m)} sMAPE {ctx['series_smape'][m][i]:.1f}" for m in models) + extra + "</sup>")
    fig = make_subplots(rows=nrow, cols=2, subplot_titles=titles, vertical_spacing=0.13, horizontal_spacing=0.06)
    for k, i in enumerate(worst):
        r, c = k // 2 + 1, k % 2 + 1
        x, y = D["x_test"][i], D["y_test"][i]
        L, H = len(x), len(y)
        fig.add_trace(go.Scatter(x=list(range(-L + 1, 1)), y=x, mode="lines", line=dict(color="#718096"), name="input",
                                 legendgroup="in", showlegend=(k == 0)), r, c)
        fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=y, mode="lines", line=dict(color="black", width=2),
                                 name="truth", legendgroup="tr", showlegend=(k == 0)), r, c)
        for m in models:
            hs = list(range(1, H + 1))
            fig.add_trace(go.Scatter(x=hs, y=ctx["series_pred"][m][i], mode="lines", line=dict(color=color(m), width=2),
                                     name=label(m), legendgroup=m, showlegend=(k == 0)), r, c)
            lo, hi = ctx["series_pred_lo"][m][i], ctx["series_pred_hi"][m][i]
            fig.add_trace(go.Scatter(x=hs + hs[::-1], y=list(hi) + list(lo[::-1]), fill="toself", fillcolor=LIGHT.get(m),
                                     line=dict(width=0), hoverinfo="skip", showlegend=False, legendgroup=m), r, c)
        fig.add_vline(x=0.5, line=dict(color="#A0AEC0", dash="dash"), row=r, col=c)
    fig.update_xaxes(title_text="steps (≤ 0 input, > 0 forecast)")
    fig.update_layout(legend=dict(orientation="h", y=-0.06), margin=dict(t=120), title=title(
        f"M4 {D['group']} · Worst series: what the models miss",
        "the series with the largest sMAPE (mean of both models); lines = mean forecast over fold models and seeds, "
        "bands = their range"))
    save(fig, out_dir, "17_worst_series", 1500, 330 * nrow + 160)
    return {"worst_series": [str(D["ids"][i]) for i in worst]}


# =========================================================================================================
# 6. The leak experiment (calendar CV only)
# =========================================================================================================
def analysis_leak(R_by_model, D, out_dir, ctx, opt):
    """Effect of M4's cross-series leak on the validation error, per model: for every fold and seed,
    leaky - clean sMAPE on the same series (half A scored while half B keeps its values after the cut-off, and
    the reverse). Placebo = the same with shuffled calendar positions. Paired Wilcoxon over series, bootstrap
    CIs, Holm; and whether KANB profits more from the leak than GELU (difference in differences)."""
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    g = D["group"]
    rng = np.random.default_rng(opt.seed + 6)
    models = list(R_by_model)
    N = len(D["train"])
    rows = []
    for m in models:
        mdir = os.path.join(ctx["run_root"], m)
        for js in sorted(glob.glob(os.path.join(mdir, "fold*__seed*.json"))):
            r = json.load(open(js))
            f, s = r["fold"], r["seed"]
            clean = np.load(js[:-5] + "_test.npz") if os.path.exists(js[:-5] + "_test.npz") else None
            for real, base_name, pair in (("real", None, ("leakA", "leakB")),
                                          ("placebo", "placebo", ("placebo_leakA", "placebo_leakB"))):
                if base_name is None:
                    if clean is None:
                        continue
                    b_idx, b_sm = clean["val_idx"], clean["val_smape"]
                else:
                    pb = os.path.join(mdir, "leak", f"fold{f}__seed{s}__{base_name}.npz")
                    if not os.path.exists(pb):
                        continue
                    zb = np.load(pb)
                    b_idx, b_sm = zb["val_idx"], zb["val_smape"]
                base = dict(zip(b_idx.tolist(), b_sm.tolist()))
                for v in pair:
                    pv = os.path.join(mdir, "leak", f"fold{f}__seed{s}__{v}.npz")
                    if not os.path.exists(pv):
                        continue
                    zv = np.load(pv)
                    for i, sm in zip(zv["val_idx"].tolist(), zv["val_smape"].tolist()):
                        if i in base:
                            rows.append({"model": m, "fold": f, "seed": s, "kind": real, "series": i,
                                         "clean": base[i], "leaky": sm, "effect": sm - base[i]})
    if not rows:
        print("  leak: no leak-experiment runs found (run run_leak9 first)")
        return {"messages": {"leak": "no leak-experiment runs yet"}}
    E = pd.DataFrame(rows)
    E.to_csv(os.path.join(out_dir, "leak_effect_per_series.csv"), index=False)
    per = E.groupby(["model", "kind", "series"]).agg(effect=("effect", "mean"), clean=("clean", "mean")).reset_index()
    tests = []
    for m in models:
        for kind in ("real", "placebo"):
            d = per[(per.model == m) & (per.kind == kind)]
            if len(d):
                r = paired_test(d.effect.values, rng)
                tests.append({"test": f"{label(m)}: leak effect ({kind})", "series": r["n"], "mean effect": r["mean"],
                              "95% CI low": r.get("mean_lo"), "95% CI high": r.get("mean_hi"),
                              "effect % of clean sMAPE": 100 * r["mean"] / d.clean.mean() if d.clean.mean() else np.nan,
                              "share of series helped by the leak": r.get("kanb_better_share"), "Wilcoxon p": r["p"]})
        a = per[(per.model == m) & (per.kind == "real")].set_index("series").effect
        b = per[(per.model == m) & (per.kind == "placebo")].set_index("series").effect
        both = a.index.intersection(b.index)
        if len(both) >= 6:
            r = paired_test((a[both] - b[both]).values, rng)
            tests.append({"test": f"{label(m)}: real minus placebo", "series": r["n"], "mean effect": r["mean"],
                          "95% CI low": r.get("mean_lo"), "95% CI high": r.get("mean_hi"),
                          "effect % of clean sMAPE": np.nan, "share of series helped by the leak": r.get("kanb_better_share"),
                          "Wilcoxon p": r["p"]})
    if len(models) == 2:
        a = per[(per.model == models[1]) & (per.kind == "real")].set_index("series").effect
        b = per[(per.model == models[0]) & (per.kind == "real")].set_index("series").effect
        both = a.index.intersection(b.index)
        if len(both) >= 6:
            r = paired_test((a[both] - b[both]).values, rng)
            tests.append({"test": "KANB minus GELU leak effect (does KANB profit more?)", "series": r["n"],
                          "mean effect": r["mean"], "95% CI low": r.get("mean_lo"), "95% CI high": r.get("mean_hi"),
                          "effect % of clean sMAPE": np.nan,
                          "share of series helped by the leak": r.get("kanb_better_share"), "Wilcoxon p": r["p"]})
    T = pd.DataFrame(tests)
    T["Holm p"] = holm(T["Wilcoxon p"].values)
    T["verdict"] = np.where(T["Holm p"] < 0.05, np.where(T["mean effect"] < 0, "lower error (leak helps)",
                                                         "higher error"), "no significant effect")
    T.to_csv(os.path.join(out_dir, "leak_effect_tests.csv"), index=False)
    cut = D["spec"].get("cal", {}).get("cutoffs", [])
    fig = make_subplots(rows=1, cols=3, column_widths=[0.32, 0.36, 0.32], horizontal_spacing=0.08,
                        subplot_titles=["mean leak effect (leaky − clean sMAPE) with 95% CI",
                                        "per fold (calendar cut-off)", "distribution over series (real leak)"])
    for kind, pat in (("real", ""), ("placebo", "/")):
        for m in models:
            t = T[T.test == f"{label(m)}: leak effect ({kind})"]
            if not len(t):
                continue
            t = t.iloc[0]
            fig.add_trace(go.Bar(x=[label(m)], y=[t["mean effect"]], name=f"{kind}", legendgroup=kind,
                                 showlegend=(m == models[0]), marker=dict(color=color(m), pattern_shape=pat,
                                                                          opacity=1 if kind == "real" else 0.55),
                                 error_y=dict(type="data", symmetric=False, array=[t["95% CI high"] - t["mean effect"]],
                                              arrayminus=[t["mean effect"] - t["95% CI low"]]),
                                 text=[f"p={t['Holm p']:.2g}"], textposition="outside"), 1, 1)
        for m in models:
            d = E[(E.model == m) & (E.kind == kind)].groupby("fold").effect.agg(["mean", "sem"]).reset_index()
            if len(d):
                xs = [cut[f][:10] if f < len(cut) else str(f) for f in d.fold]
                fig.add_trace(go.Scatter(x=xs, y=d["mean"], error_y=dict(type="data", array=1.96 * d["sem"].fillna(0)),
                                         mode="lines+markers", name=f"{label(m)} · {kind}", showlegend=False,
                                         line=dict(color=color(m), dash="solid" if kind == "real" else "dot")), 1, 2)
    for m in models:
        d = per[(per.model == m) & (per.kind == "real")]
        lo, hi = np.nanpercentile(d.effect, [1, 99]) if len(d) else (0, 1)
        fig.add_trace(go.Violin(y=np.clip(d.effect, lo, hi), name=label(m), line_color=color(m), box_visible=True,
                                meanline_visible=True, showlegend=False), 1, 3)
    for c in (1, 2, 3):
        fig.add_hline(y=0, line=dict(color="#4A5568"), row=1, col=c)
    fig.update_yaxes(title_text="sMAPE points (negative = the leak lowers the error)", row=1, col=1)
    fig.update_xaxes(title_text="fold cut-off", row=1, col=2)
    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.15), margin=dict(t=130), title=title(
        f"M4 {g} · How much does M4's cross-series leak change the error?",
        "same series, same amount of training data: half of the series keeps its values after the fold's calendar "
        "cut-off (the leak), the other half is scored, then swapped.<br>Placebo (hatched) = the same with shuffled "
        "calendar positions. Wilcoxon signed-rank over series, Holm-corrected"))
    save(fig, out_dir, "19_leak_effect", 1600, 600)
    msgs = {r.test: f"{r._3:+.3f} sMAPE points (p Holm = {r._9:.3g}) → {r.verdict}" for r in T.itertuples()}
    return {"messages": msgs, "tests": T.round(4).to_dict(orient="records")}


# =========================================================================================================
# Main
# =========================================================================================================
def calendar_bins(info, group, train, n_bins=8):
    starts = CVM.parse_dates(info, group)
    step = CVM.STEP[group]
    when = [CVM.date_at(s, step, len(x)) if not pd.isna(s) else pd.NaT for s, x in zip(starts, train)]
    when = pd.to_datetime(pd.Series(when), errors="coerce")
    ok = when.notna()
    out = np.array(["unknown"] * len(when), dtype=object)
    if ok.sum() > 0:
        q = pd.qcut(when[ok].map(pd.Timestamp.toordinal), q=min(n_bins, ok.sum()), duplicates="drop")
        lab = {c: f"{pd.Timestamp.fromordinal(int(c.left) + 1):%Y-%m}–{pd.Timestamp.fromordinal(int(c.right)):%Y-%m}"
               for c in q.cat.categories}
        out[ok.values] = [lab[c] for c in q]
    return out


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--group", required=True)
    p.add_argument("--folds_file", required=True)
    p.add_argument("--run_root", required=True, help="runs/<group> (contains <model>/fold*__seed*.json)")
    p.add_argument("--out_dir", required=True)
    p.add_argument("--root_path", default="./dataset/m4")
    p.add_argument("--models", default="TimesNet,TimesNet_KANB")
    p.add_argument("--analyses", default="all")
    p.add_argument("--folds", default="")
    p.add_argument("--seeds", default="")
    p.add_argument("--n_segments", type=int, default=8)
    p.add_argument("--shap_series", type=int, default=300)
    p.add_argument("--shap_samples", type=int, default=16)
    p.add_argument("--background", type=int, default=64)
    p.add_argument("--grad_series", type=int, default=100)
    p.add_argument("--grad_samples", type=int, default=32)
    p.add_argument("--grad_seeds", default="2021")
    p.add_argument("--faith_series", type=int, default=100)
    p.add_argument("--cam_series", type=int, default=100)
    p.add_argument("--period_series", type=int, default=2000)
    p.add_argument("--ablation_series", type=int, default=500)
    p.add_argument("--ablation_repeats", type=int, default=2)
    p.add_argument("--max_null_blocks", type=int, default=30)
    p.add_argument("--pop_series", type=int, default=3000)
    p.add_argument("--mmd_rows", type=int, default=1000)
    p.add_argument("--perms", type=int, default=199)
    p.add_argument("--clf_rows", type=int, default=3000)
    p.add_argument("--clf_iter", type=int, default=100)
    p.add_argument("--clf_perms", type=int, default=29)
    p.add_argument("--knn", type=int, default=5)
    p.add_argument("--gallery", type=int, default=6)
    p.add_argument("--seed", type=int, default=2021)
    opt = p.parse_args()
    opt.grad_seeds = [int(s) for s in opt.grad_seeds.split(",") if s]
    todo = {"periods", "shap", "gradshap", "gradshap_local", "gradcam", "faith", "ablation", "drift", "gallery",
            "leak"} if opt.analyses == "all" else \
        set(opt.analyses.split(","))
    if "faith" in todo:
        todo.add("shap")
    spec = json.load(open(opt.folds_file))
    group = spec["group"]
    assert group == opt.group, "folds file of another group"
    folds = [int(f) for f in opt.folds.split(",") if f] or None
    seeds = [int(s) for s in opt.seeds.split(",") if s] or None
    models = [m for m in opt.models.split(",") if m]
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    out_dir = opt.out_dir
    os.makedirs(out_dir, exist_ok=True)
    runs = {m: find_runs(opt.run_root, m, folds, seeds) for m in models}
    runs = {m: r for m, r in runs.items() if r}
    if len(runs) >= 2:
        keys = set.intersection(*[{(r["fold"], r["seed"]) for r in rs} for rs in runs.values()])
        for m in runs:
            dropped = [(r["fold"], r["seed"]) for r in runs[m] if (r["fold"], r["seed"]) not in keys]
            if dropped:
                print(f"note: {m}: folds/seeds {dropped} not finished for every model - left out")
            runs[m] = [r for r in runs[m] if (r["fold"], r["seed"]) in keys]
        runs = {m: r for m, r in runs.items() if r}
    if not runs:
        raise SystemExit(f"no finished runs in {opt.run_root}")
    t_start = time.time()
    train, test, ids, info = CVM.load_group(opt.root_path, group)
    H = CVM.HORIZON[group]
    L = 2 * H
    x_test, _ = CVM.last_window(train, L)
    D = {"group": group, "train": train, "y_test": np.stack(test), "x_test": x_test, "ids": ids, "spec": spec,
         "H": H, "L": L, "freq": CVM.FREQ[group]}
    R_by_model = {m: [Runner(r, D, device) for r in rs] for m, rs in runs.items()}
    rng = np.random.default_rng(opt.seed)
    N = len(train)
    ctx = {"test_year_bin": calendar_bins(info, group, train)}
    for k in ("series_smape", "series_mase", "series_pred", "series_pred_lo", "series_pred_hi"):
        ctx[k] = {}
    per_run = {}
    elig_all = np.ones(N, bool)
    elig_any = np.zeros(N, bool)
    import warnings
    warnings.filterwarnings("ignore", message="Mean of empty slice")
    warnings.filterwarnings("ignore", message="All-NaN slice encountered")
    for m, rs in runs.items():
        Z = [dict(np.load(r["npz"])) for r in rs]
        for z in Z:                           # calendar CV: a series counts only for leak-free fold models
            el = z.get("test_eligible", np.ones(N, bool)).astype(bool)
            elig_all &= el
            elig_any |= el
            z["smape"] = np.where(el, z["smape"], np.nan)
            z["mase"] = np.where(el, z["mase"], np.nan)
            z["pred"] = np.where(el[:, None], z["pred"], np.nan)
        ctx["series_smape"][m] = np.nanmean([z["smape"] for z in Z], axis=0)
        ctx["series_mase"][m] = np.nanmean([z["mase"] for z in Z], axis=0)
        P = np.stack([z["pred"] for z in Z])
        ctx["series_pred"][m], ctx["series_pred_lo"][m], ctx["series_pred_hi"][m] = \
            np.nanmean(P, 0), np.nanmin(P, 0), np.nanmax(P, 0)
        per_run[m] = {(r["fold"], r["seed"]): (z["smape"], z["mase"]) for r, z in zip(rs, Z)}
    pool = np.where(elig_all)[0] if elig_all.sum() >= 30 else np.where(elig_any)[0]
    ctx["scored"] = elig_any
    ctx["period_sample"] = np.sort(rng.choice(pool, min(len(pool), opt.period_series), replace=False))
    ctx["shap_sample"] = np.sort(rng.choice(pool, min(len(pool), opt.shap_series), replace=False))
    ctx["ablation_sample"] = np.sort(rng.choice(pool, min(len(pool), opt.ablation_series), replace=False))
    if len(runs) == 2:
        m0, m1 = models[0], models[1]
        common = sorted(set(per_run[m0]) & set(per_run[m1]))
        diffs = {mt: np.array([per_run[m1][k][j] - per_run[m0][k][j] for k in common]) for j, mt in
                 enumerate(("smape", "mase"))}
        ctx["series_diff_smape"] = np.nanmean(diffs["smape"], 0)
        ctx["series_diff_mase"] = np.nanmean(diffs["mase"], 0)
        folds_present = sorted({k[0] for k in common})

        def fold_bin_wins(sel, metric):
            wins = 0
            for f in folds_present:
                dd = np.nanmean([diffs[metric][i] for i, k in enumerate(common) if k[0] == f], axis=0)[sel]
                wins += int(np.nanmean(dd) < 0)
            return f"{wins}/{len(folds_present)}"
        ctx["fold_bin_wins"] = fold_bin_wins
    summary = {"group": group, "H": H, "models": list(runs), "runs": {m: len(r) for m, r in runs.items()},
               "test_series": N, "test_series_scored": int(elig_any.sum()), "cv": spec.get("cv", "series"),
               "mean_smape": {m: float(np.nanmean(v)) for m, v in ctx["series_smape"].items()},
               "mean_mase": {m: float(np.nanmean(v)) for m, v in ctx["series_mase"].items()}}
    order = ["periods", "drift", "shap", "gradshap", "gradshap_local", "gradcam", "faith", "ablation", "gallery", "leak"]
    ctx["run_root"] = opt.run_root
    fns = {"leak": analysis_leak, "gradshap_local": analysis_gradshap_local, "gradcam": analysis_gradcam,
           "periods": analysis_periods, "shap": analysis_shap, "gradshap": analysis_gradshap, "faith": analysis_faith,
           "ablation": analysis_ablation, "drift": analysis_drift, "gallery": analysis_gallery}
    for a in order:
        if a not in todo:
            continue
        t0 = time.time()
        print(f"== {a} ==", flush=True)
        summary[a] = fns[a](R_by_model, D, out_dir, ctx, opt)
        summary.setdefault("minutes", {})[a] = round((time.time() - t0) / 60, 2)
    sp = os.path.join(out_dir, "xai_m4_summary.json")
    if os.path.exists(sp):
        old = json.load(open(sp))
        mins = old.get("minutes", {})
        old.update({k: v for k, v in summary.items() if k != "minutes"})
        mins.update(summary.get("minutes", {}))
        summary = old
        summary["minutes"] = mins
    summary["figures"] = sorted(set(summary.get("figures", [])) | set(FIGURES))
    summary["total_minutes"] = round((time.time() - t_start) / 60, 2)
    conv = lambda o: o.item() if hasattr(o, "item") else str(o)
    with open(sp, "w") as fh:
        json.dump(summary, fh, indent=1, default=conv)
    print("XAI_M4_SUMMARY " + json.dumps({k: summary[k] for k in summary if k != "figures"}, default=conv)[:4000])


if __name__ == "__main__":
    main()

Writing xai_m4.py


In [ ]:
# ----------------------------- 8.5 Helpers ------------------------------

def s9_env():
    """Environment of the training / analysis processes. Colab sets MPLBACKEND to its inline backend, which the
    TSLib environment does not have (matplotlib then fails on import), so the scripts get the plain 'Agg' backend."""
    return dict(os.environ, CUBLAS_WORKSPACE_CONFIG = ":4096:8", MPLBACKEND = "Agg")


S9_FIXED = {"task_name", "model", "data", "seasonal_patterns", "seq_len", "label_len", "pred_len", "enc_in", "dec_in",
            "c_out", "features", "root_path", "is_training"}       # set by M4 / the protocol, not changeable


def s9_overrides(group, model):
    """The settings changed for this group and model (S9_HP, S9_HP_MODEL), in the order they are applied."""
    hp = {}
    for d in (S9_HP.get("all", {}), S9_HP.get(group, {}), S9_HP_MODEL.get(model, {}).get("all", {}),
              S9_HP_MODEL.get(model, {}).get(group, {})):
        hp.update(d or {})
    bad = set(hp) & S9_FIXED
    assert not bad, f"these settings are fixed by M4 / the protocol and cannot be changed: {sorted(bad)}"
    return hp


def s9_tag(group):
    """Name of the results folder of the current settings of this group (both models): 'paper' = no change."""
    import hashlib
    hp = {m: s9_overrides(group, m) for m in S9_MODELS}
    if not any(hp.values()):
        return "paper"
    key = json.dumps(hp, sort_keys = True)
    return "hp_" + hashlib.md5(key.encode()).hexdigest()[:8]


def s9_base(group):
    tag = s9_tag(group)
    base = os.path.join(S9_ROOT, S9_CV) if tag == "paper" else os.path.join(S9_ROOT, S9_CV, tag)
    if tag != "paper":                       # describe the settings next to the results
        os.makedirs(base, exist_ok = True)
        path = os.path.join(base, f"settings_{group}.json")
        if not os.path.exists(path):
            json.dump({m: s9_overrides(group, m) for m in S9_MODELS}, open(path, "w"), indent = 1)
    return base


def s9_command(group, model):
    """The paper command of Section 4 for this M4 group, with the model swapped and the Section 8 settings."""
    cmd = [c for c in PAPER_COMMANDS if R.arg(c, "seasonal_patterns").strip("'\"") == group]
    assert cmd, f"unknown M4 group: {group}"
    cmd = R.set_arg(cmd[0], "model", model)
    for k, v in s9_overrides(group, model).items():
        cmd = R.set_arg(cmd, k, v)
    if SMOKE:
        cmd = R.set_arg(cmd, "train_epochs", 1)
    return cmd


def s9_show_settings(group):
    """The settings both models will use for this group (changes vs the paper marked)."""
    keys = ["e_layers", "d_model", "d_ff", "top_k", "num_kernels", "dropout", "learning_rate", "batch_size",
            "train_epochs", "patience", "lradj", "loss"]
    rows = {}
    for m in S9_MODELS:
        cmd, hp = s9_command(group, m), s9_overrides(group, m)
        keys += [k for k in hp if k not in keys]
        dflt = {"num_kernels": "6", "dropout": "0.1", "patience": "3", "lradj": "type1", "e_layers": "2",
                "top_k": "5", "train_epochs": "10", "learning_rate": "0.0001", "batch_size": "32", "loss": "MSE"}
        rows[m] = {k: (f"{R.arg(cmd, k)} ✱" if k in hp else
                       (R.arg(cmd, k) or f"{dflt.get(k, '?')} (TSLib default)")) for k in keys}
    print(f"settings for {group} (✱ = changed from the paper), results folder: {s9_base(group)}")
    display(pd.DataFrame(rows))


def s9_same_settings(stem, cmd):
    """A finished run is reused only if it was trained with the current settings."""
    if not os.path.exists(stem + ".json"):
        return True
    saved = json.load(open(stem + ".json")).get("tslib_args", {})
    for i, a in enumerate(cmd):
        if not a.startswith("--") or i + 1 >= len(cmd):
            continue
        k, v = a[2:], cmd[i + 1].strip("'\"")
        if k in saved and k not in S9_FIXED:
            sv = saved[k]
            try:
                same = float(v) == float(sv)
            except (TypeError, ValueError):
                same = str(v) == str(sv)
            if not same:
                raise RuntimeError(f"{stem}.json was trained with {k}={sv}, the current settings say {k}={v}. "
                                   "Change S9_HP back, or delete that run folder")
    return True


def s9_folds_file(group):
    return os.path.join(S9_ROOT, S9_CV, "folds", f"{group}.json")


def s9_run_dir(group, model):
    return os.path.join(s9_base(group), "runs", group, model)


def s9_make_folds(group):
    path = s9_folds_file(group)
    if not os.path.exists(path):
        out = subprocess.run([PY, "-u", "cv_m4.py", "--make_folds", "--folds_file", path, "--cv", S9_CV,
                              "--n_folds", str(S9_FOLDS[group]), "--gap_h", str(S9_GAP_H),
                              "--min_train_h", str(S9_MIN_TRAIN_H), "--"] + s9_command(group, S9_MODELS[0]),
                             cwd = REPO, text = True, capture_output = True, env = s9_env())
        if out.returncode != 0:
            print(out.stdout[-2000:], out.stderr[-3000:])
            raise RuntimeError("could not create the folds")
    return json.load(open(path))


def s9_show_folds(spec):
    H, gap = spec["H"], spec["gap"]
    rows = [{"fold": f["fold"], "cut-off": f.get("cutoff", "")[:10], "validated series": f["series"],
             "training series": f.get("training_series", f["series"]),
             "validation block (per series)": f["validation_block"], "gap": f"{gap} values",
             "median training length": f["median_train_length"],
             "test series scored (leak-free)": f.get("test_series_scored", spec["series"])} for f in spec["folds"]]
    display(pd.DataFrame(rows).set_index("fold"))
    cal = spec.get("calendar", {})
    if cal.get("folds"):
        leak = {str(r["fold"]): r["share of training values after other series' target start"] for r in cal["folds"]}
        print("calendar overlap (share of training values later than other series' first predicted value): "
              + ", ".join(f"{k}: {v:.0%}" if v is not None else f"{k}: -" for k, v in leak.items()))


def run_cv9(group, models = None, seeds = None, folds = None):
    """Trains every model x fold x seed of one M4 group. Finished runs are skipped (resumable)."""
    if not S9_ACTIVE:
        print("Section 8 is switched off (S9_ACTIVE = False)")
        return
    models = models or S9_MODELS
    seeds = seeds or S9_SEEDS
    spec = s9_make_folds(group)
    print(f"== M4 {group}: {len(models)} model(s) x {spec['n_folds']} folds x {len(seeds)} seed(s); "
          f"{spec['series']} test series, horizon {spec['H']} ==")
    s9_show_folds(spec)
    s9_show_settings(group)
    for model in models:
        for seed in seeds:
            for fold in (folds if folds is not None else range(spec["n_folds"])):
                out_dir = s9_run_dir(group, model)
                stem = os.path.join(out_dir, f"fold{fold}__seed{seed}")
                if os.path.exists(stem + ".json") and os.path.exists(stem + "_test.npz") \
                        and s9_same_settings(stem, s9_command(group, model)):
                    continue
                os.makedirs(out_dir, exist_ok = True)
                log_path = os.path.join(s9_base(group), "logs", group, f"{model}_fold{fold}_seed{seed}.log")
                os.makedirs(os.path.dirname(log_path), exist_ok = True)
                print(f"   {model:14s} fold {fold} seed {seed} ...", end = "", flush = True)
                t0 = time.time()
                with open(log_path, "w") as log:
                    proc = subprocess.run([PY, "-u", "cv_m4.py", "--folds_file", s9_folds_file(group), "--fold", str(fold),
                                           "--seed", str(seed), "--out_dir", out_dir, "--"] + s9_command(group, model),
                                          cwd = REPO, text = True, stdout = log, stderr = subprocess.STDOUT,
                                          env = s9_env())
                if proc.returncode != 0 or not os.path.exists(stem + ".json"):
                    print(f" FAILED - see {log_path}")
                    print(open(log_path).read()[-2500:])
                    return
                r = json.load(open(stem + ".json"))
                print(f" sMAPE {r['smape']:.3f}  MASE {r['mase']:.3f}  OWA {r['owa']:.3f}  ({r['epochs']} epochs, "
                      f"{(time.time() - t0) / 60:.1f} min)")
    display(cv9_table(group))


def run_leak9(group, models = None, seeds = None, folds = None, placebo = None):
    """Leak experiment (calendar CV): for every model x fold x seed, retrains the fold twice with one half of the
    series keeping its values after the cut-off (A scored / B leaky, then swapped), and - with the placebo - three
    more times with shuffled calendar positions. Needs the clean fold models of run_cv9. Resumable."""
    if not S9_ACTIVE:
        print("Section 8 is switched off (S9_ACTIVE = False)")
        return
    assert S9_CV == "calendar", "the leak experiment needs S9_CV = 'calendar'"
    placebo = S9_PLACEBO if placebo is None else placebo
    variants = ["leakA", "leakB"] + (["placebo", "placebo_leakA", "placebo_leakB"] if placebo else [])
    models = models or S9_MODELS
    seeds = seeds or S9_SEEDS
    spec = s9_make_folds(group)
    print(f"== Leak experiment M4 {group}: {len(models)} model(s) x {spec['n_folds']} folds x {len(seeds)} seed(s) "
          f"x {len(variants)} variants ==")
    for model in models:
        for seed in seeds:
            for fold in (folds if folds is not None else range(spec["n_folds"])):
                out_dir = s9_run_dir(group, model)
                for v in variants:
                    stem = os.path.join(out_dir, "leak", f"fold{fold}__seed{seed}__{v}")
                    if os.path.exists(stem + ".json") and s9_same_settings(stem, s9_command(group, model)):
                        continue
                    log_path = os.path.join(s9_base(group), "logs", group, f"{model}_fold{fold}_seed{seed}_{v}.log")
                    os.makedirs(os.path.dirname(log_path), exist_ok = True)
                    print(f"   {model:14s} fold {fold} seed {seed} {v:14s} ...", end = "", flush = True)
                    t0 = time.time()
                    with open(log_path, "w") as log:
                        proc = subprocess.run([PY, "-u", "cv_m4.py", "--folds_file", s9_folds_file(group), "--fold",
                                               str(fold), "--seed", str(seed), "--variant", v, "--out_dir", out_dir,
                                               "--"] + s9_command(group, model),
                                              cwd = REPO, text = True, stdout = log, stderr = subprocess.STDOUT,
                                              env = s9_env())
                    if proc.returncode != 0 or not os.path.exists(stem + ".json"):
                        print(f" FAILED - see {log_path}")
                        print(open(log_path).read()[-2500:])
                        return
                    r = json.load(open(stem + ".json"))
                    print(f" validation sMAPE {r['val_smape']:.3f} ({(time.time() - t0) / 60:.1f} min)")
    print("done - run run_xai9(group, analyses='leak') to measure the effect")


def cv9_results(group = None):
    """Every finished run of the current CV, with the settings folder it belongs to ('paper' or 'hp_...')."""
    rows = []
    pats = [("paper", os.path.join(S9_ROOT, S9_CV, "runs", "*", "*", "fold*__seed*.json")),
            (None, os.path.join(S9_ROOT, S9_CV, "hp_*", "runs", "*", "*", "fold*__seed*.json"))]
    for tag, pat in pats:
        for js in glob.glob(pat):
            r = json.load(open(js))
            if group and r["group"] != group:
                continue
            t = tag or js.split(os.sep + "runs" + os.sep)[0].split(os.sep)[-1]
            rows.append({"settings": t, **{k: r.get(k) for k in ("group", "model", "fold", "seed", "smape", "mase", "owa",
                                                                  "val_smape", "n_test_scored", "epochs",
                                                                  "train_minutes", "n_train_series")}})
    return pd.DataFrame(rows)


def cv9_table(group = None):
    """Mean ± sd over folds x seeds per group and model, and the paired run-level comparison (same fold and
    seed for both models): KANB wins and a Wilcoxon signed-rank test over the pairs."""
    from scipy.stats import wilcoxon
    df = cv9_results(group)
    if not len(df):
        return pd.DataFrame()
    rows = []
    for (st, g), d0 in df.groupby(["settings", "group"]):
        row = {"settings": st, "group": g}
        for m in S9_MODELS:
            d = d0[d0.model == m]
            for k in ("smape", "mase", "owa"):
                row[f"{m} {k.upper()}"] = f"{d[k].mean():.3f} ± {d[k].std():.3f}" if len(d) else ""
            row[f"{m} runs"] = len(d)
        p = d0.pivot_table(index = ["fold", "seed"], columns = "model", values = "owa").dropna()
        if all(m in p for m in S9_MODELS) and len(p):
            diff = p[S9_MODELS[1]] - p[S9_MODELS[0]]
            row["KANB better OWA (pairs)"] = f"{int((diff < 0).sum())}/{len(diff)}"
            row["Wilcoxon p (OWA)"] = round(wilcoxon(diff).pvalue, 4) if len(diff) >= 6 and (diff != 0).any() else None
        rows.append(row)
    return pd.DataFrame(rows)


def run_xai9(group, analyses = "all", **overrides):
    """Section 8 analyses of one M4 group on its saved fold models (no training)."""
    if not S9_ACTIVE:
        print("Section 8 is switched off (S9_ACTIVE = False)")
        return
    opts = dict(S9_XAI, **overrides)
    out_dir = os.path.join(s9_base(group), "xai", group)
    cmd = [PY, "-u", "xai_m4.py", "--group", group, "--folds_file", s9_folds_file(group),
           "--run_root", os.path.join(s9_base(group), "runs", group), "--out_dir", out_dir,
           "--models", ",".join(S9_MODELS), "--analyses", analyses]
    for k, v in opts.items():
        cmd += [f"--{k}", str(v)]
    log_path = os.path.join(s9_base(group), "logs", group, "xai_m4.log")
    os.makedirs(os.path.dirname(log_path), exist_ok = True)
    print(f"== Section 9 analyses: M4 {group} ({analyses}) ==", flush = True)
    t0 = time.time()
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd = REPO, text = True, stdout = subprocess.PIPE, stderr = subprocess.STDOUT,
                                env = s9_env())
        for line in proc.stdout:
            log.write(line)
            if line.startswith(("==", "  SHAP", "  drift", "note")):
                print("  " + line.rstrip(), flush = True)
        proc.wait()
    if proc.returncode != 0:
        print(f"FAILED - see {log_path}")
        print(open(log_path).read()[-3000:])
        return
    print(f"done in {(time.time() - t0) / 60:.1f} min")
    show9(group, figures = False)


FIGURE_NOTES_9 = {
 "01": "**Periods.** How often TimesNet's FFT picks each period vs the series' own autocorrelation (seasonal period dotted).",
 "02": "**Periods over calendar time** (M4-info dates).",
 "03": "**Global importance** of each input segment, and per explained output.",
 "04": "**Segment × output.** Which part of the past drives which part of the forecast; right = KANB − GELU.",
 "05": "**Stability over calendar time.** Flat bands = the same part of the past matters all the time.",
 "06": "**Beeswarm.** Direction of each segment's effect on the mean forecast.",
 "07": "**Local explanation** of one series (waterfall, decision plot, the series): worst, best, largest drift.",
 "08": "**GradientSHAP per lag**, seasonal lags dotted.",
 "09": "**Faithfulness.** The SHAP order must change the forecast faster than a random order.",
 "10": "**Stability of the explanations** across seeds, folds and the two models.",
 "11": "**Segment ablation.** Test-sMAPE increase when one part of the past is replaced by real data of another series.",
 "12": "**Per-series drift.** Distribution of the drift score; share of series drifting significantly over calendar time.",
 "13": "**Train/test shift** of scale-free window features (population).",
 "14": "**Population drift.** Domain classifier AUC (inputs only vs inputs + future) with MMD p-values, and what reveals the test windows.",
 "15": "**Who copes better with drift / novelty?** sMAPE vs drift and novelty; per tercile with Wilcoxon-Holm p.",
 "16": "**Residual-based concept drift.** Do each model's errors on the test look like on its validation blocks?",
 "17": "**Worst series gallery.**",
 "18": "**Calendar overlap.** Calendar-CV folds should be 0 (no leak); the paper protocol shows M4's leak.",
 "20": "**GradientSHAP per input step** of one series: red bars raised the mean forecast, blue lowered it; they add up (approximately) to forecast − average forecast of the baselines.",
 "21": "**Grad-CAM, global.** Where TimesNet's convolutions look, folded back to time, per TimesBlock; shaded = the future part created by the linear layer.",
 "22": "**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.",
 "19": "**Leak experiment.** Leaky − clean sMAPE on the same series (negative = the leak lowers the error), real vs placebo, per model."}


def show9(group, figures = True, only = None):
    """Summary tables and the interactive figures of one M4 group."""
    import plotly.io as pio
    from IPython.display import Markdown
    d = os.path.join(s9_base(group), "xai", group)
    sp = os.path.join(d, "xai_m4_summary.json")
    if not os.path.exists(sp):
        print("no analyses yet: run run_xai9 first")
        return
    s = json.load(open(sp))
    display(Markdown(f"### M4 {group} ({s.get('cv', 'series')} CV): {s['runs']} runs, "
                     f"{s.get('test_series_scored', s['test_series'])} of {s['test_series']} test series scored"))
    display(pd.DataFrame({"mean sMAPE": s["mean_smape"], "mean MASE": s["mean_mase"]}).round(4))
    for name, ttl in (("leak_effect_tests.csv", "Leak experiment: leaky − clean sMAPE (Wilcoxon, bootstrap CI, Holm)"),
                      ("kanb_vs_gelu_by_drift.csv", "KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)"),
                      ("drift_global_tests.csv", "Population drift tests per fold (permutation)"),
                      ("domain_classifier.csv", "Domain classifier per fold"),
                      ("residual_concept_drift.csv", "Residual-based concept drift per fold model"),
                      ("train_test_shift.csv", "Train/test shift of the window features"),
                      ("shap_stability.csv", "Stability of the importance")):
        p = os.path.join(d, name)
        if os.path.exists(p):
            display(Markdown(f"**{ttl}**"))
            display(pd.read_csv(p).round(4))
    for k in ("leak", "drift", "shap", "faith", "ablation", "gradshap", "periods"):
        if k in s:
            v = s[k]
            msgs = v.get("messages", v) if isinstance(v, dict) else v
            if isinstance(msgs, dict):
                display(Markdown(f"**{k}:** " + "; ".join(f"{a}: {b}" for a, b in msgs.items())))
    if not figures:
        return
    for f in s.get("figures", []):
        if only and not any(f.startswith(o) for o in only):
            continue
        display(Markdown(FIGURE_NOTES_9.get(f[:2], "")))
        pio.read_json(os.path.join(d, f + ".json")).show()

### 8.6 Train the fold models
One cell per M4 group; finished folds are skipped, so an interrupted cell can simply be run again. For a first look
use `seeds=[2021]`. Cheapest first.

In [ ]:
S9_HP = {"all": {},
         "Hourly": {"e_layers":      2,        # TimesBlocks
                 "d_model":       32,
                 "d_ff":          32,
                 "top_k":         5,        # periods per TimesBlock
                 "num_kernels":   6,        # Inception kernels
                 "dropout":       0.1,
                 "learning_rate": 0.1,
                 "batch_size":    32,
                 "train_epochs":  20,
                 "patience":      3,        # early stopping
                 "lradj":         "type1",  # learning-rate schedule
                 "loss":          "SMAPE"},
         "Weekly": {"e_layers":      4,        # TimesBlocks
                 "d_model":       32,
                 "d_ff":          32,
                 "top_k":         5,        # periods per TimesBlock
                 "num_kernels":   6,        # Inception kernels
                 "dropout":       0.1,
                 "learning_rate": 0.2,
                 "batch_size":    32,
                 "train_epochs":  20,
                 "patience":      3,        # early stopping
                 "lradj":         "type1",  # learning-rate schedule
                 "loss":          "SMAPE"},
         "Monthly": {"e_layers":      2,        # TimesBlocks
                 "d_model":       32,
                 "d_ff":          32,
                 "top_k":         5,        # periods per TimesBlock
                 "num_kernels":   6,        # Inception kernels
                 "dropout":       0.1,
                 "learning_rate": 0.2,
                 "batch_size":    32,
                 "train_epochs":  20,
                 "patience":      3,        # early stopping
                 "lradj":         "type1",  # learning-rate schedule
                 "loss":          "SMAPE"},
         "Quarterly": {"e_layers":      2,        # TimesBlocks
                 "d_model":       32,
                 "d_ff":          32,
                 "top_k":         5,        # periods per TimesBlock
                 "num_kernels":   6,        # Inception kernels
                 "dropout":       0.1,
                 "learning_rate": 0.2,
                 "batch_size":    64,
                 "train_epochs":  20,
                 "patience":      3,        # early stopping
                 "lradj":         "type1",  # learning-rate schedule
                 "loss":          "SMAPE"},
         "Daily": {"e_layers":      2,        # TimesBlocks
                 "d_model":       32,
                 "d_ff":          32,
                 "top_k":         5,        # periods per TimesBlock
                 "num_kernels":   6,        # Inception kernels
                 "dropout":       0.1,
                 "learning_rate": 0.2,
                 "batch_size":    32,
                 "train_epochs":  20,
                 "patience":      3,        # early stopping
                 "lradj":         "type1",  # learning-rate schedule
                 "loss":          "SMAPE"},
         "Yearly": {"e_layers":      2,        # TimesBlocks
                 "d_model":       64,
                 "d_ff":          64,
                 "top_k":         5,        # periods per TimesBlock
                 "num_kernels":   6,        # Inception kernels
                 "dropout":       0.1,
                 "learning_rate": 0.2,
                 "batch_size":    64,
                 "train_epochs":  20,
                 "patience":      3,        # early stopping
                 "lradj":         "type1",  # learning-rate schedule
                 "loss":          "SMAPE"}}

# no model-specific changes: GELU and KANB use exactly the same settings
S9_HP_MODEL = {"TimesNet":      {"all": {}, "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {},
                                 "Hourly": {}},
               "TimesNet_KANB": {"all": {}, "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {},
                                 "Hourly": {}}}

#### Hourly

In [ ]:
run_cv9("Hourly")

== M4 Hourly: 2 model(s) x 5 folds x 3 seed(s); 414 test series, horizon 48 ==


,cut-off,validated series,training series,validation block (per series),gap,median training length,test series scored (leak-free)
fold,,,,,,,
0,2010-03-29,12,53,first 48 values from 2010-03-29,96 values,700,373
1,2010-05-31,73,154,first 48 values from 2010-05-31,96 values,700,335
2,2010-06-19,45,168,first 48 values from 2010-06-19,96 values,960,302
3,2010-07-18,12,177,first 48 values from 2010-07-18,96 values,960,251
4,2015-07-28,121,324,first 48 values from 2015-07-28,96 values,700,213


calendar overlap (share of training values later than other series' first predicted value): 0: 0%, 1: 0%, 2: 0%, 3: 0%, 4: 0%, M4 test (paper protocol): 45%
settings for Hourly (✱ = changed from the paper), results folder: /content/TimesNet_reproduction/table14_s9/calendar/hp_e0a459ac


,TimesNet,TimesNet_KANB
e_layers,2 ✱,2 ✱
d_model,32 ✱,32 ✱
d_ff,32 ✱,32 ✱
top_k,5 ✱,5 ✱
num_kernels,6 ✱,6 ✱
dropout,0.1 ✱,0.1 ✱
learning_rate,0.1 ✱,0.1 ✱
batch_size,32 ✱,32 ✱
train_epochs,20 ✱,20 ✱
patience,3 ✱,3 ✱


   TimesNet       fold 0 seed 2021 ... sMAPE 41.684  MASE 14.158  OWA 4.278  (20 epochs, 0.2 min)
   TimesNet       fold 1 seed 2021 ... sMAPE 33.963  MASE 8.284  OWA 2.862  (6 epochs, 0.2 min)
   TimesNet       fold 2 seed 2021 ... sMAPE 35.342  MASE 7.049  OWA 2.712  (8 epochs, 0.2 min)
   TimesNet       fold 3 seed 2021 ... sMAPE 40.028  MASE 4.561  OWA 2.284  (7 epochs, 0.2 min)
   TimesNet       fold 4 seed 2021 ... sMAPE 35.989  MASE 2.643  OWA 1.860  (5 epochs, 0.2 min)
   TimesNet       fold 0 seed 2022 ... sMAPE 33.175  MASE 9.911  OWA 3.114  (7 epochs, 0.2 min)
   TimesNet       fold 1 seed 2022 ... sMAPE 33.956  MASE 8.109  OWA 2.822  (7 epochs, 0.2 min)
   TimesNet       fold 2 seed 2022 ... sMAPE 36.631  MASE 7.290  OWA 2.807  (8 epochs, 0.2 min)
   TimesNet       fold 3 seed 2022 ... sMAPE 42.618  MASE 4.692  OWA 2.382  (4 epochs, 0.1 min)
   TimesNet       fold 4 seed 2022 ... sMAPE 39.158  MASE 3.015  OWA 2.075  (5 epochs, 0.2 min)
   TimesNet       fold 0 seed 2023 ...

,settings,group,TimesNet SMAPE,TimesNet MASE,TimesNet OWA,TimesNet runs,TimesNet_KANB SMAPE,TimesNet_KANB MASE,TimesNet_KANB OWA,TimesNet_KANB runs,KANB better OWA (pairs),Wilcoxon p (OWA)
0,hp_22e14429,Hourly,31.469 ± 6.133,5.372 ± 2.840,2.168 ± 0.716,15,36.207 ± 3.070,6.447 ± 2.535,2.575 ± 0.414,15,3/15,0.0043
1,hp_5ecd8edc,Hourly,36.015 ± 3.647,6.544 ± 3.003,2.586 ± 0.563,15,36.257 ± 3.057,6.686 ± 3.153,2.626 ± 0.612,15,7/15,0.4887
2,hp_e0a459ac,Hourly,37.215 ± 3.796,6.869 ± 3.174,2.698 ± 0.615,15,35.425 ± 2.756,6.361 ± 2.505,2.532 ± 0.417,15,9/15,0.0946
3,paper,Hourly,24.027 ± 2.964,3.193 ± 2.189,1.421 ± 0.528,15,30.466 ± 4.674,5.446 ± 2.980,2.142 ± 0.715,15,0/15,0.0001


#### Weekly

In [ ]:
run_cv9("Weekly")

== M4 Weekly: 2 model(s) x 5 folds x 3 seed(s); 359 test series, horizon 13 ==


,cut-off,validated series,training series,validation block (per series),gap,median training length,test series scored (leak-free)
fold,,,,,,,
0,2014-07-09,282,292,first 13 values from 2014-07-09,26 values,901,350
1,2015-05-20,262,292,first 13 values from 2015-05-20,26 values,946,332
2,2016-07-13,147,292,first 13 values from 2016-07-13,26 values,1006,328
3,2016-07-23,90,292,first 13 values from 2016-07-23,26 values,1008,328
4,2016-08-26,45,292,first 13 values from 2016-08-26,26 values,1013,328


calendar overlap (share of training values later than other series' first predicted value): 0: 0%, 1: 0%, 2: 0%, 3: 0%, 4: 0%, M4 test (paper protocol): 42%
settings for Weekly (✱ = changed from the paper), results folder: /content/TimesNet_reproduction/table14_s9/calendar/hp_ca6555ae


,TimesNet,TimesNet_KANB
e_layers,4 ✱,4 ✱
d_model,32 ✱,32 ✱
d_ff,32 ✱,32 ✱
top_k,5 ✱,5 ✱
num_kernels,6 ✱,6 ✱
dropout,0.1 ✱,0.1 ✱
learning_rate,0.2 ✱,0.2 ✱
batch_size,32 ✱,32 ✱
train_epochs,20 ✱,20 ✱
patience,3 ✱,3 ✱


   TimesNet       fold 0 seed 2021 ... sMAPE 19.470  MASE 6.454  OWA 2.205  (4 epochs, 0.2 min)
   TimesNet       fold 1 seed 2021 ... sMAPE 13.304  MASE 5.235  OWA 1.650  (9 epochs, 0.3 min)
   TimesNet       fold 2 seed 2021 ... sMAPE 14.936  MASE 5.544  OWA 1.790  (6 epochs, 0.2 min)
   TimesNet       fold 3 seed 2021 ... sMAPE 15.039  MASE 5.577  OWA 1.801  (4 epochs, 0.2 min)
   TimesNet       fold 4 seed 2021 ... sMAPE 11.927  MASE 4.903  OWA 1.514  (7 epochs, 0.2 min)
   TimesNet       fold 0 seed 2022 ... sMAPE 15.182  MASE 5.550  OWA 1.813  (7 epochs, 0.3 min)
   TimesNet       fold 1 seed 2022 ... sMAPE 13.405  MASE 5.253  OWA 1.659  (5 epochs, 0.2 min)
   TimesNet       fold 2 seed 2022 ... sMAPE 14.272  MASE 5.414  OWA 1.731  (8 epochs, 0.3 min)
   TimesNet       fold 3 seed 2022 ... sMAPE 16.872  MASE 5.962  OWA 1.968  (6 epochs, 0.2 min)
   TimesNet       fold 4 seed 2022 ... sMAPE 12.470  MASE 4.902  OWA 1.543  (4 epochs, 0.2 min)
   TimesNet       fold 0 seed 2023 ... s

,settings,group,TimesNet SMAPE,TimesNet MASE,TimesNet OWA,TimesNet runs,TimesNet_KANB SMAPE,TimesNet_KANB MASE,TimesNet_KANB OWA,TimesNet_KANB runs,KANB better OWA (pairs),Wilcoxon p (OWA)
0,hp_22e14429,Weekly,12.523 ± 1.547,3.966 ± 0.458,1.380 ± 0.140,15,14.159 ± 1.394,5.193 ± 0.441,1.688 ± 0.140,15,0/15,0.0001
1,hp_59e9ad1d,Weekly,14.083 ± 1.377,5.318 ± 0.300,1.706 ± 0.126,15,14.040 ± 1.762,5.250 ± 0.535,1.692 ± 0.181,15,8/15,0.6788
2,hp_ca6555ae,Weekly,14.484 ± 1.967,5.431 ± 0.428,1.748 ± 0.183,15,14.072 ± 1.461,5.352 ± 0.323,1.712 ± 0.137,15,9/15,0.2524
3,hp_e0a459ac,Weekly,14.132 ± 1.545,5.087 ± 0.486,1.667 ± 0.161,15,14.589 ± 1.542,5.252 ± 0.588,1.721 ± 0.170,15,3/15,0.1205
4,paper,Weekly,11.874 ± 0.989,3.438 ± 0.199,1.251 ± 0.081,15,13.225 ± 0.982,4.361 ± 0.425,1.489 ± 0.114,15,1/15,0.0001


#### Daily

In [ ]:
run_cv9("Daily")

== M4 Daily: 2 model(s) x 5 folds x 3 seed(s); 4227 test series, horizon 14 ==


,cut-off,validated series,training series,validation block (per series),gap,median training length,test series scored (leak-free)
fold,,,,,,,
0,1987-05-07,42,464,first 14 values from 1987-05-07,28 values,296,3806
1,2000-08-19,94,939,first 14 values from 2000-08-19,28 values,274,3383
2,2010-05-26,2369,3636,first 14 values from 2010-05-26,28 values,2402,2961
3,2012-09-10,2307,3809,first 14 values from 2012-09-10,28 values,2874,2813
4,2012-09-11,2099,3809,first 14 values from 2012-09-11,28 values,2875,2813


calendar overlap (share of training values later than other series' first predicted value): 0: 0%, 1: 0%, 2: 0%, 3: 0%, 4: 0%, M4 test (paper protocol): 46%
settings for Daily (✱ = changed from the paper), results folder: /content/TimesNet_reproduction/table14_s9/calendar/hp_59e9ad1d


,TimesNet,TimesNet_KANB
e_layers,2 ✱,2 ✱
d_model,32 ✱,32 ✱
d_ff,32 ✱,32 ✱
top_k,5 ✱,5 ✱
num_kernels,6 ✱,6 ✱
dropout,0.1 ✱,0.1 ✱
learning_rate,0.2 ✱,0.2 ✱
batch_size,32 ✱,32 ✱
train_epochs,20 ✱,20 ✱
patience,3 ✱,3 ✱


   TimesNet       fold 0 seed 2021 ... sMAPE 4.260  MASE 4.879  OWA 1.490  (6 epochs, 0.5 min)
   TimesNet       fold 1 seed 2021 ... sMAPE 3.259  MASE 3.713  OWA 1.174  (10 epochs, 0.6 min)
   TimesNet       fold 2 seed 2021 ... sMAPE 2.794  MASE 3.204  OWA 1.038  (18 epochs, 1.5 min)
   TimesNet       fold 3 seed 2021 ... sMAPE 2.990  MASE 3.452  OWA 1.137  (16 epochs, 1.5 min)
   TimesNet       fold 4 seed 2021 ... sMAPE 2.891  MASE 3.326  OWA 1.097  (16 epochs, 1.5 min)
   TimesNet       fold 0 seed 2022 ... sMAPE 3.658  MASE 4.103  OWA 1.266  (7 epochs, 0.5 min)
   TimesNet       fold 1 seed 2022 ... sMAPE 3.370  MASE 3.876  OWA 1.219  (12 epochs, 0.6 min)
   TimesNet       fold 2 seed 2022 ... sMAPE 2.995  MASE 3.496  OWA 1.122  (15 epochs, 1.3 min)
   TimesNet       fold 3 seed 2022 ... sMAPE 2.859  MASE 3.274  OWA 1.083  (14 epochs, 1.4 min)
   TimesNet       fold 4 seed 2022 ... sMAPE 3.012  MASE 3.513  OWA 1.151  (18 epochs, 1.6 min)
   TimesNet       fold 0 seed 2023 ... sMA

,settings,group,TimesNet SMAPE,TimesNet MASE,TimesNet OWA,TimesNet runs,TimesNet_KANB SMAPE,TimesNet_KANB MASE,TimesNet_KANB OWA,TimesNet_KANB runs,KANB better OWA (pairs),Wilcoxon p (OWA)
0,hp_59e9ad1d,Daily,3.263 ± 0.476,3.750 ± 0.529,1.198 ± 0.136,15,3.299 ± 0.450,3.788 ± 0.509,1.211 ± 0.134,15,6/15,0.3894


#### Yearly

In [ ]:
run_cv9("Yearly")

== M4 Yearly: 2 model(s) x 3 folds x 3 seed(s); 23000 test series, horizon 6 ==


,cut-off,validated series,training series,validation block (per series),gap,median training length,test series scored (leak-free)
fold,,,,,,,
0,1998-12-31,6563,8197,first 6 values from 1998-12-31,12 values,16,21729
1,2003-01-01,5391,8973,first 6 values from 2003-01-01,12 values,20,21286
2,2004-12-31,1929,9356,first 6 values from 2004-12-31,12 values,22,21116


calendar overlap (share of training values later than other series' first predicted value): 0: 0%, 1: 0%, 2: 0%, M4 test (paper protocol): 42%
settings for Yearly (✱ = changed from the paper), results folder: /content/TimesNet_reproduction/table14_s9/calendar/hp_fd3fd374


,TimesNet,TimesNet_KANB
e_layers,2 ✱,2 ✱
d_model,64 ✱,64 ✱
d_ff,64 ✱,64 ✱
top_k,5 ✱,5 ✱
num_kernels,6 ✱,6 ✱
dropout,0.1 ✱,0.1 ✱
learning_rate,0.2 ✱,0.2 ✱
batch_size,64 ✱,64 ✱
train_epochs,20 ✱,20 ✱
patience,3 ✱,3 ✱


   TimesNet       fold 0 seed 2021 ... sMAPE 15.261  MASE 3.418  OWA 0.895  (11 epochs, 3.6 min)
   TimesNet       fold 1 seed 2021 ... sMAPE 15.112  MASE 3.443  OWA 0.893  (5 epochs, 3.0 min)
   TimesNet       fold 2 seed 2021 ... sMAPE 13.892  MASE 3.190  OWA 0.823  (9 epochs, 3.1 min)
   TimesNet       fold 0 seed 2022 ... sMAPE 15.247  MASE 3.358  OWA 0.888  (5 epochs, 3.1 min)
   TimesNet       fold 1 seed 2022 ... sMAPE 16.177  MASE 3.774  OWA 0.967  (4 epochs, 2.9 min)
   TimesNet       fold 2 seed 2022 ... sMAPE 14.432  MASE 3.215  OWA 0.843  (6 epochs, 2.9 min)
   TimesNet       fold 0 seed 2023 ... sMAPE 14.193  MASE 3.145  OWA 0.829  (6 epochs, 3.2 min)
   TimesNet       fold 1 seed 2023 ... sMAPE 14.688  MASE 3.417  OWA 0.876  (6 epochs, 3.1 min)
   TimesNet       fold 2 seed 2023 ... sMAPE 13.828  MASE 3.131  OWA 0.814  (7 epochs, 2.9 min)
   TimesNet_KANB  fold 0 seed 2021 ... sMAPE 14.684  MASE 3.323  OWA 0.866  (8 epochs, 3.7 min)
   TimesNet_KANB  fold 1 seed 2021 ... 

,settings,group,TimesNet SMAPE,TimesNet MASE,TimesNet OWA,TimesNet runs,TimesNet_KANB SMAPE,TimesNet_KANB MASE,TimesNet_KANB OWA,TimesNet_KANB runs,KANB better OWA (pairs),Wilcoxon p (OWA)
0,hp_d303f8cc,Yearly,14.468 ± 0.674,3.280 ± 0.143,0.853 ± 0.039,9,14.660 ± 0.471,3.365 ± 0.135,0.869 ± 0.030,9,3/9,0.2500
1,hp_fd3fd374,Yearly,14.759 ± 0.765,3.344 ± 0.204,0.870 ± 0.048,9,14.384 ± 0.275,3.248 ± 0.074,0.846 ± 0.018,9,6/9,0.2031


#### Quarterly

In [ ]:
run_cv9("Quarterly")

== M4 Quarterly: 2 model(s) x 3 folds x 3 seed(s); 24000 test series, horizon 8 ==


,cut-off,validated series,training series,validation block (per series),gap,median training length,test series scored (leak-free)
fold,,,,,,,
0,2009-06-30,18252,20608,first 8 values from 2009-06-30,16 values,62,21833
1,2012-07-01,16240,22889,first 8 values from 2012-07-01,16 values,70,20631
2,2012-10-31,11274,22898,first 8 values from 2012-10-31,16 values,71,19974


calendar overlap (share of training values later than other series' first predicted value): 0: 0%, 1: 0%, 2: 0%, M4 test (paper protocol): 42%
settings for Quarterly (✱ = changed from the paper), results folder: /content/TimesNet_reproduction/table14_s9/calendar/hp_312f7e24


,TimesNet,TimesNet_KANB
e_layers,2 ✱,2 ✱
d_model,32 ✱,32 ✱
d_ff,32 ✱,32 ✱
top_k,5 ✱,5 ✱
num_kernels,6 ✱,6 ✱
dropout,0.1 ✱,0.1 ✱
learning_rate,0.2 ✱,0.2 ✱
batch_size,64 ✱,64 ✱
train_epochs,20 ✱,20 ✱
patience,3 ✱,3 ✱


   TimesNet       fold 0 seed 2021 ... sMAPE 11.037  MASE 1.263  OWA 0.955  (15 epochs, 6.1 min)
   TimesNet       fold 1 seed 2021 ... sMAPE 10.818  MASE 1.262  OWA 0.950  (10 epochs, 5.3 min)
   TimesNet       fold 2 seed 2021 ... sMAPE 10.692  MASE 1.245  OWA 0.944  (10 epochs, 4.8 min)
   TimesNet       fold 0 seed 2022 ... sMAPE 11.752  MASE 1.349  OWA 1.018  (4 epochs, 4.2 min)
   TimesNet       fold 1 seed 2022 ... sMAPE 11.313  MASE 1.301  OWA 0.987  (8 epochs, 4.9 min)
   TimesNet       fold 2 seed 2022 ... sMAPE 11.100  MASE 1.313  OWA 0.987  (7 epochs, 4.3 min)
   TimesNet       fold 0 seed 2023 ... sMAPE 11.901  MASE 1.399  OWA 1.043  (5 epochs, 4.5 min)
   TimesNet       fold 1 seed 2023 ... sMAPE 10.963  MASE 1.264  OWA 0.958  (16 epochs, 6.4 min)
   TimesNet       fold 2 seed 2023 ... sMAPE 10.809  MASE 1.259  OWA 0.954  (17 epochs, 6.1 min)
   TimesNet_KANB  fold 0 seed 2021 ... sMAPE 11.489  MASE 1.291  OWA 0.985  (8 epochs, 5.5 min)
   TimesNet_KANB  fold 1 seed 2021 

,settings,group,TimesNet SMAPE,TimesNet MASE,TimesNet OWA,TimesNet runs,TimesNet_KANB SMAPE,TimesNet_KANB MASE,TimesNet_KANB OWA,TimesNet_KANB runs,KANB better OWA (pairs),Wilcoxon p (OWA)
0,hp_312f7e24,Quarterly,11.154 ± 0.424,1.295 ± 0.051,0.977 ± 0.034,9,11.059 ± 0.268,1.270 ± 0.024,0.964 ± 0.018,9,6/9,0.25


#### Monthly

In [ ]:
run_cv9("Monthly")

== M4 Monthly: 2 model(s) x 5 folds x 3 seed(s); 48000 test series, horizon 18 ==


,cut-off,validated series,training series,validation block (per series),gap,median training length,test series scored (leak-free)
fold,,,,,,,
0,2006-05-31,28972,31016,first 18 values from 2006-05-31,36 values,148,46542
1,2011-05-28,30349,35134,first 18 values from 2011-05-28,36 values,181,43590
2,2012-08-31,28737,36345,first 18 values from 2012-08-31,36 values,177,43145
3,2013-11-30,25900,38026,first 18 values from 2013-11-30,36 values,179,41759
4,2014-02-28,8095,38080,first 18 values from 2014-02-28,36 values,182,41191


calendar overlap (share of training values later than other series' first predicted value): 0: 0%, 1: 0%, 2: 0%, 3: 0%, 4: 0%, M4 test (paper protocol): 42%
settings for Monthly (✱ = changed from the paper), results folder: /content/TimesNet_reproduction/table14_s9/calendar/hp_59e9ad1d


,TimesNet,TimesNet_KANB
e_layers,2 ✱,2 ✱
d_model,32 ✱,32 ✱
d_ff,32 ✱,32 ✱
top_k,5 ✱,5 ✱
num_kernels,6 ✱,6 ✱
dropout,0.1 ✱,0.1 ✱
learning_rate,0.2 ✱,0.2 ✱
batch_size,32 ✱,32 ✱
train_epochs,20 ✱,20 ✱
patience,3 ✱,3 ✱


   TimesNet       fold 0 seed 2021 ... sMAPE 14.381  MASE 1.089  OWA 1.016  (8 epochs, 10.4 min)
   TimesNet       fold 1 seed 2021 ... sMAPE 12.842  MASE 1.039  OWA 0.977  (16 epochs, 14.8 min)
   TimesNet       fold 2 seed 2021 ... sMAPE 12.887  MASE 1.069  OWA 0.997  (12 epochs, 12.5 min)
   TimesNet       fold 3 seed 2021 ... sMAPE 13.281  MASE 1.089  OWA 1.015  (10 epochs, 11.8 min)
   TimesNet       fold 4 seed 2021 ... sMAPE 13.525  MASE 1.102  OWA 1.029  (7 epochs, 8.7 min)
   TimesNet       fold 0 seed 2022 ... sMAPE 14.424  MASE 1.103  OWA 1.025  (9 epochs, 10.7 min)
   TimesNet       fold 1 seed 2022 ... sMAPE 13.132  MASE 1.073  OWA 1.005  (11 epochs, 12.0 min)
   TimesNet       fold 2 seed 2022 ...

KeyboardInterrupt: 

### 8.6b Leak experiment (optional)
Measures how much M4's cross-series leak changes the error, for GELU and KANB. Needs the fold models of 8.6 (same
group, seeds and folds); costs 2 extra trainings per fold model (5 with `S9_PLACEBO = True` or `placebo=True`).
Start with Yearly and Monthly (largest calendar overlap) and `seeds=[2021]`.

#### Hourly

In [ ]:
run_leak9("Hourly")

#### Weekly

In [ ]:
run_leak9("Weekly")

#### Daily

In [ ]:
run_leak9("Daily")

#### Yearly

In [ ]:
run_leak9("Yearly")

#### Quarterly

In [ ]:
run_leak9("Quarterly")

#### Monthly

In [ ]:
run_leak9("Monthly")

### 8.7 Analyse (no training)
One cell per group; uses every finished fold model and seed (only folds/seeds finished for both models).
`analyses="all"` or a subset, e.g. `analyses="drift,gallery"` (quick), `analyses="leak"` (leak experiment only) or
`analyses="periods,shap,faith"`.

#### Hourly

In [ ]:
run_xai9("Hourly")

== Section 9 analyses: M4 Hourly (all) ==
  == periods ==
  == drift ==
    drift: per-series scores (0.0 min)
    drift: fold 0 done
    drift: fold 1 done
    drift: fold 2 done
    drift: fold 3 done
    drift: fold 4 done
  == shap ==
    SHAP TimesNet (GELU) fold 0 seed 2021: 213 series
    SHAP TimesNet (GELU) fold 0 seed 2022: 213 series
    SHAP TimesNet (GELU) fold 0 seed 2023: 213 series
    SHAP TimesNet (GELU) fold 1 seed 2021: 213 series
    SHAP TimesNet (GELU) fold 1 seed 2022: 213 series
    SHAP TimesNet (GELU) fold 1 seed 2023: 213 series
    SHAP TimesNet (GELU) fold 2 seed 2021: 213 series
    SHAP TimesNet (GELU) fold 2 seed 2022: 213 series
    SHAP TimesNet (GELU) fold 2 seed 2023: 213 series
    SHAP TimesNet (GELU) fold 3 seed 2021: 213 series
    SHAP TimesNet (GELU) fold 3 seed 2022: 213 series
    SHAP TimesNet (GELU) fold 3 seed 2023: 213 series
    SHAP TimesNet (GELU) fold 4 seed 2021: 213 series
    SHAP TimesNet (GELU) fold 4 seed 2022: 213 series
    S

### M4 Hourly (calendar CV): {'TimesNet': 15, 'TimesNet_KANB': 15} runs, 373 of 414 test series scored

,mean sMAPE,mean MASE
TimesNet,33.6432,10.1753
TimesNet_KANB,31.7855,9.2351


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,113,22.5228,20.8543,-1.6685,-1.9234,-1.4358,-7.4082,1.0000,0.0,4/5,0.0,KANB better
1,smape,medium drift,129,33.0949,31.0610,-2.0339,-2.2905,-1.7997,-6.1457,1.0000,0.0,4/5,0.0,KANB better
2,smape,large drift,131,43.7755,41.9281,-1.8474,-2.1011,-1.6133,-4.2202,0.9847,0.0,4/5,0.0,KANB better
3,smape,all series,373,33.6432,31.7855,-1.8577,-1.9972,-1.7205,-5.5218,0.9946,0.0,4/5,0.0,KANB better
4,mase,small drift,113,20.2465,18.4992,-1.7473,-2.0927,-1.4329,-8.6300,1.0000,0.0,4/5,0.0,KANB better
5,mase,medium drift,129,8.8991,7.9445,-0.9546,-1.1788,-0.7564,-10.7269,1.0000,0.0,4/5,0.0,KANB better
6,mase,large drift,131,2.7448,2.5147,-0.2301,-0.2653,-0.1945,-8.3829,0.9924,0.0,4/5,0.0,KANB better
7,mase,all series,373,10.1753,9.2351,-0.9403,-1.0817,-0.8064,-9.2409,0.9973,0.0,4/5,0.0,KANB better


**Population drift tests per fold (permutation)**

... (12 outputs left out)

**faith:** TimesNet: {'auc_shap': 31.341665935122826, 'auc_random': 35.218733495931595, 'shap_better_share': 0.4653333333333333}; TimesNet_KANB: {'auc_shap': 2801.283209845036, 'auc_random': 3779.626326861104, 'shap_better_share': 0.42333333333333334}; stability: {'GELU vs KANB (same fold, seed)': 0.786, 'TimesNet (GELU): across folds (same seed)': 0.667, 'TimesNet (GELU): across seeds (same fold)': 0.69, 'TimesNet-KANB (ours): across folds (same seed)': 0.833, 'TimesNet-KANB (ours): across seeds (same fold)': 0.81}

**ablation:** TimesNet: {'most needed': 'lag 1-12', 'increase_pct': 0.22964150952185367}; TimesNet_KANB: {'most needed': 'lag 1-12', 'increase_pct': 0.8354671447587363}

**gradshap:** TimesNet: {'share of importance in the last 24 values': 0.27839750051498413}; TimesNet_KANB: {'share of importance in the last 24 values': 0.2770542502403259}

**periods:** TimesNet: {'most_chosen_period': 2, 'share': 0.23317683881064163, 'seasonal_period_chosen_share': 0.9899843505477308}; TimesNet_KANB: {'most_chosen_period': 2, 'share': 0.32754303599374024, 'seasonal_period_chosen_share': 0.9139280125195618}; acf_peaks: [24, 48, 72, 96, 120, 144]

#### Weekly

In [ ]:
run_xai9("Weekly")

== Section 9 analyses: M4 Weekly (all) ==
  == periods ==
  == drift ==
    drift: per-series scores (0.0 min)
    drift: fold 0 done
    drift: fold 1 done
    drift: fold 2 done
    drift: fold 3 done
    drift: fold 4 done
  == shap ==
    SHAP TimesNet (GELU) fold 0 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 3 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 3 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 3 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 4 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 4 seed 2022: 300 series
    S

### M4 Weekly (calendar CV): {'TimesNet': 15, 'TimesNet_KANB': 15} runs, 350 of 359 test series scored

,mean sMAPE,mean MASE
TimesNet,14.7781,5.4482
TimesNet_KANB,14.3602,5.3643


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,118,11.7815,11.2424,-0.5391,-0.7245,-0.3840,-4.5757,0.8814,0.0000,4/5,0.0000,KANB better
1,smape,medium drift,113,13.5824,13.1267,-0.4557,-0.5849,-0.3350,-3.3552,0.7788,0.0000,4/5,0.0000,KANB better
2,smape,large drift,119,18.8848,18.6230,-0.2618,-0.3789,-0.1483,-1.3864,0.7227,0.0000,5/5,0.0000,KANB better
3,smape,all series,350,14.7781,14.3602,-0.4179,-0.5063,-0.3379,-2.8278,0.7943,0.0000,4/5,0.0000,KANB better
4,mase,small drift,118,2.2224,2.1576,-0.0648,-0.0792,-0.0513,-2.9156,0.8559,0.0000,2/5,0.0000,KANB better
5,mase,medium drift,113,4.4813,4.3549,-0.1264,-0.1606,-0.0940,-2.8208,0.7788,0.0000,4/5,0.0000,KANB better
6,mase,large drift,119,9.5649,9.5027,-0.0622,-0.1384,0.0063,-0.6504,0.7227,0.0071,3/5,0.0071,KANB better
7,mase,all series,350,5.4482,5.3644,-0.0838,-0.1129,-0.0575,-1.5383,0.7857,0.0000,3/5,0.0000,KANB better


**Population drift tests per fold (permutation)**

... (12 outputs left out)

**faith:** TimesNet: {'auc_shap': 1703674.4036217334, 'auc_random': 1411897.619026595, 'shap_better_share': 0.572}; TimesNet_KANB: {'auc_shap': 3027108.249711528, 'auc_random': 2520099.2833539047, 'shap_better_share': 0.564}; stability: {'GELU vs KANB (same fold, seed)': 0.667, 'TimesNet (GELU): across folds (same seed)': 0.548, 'TimesNet (GELU): across seeds (same fold)': 0.643, 'TimesNet-KANB (ours): across folds (same seed)': 0.571, 'TimesNet-KANB (ours): across seeds (same fold)': 0.643}

**ablation:** TimesNet: {'most needed': 'lag 1-3', 'increase_pct': 2.9924633058714836}; TimesNet_KANB: {'most needed': 'lag 1-3', 'increase_pct': 3.249474970363865}

**gradshap:** TimesNet: {'share of importance in the last 6 values': 0.27717864513397217}; TimesNet_KANB: {'share of importance in the last 6 values': 0.2712610960006714}

**periods:** TimesNet: {'most_chosen_period': 39, 'share': 0.5404471544715447, 'seasonal_period_chosen_share': 0.0}; TimesNet_KANB: {'most_chosen_period': 39, 'share': 0.3326727642276423, 'seasonal_period_chosen_share': 0.0}; acf_peaks: [12, 17, 22, 26, 30, 33]

#### Daily

In [ ]:
run_xai9("Daily")

== Section 9 analyses: M4 Daily (all) ==
  == periods ==
  == drift ==
    drift: per-series scores (0.0 min)
    drift: fold 0 done
    drift: fold 1 done
    drift: fold 2 done
    drift: fold 3 done
    drift: fold 4 done
  == shap ==
    SHAP TimesNet (GELU) fold 0 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 3 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 3 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 3 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 4 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 4 seed 2022: 300 series
    SH

### M4 Daily (calendar CV): {'TimesNet': 15, 'TimesNet_KANB': 15} runs, 3806 of 4227 test series scored

,mean sMAPE,mean MASE
TimesNet,3.5640,4.0507
TimesNet_KANB,3.5609,4.0319


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,1298,2.6534,2.6448,-0.0086,-0.0332,0.0180,-0.3248,0.6063,0.0,3/5,0.0,KANB better
1,smape,medium drift,1254,2.9127,2.9534,0.0407,0.0060,0.0775,1.3972,0.3868,0.0,2/5,0.0,GELU better
2,smape,large drift,1254,5.1578,5.1166,-0.0412,-0.0887,0.0051,-0.7991,0.3557,0.0,1/5,0.0,KANB better
3,smape,all series,3806,3.5640,3.5609,-0.0031,-0.0239,0.0174,-0.0873,0.4514,0.0,2/5,0.0,KANB better
4,mase,small drift,1298,2.7124,2.7191,0.0067,-0.0140,0.0284,0.2476,0.6032,0.0,3/5,0.0,GELU better
5,mase,medium drift,1254,3.2591,3.2724,0.0133,-0.0237,0.0504,0.4070,0.3892,0.0,2/5,0.0,GELU better
6,mase,large drift,1254,6.2277,6.1502,-0.0775,-0.1350,-0.0270,-1.2447,0.3573,0.0,1/5,0.0,KANB better
7,mase,all series,3806,4.0507,4.0319,-0.0189,-0.0422,0.0031,-0.4661,0.4517,0.0,2/5,0.0,KANB better


**Population drift tests per fold (permutation)**

... (12 outputs left out)

**faith:** TimesNet: {'auc_shap': 36.62840033662925, 'auc_random': 34.152168526698404, 'shap_better_share': 0.8286666666666667}; TimesNet_KANB: {'auc_shap': 100.13132853762278, 'auc_random': 101.86904825925224, 'shap_better_share': 0.8}; stability: {'GELU vs KANB (same fold, seed)': 0.452, 'TimesNet (GELU): across folds (same seed)': 0.429, 'TimesNet (GELU): across seeds (same fold)': 0.452, 'TimesNet-KANB (ours): across folds (same seed)': 0.405, 'TimesNet-KANB (ours): across seeds (same fold)': 0.405}

**ablation:** TimesNet: {'most needed': 'lag 1-4', 'increase_pct': 34.57323292079921}; TimesNet_KANB: {'most needed': 'lag 1-4', 'increase_pct': 33.75363534297449}

**gradshap:** TimesNet: {'share of importance in the last 7 values': 0.5506694316864014}; TimesNet_KANB: {'share of importance in the last 7 values': 0.5316084027290344}

**periods:** TimesNet: {'most_chosen_period': 42, 'share': 0.57555, 'seasonal_period_chosen_share': 0.0}; TimesNet_KANB: {'most_chosen_period': 2, 'share': 0.41995, 'seasonal_period_chosen_share': 0.0}; acf_peaks: []

#### Yearly

In [ ]:
run_xai9("Yearly")

== Section 9 analyses: M4 Yearly (all) ==
  == periods ==
  == drift ==
    drift: per-series scores (0.0 min)
    drift: fold 0 done
    drift: fold 1 done
    drift: fold 2 done
  == shap ==
    SHAP TimesNet (GELU) fold 0 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2023: 300 series
    SHAP TimesNet-KANB (ours) fold 0 seed 2021: 300 series
    SHAP TimesNet-KANB (ours) fold 0 seed 2022: 300 series
    SHAP TimesNet-KANB (ours) fold 0 seed 2023: 300 series
    SHAP TimesNet-KANB (ours) fold 1 seed 2021: 300 series
    SHAP TimesNet-KANB (ours) fold 1 seed 2022: 300 series
    SHAP TimesNet-KANB (ou

### M4 Yearly (calendar CV): {'TimesNet': 9, 'TimesNet_KANB': 9} runs, 21729 of 23000 test series scored

,mean sMAPE,mean MASE
TimesNet,14.7882,3.3402
TimesNet_KANB,14.4103,3.2469


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,7156,19.2298,18.8788,-0.3509,-0.4440,-0.2605,-1.8249,0.6845,0.0000,2/3,0.0000,KANB better
1,smape,medium drift,7319,12.2795,11.5519,-0.7276,-0.7834,-0.6725,-5.9254,0.7964,0.0000,2/3,0.0000,KANB better
2,smape,large drift,7254,12.9378,12.8863,-0.0515,-0.0837,-0.0212,-0.3983,0.5087,0.3479,2/3,0.3479,no significant difference
3,smape,all series,21729,14.7882,14.4103,-0.3779,-0.4154,-0.3412,-2.5551,0.6635,0.0000,2/3,0.0000,KANB better
4,mase,small drift,7156,3.1367,3.0347,-0.1020,-0.1098,-0.0943,-3.2510,0.6944,0.0000,1/3,0.0000,KANB better
5,mase,medium drift,7319,3.0786,2.9099,-0.1687,-0.1755,-0.1619,-5.4788,0.7955,0.0000,2/3,0.0000,KANB better
6,mase,large drift,7254,3.8049,3.7961,-0.0088,-0.0152,-0.0024,-0.2307,0.4974,0.6194,2/3,0.6194,no significant difference
7,mase,all series,21729,3.3402,3.2469,-0.0933,-0.0976,-0.0890,-2.7940,0.6627,0.0000,2/3,0.0000,KANB better


**Population drift tests per fold (permutation)**

... (12 outputs left out)

**faith:** TimesNet: {'auc_shap': 1.472141115127545, 'auc_random': 0.9801350167555265, 'shap_better_share': 0.9011111111111111}; TimesNet_KANB: {'auc_shap': 1.5107482107801355, 'auc_random': 0.9958223407465572, 'shap_better_share': 0.8966666666666666}; stability: {'GELU vs KANB (same fold, seed)': 0.524, 'TimesNet (GELU): across folds (same seed)': 0.524, 'TimesNet (GELU): across seeds (same fold)': 0.524, 'TimesNet-KANB (ours): across folds (same seed)': 0.524, 'TimesNet-KANB (ours): across seeds (same fold)': 0.524}

**ablation:** TimesNet: {'most needed': 'lag 1-2', 'increase_pct': 78.33149215036008}; TimesNet_KANB: {'most needed': 'lag 1-2', 'increase_pct': 86.48387903040377}

**gradshap:** TimesNet: {'share of importance in the last 3 values': 0.6818902492523193}; TimesNet_KANB: {'share of importance in the last 3 values': 0.6582058668136597}

**periods:** TimesNet: {'most_chosen_period': 18, 'share': 0.5821944444444445, 'seasonal_period_chosen_share': 0.0}; TimesNet_KANB: {'most_chosen_period': 18, 'share': 0.6463333333333333, 'seasonal_period_chosen_share': 0.0}; acf_peaks: []

#### Quarterly

In [ ]:
run_xai9("Quarterly")

== Section 9 analyses: M4 Quarterly (all) ==
  == periods ==
  == drift ==
    drift: per-series scores (0.1 min)
    drift: fold 0 done
    drift: fold 1 done
    drift: fold 2 done
  == shap ==
    SHAP TimesNet (GELU) fold 0 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 0 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 1 seed 2023: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2021: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2022: 300 series
    SHAP TimesNet (GELU) fold 2 seed 2023: 300 series
    SHAP TimesNet-KANB (ours) fold 0 seed 2021: 300 series
    SHAP TimesNet-KANB (ours) fold 0 seed 2022: 300 series
    SHAP TimesNet-KANB (ours) fold 0 seed 2023: 300 series
    SHAP TimesNet-KANB (ours) fold 1 seed 2021: 300 series
    SHAP TimesNet-KANB (ours) fold 1 seed 2022: 300 series
    SHAP TimesNet-KANB 

### M4 Quarterly (calendar CV): {'TimesNet': 9, 'TimesNet_KANB': 9} runs, 21833 of 24000 test series scored

,mean sMAPE,mean MASE
TimesNet,11.3340,1.3037
TimesNet_KANB,11.2334,1.2778


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,7614,14.0958,14.6259,0.5302,0.4710,0.5893,3.7612,0.3980,0.0,0/3,0.0,GELU better
1,smape,medium drift,7307,10.5285,10.2069,-0.3216,-0.3902,-0.2543,-3.0545,0.6009,0.0,3/3,0.0,KANB better
2,smape,large drift,6912,9.1434,8.5813,-0.5620,-0.5969,-0.5296,-6.1470,0.7461,0.0,3/3,0.0,KANB better
3,smape,all series,21833,11.3340,11.2334,-0.1007,-0.1324,-0.0661,-0.8882,0.5761,0.0,2/3,0.0,KANB better
4,mase,small drift,7614,1.2728,1.3048,0.0320,0.0291,0.0347,2.5139,0.4010,0.0,1/3,0.0,GELU better
5,mase,medium drift,7307,1.2032,1.1790,-0.0241,-0.0274,-0.0208,-2.0047,0.6022,0.0,3/3,0.0,KANB better
6,mase,large drift,6912,1.4441,1.3526,-0.0915,-0.0953,-0.0878,-6.3369,0.7449,0.0,3/3,0.0,KANB better
7,mase,all series,21833,1.3037,1.2778,-0.0259,-0.0279,-0.0238,-1.9855,0.5772,0.0,3/3,0.0,KANB better


**Population drift tests per fold (permutation)**

... (12 outputs left out)

**faith:** TimesNet: {'auc_shap': 1.493180142779073, 'auc_random': 1.0668887705379855, 'shap_better_share': 0.8722222222222222}; TimesNet_KANB: {'auc_shap': 1.7266778405682317, 'auc_random': 1.2090674170247135, 'shap_better_share': 0.8877777777777778}; stability: {'GELU vs KANB (same fold, seed)': 0.595, 'TimesNet (GELU): across folds (same seed)': 0.548, 'TimesNet (GELU): across seeds (same fold)': 0.571, 'TimesNet-KANB (ours): across folds (same seed)': 0.571, 'TimesNet-KANB (ours): across seeds (same fold)': 0.595}

**ablation:** TimesNet: {'most needed': 'lag 1-2', 'increase_pct': 55.04028215567438}; TimesNet_KANB: {'most needed': 'lag 1-2', 'increase_pct': 61.95177172077859}

**gradshap:** TimesNet: {'share of importance in the last 4 values': 0.6499146819114685}; TimesNet_KANB: {'share of importance in the last 4 values': 0.5897092819213867}

**periods:** TimesNet: {'most_chosen_period': 24, 'share': 0.42088888888888887, 'seasonal_period_chosen_share': 1.0}; TimesNet_KANB: {'most_chosen_period': 24, 'share': 0.3661111111111111, 'seasonal_period_chosen_share': 1.0}; acf_peaks: [8, 12]

#### Monthly

In [ ]:
run_xai9("Monthly")

### 8.8 Report
Tables, automatic conclusions and every interactive figure. `only=["12", "15"]` shows only the drift figures.
HTML and PNG files are also stored in `table14_s9/xai/<group>/`.

In [ ]:
show9("Hourly")

### M4 Hourly (calendar CV): {'TimesNet': 15, 'TimesNet_KANB': 15} runs, 373 of 414 test series scored

,mean sMAPE,mean MASE
TimesNet,33.6432,10.1753
TimesNet_KANB,31.7855,9.2351


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,113,22.5228,20.8543,-1.6685,-1.9234,-1.4358,-7.4082,1.0000,0.0,4/5,0.0,KANB better
1,smape,medium drift,129,33.0949,31.0610,-2.0339,-2.2905,-1.7997,-6.1457,1.0000,0.0,4/5,0.0,KANB better
2,smape,large drift,131,43.7755,41.9281,-1.8474,-2.1011,-1.6133,-4.2202,0.9847,0.0,4/5,0.0,KANB better
3,smape,all series,373,33.6432,31.7855,-1.8577,-1.9972,-1.7205,-5.5218,0.9946,0.0,4/5,0.0,KANB better
4,mase,small drift,113,20.2465,18.4992,-1.7473,-2.0927,-1.4329,-8.6300,1.0000,0.0,4/5,0.0,KANB better
5,mase,medium drift,129,8.8991,7.9445,-0.9546,-1.1788,-0.7564,-10.7269,1.0000,0.0,4/5,0.0,KANB better
6,mase,large drift,131,2.7448,2.5147,-0.2301,-0.2653,-0.1945,-8.3829,0.9924,0.0,4/5,0.0,KANB better
7,mase,all series,373,10.1753,9.2351,-0.9403,-1.0817,-0.8064,-9.2409,0.9973,0.0,4/5,0.0,KANB better


**Population drift tests per fold (permutation)**

,fold,kind,mmd2,energy,p_mmd,p_energy
0,0,covariate,0.0826,0.6795,0.005,0.005
1,0,joint,0.0521,0.5931,0.005,0.005
2,1,covariate,0.1144,0.6723,0.005,0.005
3,1,joint,0.1065,0.7067,0.005,0.005
4,2,covariate,0.1389,0.8875,0.005,0.005
5,2,joint,0.1020,0.6774,0.005,0.005
6,3,covariate,0.1507,0.9407,0.005,0.005
7,3,joint,0.1380,0.9804,0.005,0.005
8,4,covariate,0.0224,0.1547,0.005,0.005
9,4,joint,0.0181,0.1547,0.005,0.005


... (37 outputs left out)

**GradientSHAP per input step** of one series: red bars raised the mean forecast, blue lowered it; they add up (approximately) to forecast − average forecast of the baselines.

**Grad-CAM, global.** Where TimesNet's convolutions look, folded back to time, per TimesBlock; shaded = the future part created by the linear layer.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

In [ ]:
show9("Weekly")

### M4 Weekly (calendar CV): {'TimesNet': 15, 'TimesNet_KANB': 15} runs, 350 of 359 test series scored

,mean sMAPE,mean MASE
TimesNet,14.7781,5.4482
TimesNet_KANB,14.3602,5.3643


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,118,11.7815,11.2424,-0.5391,-0.7245,-0.3840,-4.5757,0.8814,0.0000,4/5,0.0000,KANB better
1,smape,medium drift,113,13.5824,13.1267,-0.4557,-0.5849,-0.3350,-3.3552,0.7788,0.0000,4/5,0.0000,KANB better
2,smape,large drift,119,18.8848,18.6230,-0.2618,-0.3789,-0.1483,-1.3864,0.7227,0.0000,5/5,0.0000,KANB better
3,smape,all series,350,14.7781,14.3602,-0.4179,-0.5063,-0.3379,-2.8278,0.7943,0.0000,4/5,0.0000,KANB better
4,mase,small drift,118,2.2224,2.1576,-0.0648,-0.0792,-0.0513,-2.9156,0.8559,0.0000,2/5,0.0000,KANB better
5,mase,medium drift,113,4.4813,4.3549,-0.1264,-0.1606,-0.0940,-2.8208,0.7788,0.0000,4/5,0.0000,KANB better
6,mase,large drift,119,9.5649,9.5027,-0.0622,-0.1384,0.0063,-0.6504,0.7227,0.0071,3/5,0.0071,KANB better
7,mase,all series,350,5.4482,5.3644,-0.0838,-0.1129,-0.0575,-1.5383,0.7857,0.0000,3/5,0.0000,KANB better


**Population drift tests per fold (permutation)**

,fold,kind,mmd2,energy,p_mmd,p_energy
0,0,covariate,0.0171,0.1075,0.005,0.005
1,0,joint,0.0193,0.1190,0.005,0.005
2,1,covariate,0.0184,0.1045,0.005,0.005
3,1,joint,0.0152,0.1142,0.005,0.005
4,2,covariate,0.0065,0.0516,0.010,0.030
5,2,joint,0.0092,0.0651,0.005,0.005
6,3,covariate,0.0115,0.0687,0.015,0.015
7,3,joint,0.0144,0.0867,0.005,0.005
8,4,covariate,0.0071,0.0473,0.010,0.025
9,4,joint,0.0079,0.0526,0.010,0.020


... (41 outputs left out)

**Grad-CAM, global.** Where TimesNet's convolutions look, folded back to time, per TimesBlock; shaded = the future part created by the linear layer.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

In [ ]:
show9("Daily")

### M4 Daily (calendar CV): {'TimesNet': 15, 'TimesNet_KANB': 15} runs, 3806 of 4227 test series scored

,mean sMAPE,mean MASE
TimesNet,3.5640,4.0507
TimesNet_KANB,3.5609,4.0319


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,1298,2.6534,2.6448,-0.0086,-0.0332,0.0180,-0.3248,0.6063,0.0,3/5,0.0,KANB better
1,smape,medium drift,1254,2.9127,2.9534,0.0407,0.0060,0.0775,1.3972,0.3868,0.0,2/5,0.0,GELU better
2,smape,large drift,1254,5.1578,5.1166,-0.0412,-0.0887,0.0051,-0.7991,0.3557,0.0,1/5,0.0,KANB better
3,smape,all series,3806,3.5640,3.5609,-0.0031,-0.0239,0.0174,-0.0873,0.4514,0.0,2/5,0.0,KANB better
4,mase,small drift,1298,2.7124,2.7191,0.0067,-0.0140,0.0284,0.2476,0.6032,0.0,3/5,0.0,GELU better
5,mase,medium drift,1254,3.2591,3.2724,0.0133,-0.0237,0.0504,0.4070,0.3892,0.0,2/5,0.0,GELU better
6,mase,large drift,1254,6.2277,6.1502,-0.0775,-0.1350,-0.0270,-1.2447,0.3573,0.0,1/5,0.0,KANB better
7,mase,all series,3806,4.0507,4.0319,-0.0189,-0.0422,0.0031,-0.4661,0.4517,0.0,2/5,0.0,KANB better


**Population drift tests per fold (permutation)**

,fold,kind,mmd2,energy,p_mmd,p_energy
0,0,covariate,0.0027,0.0267,0.050,0.075
1,0,joint,0.0086,0.0641,0.005,0.005
2,1,covariate,0.0022,0.0169,0.030,0.030
3,1,joint,0.0039,0.0268,0.005,0.005
4,2,covariate,0.0102,0.0603,0.005,0.005
5,2,joint,0.0105,0.0621,0.005,0.005
6,3,covariate,0.0026,0.0183,0.005,0.010
7,3,joint,0.0043,0.0260,0.005,0.005
8,4,covariate,0.0007,0.0100,0.130,0.155
9,4,joint,0.0011,0.0136,0.050,0.035


... (37 outputs left out)

**GradientSHAP per input step** of one series: red bars raised the mean forecast, blue lowered it; they add up (approximately) to forecast − average forecast of the baselines.

**Grad-CAM, global.** Where TimesNet's convolutions look, folded back to time, per TimesBlock; shaded = the future part created by the linear layer.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

In [ ]:
show9("Yearly")

### M4 Yearly (calendar CV): {'TimesNet': 9, 'TimesNet_KANB': 9} runs, 21729 of 23000 test series scored

,mean sMAPE,mean MASE
TimesNet,14.7882,3.3402
TimesNet_KANB,14.4103,3.2469


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,7156,19.2298,18.8788,-0.3509,-0.4440,-0.2605,-1.8249,0.6845,0.0000,2/3,0.0000,KANB better
1,smape,medium drift,7319,12.2795,11.5519,-0.7276,-0.7834,-0.6725,-5.9254,0.7964,0.0000,2/3,0.0000,KANB better
2,smape,large drift,7254,12.9378,12.8863,-0.0515,-0.0837,-0.0212,-0.3983,0.5087,0.3479,2/3,0.3479,no significant difference
3,smape,all series,21729,14.7882,14.4103,-0.3779,-0.4154,-0.3412,-2.5551,0.6635,0.0000,2/3,0.0000,KANB better
4,mase,small drift,7156,3.1367,3.0347,-0.1020,-0.1098,-0.0943,-3.2510,0.6944,0.0000,1/3,0.0000,KANB better
5,mase,medium drift,7319,3.0786,2.9099,-0.1687,-0.1755,-0.1619,-5.4788,0.7955,0.0000,2/3,0.0000,KANB better
6,mase,large drift,7254,3.8049,3.7961,-0.0088,-0.0152,-0.0024,-0.2307,0.4974,0.6194,2/3,0.6194,no significant difference
7,mase,all series,21729,3.3402,3.2469,-0.0933,-0.0976,-0.0890,-2.7940,0.6627,0.0000,2/3,0.0000,KANB better


**Population drift tests per fold (permutation)**

,fold,kind,mmd2,energy,p_mmd,p_energy
0,0,covariate,0.2044,2.8124,0.005,0.005
1,0,joint,0.2492,3.8197,0.005,0.005
2,1,covariate,0.0427,0.3164,0.005,0.005
3,1,joint,0.0468,0.3692,0.005,0.005
4,2,covariate,0.0464,0.3132,0.005,0.005
5,2,joint,0.0640,0.4892,0.005,0.005


... (41 outputs left out)

**Grad-CAM, global.** Where TimesNet's convolutions look, folded back to time, per TimesBlock; shaded = the future part created by the linear layer.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

In [ ]:
show9("Quarterly")

### M4 Quarterly (calendar CV): {'TimesNet': 9, 'TimesNet_KANB': 9} runs, 21833 of 24000 test series scored

,mean sMAPE,mean MASE
TimesNet,11.3340,1.3037
TimesNet_KANB,11.2334,1.2778


**KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)**

,metric,drift,series,GELU,KANB,mean diff (KANB-GELU),95% CI low,95% CI high,change %,KANB better (series),Wilcoxon p,KANB better in folds,Holm p,verdict
0,smape,small drift,7614,14.0958,14.6259,0.5302,0.4710,0.5893,3.7612,0.3980,0.0,0/3,0.0,GELU better
1,smape,medium drift,7307,10.5285,10.2069,-0.3216,-0.3902,-0.2543,-3.0545,0.6009,0.0,3/3,0.0,KANB better
2,smape,large drift,6912,9.1434,8.5813,-0.5620,-0.5969,-0.5296,-6.1470,0.7461,0.0,3/3,0.0,KANB better
3,smape,all series,21833,11.3340,11.2334,-0.1007,-0.1324,-0.0661,-0.8882,0.5761,0.0,2/3,0.0,KANB better
4,mase,small drift,7614,1.2728,1.3048,0.0320,0.0291,0.0347,2.5139,0.4010,0.0,1/3,0.0,GELU better
5,mase,medium drift,7307,1.2032,1.1790,-0.0241,-0.0274,-0.0208,-2.0047,0.6022,0.0,3/3,0.0,KANB better
6,mase,large drift,6912,1.4441,1.3526,-0.0915,-0.0953,-0.0878,-6.3369,0.7449,0.0,3/3,0.0,KANB better
7,mase,all series,21833,1.3037,1.2778,-0.0259,-0.0279,-0.0238,-1.9855,0.5772,0.0,3/3,0.0,KANB better


**Population drift tests per fold (permutation)**

,fold,kind,mmd2,energy,p_mmd,p_energy
0,0,covariate,0.0013,0.0142,0.045,0.070
1,0,joint,0.0014,0.0174,0.020,0.015
2,1,covariate,0.0178,0.1191,0.005,0.005
3,1,joint,0.0153,0.1087,0.005,0.005
4,2,covariate,0.0125,0.0833,0.005,0.005
5,2,joint,0.0142,0.0968,0.005,0.005


... (41 outputs left out)

**Grad-CAM, global.** Where TimesNet's convolutions look, folded back to time, per TimesBlock; shaded = the future part created by the linear layer.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.

In [ ]:
show9("Monthly")

In [ ]:
cv9_table()          # every group trained so far

,settings,group,TimesNet SMAPE,TimesNet MASE,TimesNet OWA,TimesNet runs,TimesNet_KANB SMAPE,TimesNet_KANB MASE,TimesNet_KANB OWA,TimesNet_KANB runs,KANB better OWA (pairs),Wilcoxon p (OWA)
0,hp_312f7e24,Quarterly,11.154 ± 0.424,1.295 ± 0.051,0.977 ± 0.034,9,11.059 ± 0.268,1.270 ± 0.024,0.964 ± 0.018,9,6/9,0.2500
1,hp_59e9ad1d,Daily,3.263 ± 0.476,3.750 ± 0.529,1.198 ± 0.136,15,3.299 ± 0.450,3.788 ± 0.509,1.211 ± 0.134,15,6/15,0.3894
2,hp_59e9ad1d,Monthly,13.496 ± 0.661,1.081 ± 0.022,1.009 ± 0.018,7,,,,0,NaN,NaN
3,hp_d303f8cc,Yearly,14.468 ± 0.674,3.280 ± 0.143,0.853 ± 0.039,9,14.660 ± 0.471,3.365 ± 0.135,0.869 ± 0.030,9,3/9,0.2500
4,hp_fd3fd374,Yearly,14.759 ± 0.765,3.344 ± 0.204,0.870 ± 0.048,9,14.384 ± 0.275,3.248 ± 0.074,0.846 ± 0.018,9,6/9,0.2031


## 9. Release the GPU


In [ ]:
# Disconnect only in Colab, and only when the results are on Google Drive (otherwise they would be lost)
if AUTO_DISCONNECT and USE_DRIVE and R.in_colab():
    print("All cells finished - releasing the GPU. Results are in", ROOT)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Session kept (AUTO_DISCONNECT / USE_DRIVE is False, or not in Colab).")


In [ ]:
"""
Section 9 setup for the Table 14 notebook (same as cells 9.1 - 9.5 of TimesNet_Table14_Forecasting_S9.ipynb).
Use in Colab, after Sections 1-4 have run:
    from google.colab import files; files.upload()          # choose this file
    exec(open("section9_setup.py").read())
"""
import os
import os as _os
# ---- 9.2 - 9.4: write the model file and the two scripts into the TSLib folder ----
_S9_FILES = {
    'models/TimesNet_KANB.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.fft\nfrom layers.Embed import DataEmbedding\nfrom layers.Conv_Blocks import Inception_Block_V1\n\n# ---------------------------------------------------------------------------\n# TimesNet with KANB activations ("Ours").\n# Identical to TSLib\'s models/TimesNet.py (commit 4e938a1) except for the two\n# activation functions, which are the model\'s only non-linear activations:\n#   * KANB2 replaces nn.GELU() inside every TimesBlock (between the two Inception blocks)\n#   * KANB1 replaces F.gelu in the classification head (before the final Linear layer)\n#\n# KANB: Kanakoudis, Mandalis & Pelekis, "KANB: A Family of Parametric Activation\n# Functions With Trainable Parameters", IEEE Access 13 (2025), 146081-146087.\n#   s = sigmoid(y)\n#   KANB1(y) = 3(1-s)^2 s P1 + 3(1-s) s^2 P2 + s^3        (bounded in [0, 1])\n#   KANB2(y) = 3(1-s)^2 s P1 + 3(1-s) s^2 P2 + y s^3      (-> 0 for y -> -inf, ~y for y -> +inf)\n#   P1, P2: trainable, kept in [0, 1] (clip after every optimiser step, as in the paper),\n#           initialised at KANB1: P1 = 0.2, P2 = 0.8 and KANB2: P1 = 0.8, P2 = 0.2 (every layer).\n# ---------------------------------------------------------------------------\n\n\nclass KANB(nn.Module):\n    """KANB activation with two trainable Bezier ordinates (one pair per layer)."""\n\n    # initial (P1, P2) per variant; both parameters stay trainable\n    INIT = {1: (0.2, 0.8),          # KANB1 (classification head)\n            2: (0.8, 0.2)}          # KANB2 (inside every TimesBlock)\n\n    def __init__(self, variant):\n        super(KANB, self).__init__()\n        assert variant in (1, 2), "variant must be 1 (KANB1) or 2 (KANB2)"\n        self.variant = variant\n        # Fixed starting values, no random draw: every other weight is initialised exactly as\n        # in the original (GELU) TimesNet with the same seed -> a fair comparison.\n        p1, p2 = KANB.INIT[variant]\n        self.p1 = nn.Parameter(torch.tensor([p1]))\n        self.p2 = nn.Parameter(torch.tensor([p2]))\n\n    def forward(self, y):\n        s = torch.sigmoid(y)\n        t = 1.0 - s\n        last = s ** 3 if self.variant == 1 else y * s ** 3\n        return 3.0 * t * t * s * self.p1 + 3.0 * t * s * s * self.p2 + last\n\n    @torch.no_grad()\n    def clip_(self):\n        """Clip01 constraint of the paper: keep P1, P2 in [0, 1]."""\n        self.p1.clamp_(0.0, 1.0)\n        self.p2.clamp_(0.0, 1.0)\n\n    def extra_repr(self):\n        return f"variant=KANB{self.variant}"\n\n\ndef clip_kanb_(model):\n    """Call after every optimiser step."""\n    for m in model.modules():\n        if isinstance(m, KANB):\n            m.clip_()\n\n\ndef kanb_parameters(model):\n    """{module name: (variant, P1, P2)} for every KANB in the model."""\n    return {name: (m.variant, float(m.p1), float(m.p2))\n            for name, m in model.named_modules() if isinstance(m, KANB)}\n\n\ndef FFT_for_Period(x, k=2):\n    # [B, T, C]\n    xf = torch.fft.rfft(x, dim=1)\n    # find period by amplitudes\n    frequency_list = abs(xf).mean(0).mean(-1)\n    frequency_list[0] = 0\n    _, top_list = torch.topk(frequency_list, k)\n    top_list = top_list.detach().cpu().numpy()\n    period = x.shape[1] // top_list\n    return period, abs(xf).mean(-1)[:, top_list]\n\n\nclass TimesBlock(nn.Module):\n    def __init__(self, configs):\n        super(TimesBlock, self).__init__()\n        self.seq_len = configs.seq_len\n        self.pred_len = configs.pred_len\n        self.k = configs.top_k\n        # parameter-efficient design\n        self.conv = nn.Sequential(\n            Inception_Block_V1(configs.d_model, configs.d_ff,\n                               num_kernels=configs.num_kernels),\n            KANB(2),  # [Ours] was nn.GELU()\n            Inception_Block_V1(configs.d_ff, configs.d_model,\n                               num_kernels=configs.num_kernels)\n        )\n\n    def forward(self, x):\n        B, T, N = x.size()\n        period_list, period_weight = FFT_for_Period(x, self.k)\n\n        res = []\n        for i in range(self.k):\n            period = period_list[i]\n            # padding\n            if (self.seq_len + self.pred_len) % period != 0:\n                length = (\n                                 ((self.seq_len + self.pred_len) // period) + 1) * period\n                padding = torch.zeros([x.shape[0], (length - (self.seq_len + self.pred_len)), x.shape[2]]).to(x.device)\n                out = torch.cat([x, padding], dim=1)\n            else:\n                length = (self.seq_len + self.pred_len)\n                out = x\n            # reshape\n            out = out.reshape(B, length // period, period,\n                              N).permute(0, 3, 1, 2).contiguous()\n            # 2D conv: from 1d Variation to 2d Variation\n            out = self.conv(out)\n            # reshape back\n            out = out.permute(0, 2, 3, 1).reshape(B, -1, N)\n            res.append(out[:, :(self.seq_len + self.pred_len), :])\n        res = torch.stack(res, dim=-1)\n        # adaptive aggregation\n        period_weight = F.softmax(period_weight, dim=1)\n        period_weight = period_weight.unsqueeze(\n            1).unsqueeze(1).repeat(1, T, N, 1)\n        res = torch.sum(res * period_weight, -1)\n        # residual connection\n        res = res + x\n        return res\n\n\nclass Model(nn.Module):\n    """\n    TimesNet (paper link: https://openreview.net/pdf?id=ju_Uqw384Oq) with KANB2 in the\n    TimesBlocks and KANB1 in the classification head.\n    """\n\n    def __init__(self, configs):\n        super(Model, self).__init__()\n        self.configs = configs\n        self.task_name = configs.task_name\n        self.seq_len = configs.seq_len\n        self.label_len = configs.label_len\n        self.pred_len = configs.pred_len\n        self.model = nn.ModuleList([TimesBlock(configs)\n                                    for _ in range(configs.e_layers)])\n        self.enc_embedding = DataEmbedding(configs.enc_in, configs.d_model, configs.embed, configs.freq,\n                                           configs.dropout)\n        self.layer = configs.e_layers\n        self.layer_norm = nn.LayerNorm(configs.d_model)\n        if self.task_name == \'long_term_forecast\' or self.task_name == \'short_term_forecast\':\n            self.predict_linear = nn.Linear(\n                self.seq_len, self.pred_len + self.seq_len)\n            self.projection = nn.Linear(\n                configs.d_model, configs.c_out, bias=True)\n        if self.task_name == \'imputation\' or self.task_name == \'anomaly_detection\':\n            self.projection = nn.Linear(\n                configs.d_model, configs.c_out, bias=True)\n        if self.task_name == \'classification\':\n            self.act = KANB(1)  # [Ours] was F.gelu\n            self.dropout = nn.Dropout(configs.dropout)\n            self.projection = nn.Linear(\n                configs.d_model * configs.seq_len, configs.num_class)\n\n    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):\n        # Normalization from Non-stationary Transformer\n        means = x_enc.mean(1, keepdim=True).detach()\n        x_enc = x_enc.sub(means)\n        stdev = torch.sqrt(\n            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)\n        x_enc = x_enc.div(stdev)\n\n        # embedding\n        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]\n        enc_out = self.predict_linear(enc_out.permute(0, 2, 1)).permute(\n            0, 2, 1)  # align temporal dimension\n        # TimesNet\n        for i in range(self.layer):\n            enc_out = self.layer_norm(self.model[i](enc_out))\n        # project back\n        dec_out = self.projection(enc_out)\n\n        # De-Normalization from Non-stationary Transformer\n        dec_out = dec_out.mul(\n                  (stdev[:, 0, :].unsqueeze(1).repeat(\n                      1, self.pred_len + self.seq_len, 1)))\n        dec_out = dec_out.add(\n                  (means[:, 0, :].unsqueeze(1).repeat(\n                      1, self.pred_len + self.seq_len, 1)))\n        return dec_out\n\n    def imputation(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask):\n        # Normalization from Non-stationary Transformer\n        means = torch.sum(x_enc, dim=1) / torch.sum(mask == 1, dim=1)\n        means = means.unsqueeze(1).detach()\n        x_enc = x_enc.sub(means)\n        x_enc = x_enc.masked_fill(mask == 0, 0)\n        stdev = torch.sqrt(torch.sum(x_enc * x_enc, dim=1) /\n                           torch.sum(mask == 1, dim=1) + 1e-5)\n        stdev = stdev.unsqueeze(1).detach()\n        x_enc = x_enc.div(stdev)\n\n        # embedding\n        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]\n        # TimesNet\n        for i in range(self.layer):\n            enc_out = self.layer_norm(self.model[i](enc_out))\n        # project back\n        dec_out = self.projection(enc_out)\n\n        # De-Normalization from Non-stationary Transformer\n        dec_out = dec_out.mul(\n                  (stdev[:, 0, :].unsqueeze(1).repeat(\n                      1, self.pred_len + self.seq_len, 1)))\n        dec_out = dec_out.add(\n                  (means[:, 0, :].unsqueeze(1).repeat(\n                      1, self.pred_len + self.seq_len, 1)))\n        return dec_out\n\n    def anomaly_detection(self, x_enc):\n        # Normalization from Non-stationary Transformer\n        means = x_enc.mean(1, keepdim=True).detach()\n        x_enc = x_enc.sub(means)\n        stdev = torch.sqrt(\n            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)\n        x_enc = x_enc.div(stdev)\n\n        # embedding\n        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]\n        # TimesNet\n        for i in range(self.layer):\n            enc_out = self.layer_norm(self.model[i](enc_out))\n        # project back\n        dec_out = self.projection(enc_out)\n\n        # De-Normalization from Non-stationary Transformer\n        dec_out = dec_out.mul(\n                  (stdev[:, 0, :].unsqueeze(1).repeat(\n                      1, self.pred_len + self.seq_len, 1)))\n        dec_out = dec_out.add(\n                  (means[:, 0, :].unsqueeze(1).repeat(\n                      1, self.pred_len + self.seq_len, 1)))\n        return dec_out\n\n    def classification(self, x_enc, x_mark_enc):\n        # embedding\n        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]\n        # TimesNet\n        for i in range(self.layer):\n            enc_out = self.layer_norm(self.model[i](enc_out))\n\n        # Output\n        # the output transformer encoder/decoder embeddings don\'t include non-linearity\n        output = self.act(enc_out)\n        output = self.dropout(output)\n        # zero-out padding embeddings\n        output = output * x_mark_enc.unsqueeze(-1)\n        # (batch_size, seq_length * d_model)\n        output = output.reshape(output.shape[0], -1)\n        output = self.projection(output)  # (batch_size, num_classes)\n        return output\n\n    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):\n        if self.task_name == \'long_term_forecast\' or self.task_name == \'short_term_forecast\':\n            dec_out = self.forecast(x_enc, x_mark_enc, x_dec, x_mark_dec)\n            return dec_out[:, -self.pred_len:, :]  # [B, L, D]\n        if self.task_name == \'imputation\':\n            dec_out = self.imputation(\n                x_enc, x_mark_enc, x_dec, x_mark_dec, mask)\n            return dec_out  # [B, L, D]\n        if self.task_name == \'anomaly_detection\':\n            dec_out = self.anomaly_detection(x_enc)\n            return dec_out  # [B, L, D]\n        if self.task_name == \'classification\':\n            dec_out = self.classification(x_enc, x_mark_enc)\n            return dec_out  # [B, N]\n        return None',
    'cv_m4.py': '"""\nSliding-window cross-validation with a purge gap for TSLib short-term forecasting on M4\n(TimesNet and TimesNet_KANB), evaluated on the untouched M4 test set.\n\nProtocol (Section 9 of TimesNet_Table14, the M4 version of Section 13 of Table 13):\n* --cv calendar (default; removes M4\'s cross-series leak): one CALENDAR cut-off C_k per fold, the same for all\n  series (quantiles of the series\' last possible validation start, i.e. the date of value L - H). Fold k trains on the values before C_k - gap of every series and\n  validates on the first H values from C_k. A test series is scored only by fold models with C_k <= its forecast\n  start, so no model has seen anything later than a series\' forecast start, from any series.\n  The leak experiment (--variant leakA/leakB, placebo*) retrains a fold with one random half of the series\n  keeping its values after C_k, and scores the other half\'s validation block (see fold_sets).\n* --cv series: inside every series, K validation blocks of one horizon (H) each, consecutive and non-overlapping,\n  ending where the series\' training data ends (does not remove the cross-series leak).\n* Purge gap (default 2H = seq_len) between a fold\'s training data and its validation block: the fold trains\n  only on the series cut at  val_start - gap. The validation INPUT (the seq_len values before the block) lies\n  inside the gap, so no training target ever overlaps a validation input or target.\n* TSLib cuts M4 training examples only from the last history_size x H values of each (cut) series, so every\n  fold has the same amount of training data per series (a sliding window, as in Table 13).\n* K = 5 for Monthly, Weekly, Daily, Hourly; K = 3 for Yearly and Quarterly (short series). A series joins a\n  fold only if at least min_train_h x H values remain for training; counts are reported per fold.\n* The M4 test set is never used for training, validation or early stopping. Every fold model forecasts every\n  test series from its full training history (as TSLib), ONE SERIES AT A TIME (TimesNet picks its FFT periods\n  from the whole batch, so batch 1 makes each forecast depend only on its own input; TSLib\'s test also uses 1).\n* Preprocessing = the notebook\'s fix of Section 2c: aligned targets and first-value padding.\n* Training = TSLib\'s own Exp_Short_Term_Forecast.train (Adam, SMAPE loss, lradj, EarlyStopping). Its vali()\n  is given the fold\'s own validation inputs (the values just before each validation block).\n* KANB: P1, P2 are clipped to [0, 1] after every optimiser step (as in Table 17).\n* Metrics per series: sMAPE, MASE (M4 definitions); per group: sMAPE, MASE, OWA (vs Naive2).\n\nUsage (inside the TSLib repository):\n  python cv_m4.py --make_folds --folds_file folds_m4/Yearly.json -- <TSLib args>\n  python cv_m4.py --folds_file folds_m4/Yearly.json --fold 0 --seed 2021 --out_dir runs/Yearly/TimesNet -- <args>\n"""\nimport argparse\nimport json\nimport os\nimport random\nimport shutil\nimport sys\nimport tempfile\nimport textwrap\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nGROUPS = ["Yearly", "Quarterly", "Monthly", "Weekly", "Daily", "Hourly"]\nHORIZON = {"Yearly": 6, "Quarterly": 8, "Monthly": 18, "Weekly": 13, "Daily": 14, "Hourly": 48}\nFREQ = {"Yearly": 1, "Quarterly": 4, "Monthly": 12, "Weekly": 1, "Daily": 1, "Hourly": 24}\nHISTORY = {"Yearly": 1.5, "Quarterly": 1.5, "Monthly": 1.5, "Weekly": 10, "Daily": 10, "Hourly": 10}\nDEFAULT_FOLDS = {"Yearly": 3, "Quarterly": 3, "Monthly": 5, "Weekly": 5, "Daily": 5, "Hourly": 5}\nSTEP = {"Yearly": pd.DateOffset(years=1), "Quarterly": pd.DateOffset(months=3), "Monthly": pd.DateOffset(months=1),\n        "Weekly": pd.Timedelta(days=7), "Daily": pd.Timedelta(days=1), "Hourly": pd.Timedelta(hours=1)}\n\n\n# ---------------------------------------------------------------------------------------------------------\n# Arguments\n# ---------------------------------------------------------------------------------------------------------\ndef get_tslib_parser():\n    """TSLib\'s own argparse parser, built from run.py, so every flag and default is identical."""\n    with open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "run.py")) as f:\n        src = f.read()\n    start = src.rfind("\\n", 0, src.index("parser = argparse.ArgumentParser")) + 1\n    end = src.rfind("\\n", 0, src.index("args = parser.parse_args()")) + 1\n    namespace = {"argparse": argparse}\n    exec(textwrap.dedent(src[start:end]), namespace)\n    return namespace["parser"]\n\n\ndef parse_all_args(argv=None):\n    argv = sys.argv[1:] if argv is None else argv\n    if "--" not in argv:\n        raise SystemExit("Put TSLib arguments after a `--` separator.")\n    k = argv.index("--")\n    p = argparse.ArgumentParser(description="Sliding-window CV with purge gap on M4")\n    p.add_argument("--make_folds", action="store_true")\n    p.add_argument("--folds_file", required=True)\n    p.add_argument("--n_folds", type=int, default=0, help="0 = 3 for Yearly/Quarterly, 5 otherwise")\n    p.add_argument("--gap_h", type=float, default=2.0, help="purge gap in horizons (2 = seq_len)")\n    p.add_argument("--min_train_h", type=float, default=1.0, help="min training values per series, in horizons")\n    p.add_argument("--cv", default="calendar", choices=["calendar", "series"])\n    p.add_argument("--quantiles", default="", help="calendar CV: quantiles of the last validation starts (cut-offs)")\n    p.add_argument("--ab_seed", type=int, default=0, help="calendar CV: seed of the A/B halves and the placebo")\n    p.add_argument("--fold", type=int, default=0)\n    p.add_argument("--seed", type=int, default=2021)\n    p.add_argument("--variant", default="clean", choices=VARIANTS)\n    p.add_argument("--out_dir", default="./runs_m4")\n    p.add_argument("--save_checkpoint", type=int, default=1)\n    own = p.parse_args(argv[:k])\n    tslib = get_tslib_parser().parse_args(argv[k + 1:])\n    tslib.seasonal_patterns = tslib.seasonal_patterns.strip("\'\\"")\n    H = HORIZON[tslib.seasonal_patterns]                     # as TSLib\'s Exp_Short_Term_Forecast._build_model\n    tslib.pred_len, tslib.seq_len, tslib.label_len = H, 2 * H, H\n    tslib.frequency_map = FREQ[tslib.seasonal_patterns]\n    return own, tslib\n\n\n# ---------------------------------------------------------------------------------------------------------\n# Data\n# ---------------------------------------------------------------------------------------------------------\ndef load_group(root, group):\n    """Training series, test series (lists of arrays), ids and starting dates of one M4 group."""\n    from data_provider.m4 import M4Dataset\n    tr = M4Dataset.load(training=True, dataset_file=root)\n    te = M4Dataset.load(training=False, dataset_file=root)\n    sel = tr.groups == group\n    clean = lambda vals: [np.asarray(v, dtype=np.float64)[~np.isnan(np.asarray(v, dtype=np.float64))] for v in vals]\n    info = pd.read_csv(os.path.join(root, "M4-info.csv"))\n    info = info.set_index("M4id").loc[tr.ids[sel]]\n    return clean(tr.values[sel]), clean(te.values[sel]), np.asarray(tr.ids[sel]), info\n\n\nVARIANTS = ["clean", "leakA", "leakB", "placebo", "placebo_leakA", "placebo_leakB"]\n\n\ndef fold_sets(lengths, spec, k, variant="clean"):\n    """Who trains (and on how many values) and who is validated in fold k, for a variant of the fold.\n\n    series CV  : the last K horizons of every series (only \'clean\').\n    calendar CV: one calendar cut-off C_k for all series; training = values before C_k - gap of every series,\n                 validation = the first H values from C_k of every series that has them. A test series may be\n                 scored by this fold model only if its forecast starts at or after C_k (no leak).\n    leakA/leakB: the leak experiment. Half A (B) is validated and cut as in \'clean\'; the other half keeps its\n                 values AFTER the cut-off (up to its end) - the M4 cross-series leak, on purpose. Same series\n                 and, because TSLib only samples the last history_size x H values, the same amount of data.\n    placebo*   : the same with every series\' calendar position shuffled (start dates permuted), so the \'leak\'\n                 carries no real calendar information.\n    Returns train_idx, t1 (training length per training series), val_idx, v0 (validation start), test_eligible."""\n    H, gap, min_train = spec["H"], spec["gap"], spec["min_train"]\n    L = np.asarray(lengths)\n    N = len(L)\n    if spec.get("cv", "series") == "series":\n        if variant != "clean":\n            raise SystemExit("the leak experiment needs the calendar CV (--cv calendar)")\n        K = spec["n_folds"]\n        v0 = L - (K - k) * H\n        t1 = v0 - gap\n        idx = np.where(t1 >= min_train)[0]\n        return idx, t1[idx], idx, v0[idx], np.ones(N, bool)\n    cal = spec["cal"]\n    key = "j_placebo" if variant.startswith("placebo") else "j"\n    j = np.asarray(cal[key][k])\n    dated = np.asarray(cal["dated"], bool)\n    t1_raw = j - gap\n    t1_all = np.minimum(L, t1_raw)\n    train_mask = dated & (t1_all >= min_train)\n    val_mask = dated & (t1_raw >= min_train) & (j + H <= L)\n    eligible = dated & (L >= j)\n    t1 = t1_all.copy()\n    if variant.endswith("leakA") or variant.endswith("leakB"):\n        ab = np.asarray(cal["ab"])\n        ev = 0 if variant.endswith("leakA") else 1\n        val_mask = val_mask & (ab == ev)\n        leaky = train_mask & (ab != ev)\n        t1[leaky] = L[leaky]                        # the other half keeps its values after the cut-off\n    tr = np.where(train_mask)[0]\n    va = np.where(val_mask)[0]\n    return tr, t1[tr], va, j[va], eligible\n\n\ndef parse_dates(info, group):\n    d = pd.to_datetime(info["StartingDate"], format="%d-%m-%y %H:%M", errors="coerce")\n    return d\n\n\ndef date_at(start, step, n):\n    try:\n        return start + step * int(n)\n    except Exception:  # noqa: BLE001 (out of bounds)\n        return pd.NaT\n\n\nMONTHS = {"Yearly": 12, "Quarterly": 3, "Monthly": 1}\n\n\ndef steps_until(start, cutoff, group):\n    """Index of the first value of a series (starting at `start`) whose date is >= cutoff."""\n    step = STEP[group]\n    try:\n        if group in MONTHS:\n            md = (cutoff.year - start.year) * 12 + (cutoff.month - start.month)\n            j = int(np.floor(md / MONTHS[group])) - 1\n            while start + step * j < cutoff:\n                j += 1\n            while start + step * (j - 1) >= cutoff:\n                j -= 1\n            return j\n        return int(np.ceil((cutoff - start) / step))\n    except Exception:  # noqa: BLE001\n        return None\n\n\ndef calendar_report(train, info, spec, group, rng):\n    """How much training data lies, in calendar time, AFTER the first predicted value of OTHER series (the\n    cross-series leak of M4): share of (last training value, other series\' first predicted value) pairs where\n    the training value is later. Per fold (should be ~0 for the calendar CV) and for the paper\'s test protocol\n    (models trained on the full histories)."""\n    starts = parse_dates(info, group)\n    step = STEP[group]\n    L = np.array([len(s) for s in train])\n    out = {"unparsable_dates": int(starts.isna().sum()), "series": len(L)}\n\n    def ns(x):\n        return np.array([v.value if not pd.isna(v) else np.nan for v in x], dtype=float)\n\n    def share(train_dates, target_dates):\n        a, b = train_dates[np.isfinite(train_dates)], np.sort(target_dates[np.isfinite(target_dates)])\n        if len(a) == 0 or len(b) == 0:\n            return None\n        return float(np.mean(np.searchsorted(b, a, side="right") / len(b)))\n\n    sub = np.sort(rng.choice(len(L), min(len(L), 3000), replace=False))\n    rows = []\n    for k in list(range(spec["n_folds"])) + ["test"]:\n        if k == "test":\n            tr_idx, t1, va_idx, v0 = np.arange(len(L)), L, np.arange(len(L)), L\n        else:\n            tr_idx, t1, va_idx, v0, _ = fold_sets(L, spec, k)\n        kt, kv = np.isin(tr_idx, sub), np.isin(va_idx, sub)\n        tr_dates = ns([date_at(starts.iloc[i], step, t - 1) for i, t in zip(tr_idx[kt], t1[kt])])\n        target = ns([date_at(starts.iloc[i], step, v) for i, v in zip(va_idx[kv], v0[kv])])\n        rows.append({"fold": "M4 test (paper protocol)" if k == "test" else k,\n                     "share of training values after other series\' target start": share(tr_dates, target),\n                     "median target start": str(pd.to_datetime(np.nanmedian(target))) if np.isfinite(target).any()\n                     else None})\n    out["folds"] = rows\n    return out\n\n\ndef make_folds(args, own):\n    group = args.seasonal_patterns\n    train, test, ids, info = load_group(args.root_path, group)\n    H = HORIZON[group]\n    K = own.n_folds or DEFAULT_FOLDS[group]\n    spec = {"group": group, "cv": own.cv, "H": H, "seq_len": 2 * H, "freq": FREQ[group], "n_folds": K,\n            "gap": int(round(own.gap_h * H)), "gap_h": own.gap_h, "min_train": int(round(own.min_train_h * H)),\n            "series": len(train), "history_size": HISTORY[group]}\n    L = np.array([len(s) for s in train])\n    if own.cv == "calendar":\n        starts = parse_dates(info, group)\n        step = STEP[group]\n        dated = starts.notna().values\n        # last possible validation start of each series = the date of its value L - H. Cut-off C_k = the q_k\n        # quantile of these dates: a share 1 - q_k of the series has a full validation block after C_k, and every\n        # series whose forecast starts at or after C_k (at least that share) is scored leak-free on the M4 test.\n        last_val = pd.Series([date_at(s, step, n - H) if d else pd.NaT for s, n, d in zip(starts, L, dated)])\n        dated &= last_val.notna().values\n        qs = [float(q) for q in own.quantiles.split(",")] if own.quantiles else \\\n            ([0.10, 0.30, 0.50] if K == 3 else list(np.round(np.linspace(0.10, 0.50, K), 3)))\n        assert len(qs) == K, "one quantile per fold"\n        ordinal = np.array([t.value for t in last_val[dated]], dtype=np.int64)\n        cutoffs = [pd.Timestamp(int(np.quantile(ordinal, q, method="lower"))) for q in qs]\n        if len(set(cutoffs)) < len(cutoffs) and not own.quantiles:\n            # many M4 series share the same dates: take K distinct dates from a finer grid of quantiles instead\n            grid = np.round(np.linspace(0.05, 0.70, 131), 4)\n            cand = {}\n            for q in grid:\n                cand.setdefault(int(np.quantile(ordinal, q, method="lower")), q)\n            if len(cand) >= K:\n                keys = sorted(cand)\n                pick = [keys[int(round(i))] for i in np.linspace(0, len(keys) - 1, K)]\n                cutoffs, qs = [pd.Timestamp(v) for v in pick], [float(cand[v]) for v in pick]\n                print(f"note: cut-offs coincided (shared M4 dates); using quantiles {qs} instead")\n            else:\n                print("note: some calendar cut-offs coincide (many series share the same dates) - those folds are "\n                      "identical apart from the seed")\n        rng = np.random.default_rng(own.ab_seed)\n        perm = np.arange(len(L))\n        di = np.where(dated)[0]\n        perm[di] = rng.permutation(di)                      # placebo: start dates shuffled among dated series\n        J, JP = [], []\n        for C in cutoffs:\n            jj = [steps_until(starts.iloc[i], C, group) if dated[i] else None for i in range(len(L))]\n            jp = [steps_until(starts.iloc[perm[i]], C, group) if dated[i] else None for i in range(len(L))]\n            J.append([int(v) if v is not None else -10 ** 9 for v in jj])\n            JP.append([int(v) if v is not None else -10 ** 9 for v in jp])\n        bad = np.array([v == -10 ** 9 for v in J[0]])\n        dated &= ~bad\n        ab = rng.integers(0, 2, len(L))\n        spec["cal"] = {"quantiles": qs, "cutoffs": [str(c) for c in cutoffs], "dated": dated.astype(int).tolist(),\n                       "j": J, "j_placebo": JP, "ab": ab.astype(int).tolist(), "ab_seed": own.ab_seed}\n        spec["protocol"] = ("calendar-aware sliding CV: one calendar cut-off per fold for all series (quantiles of the "\n                            "series\' last possible validation start); training = values before cut-off - gap; validation = the first H values from the cut-off; a test series is "\n                            "scored only by fold models whose cut-off is not later than its forecast start (no "\n                            "cross-series leak)")\n    else:\n        spec["protocol"] = ("per-series sliding-window CV: K consecutive validation blocks of H values ending at the end "\n                            "of each training series; purge gap before every block; test = M4 test set, untouched")\n    folds = []\n    for k in range(K):\n        tr, t1, va, v0, el = fold_sets(L, spec, k)\n        row = {"fold": k, "series": int(len(va)), "training_series": int(len(tr)),\n               "share_of_group": float(len(va) / len(L)), "median_train_length": int(np.median(t1)) if len(t1) else 0,\n               "test_series_scored": int(el.sum())}\n        if own.cv == "calendar":\n            row["cutoff"] = spec["cal"]["cutoffs"][k]\n            row["validation_block"] = f"first {H} values from {spec[\'cal\'][\'cutoffs\'][k][:10]}"\n            assert np.all(v0 + H <= L[va]) and np.all(v0 - spec["gap"] >= spec["min_train"])\n        else:\n            row["validation_block"] = (f"values L-{(K - k) * H} .. L-{(K - k - 1) * H - 1}" if k < K - 1\n                                       else f"values L-{H} .. L-1")\n            assert np.all(t1 + spec["gap"] == v0) and np.all(v0 + H <= L[va])\n        folds.append(row)\n    spec["folds"] = folds\n    spec["calendar"] = calendar_report(train, info, spec, group, np.random.default_rng(0))\n    spec["length_quantiles"] = {str(q): int(np.quantile(L, q)) for q in (0.0, 0.25, 0.5, 0.75, 1.0)}\n    return spec\n\n\ndef window(ts, cut, seq_len, label_len, pred_len):\n    """One training example with the notebook\'s preprocessing fix (Section 2c): input = the seq_len values\n    before `cut`, padded with the first observed value; target placed so position label_len is the cut."""\n    insample = np.zeros((seq_len, 1))\n    insample_mask = np.zeros((seq_len, 1))\n    outsample = np.zeros((pred_len + label_len, 1))\n    outsample_mask = np.zeros((pred_len + label_len, 1))\n    w = ts[max(0, cut - seq_len):cut]\n    insample[:, 0] = w[0]\n    insample[-len(w):, 0] = w\n    insample_mask[-len(w):, 0] = 1.0\n    start = max(0, cut - label_len)\n    o = ts[start:min(len(ts), cut + pred_len)]\n    off = label_len - (cut - start)\n    outsample[off:off + len(o), 0] = o\n    outsample_mask[off:off + len(o), 0] = 1.0\n    return insample, outsample, insample_mask, outsample_mask\n\n\ndef last_window(series, seq_len):\n    """Inputs [n, seq_len] = the last seq_len values of each series (first-value padding), and masks."""\n    x = np.zeros((len(series), seq_len))\n    m = np.zeros((len(series), seq_len))\n    for i, ts in enumerate(series):\n        w = ts[-seq_len:]\n        x[i, :] = w[0]\n        x[i, -len(w):] = w\n        m[i, -len(w):] = 1.0\n    return x, m\n\n\nclass FoldM4:\n    """Datasets of one fold (and variant), in TSLib\'s interface (Dataset_M4)."""\n\n    def __init__(self, train, spec, k, variant="clean"):\n        tr, t1, va, v0, el = fold_sets([len(s) for s in train], spec, k, variant)\n        self.train_idx, self.t1, self.idx, self.v0, self.test_eligible = tr, t1, va, v0, el\n        self.train = [train[i][:t] for i, t in zip(tr, t1)]\n        self.val_in = [train[i][:v] for i, v in zip(va, v0)]\n        self.val_out = [train[i][v:v + spec["H"]] for i, v in zip(va, v0)]\n\n\ndef make_dataset_class(fold, args):\n    from torch.utils.data import Dataset\n    limit = int(HISTORY[args.seasonal_patterns] * args.pred_len)\n\n    class Dataset_M4_CV(Dataset):\n        def __init__(self, args, root_path, flag="train", size=None, **kw):\n            self.seq_len, self.label_len, self.pred_len = size\n            self.flag = flag\n            self.window_sampling_limit = limit\n            if flag == "train":\n                self.timeseries = fold.train\n            else:                                   # \'val\' (and never the M4 test set)\n                self.timeseries = fold.val_out\n                self.inputs = fold.val_in\n            self.ids = fold.idx if flag != "train" else fold.train_idx\n\n        def __len__(self):\n            return len(self.timeseries)\n\n        def __getitem__(self, index):                # same random cut point as TSLib\n            ts = self.timeseries[index]\n            cut = np.random.randint(low=max(1, len(ts) - self.window_sampling_limit), high=len(ts), size=1)[0]\n            return window(ts, cut, self.seq_len, self.label_len, self.pred_len)\n\n        def last_insample_window(self):\n            return last_window(self.inputs if self.flag != "train" else self.timeseries, self.seq_len)\n\n    return Dataset_M4_CV\n\n\n# ---------------------------------------------------------------------------------------------------------\n# Metrics (M4 definitions) and Naive2\n# ---------------------------------------------------------------------------------------------------------\ndef smape_series(y, f):\n    d = np.abs(y) + np.abs(f)\n    return 200 * np.mean(np.where(d > 0, np.abs(y - f) / np.where(d > 0, d, 1), 0.0), axis=-1)\n\n\ndef mase_scale(x, m):\n    if len(x) <= m:\n        return np.nan\n    return float(np.mean(np.abs(x[m:] - x[:-m])))\n\n\ndef _acf(x, k):\n    m = x.mean()\n    return np.sum((x[:-k] - m) * (x[k:] - m)) / np.sum((x - m) ** 2)\n\n\ndef naive2(x, h, m):\n    """M4\'s Naive2 benchmark (seasonal naive on the classical multiplicative decomposition, 90% test)."""\n    x = np.asarray(x, float)\n    si = np.full(m, 100.0)\n    if m > 1 and len(x) > m:\n        s = _acf(x, 1)\n        for i in range(2, m):\n            s += _acf(x, i) ** 2\n        limit = 1.645 * np.sqrt((1 + 2 * s) / len(x))\n        if abs(_acf(x, m)) > limit:\n            ts = pd.Series(x)\n            if len(x) % 2 == 0:\n                ma = ts.rolling(m, center=True).mean().rolling(2, center=True).mean()\n                ma = np.roll(ma.values, -1)\n            else:\n                ma = ts.rolling(m, center=True).mean().values\n            le = x * 100 / ma\n            le = np.hstack((le, np.full((m - (len(le) % m)) % m, np.nan)))\n            si = np.nanmean(le.reshape(-1, m), 0)\n            si = si / (np.sum(si) / (m * 100))\n    full = np.tile(si, (len(x) + h) // m + 1)[:len(x) + h]\n    des = x * 100 / full[:len(x)]\n    return des[-1] * full[len(x):] / 100\n\n\ndef naive2_forecasts(root, group, train, ids, h, m):\n    path = os.path.join(root, "submission-Naive2.csv")\n    if os.path.exists(path):\n        df = pd.read_csv(path)\n        df = df.set_index(df.columns[0])\n        if np.isin(ids, df.index).all():\n            return df.loc[ids].values[:, :h].astype(float), "submission-Naive2.csv"\n        print("note: submission-Naive2.csv ids do not match - Naive2 computed with the M4 benchmark code")\n    return np.stack([naive2(x, h, m) for x in train]), "computed (M4 benchmark code)"\n\n\n# ---------------------------------------------------------------------------------------------------------\n# Training\n# ---------------------------------------------------------------------------------------------------------\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    if torch.cuda.is_available():\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\ndef model_module(model):\n    return sys.modules[type(model).__module__]\n\n\ndef kanb_parameters(model):\n    f = getattr(model_module(model), "kanb_parameters", None)\n    return f(model) if f else None\n\n\n@torch.no_grad()\ndef forecast(model, args, x, device, batch_size=1):\n    """Forecasts [n, H] from inputs x [n, seq_len] (TSLib\'s decoder input), batch 1 by default."""\n    model.eval()\n    out = np.zeros((len(x), args.pred_len))\n    for i in range(0, len(x), batch_size):\n        xb = torch.tensor(x[i:i + batch_size], dtype=torch.float32, device=device).unsqueeze(-1)\n        dec = torch.zeros((len(xb), args.pred_len, 1), device=device)\n        dec = torch.cat([xb[:, -args.label_len:, :], dec], dim=1)\n        out[i:i + batch_size] = model(xb, None, dec, None)[:, -args.pred_len:, 0].cpu().numpy()\n    return out\n\n\ndef main():\n    own, args = parse_all_args()\n    if own.make_folds:\n        spec = make_folds(args, own)\n        os.makedirs(os.path.dirname(os.path.abspath(own.folds_file)), exist_ok=True)\n        with open(own.folds_file, "w") as fh:\n            json.dump(spec, fh, indent=1)\n        print("FOLDS", json.dumps({k: spec[k] for k in ("group", "H", "n_folds", "gap", "series")}),\n              "| series per fold:", [f["series"] for f in spec["folds"]])\n        return\n\n    spec = json.load(open(own.folds_file))\n    group = args.seasonal_patterns\n    assert spec["group"] == group, "folds file of another group"\n    args.use_gpu = bool(torch.cuda.is_available() and args.use_gpu)\n    args.device = torch.device("cuda:0" if args.use_gpu else "cpu")\n    set_seed(own.seed)\n    train, test, ids, info = load_group(args.root_path, group)\n    fold = FoldM4(train, spec, own.fold, own.variant)\n    tmp = tempfile.mkdtemp(prefix="cvm4_")\n    args.checkpoints = tmp\n\n    import data_provider.data_factory as DF\n    import exp.exp_short_term_forecasting as EST\n    import utils.tools as T\n    DF.data_dict["m4"] = make_dataset_class(fold, args)\n\n    def vali(self, train_loader, vali_loader, criterion):\n        """TSLib\'s vali(), with the fold\'s own validation inputs (the values just before each block)."""\n        x, _ = vali_loader.dataset.last_insample_window()\n        y = vali_loader.dataset.timeseries\n        x = torch.tensor(x, dtype=torch.float32).to(self.device).unsqueeze(-1)\n        self.model.eval()\n        with torch.no_grad():\n            B, _, C = x.shape\n            dec = torch.zeros((B, self.args.pred_len, C)).float().to(self.device)\n            dec = torch.cat([x[:, -self.args.label_len:, :], dec], dim=1).float()\n            out = torch.zeros((B, self.args.pred_len, C)).float()\n            ids_ = np.append(np.arange(0, B, 500), B)\n            for i in range(len(ids_) - 1):\n                out[ids_[i]:ids_[i + 1]] = self.model(x[ids_[i]:ids_[i + 1]], None, dec[ids_[i]:ids_[i + 1]],\n                                                      None).detach().cpu()[:, -self.args.pred_len:, :]\n            true = torch.from_numpy(np.array(y))\n            loss = criterion(x.detach().cpu()[:, :, 0], self.args.frequency_map, out[:, :, 0], true,\n                             torch.ones(true.shape))\n        self.model.train()\n        return loss\n\n    EST.Exp_Short_Term_Forecast.vali = vali\n    history = {"vali": []}\n    original_call = T.EarlyStopping.__call__\n\n    def recording_call(self, val_loss, model, path):\n        history["vali"].append(float(val_loss))\n        return original_call(self, val_loss, model, path)\n\n    EST.EarlyStopping.__call__ = recording_call\n    original_opt = EST.Exp_Short_Term_Forecast._select_optimizer\n\n    def select_optimizer(self):\n        opt = original_opt(self)\n        clip = getattr(model_module(self.model), "clip_kanb_", None)\n        if clip is not None:\n            opt.register_step_post_hook(lambda *_: clip(self.model))\n        return opt\n\n    EST.Exp_Short_Term_Forecast._select_optimizer = select_optimizer\n\n    t0 = time.time()\n    exp = EST.Exp_Short_Term_Forecast(args)\n    n_params = sum(p.numel() for p in exp.model.parameters())\n    kanb_start = kanb_parameters(exp.model)\n    setting = f"m4_{group}_{args.model}_fold{own.fold}_seed{own.seed}_{own.variant}"\n    exp.train(setting)\n    train_min = (time.time() - t0) / 60\n    model = exp.model\n\n    H, m = args.pred_len, FREQ[group]\n    # ---- validation blocks of this fold (early-stopping data; also the leak experiment\'s evaluation)\n    x_val, _ = last_window(fold.val_in, args.seq_len)\n    vpred = forecast(model, args, x_val, exp.device)\n    vy = np.stack(fold.val_out)\n    vscale = np.array([mase_scale(s, m) for s in fold.val_in])\n    v_sm = smape_series(vy, vpred)\n    v_ms = np.mean(np.abs(vy - vpred), axis=1) / vscale\n    common = {"group": group, "model": args.model, "fold": own.fold, "seed": own.seed, "variant": own.variant, "H": H,\n              "cv": spec.get("cv", "series"), "val_smape": float(np.mean(v_sm)), "val_mase": float(np.nanmean(v_ms)),\n              "n_val_series": int(len(fold.idx)), "n_train_series": int(len(fold.train_idx)),\n              "epochs": len(history["vali"]), "vali_losses": history["vali"],\n              "best_vali": float(min(history["vali"])) if history["vali"] else None, "train_minutes": train_min,\n              "n_params": int(n_params), "kanb_start": kanb_start, "kanb_end": kanb_parameters(model),\n              "gap": spec["gap"], "tslib_args": {k: v for k, v in vars(args).items()\n                                                 if isinstance(v, (int, float, str, bool, type(None)))}}\n    if own.variant != "clean":\n        # leak experiment / placebo: only the validation half is scored; nothing else is needed\n        out_dir = os.path.join(own.out_dir, "leak")\n        os.makedirs(out_dir, exist_ok=True)\n        stem = os.path.join(out_dir, f"fold{own.fold}__seed{own.seed}__{own.variant}")\n        np.savez_compressed(stem + ".npz", val_idx=fold.idx, val_smape=v_sm.astype(np.float32),\n                            val_mase=v_ms.astype(np.float32), val_pred=vpred.astype(np.float32))\n        with open(stem + ".json.part", "w") as fh:\n            json.dump(common, fh, indent=1)\n        os.replace(stem + ".json.part", stem + ".json")\n        shutil.rmtree(tmp, ignore_errors=True)\n        print("FOLD_RESULT", json.dumps({k: common[k] for k in ("model", "group", "fold", "seed", "variant", "val_smape",\n                                                                 "epochs", "train_minutes")}))\n        return\n    # ---- test: every series of the group, from its full training history, one series at a time; a series counts\n    #      only if this fold model is leak-free for it (calendar CV: its forecast starts at or after the cut-off)\n    t1 = time.time()\n    x_test, _ = last_window(train, args.seq_len)\n    pred = forecast(model, args, x_test, exp.device)\n    test_min = (time.time() - t1) / 60\n    y = np.stack(test)\n    el = fold.test_eligible\n    scale = np.array([mase_scale(s, m) for s in train])\n    sm = smape_series(y, pred)\n    ms = np.mean(np.abs(y - pred), axis=1) / scale\n    n2, n2_src = naive2_forecasts(args.root_path, group, train, ids, H, m)\n    sm_n2, ms_n2 = smape_series(y, n2), np.mean(np.abs(y - n2), axis=1) / scale\n    owa = 0.5 * (np.nanmean(sm[el]) / np.nanmean(sm_n2[el]) + np.nanmean(ms[el]) / np.nanmean(ms_n2[el]))\n    res = dict(common, smape=float(np.mean(sm[el])), mase=float(np.nanmean(ms[el])), owa=float(owa),\n               naive2_smape=float(np.mean(sm_n2[el])), naive2_mase=float(np.nanmean(ms_n2[el])),\n               naive2_source=n2_src, n_test_series=int(len(train)), n_test_scored=int(el.sum()),\n               test_minutes=test_min, test_batch_size=1)\n    os.makedirs(own.out_dir, exist_ok=True)\n    stem = os.path.join(own.out_dir, f"fold{own.fold}__seed{own.seed}")\n    np.savez_compressed(stem + "_test.npz", pred=pred.astype(np.float32), smape=sm.astype(np.float32),\n                        mase=ms.astype(np.float32), scale=scale.astype(np.float32), test_eligible=el,\n                        err_scaled=((pred - y) / scale[:, None]).mean(1).astype(np.float32),\n                        val_idx=fold.idx, val_pred=vpred.astype(np.float32), val_smape=v_sm.astype(np.float32),\n                        val_mase=v_ms.astype(np.float32),\n                        val_err_scaled=((vpred - vy) / vscale[:, None]).mean(1).astype(np.float32),\n                        naive2_smape=sm_n2.astype(np.float32), naive2_mase=ms_n2.astype(np.float32))\n    if own.save_checkpoint:\n        shutil.copy2(os.path.join(tmp, setting, "checkpoint.pth"), stem + ".pth")\n    with open(stem + ".json.part", "w") as fh:\n        json.dump(res, fh, indent=1)\n    os.replace(stem + ".json.part", stem + ".json")\n    shutil.rmtree(tmp, ignore_errors=True)\n    print("FOLD_RESULT", json.dumps({k: res[k] for k in ("model", "group", "fold", "seed", "smape", "mase", "owa",\n                                                          "n_test_scored", "epochs", "train_minutes")}))\n\n\nif __name__ == "__main__":\n    main()',
    'xai_m4.py': '"""\nSection 9 of TimesNet_Table14 (M4): explainability, drift and TimesNet (GELU) vs TimesNet-KANB, for the fold\nmodels of cv_m4.py. The M4 version of Section 13 of the Table 13 notebook. No training happens here.\n\nM4 differs from ETTh1 in three ways, and every analysis follows from them:\n* one variable per series -> the features are TIME SEGMENTS of the input (lags), not variables;\n* ~100,000 short, separate series of very different scales -> everything is expressed in each series\'\n  own normalised units (value - input mean) / input std, exactly the units TimesNet works in internally,\n  and replacements for SHAP are real training windows of OTHER series, rescaled to the series\' level/scale;\n* the unit of comparison is the SERIES (independent) -> paired Wilcoxon signed-rank tests with bootstrap\n  confidence intervals and Holm\'s correction (Diebold-Mariano needs a time order, which series do not have).\n\nAnalyses (--analyses, comma separated or "all"):\n  periods   which periods TimesNet\'s FFT picks per series vs the group\'s seasonality, and over calendar time\n  shap      permutation SHAP on time segments of the input (exact additivity), global, over calendar time,\n            beeswarm, local waterfall + decision plots\n  gradshap  GradientSHAP per lag (fine grained), seasonal lags marked\n  gradshap_local  GradientSHAP of single series: one signed bar per input step (red raised, blue lowered)\n  gradcam   Grad-CAM on TimesNet\'s 2D period images (rows = cycles, columns = position in the period),\n            folded back to time, future part shaded; global profile and single series\n  faith     faithfulness (deletion with real replacements) + stability across seeds, folds and models\n  ablation  segment ablation: one input segment replaced by the same segment of another real series\n  drift     per-series drift (conformal p-values against the series\' own history, Benjamini-Hochberg),\n            population covariate and concept drift (MMD, energy distance, domain classifier with SHAP),\n            novelty, residual-based concept drift, drift terciles with Wilcoxon + bootstrap + Holm, calendar leak\n  gallery   worst series: input, truth and both models\' forecasts\n"""\nimport argparse\nimport glob\nimport importlib\nimport json\nimport os\nimport time\nfrom argparse import Namespace\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nimport cv_m4 as CVM\n\nCOLORS = {"TimesNet": "#2B6CB0", "TimesNet_KANB": "#C05621"}\nLIGHT = {"TimesNet": "rgba(43,108,176,0.18)", "TimesNet_KANB": "rgba(192,86,33,0.18)"}\nLABEL = {"TimesNet": "TimesNet (GELU)", "TimesNet_KANB": "TimesNet-KANB (ours)"}\nPAL = ["#2B6CB0", "#DD6B20", "#2F855A", "#6B46C1", "#B7791F", "#C53030", "#319795", "#D53F8C", "#4A5568",\n       "#975A16", "#2C5282", "#9C4221"]\nlabel = lambda m: LABEL.get(m, m)\ncolor = lambda m: COLORS.get(m, "#4A5568")\nFIGURES = []\n\n\ndef save(fig, out_dir, name, width=1200, height=650):\n    os.makedirs(out_dir, exist_ok=True)\n    fig.update_layout(template="plotly_white", font=dict(family="Arial", size=12), title_font=dict(size=16),\n                      width=width, height=height)\n    fig.write_html(os.path.join(out_dir, name + ".html"), include_plotlyjs="directory")\n    fig.write_json(os.path.join(out_dir, name + ".json"))\n    try:\n        fig.write_image(os.path.join(out_dir, name + ".png"), scale=2)\n    except Exception as e:  # noqa: BLE001\n        print(f"PNG export failed for {name}: {e}")\n    FIGURES.append(name)\n\n\ndef title(main, sub):\n    return f"<b>{main}</b><br><sup>{sub}</sup>"\n\n\n# =========================================================================================================\n# Statistics\n# =========================================================================================================\ndef bh(p):\n    p = np.asarray(p, float)\n    q = np.full_like(p, np.nan)\n    ok = np.isfinite(p)\n    if ok.sum() == 0:\n        return q\n    pv = p[ok]\n    order = np.argsort(pv)\n    ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)\n    ranked = np.minimum.accumulate(ranked[::-1])[::-1]\n    out = np.empty_like(pv)\n    out[order] = np.minimum(ranked, 1)\n    q[ok] = out\n    return q\n\n\ndef holm(p):\n    p = np.asarray(p, float)\n    out = np.full_like(p, np.nan)\n    ok = np.where(np.isfinite(p))[0]\n    order = ok[np.argsort(p[ok])]\n    run = 0.0\n    for r, i in enumerate(order):\n        run = max(run, min(1.0, (len(order) - r) * p[i]))\n        out[i] = run\n    return out\n\n\ndef paired_test(d, rng, n_boot=2000):\n    """Paired comparison on per-series differences d (KANB - GELU): Wilcoxon signed-rank p, mean and median\n    difference with bootstrap 95% CIs (over series), share of series where KANB is better."""\n    from scipy.stats import wilcoxon\n    d = np.asarray(d, float)\n    d = d[np.isfinite(d)]\n    n = len(d)\n    if n < 6:\n        return {"n": n, "mean": float(np.mean(d)) if n else np.nan, "p": np.nan}\n    p = wilcoxon(d, zero_method="zsplit").pvalue if np.any(d != 0) else 1.0\n    idx = rng.integers(0, n, size=(n_boot, n))\n    bm = d[idx].mean(1)\n    bmed = np.median(d[idx], axis=1)\n    return {"n": n, "mean": float(d.mean()), "mean_lo": float(np.quantile(bm, 0.025)),\n            "mean_hi": float(np.quantile(bm, 0.975)), "median": float(np.median(d)),\n            "median_lo": float(np.quantile(bmed, 0.025)), "median_hi": float(np.quantile(bmed, 0.975)),\n            "kanb_better_share": float((d < 0).mean()), "p": float(p)}\n\n\ndef median_gamma(ref, rng, max_n=600):\n    from scipy.spatial.distance import pdist\n    r = ref[rng.choice(len(ref), min(len(ref), max_n), replace=False)]\n    d = pdist(r, "sqeuclidean")\n    med = np.median(d[d > 0]) if (d > 0).any() else 1.0\n    return 1.0 / med\n\n\ndef mmd2(A, B, gamma):\n    from scipy.spatial.distance import cdist\n    Kaa = np.exp(-gamma * cdist(A, A, "sqeuclidean"))\n    Kbb = np.exp(-gamma * cdist(B, B, "sqeuclidean"))\n    Kab = np.exp(-gamma * cdist(A, B, "sqeuclidean"))\n    m, n = len(A), len(B)\n    return float((Kaa.sum() - np.trace(Kaa)) / (m * (m - 1)) + (Kbb.sum() - np.trace(Kbb)) / (n * (n - 1))\n                 - 2 * Kab.mean())\n\n\ndef energy(A, B):\n    from scipy.spatial.distance import cdist\n    A = A.reshape(len(A), -1)\n    B = B.reshape(len(B), -1)\n    return float(2 * cdist(A, B).mean() - cdist(A, A).mean() - cdist(B, B).mean())\n\n\ndef energy_1d(a, b):\n    """Energy distance between two samples of numbers: 2E|A-B| - E|A-A\'| - E|B-B\'|."""\n    a, b = np.asarray(a, float).ravel(), np.asarray(b, float).ravel()\n    return float(2 * np.abs(a[:, None] - b[None, :]).mean() - np.abs(a[:, None] - a[None, :]).mean()\n                 - np.abs(b[:, None] - b[None, :]).mean())\n\n\ndef lowess_line(x, y, frac=0.5):\n    try:\n        from statsmodels.nonparametric.smoothers_lowess import lowess\n        r = lowess(y, x, frac=frac)\n        return r[:, 0], r[:, 1]\n    except Exception:  # noqa: BLE001\n        o = np.argsort(x)\n        k = max(3, len(x) // 5)\n        return np.asarray(x)[o], pd.Series(np.asarray(y)[o]).rolling(k, center=True, min_periods=1).mean().values\n\n\n# =========================================================================================================\n# Runs, models and TimesNet\'s period selection\n# =========================================================================================================\ndef find_runs(run_root, model, folds=None, seeds=None):\n    out = []\n    for js in sorted(glob.glob(os.path.join(run_root, model, "fold*__seed*.json"))):\n        stem = js[:-5]\n        res = json.load(open(js))\n        if folds is not None and res["fold"] not in folds:\n            continue\n        if seeds is not None and res["seed"] not in seeds:\n            continue\n        if os.path.exists(stem + ".pth") and os.path.exists(stem + "_test.npz"):\n            out.append({"model": model, "fold": res["fold"], "seed": res["seed"], "res": res,\n                        "pth": stem + ".pth", "npz": stem + "_test.npz"})\n    return out\n\n\nPER = {"mode": "normal", "rec": [], "i": 0, "log": None}\n\n\ndef fft_for_period(x, k=2):\n    xf = torch.fft.rfft(x, dim=1)\n    amp = abs(xf).mean(-1)\n    if PER["mode"] == "force":\n        top = PER["rec"][PER["i"]]\n        PER["i"] += 1\n    else:                                                       # TSLib\'s own rule, unchanged\n        fl = abs(xf).mean(0).mean(-1)\n        fl[0] = 0\n        _, top = torch.topk(fl, k)\n        top = top.detach().cpu().numpy()\n        if PER["mode"] == "record":\n            PER["rec"].append(top)\n    period = x.shape[1] // top\n    if PER["log"] is not None:\n        w = torch.softmax(amp[:, top], dim=1)[0].detach().cpu().numpy()\n        PER["log"].append((period.tolist(), w.tolist()))\n    return period, amp[:, top]\n\n\ndef forecast_full_gradient(self, x_enc, x_mark_enc, x_dec, x_mark_dec):\n    """TSLib\'s forecast() without .detach() on the window mean (same values; full gradient for SHAP)."""\n    means = x_enc.mean(1, keepdim=True)\n    x_enc = x_enc.sub(means)\n    stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)\n    x_enc = x_enc.div(stdev)\n    enc_out = self.enc_embedding(x_enc, x_mark_enc)\n    enc_out = self.predict_linear(enc_out.permute(0, 2, 1)).permute(0, 2, 1)\n    for i in range(self.layer):\n        enc_out = self.layer_norm(self.model[i](enc_out))\n    dec_out = self.projection(enc_out)\n    dec_out = dec_out.mul(stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len + self.seq_len, 1))\n    dec_out = dec_out.add(means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len + self.seq_len, 1))\n    return dec_out\n\n\ndef patch_module(mod):\n    if getattr(mod, "_xaim4", False):\n        return\n    mod.FFT_for_Period = fft_for_period\n    mod._original_forecast = mod.Model.forecast\n    mod._xaim4 = True\n\n\nclass Runner:\n    """One fold model and forward passes in each series\' own normalised units."""\n\n    def __init__(self, run, D, device):\n        self.run, self.D, self.device = run, D, device\n        args = Namespace(**run["res"]["tslib_args"])\n        args.use_gpu = torch.cuda.is_available()\n        self.args = args\n        self.mod = importlib.import_module(f"models.{args.model}")\n        patch_module(self.mod)\n        self.model = self.mod.Model(args).float().to(device)\n        self.model.load_state_dict(torch.load(run["pth"], map_location=device, weights_only=True))\n        self.model.eval()\n        self.L, self.H = args.seq_len, args.pred_len\n        self.fold = CVM.FoldM4(D["train"], D["spec"], run["fold"])\n\n    def forward(self, xs, mode="normal", rec=None, grad=False, chunk=512):\n        """Forecasts [B, H] (original units) for inputs xs [B, L] (numpy or tensor)."""\n        self.mod.Model.forecast = forecast_full_gradient if grad else self.mod._original_forecast\n        outs = []\n        for i in range(0, len(xs), chunk):\n            xb = xs[i:i + chunk]\n            xt = xb if torch.is_tensor(xb) else torch.tensor(np.asarray(xb), dtype=torch.float32, device=self.device)\n            xt = xt.unsqueeze(-1)\n            dec = torch.zeros((len(xt), self.L // 2 + self.H, 1), device=self.device)\n            PER.update(mode=mode, rec=rec if rec is not None else [], i=0)\n            with torch.set_grad_enabled(grad):\n                o = self.model(xt, None, dec, None)[:, -self.H:, 0]\n            if mode == "record":\n                self.last_rec = list(PER["rec"])\n            PER.update(mode="normal", rec=[], i=0)\n            outs.append(o)\n        self.mod.Model.forecast = self.mod._original_forecast\n        return torch.cat(outs)\n\n    def own_periods(self, x):\n        with torch.no_grad():\n            self.forward(x[None], mode="record")\n        return self.last_rec\n\n    def train_windows(self, rng, n):\n        """Real training windows (input L) of this fold\'s training data, z-normalised: [n, L]."""\n        out = []\n        series = self.fold.train\n        lim = int(CVM.HISTORY[self.D["group"]] * self.H)\n        while len(out) < n:\n            ts = series[rng.integers(len(series))]\n            if len(ts) < self.L + 1:\n                continue\n            c = rng.integers(max(self.L, len(ts) - lim), len(ts) + 1)\n            w = ts[c - self.L:c]\n            sd = w.std()\n            if sd <= 1e-8:\n                continue\n            out.append((w - w.mean()) / sd)\n        return np.stack(out)\n\n\ndef norm_stats(x):\n    mu = x.mean(-1, keepdims=True)\n    sd = np.sqrt(((x - mu) ** 2).mean(-1, keepdims=True) + 1e-5)\n    return mu, sd\n\n\ndef outputs_of(pred_n, H):\n    """Explained outputs from normalised forecasts [B, H]: mean over the horizon, and steps 1, H/2, H."""\n    steps = sorted({0, H // 2 - 1, H - 1})\n    return torch.cat([pred_n.mean(1, keepdim=True), pred_n[:, steps]], dim=1)\n\n\ndef output_names(H):\n    return ["mean forecast"] + [f"t+{h}" for h in sorted({1, H // 2, H})]\n\n\ndef segments(L, n_seg):\n    n_seg = min(n_seg, L)\n    edges = np.linspace(0, L, n_seg + 1).round().astype(int)\n    names = [f"lag {L - edges[j + 1] + 1}-{L - edges[j]}" for j in range(n_seg)]\n    masks = []\n    for j in range(n_seg):\n        m = np.zeros(L, bool)\n        m[edges[j]:edges[j + 1]] = True\n        masks.append(m)\n    return names, masks, edges\n\n\n# =========================================================================================================\n# 0. Periods\n# =========================================================================================================\ndef analysis_periods(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    g, L, H, m_ = D["group"], D["L"], D["H"], D["freq"]\n    sample = ctx["period_sample"]\n    rows = []\n    for m, runners in R_by_model.items():\n        for R in runners:\n            for i in sample:\n                PER["log"] = []\n                with torch.no_grad():\n                    R.forward(D["x_test"][i][None])\n                for layer, (periods, weights) in enumerate(PER["log"]):\n                    for rank, (p, w) in enumerate(zip(periods, weights)):\n                        rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "series": int(i),\n                                     "layer": layer + 1, "rank": rank + 1, "period": int(p), "weight": float(w)})\n                PER["log"] = None\n    P = pd.DataFrame(rows)\n    P.to_csv(os.path.join(out_dir, "periods.csv"), index=False)\n    models = list(R_by_model)\n    # the data\'s own cycles: mean autocorrelation of the (normalised) last 4H values of the series\n    xs = [s[-min(len(s), 4 * L):] for s in D["train"]]\n    max_lag = min(2 * L, min(len(x) for x in xs) - 2)\n    xs = [(x - x.mean()) / (x.std() + 1e-9) for x in xs[:3000]]\n    acf = [1.0]\n    for k in range(1, max(2, max_lag) + 1):\n        acf.append(float(np.mean([np.corrcoef(x[:-k], x[k:])[0, 1] for x in xs if len(x) > k + 2 and x[k:].std() > 0\n                                  and x[:-k].std() > 0])))\n    acf = np.array(acf)\n    peaks = [k for k in range(2, len(acf) - 1) if acf[k] > acf[k - 1] and acf[k] >= acf[k + 1] and acf[k] > 0.1]\n    top = P.groupby("period").size().sort_values(ascending=False).index[:12]\n    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.1,\n                        subplot_titles=["share of series choosing each period (any rank, any layer)",\n                                        "data\'s own cycles: mean autocorrelation of the series"])\n    for m in models:\n        d = P[P.model == m].drop_duplicates(["fold", "seed", "series", "layer", "period"])   # 3H//k can repeat\n        n_w = d.groupby(["fold", "seed", "series", "layer"]).ngroups\n        share = d.groupby("period").size().reindex(top).fillna(0) / max(n_w, 1)\n        fig.add_trace(go.Bar(x=[str(p) for p in top], y=share.values, name=label(m), marker_color=color(m)), 1, 1)\n    fig.add_trace(go.Scatter(x=list(range(len(acf))), y=acf, mode="lines", name="autocorrelation",\n                             line=dict(color="#4A5568")), 1, 2)\n    if m_ > 1:\n        for k in range(m_, len(acf), m_):\n            fig.add_vline(x=k, line=dict(color="#C05621", dash="dot"), row=1, col=2)\n    for k in peaks[:6]:\n        fig.add_annotation(x=k, y=acf[k], text=str(k), showarrow=False, yshift=10, row=1, col=2)\n    fig.update_xaxes(title_text="period (time steps)", type="category", row=1, col=1)\n    fig.update_xaxes(title_text="lag (time steps)", row=1, col=2)\n    fig.update_yaxes(title_text="share of series", row=1, col=1)\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.2), margin=dict(t=120), title=title(\n        f"M4 {g} · Which cycles does TimesNet use?",\n        f"left: the periods TimesNet\'s FFT picks for {len(sample)} test series (top-k, both TimesBlocks; fold models and "\n        f"seeds pooled). Right: the series\' own autocorrelation" + (f"; dotted = multiples of the seasonal period {m_}"\n                                                                    if m_ > 1 else "") +\n        "<br>Note: TimesNet\'s FFT runs on 2H + H = 3H steps, so it can only pick periods 3H/k"))\n    save(fig, out_dir, "01_periods_chosen", 1400, 560)\n    # (b) over calendar time\n    yr = ctx["test_year_bin"]\n    fig = make_subplots(rows=1, cols=len(models), shared_yaxes=True, subplot_titles=[label(m) for m in models])\n    for j, m in enumerate(models, 1):\n        d = P[(P.model == m) & (P.layer == 1)].copy()\n        d["when"] = yr[d.series.values]\n        piv = d.groupby(["when", "period"]).size().unstack(fill_value=0)\n        piv = piv.div(piv.sum(1), axis=0).reindex(columns=list(top), fill_value=0.0)\n        fig.add_trace(go.Heatmap(z=piv.values.T, x=[str(w) for w in piv.index], y=[str(p) for p in piv.columns],\n                                 colorscale="Blues", zmin=0, zmax=1, showscale=(j == 1), colorbar=dict(title="share")),\n                      1, j)\n    fig.update_yaxes(type="category")\n    fig.update_yaxes(title_text="period", row=1, col=1)\n    fig.update_xaxes(title_text="calendar period of the forecast (M4-info dates)")\n    fig.update_layout(margin=dict(t=110), title=title(\n        f"M4 {g} · Period choice over calendar time (first TimesBlock)",\n        "share of series (fold models and seeds pooled) whose forecast starts in each calendar period that pick each "\n        "period. M4 series are not aligned in time; their starting dates place them on one calendar"))\n    save(fig, out_dir, "02_periods_over_time", 1400, 520)\n    out = {}\n    for m in models:\n        d = P[(P.model == m) & (P["rank"] == 1)]\n        if len(d):\n            t = d.period.value_counts(normalize=True)\n            out[m] = {"most_chosen_period": int(t.index[0]), "share": float(t.iloc[0]),\n                      "seasonal_period_chosen_share": float((P[(P.model == m)].groupby(["fold", "seed", "series"])\n                                                             .period.apply(lambda s: (s % m_ == 0).any() if m_ > 1\n                                                                           else False)).mean())}\n    out["acf_peaks"] = peaks[:6]\n    return out\n\n\n# =========================================================================================================\n# 1. SHAP\n# =========================================================================================================\ndef perm_shap_series(R, x, masks, bank, samples, rng, H):\n    """Permutation SHAP (antithetic orders) of one series; replacements = real training windows of other\n    series rescaled to this series\' input mean / std. Outputs in this series\' normalised units."""\n    mu, sd = norm_stats(x)\n    rec = R.own_periods(x)\n    F = len(masks)\n    inputs, orders = [], []\n    for _ in range(max(1, samples // 2)):\n        b = bank[rng.integers(len(bank))] * sd + mu\n        o = rng.permutation(F)\n        for order in (o, o[::-1]):\n            cur = b.copy()\n            seq = [cur.copy()]\n            for j in order:\n                cur = np.where(masks[j], x, cur)\n                seq.append(cur.copy())\n            inputs.extend(seq)\n            orders.append(order)\n    with torch.no_grad():\n        pred = R.forward(np.stack(inputs), mode="force", rec=rec).cpu().numpy()\n    out = outputs_of(torch.tensor((pred - mu) / sd), H).numpy().reshape(len(orders), F + 1, -1)\n    phi = np.zeros((F, out.shape[-1]))\n    for k, order in enumerate(orders):\n        phi[order] += out[k, 1:] - out[k, :-1]\n    phi /= len(orders)\n    return phi, out[0, -1], out[:, 0].mean(0)\n\n\ndef analysis_shap(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    from scipy.stats import spearmanr\n    g, L, H = D["group"], D["L"], D["H"]\n    names, masks, edges = segments(L, opt.n_segments)\n    F = len(names)\n    onames = output_names(H)\n    sample = ctx["shap_sample"]\n    rng0 = np.random.default_rng(opt.seed)\n    store, checks = {}, []\n    for m, runners in R_by_model.items():\n        for R in runners:\n            rng = np.random.default_rng(rng0.integers(1 << 31))\n            bank = R.train_windows(rng, opt.background)\n            phis, fulls, bases = [], [], []\n            for i in sample:\n                phi, full, base = perm_shap_series(R, D["x_test"][i], masks, bank, opt.shap_samples, rng, H)\n                phis.append(phi); fulls.append(full); bases.append(base)\n                checks.append(float(np.abs(phi.sum(0) - (full - base)).max()))\n            store[(m, R.run["fold"], R.run["seed"])] = {"phi": np.stack(phis), "full": np.stack(fulls),\n                                                        "base": np.stack(bases)}\n            print(f"  SHAP {label(m)} fold {R.run[\'fold\']} seed {R.run[\'seed\']}: {len(phis)} series", flush=True)\n    np.savez_compressed(os.path.join(out_dir, "shap_values.npz"), series=np.array(sample),\n                        **{f"{m}__fold{f}__seed{s}__{k}": v[k] for (m, f, s), v in store.items() for k in v},\n                        feature_names=np.array(names), output_names=np.array(onames))\n    models = list(R_by_model)\n    rows = []\n    for (m, f, s), v in store.items():\n        imp = np.abs(v["phi"]).mean(axis=(0, 2))\n        share = imp / imp.sum()\n        for j in range(F):\n            rows.append({"model": m, "fold": f, "seed": s, "segment": j, "lags": names[j], "importance": imp[j],\n                         "share": share[j]})\n    I = pd.DataFrame(rows)\n    I.to_csv(os.path.join(out_dir, "shap_importance.csv"), index=False)\n    # (a) global importance per segment, and per explained output\n    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,\n                        subplot_titles=["share of importance per input segment", "importance per explained output"])\n    for m in models:\n        ps = I[I.model == m].pivot_table(index=["fold", "seed"], columns="segment", values="share")\n        fig.add_trace(go.Bar(x=names, y=ps.mean().values, error_y=dict(type="data", array=ps.std().fillna(0).values),\n                             name=label(m), marker_color=color(m), legendgroup=m), 1, 1)\n        po = np.mean([np.abs(v["phi"]).sum(1).mean(0) for (mm, f, s), v in store.items() if mm == m], axis=0)\n        fig.add_trace(go.Bar(x=onames, y=po, name=label(m), marker_color=color(m), legendgroup=m, showlegend=False), 1, 2)\n    fig.update_yaxes(tickformat=".0%", row=1, col=1)\n    fig.update_yaxes(title_text="total |SHAP| (normalised units)", row=1, col=2)\n    fig.update_xaxes(title_text="input segment (lag 1 = most recent value)", row=1, col=1)\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.22), margin=dict(t=120), title=title(\n        f"M4 {g} · Global importance of the past (permutation SHAP)",\n        f"{len(sample)} test series; mean ± sd over the {I.groupby([\'fold\', \'seed\']).ngroups} fold models and seeds of "\n        f"each model. Features = {F} segments of the {L}-step input; replacements = real training windows of other "\n        "series rescaled to the series\' level and scale; units = the series\' own normalised units"))\n    save(fig, out_dir, "03_shap_global_importance", 1400, 560)\n    # (b) segment x output heatmaps + difference\n    nm = len(models)\n    fig = make_subplots(rows=1, cols=nm + (1 if nm == 2 else 0), horizontal_spacing=0.07,\n                        subplot_titles=[label(m) for m in models] + (["KANB − GELU"] if nm == 2 else []))\n    mats = {}\n    for j, m in enumerate(models, 1):\n        mat = np.mean([np.abs(v["phi"]).mean(0) for (mm, f, s), v in store.items() if mm == m], axis=0)  # [F, out]\n        mat = mat / mat.sum(0, keepdims=True)\n        mats[m] = mat\n        fig.add_trace(go.Heatmap(z=100 * mat, x=onames, y=names, colorscale="Viridis", showscale=(j == 1),\n                                 colorbar=dict(title="% of<br>output", x=-0.1),\n                                 hovertemplate="%{y} → %{x}<br>%{z:.1f}%<extra></extra>"), 1, j)\n    if nm == 2:\n        dm = 100 * (mats[models[1]] - mats[models[0]])\n        zmax = max(1e-9, float(np.abs(dm).max()))\n        fig.add_trace(go.Heatmap(z=dm, x=onames, y=names, colorscale="RdBu_r", zmin=-zmax, zmax=zmax,\n                                 colorbar=dict(title="points"), hovertemplate="%{y} → %{x}<br>%{z:+.1f}<extra></extra>"),\n                      1, 3)\n    fig.update_layout(margin=dict(t=120), title=title(\n        f"M4 {g} · Which part of the past drives which part of the forecast?",\n        "share of each explained output\'s importance per input segment. Right: KANB minus GELU (red = KANB relies more "\n        "on that segment)"))\n    save(fig, out_dir, "04_shap_segment_output", 1500, 520)\n    # (c) over calendar time + stability\n    when = ctx["test_year_bin"][sample]\n    bins = sorted(set(when))\n    fig = make_subplots(rows=nm, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=[label(m) for m in models])\n    stab = []\n    for r, m in enumerate(models, 1):\n        per = np.mean([np.abs(v["phi"]).mean(2) for (mm, f, s), v in store.items() if mm == m], axis=0)   # [n, F]\n        share = per / per.sum(1, keepdims=True)\n        for j in range(F):\n            y = [share[when == b, j].mean() for b in bins]\n            fig.add_trace(go.Scatter(x=[str(b) for b in bins], y=y, mode="lines", stackgroup=f"g{r}", name=names[j],\n                                     legendgroup=names[j], showlegend=(r == 1),\n                                     line=dict(width=0.5, color=PAL[j % len(PAL)])), r, 1)\n        sub = per[rng0.choice(len(per), min(len(per), 200), replace=False)]\n        rho = [spearmanr(sub[a], sub[b]).statistic for a in range(len(sub)) for b in range(a + 1, len(sub))]\n        cv = share.std(0) / np.maximum(share.mean(0), 1e-12)\n        stab.append({"model": m, "median Spearman between series": float(np.nanmedian(rho)),\n                     **{f"CV share {n}": float(c) for n, c in zip(names, cv)}})\n    fig.update_yaxes(tickformat=".0%", title_text="share of importance")\n    fig.update_xaxes(title_text="calendar period of the forecast")\n    fig.update_layout(margin=dict(t=120), legend=dict(orientation="h", y=-0.12), title=title(\n        f"M4 {g} · Is the same part of the past important over calendar time?",\n        "share of importance of each input segment for series whose forecast starts in each calendar period (mean over "\n        "fold models and seeds). Flat bands = stable explanations"))\n    save(fig, out_dir, "05_shap_importance_over_time", 1400, 300 + 280 * nm)\n    S = pd.DataFrame(stab).set_index("model")\n    S.round(4).to_csv(os.path.join(out_dir, "shap_stability.csv"))\n    # (d) beeswarm for the mean forecast\n    fig = make_subplots(rows=1, cols=nm, shared_yaxes=True, horizontal_spacing=0.06, subplot_titles=[label(m) for m in models])\n    xn = (D["x_test"][sample] - norm_stats(D["x_test"][sample])[0]) / norm_stats(D["x_test"][sample])[1]\n    for j, m in enumerate(models, 1):\n        keys = [k for k in store if k[0] == m]\n        for f_ in range(F):\n            xs = np.concatenate([store[k]["phi"][:, f_, 0] for k in keys])\n            vals = np.tile(xn[:, masks[f_]].mean(1), len(keys))\n            z = np.clip(vals, -2, 2)\n            jit = np.random.default_rng(f_).uniform(-0.3, 0.3, len(xs))\n            fig.add_trace(go.Scatter(x=xs, y=f_ + jit, mode="markers", showlegend=False,\n                                     marker=dict(size=4, color=z, colorscale="RdBu_r", cmin=-2, cmax=2, opacity=0.6,\n                                                 showscale=(j == 1 and f_ == 0),\n                                                 colorbar=dict(title="segment<br>level (z)"))), 1, j)\n        fig.add_vline(x=0, line=dict(color="#A0AEC0"), row=1, col=j)\n        fig.update_xaxes(title_text="SHAP on the mean forecast (normalised units)", row=1, col=j)\n    fig.update_yaxes(tickvals=list(range(F)), ticktext=names, col=1)\n    fig.update_layout(margin=dict(t=120, l=120), title=title(\n        f"M4 {g} · Beeswarm: how each part of the past moves the forecast",\n        "one dot = one series of one fold model / seed; colour = the segment\'s level relative to the input window "\n        "(red above the window mean). Red on the right = a high level there raises the forecast"))\n    save(fig, out_dir, "06_shap_beeswarm", 1500, 600)\n    # (e) local\n    ctx["shap_store"], ctx["seg_names"], ctx["onames"] = store, names, onames\n    for m in models:\n        e = ctx["series_smape"][m][sample]\n        picks = [("worst series", int(np.argmax(e))), ("best series", int(np.argmin(e)))]\n        if "drift_series" in ctx:\n            picks.append(("largest drift", int(np.nanargmax(ctx["drift_series"][sample]))))\n        for why, w in picks:\n            local_figure(m, w, why, store, names, onames, D, ctx, out_dir)\n    msg = {}\n    for m in models:\n        d = I[I.model == m].groupby("lags").share.mean().sort_values(ascending=False)\n        msg[m] = f"most important segment {d.index[0]} ({100 * d.iloc[0]:.0f}% of importance)"\n    return {"messages": msg, "efficiency_max_error": float(np.max(checks)) if checks else None,\n            "stability": S.round(4).to_dict(orient="index")}\n\n\ndef local_figure(m, w, why, store, names, onames, D, ctx, out_dir):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    keys = [k for k in store if k[0] == m]\n    phi = np.mean([store[k]["phi"][w] for k in keys], axis=0)\n    full = np.mean([store[k]["full"][w] for k in keys], axis=0)\n    base = np.mean([store[k]["base"][w] for k in keys], axis=0)\n    i = ctx["shap_sample"][w]\n    t = 0\n    order = np.argsort(-np.abs(phi[:, t]))\n    xs = ["average forecast<br>(training windows)"] + [names[j] for j in order] + ["forecast"]\n    ys = [base[t]] + [phi[j, t] for j in order] + [0.0]\n    meas = ["absolute"] + ["relative"] * len(order) + ["total"]\n    texts = [f"{v:+.2f}" if mm == "relative" else f"{v:.2f}" for v, mm in zip(ys, meas)]\n    texts[-1] = f"{full[t]:.2f}"\n    run_ = np.cumsum([base[t]] + [phi[j, t] for j in order])\n    lo, hi = min(0.0, run_.min()), max(0.0, run_.max())\n    pad = 0.2 * (hi - lo + 1e-9)\n    fig = make_subplots(rows=1, cols=3, column_widths=[0.34, 0.33, 0.33], horizontal_spacing=0.09,\n                        subplot_titles=["waterfall: mean forecast (normalised units)",\n                                        "decision plot: every explained output", "the series"])\n    fig.add_trace(go.Waterfall(orientation="h", y=xs, x=ys, measure=meas, text=texts, textposition="outside",\n                               cliponaxis=False, increasing=dict(marker=dict(color="#C53030")),\n                               decreasing=dict(marker=dict(color="#2B6CB0")), totals=dict(marker=dict(color="#2D3748")),\n                               showlegend=False), 1, 1)\n    fig.update_xaxes(range=[lo - pad, hi + pad], row=1, col=1)\n    fig.update_yaxes(autorange="reversed", row=1, col=1)\n    imp = np.argsort(-np.abs(phi).sum(1))[::-1]\n    labs = ["average"] + [names[j] for j in imp]\n    for k, on in enumerate(onames):\n        steps = [base[k]] + list(base[k] + np.cumsum(phi[imp, k]))\n        fig.add_trace(go.Scatter(x=steps, y=labs, mode="lines+markers", name=on,\n                                 line=dict(color=PAL[k % len(PAL)], width=3 if k == 0 else 1.3)), 1, 2)\n    x = D["x_test"][i]\n    y = D["y_test"][i]\n    L, H = len(x), len(y)\n    fig.add_trace(go.Scatter(x=list(range(-L + 1, 1)), y=x, mode="lines", line=dict(color="#718096"), name="input"), 1, 3)\n    fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=y, mode="lines", line=dict(color="black", width=2), name="truth"),\n                  1, 3)\n    fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=ctx["series_pred"][m][i], mode="lines",\n                             line=dict(color=color(m), width=2), name=label(m)), 1, 3)\n    fig.update_layout(margin=dict(t=130, l=150), legend=dict(orientation="h", y=-0.15), title=title(\n        f"M4 {D[\'group\']} · {label(m)} · series {D[\'ids\'][i]}: {why}",\n        f"sMAPE {ctx[\'series_smape\'][m][i]:.1f} (mean over fold models and seeds). From the average forecast on real "\n        "training windows, each input segment adds (red) or removes (blue); the bars add up exactly to the forecast"))\n    save(fig, out_dir, f"07_shap_local_{m}_{D[\'ids\'][i]}", 1700, 620)\n\n\ndef analysis_gradshap(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    g, L, H, mf = D["group"], D["L"], D["H"], D["freq"]\n    rng = np.random.default_rng(opt.seed + 1)\n    sample = ctx["shap_sample"][:opt.grad_series]\n    maps = {}\n    for m, runners in R_by_model.items():\n        acc = []\n        for R in runners:\n            if R.run["seed"] not in opt.grad_seeds:\n                continue\n            bank = R.train_windows(rng, opt.background)\n            for i in sample:\n                x = D["x_test"][i]\n                mu, sd = norm_stats(x)\n                rec = R.own_periods(x)\n                pick = bank[rng.integers(0, len(bank), opt.grad_samples)] * sd + mu\n                alpha = rng.uniform(size=(opt.grad_samples, 1))\n                diff = x[None] - pick\n                xb = torch.tensor(pick + alpha * diff, dtype=torch.float32, device=R.device, requires_grad=True)\n                pred = R.forward(xb, mode="force", rec=rec, grad=True, chunk=10 ** 6)\n                outs = outputs_of((pred - mu.item()) / sd.item(), H)\n                ph = []\n                for t in range(outs.shape[1]):\n                    gr, = torch.autograd.grad(outs[:, t].sum(), xb, retain_graph=t < outs.shape[1] - 1)\n                    ph.append((torch.tensor(diff, dtype=torch.float32, device=R.device) * gr).mean(0).detach().cpu().numpy())\n                acc.append(np.abs(np.stack(ph, -1)))                       # [L, out]\n        if acc:\n            maps[m] = np.mean(acc, axis=0)\n    if not maps:\n        return {}\n    np.savez_compressed(os.path.join(out_dir, "gradshap_maps.npz"), **maps)\n    models = list(maps)\n    onames = output_names(H)\n    fig = make_subplots(rows=2, cols=len(models), row_heights=[0.55, 0.45], vertical_spacing=0.14,\n                        horizontal_spacing=0.08, subplot_titles=[label(m) for m in models] + [""] * len(models))\n    for j, m in enumerate(models, 1):\n        mp = maps[m][::-1]                                                # row 0 = lag 1\n        share = mp / mp.sum(0, keepdims=True)\n        fig.add_trace(go.Heatmap(z=100 * share.T, x=list(range(1, L + 1)), y=onames, colorscale="Viridis",\n                                 showscale=(j == 1), colorbar=dict(title="%", len=0.5, y=0.78),\n                                 hovertemplate="lag %{x} → %{y}<br>%{z:.2f}%<extra></extra>"), 1, j)\n        fig.add_trace(go.Bar(x=list(range(1, L + 1)), y=100 * share[:, 0], marker_color=color(m), showlegend=False), 2, j)\n        if mf > 1:\n            for k in range(mf, L + 1, mf):\n                fig.add_vline(x=k, line=dict(color="#A0AEC0", dash="dot"), row=2, col=j)\n        fig.update_xaxes(title_text="lag (1 = most recent value)", row=2, col=j)\n    fig.update_yaxes(title_text="% of importance (mean forecast)", row=2, col=1)\n    fig.update_layout(margin=dict(t=120), title=title(\n        f"M4 {g} · Importance of every past value (GradientSHAP)",\n        f"expected gradients with rescaled real training windows as references ({len(sample)} series; seed(s) "\n        + ", ".join(map(str, opt.grad_seeds)) + ")." + (f" Dotted = multiples of the seasonal period {mf}" if mf > 1 else "")))\n    save(fig, out_dir, "08_gradshap_lags", 1500, 720)\n    out = {}\n    for m in models:\n        prof = maps[m][::-1][:, 0]\n        k = mf if mf > 1 else max(1, H // 2)\n        out[m] = {f"share of importance in the last {k} values": float(prof[:k].sum() / prof.sum())}\n    return out\n\n\n# =========================================================================================================\n# 1b. Local GradientSHAP (one signed bar per input step) and Grad-CAM on TimesNet\'s 2D period images\n# =========================================================================================================\ndef local_picks(D, ctx, models, k_extra=True):\n    """Series explained locally: worst, best and largest-drift series of the SHAP sample (mean of both models)."""\n    pool = np.asarray(ctx["shap_sample"])\n    e = np.nanmean([ctx["series_smape"][m][pool] for m in models], axis=0)\n    picks = [("worst series", int(pool[np.nanargmax(e)])), ("best series", int(pool[np.nanargmin(e)]))]\n    if k_extra and "drift_series" in ctx:\n        picks.append(("largest drift", int(pool[np.nanargmax(ctx["drift_series"][pool])])))\n    out, seen = [], set()\n    for why, i in picks:\n        if i not in seen:\n            out.append((why, i))\n            seen.add(i)\n    return out\n\n\ndef gradshap_signed(R, x, bank, n, rng):\n    """Expected gradients of the mean forecast (series\' normalised units) w.r.t. every input step, signed.\n    Returns phi [L], f(x), mean f(baselines) (all normalised); phi sums approximately to f(x) - E f(baseline)."""\n    mu, sd = norm_stats(x)\n    rec = R.own_periods(x)\n    pick = bank[rng.integers(0, len(bank), n)] * sd + mu\n    alpha = rng.uniform(size=(n, 1))\n    diff = x[None] - pick\n    xb = torch.tensor(pick + alpha * diff, dtype=torch.float32, device=R.device, requires_grad=True)\n    pred = R.forward(xb, mode="force", rec=rec, grad=True, chunk=10 ** 6)\n    out = ((pred - mu.item()) / sd.item()).mean(1)\n    gr, = torch.autograd.grad(out.sum(), xb)\n    phi = (torch.tensor(diff, dtype=torch.float32, device=R.device) * gr).mean(0).detach().cpu().numpy()\n    with torch.no_grad():\n        fx = ((R.forward(x[None], mode="force", rec=rec).cpu().numpy() - mu) / sd).mean()\n        fb = ((R.forward(pick, mode="force", rec=rec).cpu().numpy() - mu) / sd).mean(1).mean()\n    return phi, float(fx), float(fb)\n\n\ndef analysis_gradshap_local(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    g, L, H, mf = D["group"], D["L"], D["H"], D["freq"]\n    models = list(R_by_model)\n    rng = np.random.default_rng(opt.seed + 7)\n    picks = local_picks(D, ctx, models)\n    out = {}\n    for why, i in picks:\n        x, y = D["x_test"][i], D["y_test"][i]\n        res = {}\n        for m in models:\n            phis, fxs, fbs = [], [], []\n            for R in R_by_model[m]:\n                if R.run["seed"] not in opt.grad_seeds:\n                    continue\n                bank = R.train_windows(rng, opt.background)\n                phi, fx, fb = gradshap_signed(R, x, bank, opt.grad_samples * 4, rng)\n                phis.append(phi); fxs.append(fx); fbs.append(fb)\n            if phis:\n                res[m] = (np.mean(phis, 0), float(np.mean(fxs)), float(np.mean(fbs)))\n        if not res:\n            continue\n        cols = list(res)\n        fig = make_subplots(rows=2, cols=len(cols), row_heights=[0.45, 0.55], shared_xaxes=True, vertical_spacing=0.1,\n                            horizontal_spacing=0.07, subplot_titles=[label(m) for m in cols] + [""] * len(cols))\n        mu, sd = norm_stats(x)\n        steps_in = list(range(-L + 1, 1))\n        for j, m in enumerate(cols, 1):\n            phi, fx, fb = res[m]\n            fig.add_trace(go.Scatter(x=steps_in, y=(x - mu) / sd, mode="lines", line=dict(color="#718096"), name="input",\n                                     legendgroup="in", showlegend=(j == 1)), 1, j)\n            fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=(y - mu) / sd, mode="lines", line=dict(color="black"),\n                                     name="truth", legendgroup="tr", showlegend=(j == 1)), 1, j)\n            fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=(ctx["series_pred"][m][i] - mu) / sd, mode="lines",\n                                     line=dict(color=color(m), width=2), name=label(m)), 1, j)\n            fig.add_trace(go.Bar(x=steps_in, y=phi, marker_color=np.where(phi >= 0, "#C53030", "#2B6CB0"),\n                                 showlegend=False, hovertemplate="step %{x}<br>SHAP %{y:+.3f}<extra></extra>"), 2, j)\n            if mf > 1:\n                for k in range(0, -L, -mf):\n                    fig.add_vline(x=k, line=dict(color="#CBD5E0", dash="dot"), row=2, col=j)\n            fig.layout.annotations[j - 1].text = (f"{label(m)}<br><sup>sum of bars {phi.sum():+.3f} vs forecast − "\n                                                  f"average forecast {fx - fb:+.3f}</sup>")\n        fig.update_yaxes(title_text="normalised units", row=1, col=1)\n        fig.update_yaxes(title_text="SHAP on the mean forecast", row=2, col=1)\n        fig.update_xaxes(title_text="input step (0 = last observed value)", row=2)\n        fig.update_layout(legend=dict(orientation="h", y=-0.12), margin=dict(t=140), title=title(\n            f"M4 {g} · series {D[\'ids\'][i]} ({why}): which past steps pushed the forecast up or down?",\n            "GradientSHAP (expected gradients), one bar per input step: red = raised the mean forecast, blue = lowered "\n            "it.<br>Baselines = other series\' training windows at this series\' level and scale; the bars add up "\n            "(approximately) to forecast − average forecast of the baselines. Mean over the fold models of seed(s) "\n            + ", ".join(map(str, opt.grad_seeds)) + (f"; dotted = every {mf} steps (seasonal period)" if mf > 1 else "")))\n        save(fig, out_dir, f"20_gradshap_steps_{D[\'ids\'][i]}", 1600, 700)\n        out[str(D["ids"][i])] = {why: {m: {"sum": float(res[m][0].sum()), "forecast_minus_baseline":\n                                             res[m][1] - res[m][2]} for m in res}}\n    return out\n\n\ndef gradcam_series(R, x):\n    """Grad-CAM of the mean forecast on every 2D period image of every TimesBlock (one series, its own periods).\n    Returns a list per TimesBlock of (period, weight, cam [rows, period]) and the time profile per block [L+H]."""\n    acts = []\n    hooks = []\n    for blk in R.model.model:\n        def hook(module, inp, out):\n            out.retain_grad()\n            acts.append(out)\n        hooks.append(blk.conv.register_forward_hook(hook))\n    PER["log"] = []\n    try:\n        xt = torch.tensor(x[None], dtype=torch.float32, device=R.device)\n        R.mod.Model.forecast = R.mod._original_forecast\n        with torch.set_grad_enabled(True):\n            o = R.model(xt.unsqueeze(-1), None, torch.zeros((1, R.L // 2 + R.H, 1), device=R.device), None)\n            o[:, -R.H:, 0].mean().backward()\n    finally:\n        for h in hooks:\n            h.remove()\n        log, PER["log"] = PER["log"], None\n    T = R.L + R.H\n    blocks, profiles, a = [], [], 0\n    for periods, weights in log:\n        items, prof = [], np.zeros(T)\n        for p, w in zip(periods, weights):\n            A, G = acts[a], acts[a].grad\n            a += 1\n            alpha = G.mean(dim=(2, 3), keepdim=True)\n            cam = torch.relu((alpha * A).sum(1))[0].detach().cpu().numpy()          # [rows, period]\n            items.append((int(p), float(w), cam))\n            prof += w * cam.reshape(-1)[:T]\n        blocks.append(items)\n        profiles.append(prof)\n    return blocks, profiles\n\n\ndef analysis_gradcam(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    g, L, H = D["group"], D["L"], D["H"]\n    T = L + H\n    models = list(R_by_model)\n    # ---- global: mean time profile over the SHAP sample, per model and TimesBlock\n    sample = np.asarray(ctx["shap_sample"])[:opt.cam_series]\n    glob = {}\n    for m in models:\n        profs = []\n        for R in R_by_model[m]:\n            for i in sample:\n                _, pr = gradcam_series(R, D["x_test"][i])\n                profs.append([p / (p.sum() + 1e-12) for p in pr])\n        glob[m] = np.mean(profs, axis=0)                                         # [blocks, T]\n    nb = glob[models[0]].shape[0]\n    fig = make_subplots(rows=1, cols=nb, shared_yaxes=True, horizontal_spacing=0.06,\n                        subplot_titles=[f"TimesBlock {b + 1}" for b in range(nb)])\n    out = {}\n    for m in models:\n        for b in range(nb):\n            fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=100 * glob[m][b], mode="lines",\n                                     line=dict(color=color(m), width=2.5), name=label(m), legendgroup=m,\n                                     showlegend=(b == 0)), 1, b + 1)\n        out[m] = {f"block {b + 1}: share of Grad-CAM on the future part": float(glob[m][b][L:].sum()) for b in range(nb)}\n    for b in range(nb):\n        fig.add_vrect(x0=0.5, x1=H + 0.5, fillcolor="#EDF2F7", opacity=0.7, line_width=0, layer="below", row=1, col=b + 1)\n        fig.update_xaxes(title_text="time step (≤ 0 input, > 0 future part)", row=1, col=b + 1)\n    fig.update_yaxes(title_text="% of Grad-CAM", row=1, col=1)\n    fig.update_layout(legend=dict(orientation="h", y=-0.2), margin=dict(t=120), title=title(\n        f"M4 {g} · Where do TimesNet\'s convolutions look? (Grad-CAM, folded back to time)",\n        f"{len(sample)} test series, all fold models and seeds; each period image\'s Grad-CAM is folded back to time and "\n        "weighted by TimesNet\'s period weight. Shaded = the future part, which TimesNet\'s linear layer creates before "\n        "the convolutions"))\n    save(fig, out_dir, "21_gradcam_global", 1400, 520)\n    # ---- local: the 2D images of chosen series\n    rng = np.random.default_rng(opt.seed + 8)\n    for why, i in local_picks(D, ctx, models):\n        x, y = D["x_test"][i], D["y_test"][i]\n        mu, sd = norm_stats(x)\n        ncols = 3\n        comp, titles = {}, []\n        for m in models:\n            Rs = [R for R in R_by_model[m] if R.run["seed"] in opt.grad_seeds] or R_by_model[m]\n            blocks, profs = gradcam_series(Rs[0], x)\n            items = sorted(blocks[0], key=lambda t: -t[1])[:ncols]\n            comp[m] = (items, profs)\n            titles += [f"{label(m)} · TimesBlock 1 · period {p} (weight {w:.2f})" for p, w, _ in items]\n            titles += [""] * (ncols - len(items))\n            titles.append(f"{label(m)} · folded back to time (shaded = future part)")\n        fig = make_subplots(rows=2 * len(models), cols=ncols, vertical_spacing=0.07, horizontal_spacing=0.06,\n                            specs=[[{}] * ncols, [{"colspan": ncols}, None, None]] * len(models),\n                            row_heights=[0.6, 0.4] * len(models), subplot_titles=titles)\n        for r_, m in enumerate(models):\n            items, profs = comp[m]\n            row = 2 * r_ + 1\n            for c, (p, w, cam) in enumerate(items, 1):\n                z = cam / (cam.max() + 1e-12)\n                fig.add_trace(go.Heatmap(z=z, colorscale="Inferno", zmin=0, zmax=1, showscale=(r_ == 0 and c == 1),\n                                         colorbar=dict(title="Grad-CAM", len=0.3, y=0.85),\n                                         hovertemplate="cycle %{y} · position %{x}<br>%{z:.2f}<extra></extra>"), row, c)\n                fig.update_xaxes(title_text="position inside one period", row=row, col=c)\n                fig.update_yaxes(title_text="cycle (row)", autorange="reversed", row=row, col=c)\n            prof = profs[0] / (profs[0].max() + 1e-12)\n            fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=prof, mode="lines", fill="tozeroy",\n                                     line=dict(color=color(m)), name=f"{label(m)} Grad-CAM (block 1)"), row + 1, 1)\n            if len(profs) > 1:\n                p2 = profs[1] / (profs[1].max() + 1e-12)\n                fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=p2, mode="lines",\n                                         line=dict(color=color(m), dash="dot"), name=f"{label(m)} Grad-CAM (block 2)"),\n                              row + 1, 1)\n            series = np.r_[(x - mu) / sd, (y - mu) / sd]\n            sn = (series - series.min()) / (series.max() - series.min() + 1e-12)\n            fig.add_trace(go.Scatter(x=list(range(-L + 1, H + 1)), y=sn, mode="lines", line=dict(color="#A0AEC0"),\n                                     name="series (input + truth, rescaled)", showlegend=(r_ == 0)), row + 1, 1)\n            fig.add_vrect(x0=0.5, x1=H + 0.5, fillcolor="#EDF2F7", opacity=0.6, line_width=0, layer="below", row=row + 1, col=1)\n        fig.update_layout(legend=dict(orientation="h", y=-0.06), margin=dict(t=130), title=title(\n            f"M4 {g} · series {D[\'ids\'][i]} ({why}): which part of the 2D period images did the convolutions use?",\n            "TimesNet folds the (embedded) series into 2D images, rows = cycles, columns = position inside one period, "\n            "before its Conv2d blocks.<br>Bright = important for the mean forecast (Grad-CAM on the convolution output). "\n            "The three periods with the largest weights in TimesBlock 1; one fold model (first seed)"))\n        save(fig, out_dir, f"22_gradcam_{D[\'ids\'][i]}", 1500, 520 * len(models) + 120)\n    _ = rng\n    return out\n\n\n# =========================================================================================================\n# 2. Faithfulness and stability\n# =========================================================================================================\ndef analysis_faith(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    from scipy.stats import spearmanr\n    from itertools import combinations\n    store = ctx["shap_store"]\n    g, L, H = D["group"], D["L"], D["H"]\n    names, masks, _ = segments(L, opt.n_segments)\n    F = len(masks)\n    sample = ctx["shap_sample"][:opt.faith_series]\n    rng = np.random.default_rng(opt.seed + 2)\n    rows = []\n    for m, runners in R_by_model.items():\n        for R in runners:\n            v = store[(m, R.run["fold"], R.run["seed"])]\n            bank = R.train_windows(rng, 3)\n            for w, i in enumerate(sample):\n                x = D["x_test"][i]\n                mu, sd = norm_stats(x)\n                rec = R.own_periods(x)\n                imp = np.abs(v["phi"][w]).sum(1)\n                orders = {"SHAP order": list(np.argsort(-imp)), "random order": list(rng.permutation(F))}\n                inputs, tags = [], []\n                for kind, order in orders.items():\n                    for b in bank:\n                        b = b * sd + mu\n                        for n in range(F + 1):\n                            cur = x.copy()\n                            for j in order[:n]:\n                                cur = np.where(masks[j], b, cur)\n                            inputs.append(cur)\n                            tags.append((kind, n))\n                with torch.no_grad():\n                    pred = R.forward(np.stack(inputs), mode="force", rec=rec).cpu().numpy()\n                out = outputs_of(torch.tensor((pred - mu) / sd), H).numpy()\n                fx = out[0]\n                ref = np.mean([out[k] for k, (kind, n) in enumerate(tags) if n == F and kind == "SHAP order"], axis=0)\n                scale = np.linalg.norm(ref - fx) + 1e-9\n                for k, (kind, n) in enumerate(tags):\n                    rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "series": w, "order": kind,\n                                 "fraction": n / F, "change": float(np.linalg.norm(out[k] - fx) / scale)})\n    Df = pd.DataFrame(rows).groupby(["model", "fold", "seed", "series", "order", "fraction"]).change.mean().reset_index()\n    Df.to_csv(os.path.join(out_dir, "faithfulness.csv"), index=False)\n    models = list(R_by_model)\n    res = {}\n    fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.12,\n                        subplot_titles=["forecast change while segments are replaced", "area under the curve"])\n    for m in models:\n        d = Df[Df.model == m]\n        for kind, dash in (("SHAP order", "solid"), ("random order", "dot")):\n            c = d[d.order == kind].groupby("fraction").change.mean()\n            fig.add_trace(go.Scatter(x=100 * c.index, y=c.values, mode="lines+markers", name=f"{label(m)} · {kind}",\n                                     line=dict(color=color(m), dash=dash, width=3 if dash == "solid" else 2)), 1, 1)\n        auc = d.groupby(["order", "fold", "seed", "series"]).change.mean().unstack("order")\n        res[m] = {"auc_shap": float(auc["SHAP order"].mean()), "auc_random": float(auc["random order"].mean()),\n                  "shap_better_share": float((auc["SHAP order"] > auc["random order"]).mean())}\n    fig.add_trace(go.Bar(x=[label(m) for m in res], y=[res[m]["auc_shap"] for m in res], name="SHAP order",\n                         marker_color="#2D3748"), 1, 2)\n    fig.add_trace(go.Bar(x=[label(m) for m in res], y=[res[m]["auc_random"] for m in res], name="random order",\n                         marker_color="#CBD5E0"), 1, 2)\n    fig.update_xaxes(title_text="% of input segments replaced by real (rescaled) training values", row=1, col=1)\n    fig.update_yaxes(title_text="forecast change (1 = all replaced)", row=1, col=1)\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.22), margin=dict(t=120), title=title(\n        f"M4 {g} · Faithfulness: does SHAP find what the model really uses?",\n        "segments replaced most-important-first vs at random; a faster rise for the SHAP order = trustworthy. Share of "\n        "series where SHAP beats random: " + ", ".join(f"{label(m)} {100 * r[\'shap_better_share\']:.0f}%" for m, r in res.items())))\n    save(fig, out_dir, "09_faithfulness", 1400, 560)\n    imp = {k: np.abs(v["phi"]).mean(axis=2) for k, v in store.items()}\n    rows = []\n    for m in models:\n        keys = [k for k in imp if k[0] == m]\n        for a, b in combinations(keys, 2):\n            kind = "across seeds (same fold)" if a[1] == b[1] else ("across folds (same seed)" if a[2] == b[2] else None)\n            if kind:\n                for w in range(imp[a].shape[0]):\n                    rows.append({"pair": f"{label(m)}: {kind}", "rho": spearmanr(imp[a][w], imp[b][w]).statistic})\n    if len(models) == 2:\n        for k in [k for k in imp if k[0] == models[0]]:\n            k2 = (models[1], k[1], k[2])\n            if k2 in imp:\n                for w in range(imp[k].shape[0]):\n                    rows.append({"pair": "GELU vs KANB (same fold, seed)", "rho": spearmanr(imp[k][w], imp[k2][w]).statistic})\n    S = pd.DataFrame(rows).dropna()\n    fig = go.Figure()\n    for j, (pair, gg) in enumerate(S.groupby("pair", sort=False)):\n        fig.add_trace(go.Box(y=gg.rho, name=pair, boxmean=True, marker_color=PAL[j % len(PAL)]))\n    fig.add_hline(y=0, line=dict(color="#A0AEC0", dash="dash"))\n    fig.update_yaxes(range=[-1, 1], title_text="Spearman correlation of the segment rankings (per series)")\n    fig.update_layout(showlegend=False, margin=dict(t=120), title=title(\n        f"M4 {g} · Stability of the explanations",\n        "1 = the two explanations rank the input segments identically for the same series"))\n    save(fig, out_dir, "10_explanation_stability", 1300, 560)\n    res["stability"] = S.groupby("pair").rho.median().round(3).to_dict() if len(S) else {}\n    return res\n\n\n# =========================================================================================================\n# 3. Segment ablation\n# =========================================================================================================\ndef analysis_ablation(R_by_model, D, out_dir, ctx, opt):\n    import plotly.graph_objects as go\n    g, L, H = D["group"], D["L"], D["H"]\n    names, masks, _ = segments(L, opt.n_segments)\n    sample = np.array(ctx["ablation_sample"])\n    rng = np.random.default_rng(opt.seed + 3)\n    X, Y = D["x_test"][sample], D["y_test"][sample]\n    mu, sd = norm_stats(X)\n    rows = []\n    for m, runners in R_by_model.items():\n        for R in runners:\n            with torch.no_grad():\n                base = R.forward(X).cpu().numpy()\n            base_s = CVM.smape_series(Y, base).mean()\n            for j, nm in enumerate(names):\n                for rep in range(opt.ablation_repeats):\n                    donor = rng.permutation(len(X))\n                    Xd = X.copy()\n                    zd = (X[donor] - mu[donor]) / sd[donor]\n                    Xd[:, masks[j]] = (zd * sd + mu)[:, masks[j]]\n                    with torch.no_grad():\n                        p = np.concatenate([R.forward(Xd[k:k + 1]).cpu().numpy() for k in range(len(Xd))])\n                    s = CVM.smape_series(Y, p).mean()\n                    rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "segment": nm, "repeat": rep,\n                                 "base_smape": base_s, "smape": s, "increase_pct": 100 * (s - base_s) / base_s})\n    A = pd.DataFrame(rows)\n    A.to_csv(os.path.join(out_dir, "segment_ablation.csv"), index=False)\n    gA = A.groupby(["model", "fold", "seed", "segment"]).increase_pct.mean().reset_index()\n    fig = go.Figure()\n    for m in R_by_model:\n        d = gA[gA.model == m].groupby("segment").increase_pct\n        mean, lo, hi = d.mean().reindex(names), d.quantile(0.025).reindex(names), d.quantile(0.975).reindex(names)\n        fig.add_trace(go.Bar(x=names, y=mean.values, name=label(m), marker_color=color(m),\n                             error_y=dict(type="data", symmetric=False, array=(hi - mean).values, arrayminus=(mean - lo).values)))\n    fig.add_hline(y=0, line=dict(color="#4A5568"))\n    fig.update_yaxes(title_text="test sMAPE increase when the segment is replaced (%)")\n    fig.update_xaxes(title_text="input segment (lag 1 = most recent value)")\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.18), margin=dict(t=120), title=title(\n        f"M4 {g} · Which part of the past does each model need? (segment ablation)",\n        f"{len(sample)} test series; each segment is replaced by the same segment of another real series, rescaled to "\n        f"the series\' level and scale ({opt.ablation_repeats} repeats). Bars = mean over fold models and seeds, "\n        "whiskers = 2.5–97.5% range"))\n    save(fig, out_dir, "11_segment_ablation", 1300, 560)\n    out = {}\n    for m in R_by_model:\n        d = gA[gA.model == m].groupby("segment").increase_pct.mean().sort_values(ascending=False)\n        out[m] = {"most needed": d.index[0], "increase_pct": float(d.iloc[0])}\n    return out\n\n\n# =========================================================================================================\n# 4. Drift, novelty, concept drift, KANB vs GELU per drift tercile\n# =========================================================================================================\ndef window_features(xin, fut=None, m=1):\n    """Scale-free features of input windows [n, L] (and their futures [n, H]) in input-normalised units."""\n    mu, sd = norm_stats(xin)\n    z = (xin - mu) / sd\n    L = z.shape[1]\n    t = np.arange(L) - (L - 1) / 2\n    slope = (z * t).sum(1) / (t ** 2).sum()\n    d = np.diff(z, axis=1)\n    half = L // 2\n    lvl = z[:, half:].mean(1) - z[:, :half].mean(1)\n    vol = np.log((z[:, half:].std(1) + 1e-3) / (z[:, :half].std(1) + 1e-3))\n\n    def ac(k):\n        if k >= L - 2:\n            return np.zeros(len(z))\n        a, b = z[:, :-k], z[:, k:]\n        a = a - a.mean(1, keepdims=True)\n        b = b - b.mean(1, keepdims=True)\n        return (a * b).sum(1) / (np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1)) + 1e-9)\n\n    cols = {"slope": slope, "noise (std of changes)": d.std(1), "level change (2nd vs 1st half)": lvl,\n            "volatility change": vol, "autocorrelation lag 1": ac(1), "last value": z[:, -1],\n            "max |value|": np.abs(z).max(1)}\n    if m > 1:\n        cols[f"autocorrelation lag {m}"] = ac(m)\n    marg = pd.DataFrame(cols)\n    if fut is None:\n        return marg, None\n    zf = (fut - mu) / sd\n    tf = np.arange(zf.shape[1]) - (zf.shape[1] - 1) / 2\n    joint = marg.copy()\n    joint["future level"] = zf.mean(1)\n    joint["future slope"] = (zf * tf).sum(1) / (tf ** 2).sum()\n    joint["future spread"] = zf.std(1)\n    return marg, joint\n\n\ndef series_drift(D, opt):\n    """Per series: drift of the forecast period vs the input (energy distance of the values, in input-normalised\n    units) and a conformal p-value against the same score for earlier blocks of the series\' own history."""\n    L, H = D["L"], D["H"]\n    scores, pvals, nnull = [], [], []\n    for x, y in zip(D["train"], D["y_test"]):\n        w = x[-L:]\n        mu, sd = norm_stats(w)\n        s = energy_1d((w - mu) / sd, (y - mu) / sd)\n        null = []\n        for c in range(len(x) - H, L - 1, -H):\n            if len(null) >= opt.max_null_blocks:\n                break\n            wi = x[c - L:c]\n            m_, s_ = norm_stats(wi)\n            null.append(energy_1d((wi - m_) / s_, (x[c:c + H] - m_) / s_))\n        null = np.array(null)\n        scores.append(s)\n        pvals.append((1 + (null >= s).sum()) / (1 + len(null)) if len(null) else np.nan)\n        nnull.append(len(null))\n    return np.array(scores), np.array(pvals), np.array(nnull)\n\n\ndef analysis_drift(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    from scipy.stats import gaussian_kde\n    from scipy.spatial.distance import cdist\n    g, L, H, mf = D["group"], D["L"], D["H"], D["freq"]\n    rng = np.random.default_rng(opt.seed + 4)\n    models = list(R_by_model)\n    # ---- per-series drift\n    t0 = time.time()\n    score, pval, nnull = series_drift(D, opt)\n    qval = bh(pval)\n    ctx["drift_series"] = score\n    q1, q2 = np.nanquantile(score, [1 / 3, 2 / 3])\n    bins = np.where(score <= q1, "small drift", np.where(score <= q2, "medium drift", "large drift"))\n    print(f"  drift: per-series scores ({(time.time() - t0) / 60:.1f} min)", flush=True)\n    # ---- population drift: training windows (per fold) vs test windows\n    folds = sorted({R.run["fold"] for rs in R_by_model.values() for R in rs})\n    test_idx = rng.choice(len(D["train"]), min(opt.pop_series, len(D["train"])), replace=False)\n    te_m, te_j = window_features(D["x_test"][test_idx], D["y_test"][test_idx], mf)\n    glob_rows, clf_rows, shap_rows, feat_store = [], [], [], {}\n    for f in folds:\n        fo = CVM.FoldM4(D["train"], D["spec"], f)\n        ok = [k for k, ts in enumerate(fo.train) if len(ts) >= L + H]\n        ok = rng.choice(ok, min(opt.pop_series, len(ok)), replace=False)\n        xin = np.stack([fo.train[k][-H - L:-H] for k in ok])          # training input with its (training) future\n        fut = np.stack([fo.train[k][-H:] for k in ok])\n        tr_m, tr_j = window_features(xin, fut, mf)\n        feat_store[f] = (tr_m, tr_j)\n        for kind, A, B in (("covariate", tr_m, te_m), ("joint", tr_j, te_j)):\n            mu, sd = A.values.mean(0), A.values.std(0) + 1e-9\n            Az, Bz = (A.values - mu) / sd, (B.values - mu) / sd\n            n = min(len(Az), len(Bz), opt.mmd_rows)\n            Az, Bz = Az[rng.choice(len(Az), n, replace=False)], Bz[rng.choice(len(Bz), n, replace=False)]\n            gamma = median_gamma(np.vstack([Az, Bz]), rng)\n            obs = (mmd2(Az, Bz, gamma), energy(Az, Bz))\n            pool = np.vstack([Az, Bz])\n            perm = []\n            for _ in range(opt.perms):\n                o = rng.permutation(len(pool))\n                perm.append((mmd2(pool[o[:n]], pool[o[n:]], gamma), energy(pool[o[:n]], pool[o[n:]])))\n            perm = np.array(perm)\n            glob_rows.append({"fold": f, "kind": kind, "mmd2": obs[0], "energy": obs[1],\n                              "p_mmd": (1 + (perm[:, 0] >= obs[0]).sum()) / (1 + opt.perms),\n                              "p_energy": (1 + (perm[:, 1] >= obs[1]).sum()) / (1 + opt.perms)})\n            auc, p_auc, sh = domain_classifier(A.values, B.values, opt, rng)\n            clf_rows.append({"fold": f, "kind": kind, "auc": auc, "p": p_auc})\n            if sh is not None:\n                for nm, v in zip(A.columns, sh):\n                    shap_rows.append({"fold": f, "kind": kind, "feature": nm, "mean_abs_shap": v})\n        print(f"  drift: fold {f} done", flush=True)\n    G, CL, SH = pd.DataFrame(glob_rows), pd.DataFrame(clf_rows), pd.DataFrame(shap_rows)\n    G.to_csv(os.path.join(out_dir, "drift_global_tests.csv"), index=False)\n    CL.to_csv(os.path.join(out_dir, "domain_classifier.csv"), index=False)\n    # ---- novelty of each test input (kNN in feature space vs the training windows, leave-series-out null)\n    tr_m = pd.concat([feat_store[f][0] for f in folds])\n    ref = tr_m.values[rng.choice(len(tr_m), min(len(tr_m), 4000), replace=False)]\n    mu, sd = ref.mean(0), ref.std(0) + 1e-9\n    refz = (ref - mu) / sd\n    allm, _ = window_features(D["x_test"], None, mf)\n    tz = (allm.values - mu) / sd\n    k = opt.knn\n    Dr = cdist(refz, refz)\n    np.fill_diagonal(Dr, np.inf)\n    null = np.sort(Dr, axis=1)[:, :k].mean(1)\n    nov = np.empty(len(tz))\n    for s0 in range(0, len(tz), 2000):\n        dt = np.sort(cdist(tz[s0:s0 + 2000], refz), axis=1)[:, :k].mean(1)\n        nov[s0:s0 + 2000] = np.searchsorted(np.sort(null), dt) / len(null)\n    # ---- per-series table, KANB vs GELU per drift tercile\n    W = pd.DataFrame({"id": D["ids"], "length": [len(s) for s in D["train"]], "drift_score": score,\n                      "drift_p": pval, "drift_q": qval, "null_blocks": nnull, "drift_bin": bins, "novelty": nov,\n                      "forecast_period": ctx["test_year_bin"]})\n    for m in models:\n        W[f"smape_{m}"] = ctx["series_smape"][m]\n        W[f"mase_{m}"] = ctx["series_mase"][m]\n    rows = []\n    if len(models) == 2:\n        for metric in ("smape", "mase"):\n            d_all = ctx[f"series_diff_{metric}"]\n            for bname in ("small drift", "medium drift", "large drift", "all series"):\n                sel = np.ones(len(d_all), bool) if bname == "all series" else (bins == bname)\n                r = paired_test(d_all[sel], rng)\n                base = np.nanmean(ctx[f"series_{metric}"][models[0]][sel])\n                rows.append({"metric": metric, "drift": bname, "series": r["n"], "GELU": base,\n                             "KANB": np.nanmean(ctx[f"series_{metric}"][models[1]][sel]),\n                             "mean diff (KANB-GELU)": r["mean"], "95% CI low": r.get("mean_lo"),\n                             "95% CI high": r.get("mean_hi"), "change %": 100 * r["mean"] / base if base else np.nan,\n                             "KANB better (series)": r.get("kanb_better_share"), "Wilcoxon p": r["p"],\n                             "KANB better in folds": ctx["fold_bin_wins"](sel, metric)})\n        T = pd.DataFrame(rows)\n        T["Holm p"] = np.nan\n        for metric in ("smape", "mase"):\n            s = T.metric == metric\n            T.loc[s, "Holm p"] = holm(T.loc[s, "Wilcoxon p"].values)\n        T["verdict"] = np.where((T["Holm p"] < 0.05) & (T["mean diff (KANB-GELU)"] < 0), "KANB better",\n                                np.where((T["Holm p"] < 0.05) & (T["mean diff (KANB-GELU)"] > 0), "GELU better",\n                                         "no significant difference"))\n    else:\n        T = pd.DataFrame()\n    T.to_csv(os.path.join(out_dir, "kanb_vs_gelu_by_drift.csv"), index=False)\n    W.to_csv(os.path.join(out_dir, "series_drift_novelty.csv"), index=False)\n    # ---- residual-based concept drift: each model\'s validation errors vs its test errors\n    res_rows = []\n    for m, runners in R_by_model.items():\n        for R in runners:\n            z = np.load(R.run["npz"])\n            a = z["val_err_scaled"]\n            b = z["err_scaled"][z["val_idx"]]\n            ok = np.isfinite(a) & np.isfinite(b)\n            a, b = a[ok], b[ok]\n            lo, hi = np.quantile(np.r_[a, b], [0.01, 0.99])\n            a, b = np.clip(a, lo, hi), np.clip(b, lo, hi)\n            n = min(len(a), opt.mmd_rows)\n            sel = rng.choice(len(a), n, replace=False)\n            obs = energy_1d(a[sel], b[sel])\n            pool = np.r_[a[sel], b[sel]]\n            perm = []\n            for _ in range(opt.perms):\n                o = rng.permutation(len(pool))\n                perm.append(energy_1d(pool[o[:n]], pool[o[n:]]))\n            res_rows.append({"model": m, "fold": R.run["fold"], "seed": R.run["seed"], "energy": obs,\n                             "p": (1 + (np.array(perm) >= obs).sum()) / (1 + opt.perms),\n                             "validation sMAPE": float(np.mean(z["val_smape"])),\n                             "test sMAPE (same series)": float(np.mean(z["smape"][z["val_idx"]]))})\n    RS = pd.DataFrame(res_rows)\n    if len(RS):\n        RS["q"] = bh(RS.p)\n    RS.to_csv(os.path.join(out_dir, "residual_concept_drift.csv"), index=False)\n    drift_figures(W, G, CL, SH, T, RS, feat_store, te_m, D, ctx, out_dir, models, folds)\n    p_min = np.nanmedian(1 / (1 + nnull[nnull > 0])) if (nnull > 0).any() else np.nan\n    msgs = {"series": f"{100 * np.nanmean(pval <= 0.05):.1f}% of series have conformal p ≤ 0.05 (5% expected without "\n                      f"drift); {100 * np.nanmean(qval < 0.05):.1f}% after Benjamini-Hochberg (the smallest possible p "\n                      f"is 1/(1 + history blocks), median {p_min:.3f})"}\n    if len(T):\n        for r in T[T.metric == "smape"].itertuples():\n            msgs[f"sMAPE {r.drift}"] = (f"KANB {r._9:+.2f}% sMAPE vs GELU (Wilcoxon-Holm p = {r._13:.3g}, "\n                                        f"{r.series} series) → {r.verdict}")\n    msgs["population"] = "; ".join(f"fold {r.fold} {r.kind}: MMD p = {r.p_mmd:.3f}" for r in G.itertuples())\n    msgs["classifier"] = "; ".join(f"fold {r.fold} {r.kind}: AUC {r.auc:.2f} (p = {r.p:.3f})" for r in CL.itertuples())\n    return {"messages": msgs, "comparison": T.round(4).to_dict(orient="records") if len(T) else []}\n\n\ndef domain_classifier(A, B, opt, rng):\n    """Training windows (0) vs test windows (1), one row per series: 5-fold CV AUC, permutation null, SHAP."""\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    from sklearn.metrics import roc_auc_score\n    from sklearn.model_selection import StratifiedKFold\n    n = min(len(A), len(B), opt.clf_rows)\n    X = np.vstack([A[rng.choice(len(A), n, replace=False)], B[rng.choice(len(B), n, replace=False)]])\n    y = np.r_[np.zeros(n), np.ones(n)]\n\n    def oof(lab):\n        p = np.zeros(len(lab))\n        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, lab):\n            clf = HistGradientBoostingClassifier(max_iter=opt.clf_iter, max_depth=3, learning_rate=0.1, random_state=0)\n            clf.fit(X[tr], lab[tr])\n            p[te] = clf.predict_proba(X[te])[:, 1]\n        return p\n\n    auc = roc_auc_score(y, oof(y))\n    null = [roc_auc_score(yp, oof(yp)) for yp in (rng.permutation(y) for _ in range(opt.clf_perms))]\n    p_auc = (1 + (np.array(null) >= auc).sum()) / (1 + opt.clf_perms)\n    sh = None\n    try:\n        import shap\n        clf = HistGradientBoostingClassifier(max_iter=opt.clf_iter, max_depth=3, learning_rate=0.1, random_state=0)\n        clf.fit(X, y)\n        sv = shap.TreeExplainer(clf).shap_values(X[rng.choice(len(X), min(len(X), 800), replace=False)])\n        sv = sv[1] if isinstance(sv, list) else sv\n        sh = np.abs(sv).mean(0)\n    except Exception as e:  # noqa: BLE001\n        print("domain-classifier SHAP unavailable:", e)\n    return float(auc), float(p_auc), sh\n\n\ndef drift_figures(W, G, CL, SH, T, RS, feat_store, te_m, D, ctx, out_dir, models, folds):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    from scipy.stats import gaussian_kde\n    g = D["group"]\n    # (a) per-series drift: distribution, significance, over calendar time\n    per = W.groupby("forecast_period").agg(sig=("drift_p", lambda q: float(np.nanmean(q <= 0.05))),\n                                           sig_bh=("drift_q", lambda q: float(np.nanmean(q < 0.05))),\n                                           score=("drift_score", "median"), n=("drift_score", "size")).reset_index()\n    fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], horizontal_spacing=0.1,\n                        subplot_titles=["drift score of every series (forecast period vs its input)",\n                                        "over calendar time: share of series drifting significantly (dashed = 5% expected without drift)"])\n    fig.add_trace(go.Histogram(x=W.drift_score, nbinsx=60, marker_color="#4A5568", name="all series"), 1, 1)\n    for q, nm in zip(np.nanquantile(W.drift_score, [1 / 3, 2 / 3]), ["tercile 1/3", "tercile 2/3"]):\n        fig.add_vline(x=q, line=dict(color="#C05621", dash="dash"), row=1, col=1)\n    fig.add_trace(go.Bar(x=per.forecast_period.astype(str), y=per.sig, marker_color="#C05621", name="p ≤ 0.05",\n                         text=[f"n={n}" for n in per.n], textposition="outside"), 1, 2)\n    fig.add_trace(go.Bar(x=per.forecast_period.astype(str), y=per.sig_bh, marker_color="#2D3748",\n                         name="BH q < 0.05"), 1, 2)\n    fig.add_hline(y=0.05, line=dict(color="#A0AEC0", dash="dash"), row=1, col=2)\n    fig.update_xaxes(title_text="drift score (energy distance, normalised units)", row=1, col=1)\n    fig.update_yaxes(title_text="series", row=1, col=1)\n    fig.update_yaxes(title_text="share of series (BH q < 0.05)", tickformat=".0%", row=1, col=2)\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.18), margin=dict(t=120), title=title(\n        f"M4 {g} · How much does each series drift in its forecast period?",\n        "score = energy distance between the forecast period\'s values and the input window (both in the input\'s "\n        "normalised units); p-value = rank of the score among the same score for earlier blocks of the SAME series "\n        "(conformal), Benjamini-Hochberg over series. Dashed = tercile borders (small / medium / large drift)"))\n    save(fig, out_dir, "12_drift_series", 1400, 560)\n    # (b) train/test shift of the window features (population)\n    tr = feat_store[folds[-1]][0]\n    cols = list(tr.columns)\n    ncol = 4\n    nrow = -(-len(cols) // ncol)\n    fig = make_subplots(rows=nrow, cols=ncol, subplot_titles=cols, horizontal_spacing=0.06, vertical_spacing=0.16)\n    shift = []\n    for i, c in enumerate(cols):\n        a, b = tr[c].values, te_m[c].values\n        lo, hi = np.percentile(np.r_[a, b], [1, 99])\n        if hi <= lo:\n            continue\n        grid = np.linspace(lo, hi, 200)\n        try:\n            fa, fb = gaussian_kde(a)(grid), gaussian_kde(b)(grid)\n        except Exception:  # noqa: BLE001\n            continue\n        ovl = float(np.trapezoid(np.minimum(fa, fb), grid))\n        shift.append({"feature": c, "overlap (OVL)": ovl, "train median": float(np.median(a)), "test median": float(np.median(b))})\n        r, cc = i // ncol + 1, i % ncol + 1\n        fig.add_trace(go.Scatter(x=grid, y=fa, fill="tozeroy", name="training windows (last fold)", legendgroup="a",\n                                 showlegend=(i == 0), line=dict(color="#2B6CB0")), r, cc)\n        fig.add_trace(go.Scatter(x=grid, y=fb, fill="tozeroy", name="test inputs", legendgroup="b", showlegend=(i == 0),\n                                 line=dict(color="#C53030")), r, cc)\n        fig.layout.annotations[i].text = f"{c} · OVL {ovl:.2f}"\n    pd.DataFrame(shift).round(4).to_csv(os.path.join(out_dir, "train_test_shift.csv"), index=False)\n    fig.update_layout(legend=dict(orientation="h", y=-0.08), margin=dict(t=120), title=title(\n        f"M4 {g} · Train/test shift of the input windows (no model)",\n        "scale-free features of the training windows of the last fold vs the test inputs. OVL = overlap of the two "\n        "densities (1 = identical)"))\n    save(fig, out_dir, "13_train_test_shift", 1400, 260 * nrow + 160)\n    # (c) population tests + domain classifier\n    fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6], horizontal_spacing=0.16,\n                        subplot_titles=["domain classifier AUC per fold (0.5 = no drift)",\n                                        "what gives the test windows away (SHAP of the classifier)"])\n    for kind, col_ in (("covariate", "#2B6CB0"), ("joint", "#C05621")):\n        d = CL[CL.kind == kind]\n        gg = G[G.kind == kind].set_index("fold")\n        fig.add_trace(go.Bar(x=[f"fold {f}" for f in d.fold], y=d.auc, marker_color=col_,\n                             name="inputs only" if kind == "covariate" else "inputs + future",\n                             text=[f"p={p:.2f}<br>MMD p={gg.loc[f, \'p_mmd\']:.2f}" for f, p in zip(d.fold, d.p)],\n                             textposition="outside"), 1, 1)\n    fig.add_hline(y=0.5, line=dict(color="#A0AEC0", dash="dash"), row=1, col=1)\n    if len(SH):\n        s = SH.groupby(["kind", "feature"]).mean_abs_shap.mean().reset_index()\n        for kind, col_ in (("covariate", "#2B6CB0"), ("joint", "#C05621")):\n            d = s[s.kind == kind].sort_values("mean_abs_shap")\n            fig.add_trace(go.Bar(y=d.feature, x=d.mean_abs_shap, orientation="h", marker_color=col_, showlegend=False), 1, 2)\n    fig.update_yaxes(range=[0, 1.15], row=1, col=1)\n    fig.update_xaxes(title_text="mean |SHAP| (log-odds of \'test\')", row=1, col=2)\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.18), margin=dict(t=120, l=60), title=title(\n        f"M4 {g} · Population drift: training windows vs test windows",\n        "gradient-boosted trees on scale-free window features, 5-fold CV, p from label permutations (series are "\n        "independent); MMD p from permutations. \'Inputs + future\' clearly above \'inputs only\' = the relation "\n        "past → future changed (concept drift)"))\n    save(fig, out_dir, "14_population_drift", 1400, 600)\n    # (d) error vs drift / novelty, terciles\n    fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.36, 0.28], horizontal_spacing=0.08,\n                        subplot_titles=["sMAPE vs drift score (binned)", "sMAPE vs novelty of the input (binned)",\n                                        "sMAPE per drift tercile"])\n    for mdl in models:\n        y = W[f"smape_{mdl}"].values\n        for j, xcol in ((1, "drift_score"), (2, "novelty")):\n            x = W[xcol].values\n            if xcol == "drift_score":\n                x = np.maximum(x, 1e-4)\n            ok = np.isfinite(x) & np.isfinite(y)\n            qs = np.unique(np.nanquantile(x[ok], np.linspace(0, 1, 21)))\n            idx = np.clip(np.searchsorted(qs, x[ok], side="right") - 1, 0, len(qs) - 2)\n            bx = [x[ok][idx == b].mean() for b in range(len(qs) - 1) if (idx == b).any()]\n            by = [y[ok][idx == b].mean() for b in range(len(qs) - 1) if (idx == b).any()]\n            be = [1.96 * y[ok][idx == b].std() / np.sqrt((idx == b).sum()) for b in range(len(qs) - 1) if (idx == b).any()]\n            fig.add_trace(go.Scatter(x=bx, y=by, mode="lines+markers", error_y=dict(type="data", array=be),\n                                     line=dict(color=color(mdl), width=2.5), name=label(mdl), legendgroup=mdl,\n                                     showlegend=(j == 1)), 1, j)\n        order = ["small drift", "medium drift", "large drift"]\n        gm = W.groupby("drift_bin")[f"smape_{mdl}"].mean().reindex(order)\n        fig.add_trace(go.Bar(x=order, y=gm.values, marker_color=color(mdl), showlegend=False, legendgroup=mdl), 1, 3)\n    if len(T):\n        for r in T[(T.metric == "smape") & (T.drift != "all series")].itertuples():\n            fig.add_annotation(x=r.drift, y=max(r.GELU, r.KANB), text=f"{r._9:+.1f}%<br>p={r._13:.2g}",\n                               showarrow=False, yshift=22, row=1, col=3, font=dict(size=10))\n    fig.update_xaxes(title_text="drift score (energy distance, log scale)", type="log", row=1, col=1)\n    fig.update_xaxes(title_text="novelty (share of training windows less unusual)", row=1, col=2)\n    fig.update_yaxes(title_text="sMAPE (mean over fold models and seeds)", row=1, col=1)\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.2), margin=dict(t=130), title=title(\n        f"M4 {g} · Which model copes better with drift and novelty?",\n        "every test series, sMAPE averaged over the fold models and seeds; 20 equal-count bins with 95% CIs. Right: "\n        "sMAPE per drift tercile, KANB\'s change vs GELU and the Wilcoxon signed-rank p (Holm-corrected)"))\n    save(fig, out_dir, "15_error_vs_drift", 1600, 600)\n    # (e) residual-based concept drift\n    if len(RS):\n        fig = go.Figure()\n        for mdl in models:\n            d = RS[RS.model == mdl]\n            fig.add_trace(go.Box(y=d.energy, name=label(mdl), marker_color=color(mdl), boxpoints="all", jitter=0.4,\n                                 pointpos=0, hovertext=[f"fold {f} seed {s}: p={p:.3f}" for f, s, p in zip(d.fold, d.seed, d.p)]))\n        fig.update_yaxes(title_text="energy distance: test vs validation scaled errors")\n        fig.update_layout(showlegend=False, margin=dict(t=120), title=title(\n            f"M4 {g} · Did the relation each model learned break on the test period?",\n            "per fold model and seed, on the same series: mean scaled error (error / MASE scale) of the validation "\n            "block vs of the test period; permutation p-values in the hover. Lower = errors behave as on validation"))\n        save(fig, out_dir, "16_residual_concept_drift", 1100, 540)\n    # (f) calendar leak\n    cal = D["spec"].get("calendar", {})\n    if cal.get("folds"):\n        c = pd.DataFrame(cal["folds"])\n        fig = go.Figure(go.Bar(x=c.fold.astype(str), y=c.iloc[:, 1], marker_color="#6B46C1",\n                               text=[f"{v:.0%}" if v is not None else "" for v in c.iloc[:, 1]], textposition="outside"))\n        fig.update_yaxes(tickformat=".0%", range=[0, 1], title_text="share of (training value, other series) pairs")\n        fig.update_xaxes(title_text="fold (and the M4 test)")\n        fig.update_layout(margin=dict(t=120), title=title(\n            f"M4 {g} · Calendar overlap: the cross-series leak of M4",\n            "share of training values that lie, in calendar time, after the first predicted value of ANOTHER series.<br>"\n            "Calendar-CV folds should be 0 (no leak); the paper protocol (full histories, M4 test) shows the leak. "\n            f"{cal.get(\'unparsable_dates\', 0)} series with unreadable dates left out"))\n        save(fig, out_dir, "18_calendar_overlap", 1000, 520)\n\n\n# =========================================================================================================\n# 5. Worst series\n# =========================================================================================================\ndef analysis_gallery(R_by_model, D, out_dir, ctx, opt):\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    models = list(R_by_model)\n    ens = np.nanmean([ctx["series_smape"][m] for m in models], axis=0)\n    worst = list(np.argsort(-np.nan_to_num(ens, nan=-np.inf))[:opt.gallery])\n    nrow = -(-len(worst) // 2)\n    titles = []\n    for i in worst:\n        extra = f" · drift {ctx[\'drift_series\'][i]:.2f}" if "drift_series" in ctx else ""\n        titles.append(f"{D[\'ids\'][i]} (length {len(D[\'train\'][i])})<br><sup>" +\n                      " · ".join(f"{label(m)} sMAPE {ctx[\'series_smape\'][m][i]:.1f}" for m in models) + extra + "</sup>")\n    fig = make_subplots(rows=nrow, cols=2, subplot_titles=titles, vertical_spacing=0.13, horizontal_spacing=0.06)\n    for k, i in enumerate(worst):\n        r, c = k // 2 + 1, k % 2 + 1\n        x, y = D["x_test"][i], D["y_test"][i]\n        L, H = len(x), len(y)\n        fig.add_trace(go.Scatter(x=list(range(-L + 1, 1)), y=x, mode="lines", line=dict(color="#718096"), name="input",\n                                 legendgroup="in", showlegend=(k == 0)), r, c)\n        fig.add_trace(go.Scatter(x=list(range(1, H + 1)), y=y, mode="lines", line=dict(color="black", width=2),\n                                 name="truth", legendgroup="tr", showlegend=(k == 0)), r, c)\n        for m in models:\n            hs = list(range(1, H + 1))\n            fig.add_trace(go.Scatter(x=hs, y=ctx["series_pred"][m][i], mode="lines", line=dict(color=color(m), width=2),\n                                     name=label(m), legendgroup=m, showlegend=(k == 0)), r, c)\n            lo, hi = ctx["series_pred_lo"][m][i], ctx["series_pred_hi"][m][i]\n            fig.add_trace(go.Scatter(x=hs + hs[::-1], y=list(hi) + list(lo[::-1]), fill="toself", fillcolor=LIGHT.get(m),\n                                     line=dict(width=0), hoverinfo="skip", showlegend=False, legendgroup=m), r, c)\n        fig.add_vline(x=0.5, line=dict(color="#A0AEC0", dash="dash"), row=r, col=c)\n    fig.update_xaxes(title_text="steps (≤ 0 input, > 0 forecast)")\n    fig.update_layout(legend=dict(orientation="h", y=-0.06), margin=dict(t=120), title=title(\n        f"M4 {D[\'group\']} · Worst series: what the models miss",\n        "the series with the largest sMAPE (mean of both models); lines = mean forecast over fold models and seeds, "\n        "bands = their range"))\n    save(fig, out_dir, "17_worst_series", 1500, 330 * nrow + 160)\n    return {"worst_series": [str(D["ids"][i]) for i in worst]}\n\n\n# =========================================================================================================\n# 6. The leak experiment (calendar CV only)\n# =========================================================================================================\ndef analysis_leak(R_by_model, D, out_dir, ctx, opt):\n    """Effect of M4\'s cross-series leak on the validation error, per model: for every fold and seed,\n    leaky - clean sMAPE on the same series (half A scored while half B keeps its values after the cut-off, and\n    the reverse). Placebo = the same with shuffled calendar positions. Paired Wilcoxon over series, bootstrap\n    CIs, Holm; and whether KANB profits more from the leak than GELU (difference in differences)."""\n    from plotly.subplots import make_subplots\n    import plotly.graph_objects as go\n    g = D["group"]\n    rng = np.random.default_rng(opt.seed + 6)\n    models = list(R_by_model)\n    N = len(D["train"])\n    rows = []\n    for m in models:\n        mdir = os.path.join(ctx["run_root"], m)\n        for js in sorted(glob.glob(os.path.join(mdir, "fold*__seed*.json"))):\n            r = json.load(open(js))\n            f, s = r["fold"], r["seed"]\n            clean = np.load(js[:-5] + "_test.npz") if os.path.exists(js[:-5] + "_test.npz") else None\n            for real, base_name, pair in (("real", None, ("leakA", "leakB")),\n                                          ("placebo", "placebo", ("placebo_leakA", "placebo_leakB"))):\n                if base_name is None:\n                    if clean is None:\n                        continue\n                    b_idx, b_sm = clean["val_idx"], clean["val_smape"]\n                else:\n                    pb = os.path.join(mdir, "leak", f"fold{f}__seed{s}__{base_name}.npz")\n                    if not os.path.exists(pb):\n                        continue\n                    zb = np.load(pb)\n                    b_idx, b_sm = zb["val_idx"], zb["val_smape"]\n                base = dict(zip(b_idx.tolist(), b_sm.tolist()))\n                for v in pair:\n                    pv = os.path.join(mdir, "leak", f"fold{f}__seed{s}__{v}.npz")\n                    if not os.path.exists(pv):\n                        continue\n                    zv = np.load(pv)\n                    for i, sm in zip(zv["val_idx"].tolist(), zv["val_smape"].tolist()):\n                        if i in base:\n                            rows.append({"model": m, "fold": f, "seed": s, "kind": real, "series": i,\n                                         "clean": base[i], "leaky": sm, "effect": sm - base[i]})\n    if not rows:\n        print("  leak: no leak-experiment runs found (run run_leak9 first)")\n        return {"messages": {"leak": "no leak-experiment runs yet"}}\n    E = pd.DataFrame(rows)\n    E.to_csv(os.path.join(out_dir, "leak_effect_per_series.csv"), index=False)\n    per = E.groupby(["model", "kind", "series"]).agg(effect=("effect", "mean"), clean=("clean", "mean")).reset_index()\n    tests = []\n    for m in models:\n        for kind in ("real", "placebo"):\n            d = per[(per.model == m) & (per.kind == kind)]\n            if len(d):\n                r = paired_test(d.effect.values, rng)\n                tests.append({"test": f"{label(m)}: leak effect ({kind})", "series": r["n"], "mean effect": r["mean"],\n                              "95% CI low": r.get("mean_lo"), "95% CI high": r.get("mean_hi"),\n                              "effect % of clean sMAPE": 100 * r["mean"] / d.clean.mean() if d.clean.mean() else np.nan,\n                              "share of series helped by the leak": r.get("kanb_better_share"), "Wilcoxon p": r["p"]})\n        a = per[(per.model == m) & (per.kind == "real")].set_index("series").effect\n        b = per[(per.model == m) & (per.kind == "placebo")].set_index("series").effect\n        both = a.index.intersection(b.index)\n        if len(both) >= 6:\n            r = paired_test((a[both] - b[both]).values, rng)\n            tests.append({"test": f"{label(m)}: real minus placebo", "series": r["n"], "mean effect": r["mean"],\n                          "95% CI low": r.get("mean_lo"), "95% CI high": r.get("mean_hi"),\n                          "effect % of clean sMAPE": np.nan, "share of series helped by the leak": r.get("kanb_better_share"),\n                          "Wilcoxon p": r["p"]})\n    if len(models) == 2:\n        a = per[(per.model == models[1]) & (per.kind == "real")].set_index("series").effect\n        b = per[(per.model == models[0]) & (per.kind == "real")].set_index("series").effect\n        both = a.index.intersection(b.index)\n        if len(both) >= 6:\n            r = paired_test((a[both] - b[both]).values, rng)\n            tests.append({"test": "KANB minus GELU leak effect (does KANB profit more?)", "series": r["n"],\n                          "mean effect": r["mean"], "95% CI low": r.get("mean_lo"), "95% CI high": r.get("mean_hi"),\n                          "effect % of clean sMAPE": np.nan,\n                          "share of series helped by the leak": r.get("kanb_better_share"), "Wilcoxon p": r["p"]})\n    T = pd.DataFrame(tests)\n    T["Holm p"] = holm(T["Wilcoxon p"].values)\n    T["verdict"] = np.where(T["Holm p"] < 0.05, np.where(T["mean effect"] < 0, "lower error (leak helps)",\n                                                         "higher error"), "no significant effect")\n    T.to_csv(os.path.join(out_dir, "leak_effect_tests.csv"), index=False)\n    cut = D["spec"].get("cal", {}).get("cutoffs", [])\n    fig = make_subplots(rows=1, cols=3, column_widths=[0.32, 0.36, 0.32], horizontal_spacing=0.08,\n                        subplot_titles=["mean leak effect (leaky − clean sMAPE) with 95% CI",\n                                        "per fold (calendar cut-off)", "distribution over series (real leak)"])\n    for kind, pat in (("real", ""), ("placebo", "/")):\n        for m in models:\n            t = T[T.test == f"{label(m)}: leak effect ({kind})"]\n            if not len(t):\n                continue\n            t = t.iloc[0]\n            fig.add_trace(go.Bar(x=[label(m)], y=[t["mean effect"]], name=f"{kind}", legendgroup=kind,\n                                 showlegend=(m == models[0]), marker=dict(color=color(m), pattern_shape=pat,\n                                                                          opacity=1 if kind == "real" else 0.55),\n                                 error_y=dict(type="data", symmetric=False, array=[t["95% CI high"] - t["mean effect"]],\n                                              arrayminus=[t["mean effect"] - t["95% CI low"]]),\n                                 text=[f"p={t[\'Holm p\']:.2g}"], textposition="outside"), 1, 1)\n        for m in models:\n            d = E[(E.model == m) & (E.kind == kind)].groupby("fold").effect.agg(["mean", "sem"]).reset_index()\n            if len(d):\n                xs = [cut[f][:10] if f < len(cut) else str(f) for f in d.fold]\n                fig.add_trace(go.Scatter(x=xs, y=d["mean"], error_y=dict(type="data", array=1.96 * d["sem"].fillna(0)),\n                                         mode="lines+markers", name=f"{label(m)} · {kind}", showlegend=False,\n                                         line=dict(color=color(m), dash="solid" if kind == "real" else "dot")), 1, 2)\n    for m in models:\n        d = per[(per.model == m) & (per.kind == "real")]\n        lo, hi = np.nanpercentile(d.effect, [1, 99]) if len(d) else (0, 1)\n        fig.add_trace(go.Violin(y=np.clip(d.effect, lo, hi), name=label(m), line_color=color(m), box_visible=True,\n                                meanline_visible=True, showlegend=False), 1, 3)\n    for c in (1, 2, 3):\n        fig.add_hline(y=0, line=dict(color="#4A5568"), row=1, col=c)\n    fig.update_yaxes(title_text="sMAPE points (negative = the leak lowers the error)", row=1, col=1)\n    fig.update_xaxes(title_text="fold cut-off", row=1, col=2)\n    fig.update_layout(barmode="group", legend=dict(orientation="h", y=-0.15), margin=dict(t=130), title=title(\n        f"M4 {g} · How much does M4\'s cross-series leak change the error?",\n        "same series, same amount of training data: half of the series keeps its values after the fold\'s calendar "\n        "cut-off (the leak), the other half is scored, then swapped.<br>Placebo (hatched) = the same with shuffled "\n        "calendar positions. Wilcoxon signed-rank over series, Holm-corrected"))\n    save(fig, out_dir, "19_leak_effect", 1600, 600)\n    msgs = {r.test: f"{r._3:+.3f} sMAPE points (p Holm = {r._9:.3g}) → {r.verdict}" for r in T.itertuples()}\n    return {"messages": msgs, "tests": T.round(4).to_dict(orient="records")}\n\n\n# =========================================================================================================\n# Main\n# =========================================================================================================\ndef calendar_bins(info, group, train, n_bins=8):\n    starts = CVM.parse_dates(info, group)\n    step = CVM.STEP[group]\n    when = [CVM.date_at(s, step, len(x)) if not pd.isna(s) else pd.NaT for s, x in zip(starts, train)]\n    when = pd.to_datetime(pd.Series(when), errors="coerce")\n    ok = when.notna()\n    out = np.array(["unknown"] * len(when), dtype=object)\n    if ok.sum() > 0:\n        q = pd.qcut(when[ok].map(pd.Timestamp.toordinal), q=min(n_bins, ok.sum()), duplicates="drop")\n        lab = {c: f"{pd.Timestamp.fromordinal(int(c.left) + 1):%Y-%m}–{pd.Timestamp.fromordinal(int(c.right)):%Y-%m}"\n               for c in q.cat.categories}\n        out[ok.values] = [lab[c] for c in q]\n    return out\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--group", required=True)\n    p.add_argument("--folds_file", required=True)\n    p.add_argument("--run_root", required=True, help="runs/<group> (contains <model>/fold*__seed*.json)")\n    p.add_argument("--out_dir", required=True)\n    p.add_argument("--root_path", default="./dataset/m4")\n    p.add_argument("--models", default="TimesNet,TimesNet_KANB")\n    p.add_argument("--analyses", default="all")\n    p.add_argument("--folds", default="")\n    p.add_argument("--seeds", default="")\n    p.add_argument("--n_segments", type=int, default=8)\n    p.add_argument("--shap_series", type=int, default=300)\n    p.add_argument("--shap_samples", type=int, default=16)\n    p.add_argument("--background", type=int, default=64)\n    p.add_argument("--grad_series", type=int, default=100)\n    p.add_argument("--grad_samples", type=int, default=32)\n    p.add_argument("--grad_seeds", default="2021")\n    p.add_argument("--faith_series", type=int, default=100)\n    p.add_argument("--cam_series", type=int, default=100)\n    p.add_argument("--period_series", type=int, default=2000)\n    p.add_argument("--ablation_series", type=int, default=500)\n    p.add_argument("--ablation_repeats", type=int, default=2)\n    p.add_argument("--max_null_blocks", type=int, default=30)\n    p.add_argument("--pop_series", type=int, default=3000)\n    p.add_argument("--mmd_rows", type=int, default=1000)\n    p.add_argument("--perms", type=int, default=199)\n    p.add_argument("--clf_rows", type=int, default=3000)\n    p.add_argument("--clf_iter", type=int, default=100)\n    p.add_argument("--clf_perms", type=int, default=29)\n    p.add_argument("--knn", type=int, default=5)\n    p.add_argument("--gallery", type=int, default=6)\n    p.add_argument("--seed", type=int, default=2021)\n    opt = p.parse_args()\n    opt.grad_seeds = [int(s) for s in opt.grad_seeds.split(",") if s]\n    todo = {"periods", "shap", "gradshap", "gradshap_local", "gradcam", "faith", "ablation", "drift", "gallery",\n            "leak"} if opt.analyses == "all" else \\\n        set(opt.analyses.split(","))\n    if "faith" in todo:\n        todo.add("shap")\n    spec = json.load(open(opt.folds_file))\n    group = spec["group"]\n    assert group == opt.group, "folds file of another group"\n    folds = [int(f) for f in opt.folds.split(",") if f] or None\n    seeds = [int(s) for s in opt.seeds.split(",") if s] or None\n    models = [m for m in opt.models.split(",") if m]\n    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n    out_dir = opt.out_dir\n    os.makedirs(out_dir, exist_ok=True)\n    runs = {m: find_runs(opt.run_root, m, folds, seeds) for m in models}\n    runs = {m: r for m, r in runs.items() if r}\n    if len(runs) >= 2:\n        keys = set.intersection(*[{(r["fold"], r["seed"]) for r in rs} for rs in runs.values()])\n        for m in runs:\n            dropped = [(r["fold"], r["seed"]) for r in runs[m] if (r["fold"], r["seed"]) not in keys]\n            if dropped:\n                print(f"note: {m}: folds/seeds {dropped} not finished for every model - left out")\n            runs[m] = [r for r in runs[m] if (r["fold"], r["seed"]) in keys]\n        runs = {m: r for m, r in runs.items() if r}\n    if not runs:\n        raise SystemExit(f"no finished runs in {opt.run_root}")\n    t_start = time.time()\n    train, test, ids, info = CVM.load_group(opt.root_path, group)\n    H = CVM.HORIZON[group]\n    L = 2 * H\n    x_test, _ = CVM.last_window(train, L)\n    D = {"group": group, "train": train, "y_test": np.stack(test), "x_test": x_test, "ids": ids, "spec": spec,\n         "H": H, "L": L, "freq": CVM.FREQ[group]}\n    R_by_model = {m: [Runner(r, D, device) for r in rs] for m, rs in runs.items()}\n    rng = np.random.default_rng(opt.seed)\n    N = len(train)\n    ctx = {"test_year_bin": calendar_bins(info, group, train)}\n    for k in ("series_smape", "series_mase", "series_pred", "series_pred_lo", "series_pred_hi"):\n        ctx[k] = {}\n    per_run = {}\n    elig_all = np.ones(N, bool)\n    elig_any = np.zeros(N, bool)\n    import warnings\n    warnings.filterwarnings("ignore", message="Mean of empty slice")\n    warnings.filterwarnings("ignore", message="All-NaN slice encountered")\n    for m, rs in runs.items():\n        Z = [dict(np.load(r["npz"])) for r in rs]\n        for z in Z:                           # calendar CV: a series counts only for leak-free fold models\n            el = z.get("test_eligible", np.ones(N, bool)).astype(bool)\n            elig_all &= el\n            elig_any |= el\n            z["smape"] = np.where(el, z["smape"], np.nan)\n            z["mase"] = np.where(el, z["mase"], np.nan)\n            z["pred"] = np.where(el[:, None], z["pred"], np.nan)\n        ctx["series_smape"][m] = np.nanmean([z["smape"] for z in Z], axis=0)\n        ctx["series_mase"][m] = np.nanmean([z["mase"] for z in Z], axis=0)\n        P = np.stack([z["pred"] for z in Z])\n        ctx["series_pred"][m], ctx["series_pred_lo"][m], ctx["series_pred_hi"][m] = \\\n            np.nanmean(P, 0), np.nanmin(P, 0), np.nanmax(P, 0)\n        per_run[m] = {(r["fold"], r["seed"]): (z["smape"], z["mase"]) for r, z in zip(rs, Z)}\n    pool = np.where(elig_all)[0] if elig_all.sum() >= 30 else np.where(elig_any)[0]\n    ctx["scored"] = elig_any\n    ctx["period_sample"] = np.sort(rng.choice(pool, min(len(pool), opt.period_series), replace=False))\n    ctx["shap_sample"] = np.sort(rng.choice(pool, min(len(pool), opt.shap_series), replace=False))\n    ctx["ablation_sample"] = np.sort(rng.choice(pool, min(len(pool), opt.ablation_series), replace=False))\n    if len(runs) == 2:\n        m0, m1 = models[0], models[1]\n        common = sorted(set(per_run[m0]) & set(per_run[m1]))\n        diffs = {mt: np.array([per_run[m1][k][j] - per_run[m0][k][j] for k in common]) for j, mt in\n                 enumerate(("smape", "mase"))}\n        ctx["series_diff_smape"] = np.nanmean(diffs["smape"], 0)\n        ctx["series_diff_mase"] = np.nanmean(diffs["mase"], 0)\n        folds_present = sorted({k[0] for k in common})\n\n        def fold_bin_wins(sel, metric):\n            wins = 0\n            for f in folds_present:\n                dd = np.nanmean([diffs[metric][i] for i, k in enumerate(common) if k[0] == f], axis=0)[sel]\n                wins += int(np.nanmean(dd) < 0)\n            return f"{wins}/{len(folds_present)}"\n        ctx["fold_bin_wins"] = fold_bin_wins\n    summary = {"group": group, "H": H, "models": list(runs), "runs": {m: len(r) for m, r in runs.items()},\n               "test_series": N, "test_series_scored": int(elig_any.sum()), "cv": spec.get("cv", "series"),\n               "mean_smape": {m: float(np.nanmean(v)) for m, v in ctx["series_smape"].items()},\n               "mean_mase": {m: float(np.nanmean(v)) for m, v in ctx["series_mase"].items()}}\n    order = ["periods", "drift", "shap", "gradshap", "gradshap_local", "gradcam", "faith", "ablation", "gallery", "leak"]\n    ctx["run_root"] = opt.run_root\n    fns = {"leak": analysis_leak, "gradshap_local": analysis_gradshap_local, "gradcam": analysis_gradcam,\n           "periods": analysis_periods, "shap": analysis_shap, "gradshap": analysis_gradshap, "faith": analysis_faith,\n           "ablation": analysis_ablation, "drift": analysis_drift, "gallery": analysis_gallery}\n    for a in order:\n        if a not in todo:\n            continue\n        t0 = time.time()\n        print(f"== {a} ==", flush=True)\n        summary[a] = fns[a](R_by_model, D, out_dir, ctx, opt)\n        summary.setdefault("minutes", {})[a] = round((time.time() - t0) / 60, 2)\n    sp = os.path.join(out_dir, "xai_m4_summary.json")\n    if os.path.exists(sp):\n        old = json.load(open(sp))\n        mins = old.get("minutes", {})\n        old.update({k: v for k, v in summary.items() if k != "minutes"})\n        mins.update(summary.get("minutes", {}))\n        summary = old\n        summary["minutes"] = mins\n    summary["figures"] = sorted(set(summary.get("figures", [])) | set(FIGURES))\n    summary["total_minutes"] = round((time.time() - t_start) / 60, 2)\n    conv = lambda o: o.item() if hasattr(o, "item") else str(o)\n    with open(sp, "w") as fh:\n        json.dump(summary, fh, indent=1, default=conv)\n    print("XAI_M4_SUMMARY " + json.dumps({k: summary[k] for k in summary if k != "figures"}, default=conv)[:4000])\n\n\nif __name__ == "__main__":\n    main()',
}
for _name, _src in _S9_FILES.items():
    _p = _os.path.join(REPO, _name)
    _os.makedirs(_os.path.dirname(_p), exist_ok=True)
    with open(_p, "w") as _fh:
        _fh.write(_src)
    print("written:", _p)

# ------------------------------ 9.1 Settings ------------------------------

import subprocess, json, time, glob
import numpy as np
import pandas as pd

S9_ACTIVE = True                                # False = skip Section 9 (e.g. when using "Run all")
S9_MODELS = ["TimesNet", "TimesNet_KANB"]       # first = reference (GELU), second = ours
S9_SEEDS = SEEDS                                # [2021, 2022, 2023]
S9_CV = "calendar"                              # "calendar" = leak-free calendar CV; "series" = per-series CV
S9_GAP_H = 2.0                                  # purge gap in horizons (2 = seq_len)
S9_PLACEBO = False                              # leak experiment: also run the placebo (shuffled calendar)
S9_MIN_TRAIN_H = 1.0                            # a series joins a fold only if >= this x H training values remain
S9_FOLDS = {"Yearly": 3, "Quarterly": 3, "Monthly": 5, "Weekly": 5, "Daily": 5, "Hourly": 5}

# Analysis settings (passed to xai_m4.py); larger = more precise, slower
S9_XAI = {"n_segments": 8,        # input segments (SHAP features)
          "shap_series": 300,     # test series explained with permutation SHAP (same series for every model)
          "shap_samples": 16,     # random orders per series (antithetic pairs)
          "background": 64,       # real training windows used as replacements
          "grad_series": 100,     # series for GradientSHAP
          "grad_samples": 32,
          "grad_seeds": "2021",
          "faith_series": 100,    # series for the faithfulness curves
          "cam_series": 100,      # series for the global Grad-CAM profile
          "period_series": 2000,  # series for the period analysis
          "ablation_series": 500, # series for the segment ablation
          "ablation_repeats": 2,
          "pop_series": 3000,     # series per side for the population drift tests
          "perms": 199,           # permutations for the drift tests
          "clf_perms": 29,        # permutations for the domain classifier's AUC
          "gallery": 6}

# Model settings for the comparison (Section 9 only; Sections 5-8 keep the paper's settings).
# Empty = the paper's settings of Section 4. Any TSLib flag works, e.g. d_model, d_ff, e_layers, top_k,
# num_kernels, dropout, learning_rate, batch_size, train_epochs, patience, lradj, loss.
# Applied in this order (later wins): paper -> S9_HP["all"] -> S9_HP[group] -> S9_HP_MODEL[model]["all"]
#                                     -> S9_HP_MODEL[model][group]
# Every combination of settings gets its own results folder, so runs with different settings never mix.
S9_HP = {"all": {},                      # both models, every group, e.g. {"train_epochs": 20}
         "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {}, "Hourly": {}}
S9_HP_MODEL = {"TimesNet":      {"all": {}, "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {},
                                 "Hourly": {}},
               "TimesNet_KANB": {"all": {}, "Yearly": {}, "Quarterly": {}, "Monthly": {}, "Weekly": {}, "Daily": {},
                                 "Hourly": {}}}

S9_ROOT = R.results_root(name = "table14_s9", use_drive = USE_DRIVE)
print("Results folder:", S9_ROOT)
if S9_ACTIVE:
    R.sh(f"uv pip install --python {PY} -q plotly==5.24.1 kaleido==0.2.1 shap==0.48.0 statsmodels")

# ----------------------------- 9.5 Helpers ------------------------------

def s9_env():
    """Environment of the training / analysis processes. Colab sets MPLBACKEND to its inline backend, which the
    TSLib environment does not have (matplotlib then fails on import), so the scripts get the plain 'Agg' backend."""
    return dict(os.environ, CUBLAS_WORKSPACE_CONFIG = ":4096:8", MPLBACKEND = "Agg")


S9_FIXED = {"task_name", "model", "data", "seasonal_patterns", "seq_len", "label_len", "pred_len", "enc_in", "dec_in",
            "c_out", "features", "root_path", "is_training"}       # set by M4 / the protocol, not changeable


def s9_overrides(group, model):
    """The settings changed for this group and model (S9_HP, S9_HP_MODEL), in the order they are applied."""
    hp = {}
    for d in (S9_HP.get("all", {}), S9_HP.get(group, {}), S9_HP_MODEL.get(model, {}).get("all", {}),
              S9_HP_MODEL.get(model, {}).get(group, {})):
        hp.update(d or {})
    bad = set(hp) & S9_FIXED
    assert not bad, f"these settings are fixed by M4 / the protocol and cannot be changed: {sorted(bad)}"
    return hp


def s9_tag(group):
    """Name of the results folder of the current settings of this group (both models): 'paper' = no change."""
    import hashlib
    hp = {m: s9_overrides(group, m) for m in S9_MODELS}
    if not any(hp.values()):
        return "paper"
    key = json.dumps(hp, sort_keys = True)
    return "hp_" + hashlib.md5(key.encode()).hexdigest()[:8]


def s9_base(group):
    tag = s9_tag(group)
    base = os.path.join(S9_ROOT, S9_CV) if tag == "paper" else os.path.join(S9_ROOT, S9_CV, tag)
    if tag != "paper":                       # describe the settings next to the results
        os.makedirs(base, exist_ok = True)
        path = os.path.join(base, f"settings_{group}.json")
        if not os.path.exists(path):
            json.dump({m: s9_overrides(group, m) for m in S9_MODELS}, open(path, "w"), indent = 1)
    return base


def s9_command(group, model):
    """The paper command of Section 4 for this M4 group, with the model swapped and the Section 9 settings."""
    cmd = [c for c in PAPER_COMMANDS if R.arg(c, "seasonal_patterns").strip("'\"") == group]
    assert cmd, f"unknown M4 group: {group}"
    cmd = R.set_arg(cmd[0], "model", model)
    for k, v in s9_overrides(group, model).items():
        cmd = R.set_arg(cmd, k, v)
    if SMOKE:
        cmd = R.set_arg(cmd, "train_epochs", 1)
    return cmd


def s9_show_settings(group):
    """The settings both models will use for this group (changes vs the paper marked)."""
    keys = ["e_layers", "d_model", "d_ff", "top_k", "num_kernels", "dropout", "learning_rate", "batch_size",
            "train_epochs", "patience", "lradj", "loss"]
    rows = {}
    for m in S9_MODELS:
        cmd, hp = s9_command(group, m), s9_overrides(group, m)
        keys += [k for k in hp if k not in keys]
        dflt = {"num_kernels": "6", "dropout": "0.1", "patience": "3", "lradj": "type1", "e_layers": "2",
                "top_k": "5", "train_epochs": "10", "learning_rate": "0.0001", "batch_size": "32", "loss": "MSE"}
        rows[m] = {k: (f"{R.arg(cmd, k)} ✱" if k in hp else
                       (R.arg(cmd, k) or f"{dflt.get(k, '?')} (TSLib default)")) for k in keys}
    print(f"settings for {group} (✱ = changed from the paper), results folder: {s9_base(group)}")
    display(pd.DataFrame(rows))


def s9_same_settings(stem, cmd):
    """A finished run is reused only if it was trained with the current settings."""
    if not os.path.exists(stem + ".json"):
        return True
    saved = json.load(open(stem + ".json")).get("tslib_args", {})
    for i, a in enumerate(cmd):
        if not a.startswith("--") or i + 1 >= len(cmd):
            continue
        k, v = a[2:], cmd[i + 1].strip("'\"")
        if k in saved and k not in S9_FIXED:
            sv = saved[k]
            try:
                same = float(v) == float(sv)
            except (TypeError, ValueError):
                same = str(v) == str(sv)
            if not same:
                raise RuntimeError(f"{stem}.json was trained with {k}={sv}, the current settings say {k}={v}. "
                                   "Change S9_HP back, or delete that run folder")
    return True


def s9_folds_file(group):
    return os.path.join(S9_ROOT, S9_CV, "folds", f"{group}.json")


def s9_run_dir(group, model):
    return os.path.join(s9_base(group), "runs", group, model)


def s9_make_folds(group):
    path = s9_folds_file(group)
    if not os.path.exists(path):
        out = subprocess.run([PY, "-u", "cv_m4.py", "--make_folds", "--folds_file", path, "--cv", S9_CV,
                              "--n_folds", str(S9_FOLDS[group]), "--gap_h", str(S9_GAP_H),
                              "--min_train_h", str(S9_MIN_TRAIN_H), "--"] + s9_command(group, S9_MODELS[0]),
                             cwd = REPO, text = True, capture_output = True, env = s9_env())
        if out.returncode != 0:
            print(out.stdout[-2000:], out.stderr[-3000:])
            raise RuntimeError("could not create the folds")
    return json.load(open(path))


def s9_show_folds(spec):
    H, gap = spec["H"], spec["gap"]
    rows = [{"fold": f["fold"], "cut-off": f.get("cutoff", "")[:10], "validated series": f["series"],
             "training series": f.get("training_series", f["series"]),
             "validation block (per series)": f["validation_block"], "gap": f"{gap} values",
             "median training length": f["median_train_length"],
             "test series scored (leak-free)": f.get("test_series_scored", spec["series"])} for f in spec["folds"]]
    display(pd.DataFrame(rows).set_index("fold"))
    cal = spec.get("calendar", {})
    if cal.get("folds"):
        leak = {str(r["fold"]): r["share of training values after other series' target start"] for r in cal["folds"]}
        print("calendar overlap (share of training values later than other series' first predicted value): "
              + ", ".join(f"{k}: {v:.0%}" if v is not None else f"{k}: -" for k, v in leak.items()))


def run_cv9(group, models = None, seeds = None, folds = None):
    """Trains every model x fold x seed of one M4 group. Finished runs are skipped (resumable)."""
    if not S9_ACTIVE:
        print("Section 9 is switched off (S9_ACTIVE = False)")
        return
    models = models or S9_MODELS
    seeds = seeds or S9_SEEDS
    spec = s9_make_folds(group)
    print(f"== M4 {group}: {len(models)} model(s) x {spec['n_folds']} folds x {len(seeds)} seed(s); "
          f"{spec['series']} test series, horizon {spec['H']} ==")
    s9_show_folds(spec)
    s9_show_settings(group)
    for model in models:
        for seed in seeds:
            for fold in (folds if folds is not None else range(spec["n_folds"])):
                out_dir = s9_run_dir(group, model)
                stem = os.path.join(out_dir, f"fold{fold}__seed{seed}")
                if os.path.exists(stem + ".json") and os.path.exists(stem + "_test.npz") \
                        and s9_same_settings(stem, s9_command(group, model)):
                    continue
                os.makedirs(out_dir, exist_ok = True)
                log_path = os.path.join(s9_base(group), "logs", group, f"{model}_fold{fold}_seed{seed}.log")
                os.makedirs(os.path.dirname(log_path), exist_ok = True)
                print(f"   {model:14s} fold {fold} seed {seed} ...", end = "", flush = True)
                t0 = time.time()
                with open(log_path, "w") as log:
                    proc = subprocess.run([PY, "-u", "cv_m4.py", "--folds_file", s9_folds_file(group), "--fold", str(fold),
                                           "--seed", str(seed), "--out_dir", out_dir, "--"] + s9_command(group, model),
                                          cwd = REPO, text = True, stdout = log, stderr = subprocess.STDOUT,
                                          env = s9_env())
                if proc.returncode != 0 or not os.path.exists(stem + ".json"):
                    print(f" FAILED - see {log_path}")
                    print(open(log_path).read()[-2500:])
                    return
                r = json.load(open(stem + ".json"))
                print(f" sMAPE {r['smape']:.3f}  MASE {r['mase']:.3f}  OWA {r['owa']:.3f}  ({r['epochs']} epochs, "
                      f"{(time.time() - t0) / 60:.1f} min)")
    display(cv9_table(group))


def run_leak9(group, models = None, seeds = None, folds = None, placebo = None):
    """Leak experiment (calendar CV): for every model x fold x seed, retrains the fold twice with one half of the
    series keeping its values after the cut-off (A scored / B leaky, then swapped), and - with the placebo - three
    more times with shuffled calendar positions. Needs the clean fold models of run_cv9. Resumable."""
    if not S9_ACTIVE:
        print("Section 9 is switched off (S9_ACTIVE = False)")
        return
    assert S9_CV == "calendar", "the leak experiment needs S9_CV = 'calendar'"
    placebo = S9_PLACEBO if placebo is None else placebo
    variants = ["leakA", "leakB"] + (["placebo", "placebo_leakA", "placebo_leakB"] if placebo else [])
    models = models or S9_MODELS
    seeds = seeds or S9_SEEDS
    spec = s9_make_folds(group)
    print(f"== Leak experiment M4 {group}: {len(models)} model(s) x {spec['n_folds']} folds x {len(seeds)} seed(s) "
          f"x {len(variants)} variants ==")
    for model in models:
        for seed in seeds:
            for fold in (folds if folds is not None else range(spec["n_folds"])):
                out_dir = s9_run_dir(group, model)
                for v in variants:
                    stem = os.path.join(out_dir, "leak", f"fold{fold}__seed{seed}__{v}")
                    if os.path.exists(stem + ".json") and s9_same_settings(stem, s9_command(group, model)):
                        continue
                    log_path = os.path.join(s9_base(group), "logs", group, f"{model}_fold{fold}_seed{seed}_{v}.log")
                    os.makedirs(os.path.dirname(log_path), exist_ok = True)
                    print(f"   {model:14s} fold {fold} seed {seed} {v:14s} ...", end = "", flush = True)
                    t0 = time.time()
                    with open(log_path, "w") as log:
                        proc = subprocess.run([PY, "-u", "cv_m4.py", "--folds_file", s9_folds_file(group), "--fold",
                                               str(fold), "--seed", str(seed), "--variant", v, "--out_dir", out_dir,
                                               "--"] + s9_command(group, model),
                                              cwd = REPO, text = True, stdout = log, stderr = subprocess.STDOUT,
                                              env = s9_env())
                    if proc.returncode != 0 or not os.path.exists(stem + ".json"):
                        print(f" FAILED - see {log_path}")
                        print(open(log_path).read()[-2500:])
                        return
                    r = json.load(open(stem + ".json"))
                    print(f" validation sMAPE {r['val_smape']:.3f} ({(time.time() - t0) / 60:.1f} min)")
    print("done - run run_xai9(group, analyses='leak') to measure the effect")


def cv9_results(group = None):
    """Every finished run of the current CV, with the settings folder it belongs to ('paper' or 'hp_...')."""
    rows = []
    pats = [("paper", os.path.join(S9_ROOT, S9_CV, "runs", "*", "*", "fold*__seed*.json")),
            (None, os.path.join(S9_ROOT, S9_CV, "hp_*", "runs", "*", "*", "fold*__seed*.json"))]
    for tag, pat in pats:
        for js in glob.glob(pat):
            r = json.load(open(js))
            if group and r["group"] != group:
                continue
            t = tag or js.split(os.sep + "runs" + os.sep)[0].split(os.sep)[-1]
            rows.append({"settings": t, **{k: r.get(k) for k in ("group", "model", "fold", "seed", "smape", "mase", "owa",
                                                                  "val_smape", "n_test_scored", "epochs",
                                                                  "train_minutes", "n_train_series")}})
    return pd.DataFrame(rows)


def cv9_table(group = None):
    """Mean ± sd over folds x seeds per group and model, and the paired run-level comparison (same fold and
    seed for both models): KANB wins and a Wilcoxon signed-rank test over the pairs."""
    from scipy.stats import wilcoxon
    df = cv9_results(group)
    if not len(df):
        return pd.DataFrame()
    rows = []
    for (st, g), d0 in df.groupby(["settings", "group"]):
        row = {"settings": st, "group": g}
        for m in S9_MODELS:
            d = d0[d0.model == m]
            for k in ("smape", "mase", "owa"):
                row[f"{m} {k.upper()}"] = f"{d[k].mean():.3f} ± {d[k].std():.3f}" if len(d) else ""
            row[f"{m} runs"] = len(d)
        p = d0.pivot_table(index = ["fold", "seed"], columns = "model", values = "owa").dropna()
        if all(m in p for m in S9_MODELS) and len(p):
            diff = p[S9_MODELS[1]] - p[S9_MODELS[0]]
            row["KANB better OWA (pairs)"] = f"{int((diff < 0).sum())}/{len(diff)}"
            row["Wilcoxon p (OWA)"] = round(wilcoxon(diff).pvalue, 4) if len(diff) >= 6 and (diff != 0).any() else None
        rows.append(row)
    return pd.DataFrame(rows)


def run_xai9(group, analyses = "all", **overrides):
    """Section 9 analyses of one M4 group on its saved fold models (no training)."""
    if not S9_ACTIVE:
        print("Section 9 is switched off (S9_ACTIVE = False)")
        return
    opts = dict(S9_XAI, **overrides)
    out_dir = os.path.join(s9_base(group), "xai", group)
    cmd = [PY, "-u", "xai_m4.py", "--group", group, "--folds_file", s9_folds_file(group),
           "--run_root", os.path.join(s9_base(group), "runs", group), "--out_dir", out_dir,
           "--models", ",".join(S9_MODELS), "--analyses", analyses]
    for k, v in opts.items():
        cmd += [f"--{k}", str(v)]
    log_path = os.path.join(s9_base(group), "logs", group, "xai_m4.log")
    os.makedirs(os.path.dirname(log_path), exist_ok = True)
    print(f"== Section 9 analyses: M4 {group} ({analyses}) ==", flush = True)
    t0 = time.time()
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd = REPO, text = True, stdout = subprocess.PIPE, stderr = subprocess.STDOUT,
                                env = s9_env())
        for line in proc.stdout:
            log.write(line)
            if line.startswith(("==", "  SHAP", "  drift", "note")):
                print("  " + line.rstrip(), flush = True)
        proc.wait()
    if proc.returncode != 0:
        print(f"FAILED - see {log_path}")
        print(open(log_path).read()[-3000:])
        return
    print(f"done in {(time.time() - t0) / 60:.1f} min")
    show9(group, figures = False)


FIGURE_NOTES_9 = {
 "01": "**Periods.** How often TimesNet's FFT picks each period vs the series' own autocorrelation (seasonal period dotted).",
 "02": "**Periods over calendar time** (M4-info dates).",
 "03": "**Global importance** of each input segment, and per explained output.",
 "04": "**Segment × output.** Which part of the past drives which part of the forecast; right = KANB − GELU.",
 "05": "**Stability over calendar time.** Flat bands = the same part of the past matters all the time.",
 "06": "**Beeswarm.** Direction of each segment's effect on the mean forecast.",
 "07": "**Local explanation** of one series (waterfall, decision plot, the series): worst, best, largest drift.",
 "08": "**GradientSHAP per lag**, seasonal lags dotted.",
 "09": "**Faithfulness.** The SHAP order must change the forecast faster than a random order.",
 "10": "**Stability of the explanations** across seeds, folds and the two models.",
 "11": "**Segment ablation.** Test-sMAPE increase when one part of the past is replaced by real data of another series.",
 "12": "**Per-series drift.** Distribution of the drift score; share of series drifting significantly over calendar time.",
 "13": "**Train/test shift** of scale-free window features (population).",
 "14": "**Population drift.** Domain classifier AUC (inputs only vs inputs + future) with MMD p-values, and what reveals the test windows.",
 "15": "**Who copes better with drift / novelty?** sMAPE vs drift and novelty; per tercile with Wilcoxon-Holm p.",
 "16": "**Residual-based concept drift.** Do each model's errors on the test look like on its validation blocks?",
 "17": "**Worst series gallery.**",
 "18": "**Calendar overlap.** Calendar-CV folds should be 0 (no leak); the paper protocol shows M4's leak.",
 "20": "**GradientSHAP per input step** of one series: red bars raised the mean forecast, blue lowered it; they add up (approximately) to forecast − average forecast of the baselines.",
 "21": "**Grad-CAM, global.** Where TimesNet's convolutions look, folded back to time, per TimesBlock; shaded = the future part created by the linear layer.",
 "22": "**Grad-CAM, one series.** The 2D period images (rows = cycles, columns = position in the period) of TimesBlock 1 with Grad-CAM, and the curve folded back to time.",
 "19": "**Leak experiment.** Leaky − clean sMAPE on the same series (negative = the leak lowers the error), real vs placebo, per model."}


def show9(group, figures = True, only = None):
    """Summary tables and the interactive figures of one M4 group."""
    import plotly.io as pio
    from IPython.display import Markdown
    d = os.path.join(s9_base(group), "xai", group)
    sp = os.path.join(d, "xai_m4_summary.json")
    if not os.path.exists(sp):
        print("no analyses yet: run run_xai9 first")
        return
    s = json.load(open(sp))
    display(Markdown(f"### M4 {group} ({s.get('cv', 'series')} CV): {s['runs']} runs, "
                     f"{s.get('test_series_scored', s['test_series'])} of {s['test_series']} test series scored"))
    display(pd.DataFrame({"mean sMAPE": s["mean_smape"], "mean MASE": s["mean_mase"]}).round(4))
    for name, ttl in (("leak_effect_tests.csv", "Leak experiment: leaky − clean sMAPE (Wilcoxon, bootstrap CI, Holm)"),
                      ("kanb_vs_gelu_by_drift.csv", "KANB vs GELU per drift tercile (Wilcoxon, bootstrap CI, Holm)"),
                      ("drift_global_tests.csv", "Population drift tests per fold (permutation)"),
                      ("domain_classifier.csv", "Domain classifier per fold"),
                      ("residual_concept_drift.csv", "Residual-based concept drift per fold model"),
                      ("train_test_shift.csv", "Train/test shift of the window features"),
                      ("shap_stability.csv", "Stability of the importance")):
        p = os.path.join(d, name)
        if os.path.exists(p):
            display(Markdown(f"**{ttl}**"))
            display(pd.read_csv(p).round(4))
    for k in ("leak", "drift", "shap", "faith", "ablation", "gradshap", "periods"):
        if k in s:
            v = s[k]
            msgs = v.get("messages", v) if isinstance(v, dict) else v
            if isinstance(msgs, dict):
                display(Markdown(f"**{k}:** " + "; ".join(f"{a}: {b}" for a, b in msgs.items())))
    if not figures:
        return
    for f in s.get("figures", []):
        if only and not any(f.startswith(o) for o in only):
            continue
        display(Markdown(FIGURE_NOTES_9.get(f[:2], "")))
        pio.read_json(os.path.join(d, f + ".json")).show()

print("Section 9 ready:", "s9_overrides" in globals())

In [ ]:
from google.colab import files
files.upload()                                   # choose section9_setup.py
exec(open("section9_setup.py").read())           # = cells 9.1, the 3 %%writefile cells and 9.5
print("s9_overrides" in globals())

Saving section9_setup.py to section9_setup.py
written: /content/Time-Series-Library/models/TimesNet_KANB.py
written: /content/Time-Series-Library/cv_m4.py
written: /content/Time-Series-Library/xai_m4.py
Results folder: /content/TimesNet_reproduction/table14_s9
$ uv pip install --python /content/tslib_env_blackwell/bin/python -q plotly==5.24.1 kaleido==0.2.1 shap==0.48.0 statsmodels
Section 9 ready: True
True


In [ ]:
# ---- Download all Table 14 results as one zip ----
import os, zipfile
from google.colab import files

INCLUDE_CHECKPOINTS = True      # True = also the trained models (.pth); makes the zip much bigger

folders = {"table14_reproduction": ROOT}                 # Sections 5-7 (forecasts, scores, tables, logs)
if "S9_ROOT" in globals():
    folders["table14_s9"] = S9_ROOT                      # Section 8 (folds, runs, figures, tables, logs)

zip_path = "/content/table14_results.zip"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name, root in folders.items():
        if not os.path.exists(root):
            print("not found, skipped:", root)
            continue
        for dirpath, _, filenames in os.walk(root):
            for f in filenames:
                if f.endswith(".pth") and not INCLUDE_CHECKPOINTS:
                    continue
                full = os.path.join(dirpath, f)
                z.write(full, os.path.join(name, os.path.relpath(full, root)))
                n += 1

print(f"{n} files, {os.path.getsize(zip_path) / 1e6:.1f} MB -> {zip_path}")
files.download(zip_path)

741 files, 1132.8 MB -> /content/table14_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>